# Modelling pipeline

Data loading and preprocessing, benchmarking of the sixteen algorithms,
multi-split robustness analysis, SHAP interpretability, validation on the
parallel stream, and the feasible-region analysis.

Set `BASE_DIR` in the next cell, place the operating record at
`BASE_DIR/data/UF_data.csv`, then run the cells in order.


In [ ]:
# Paths. BASE_DIR is the repository root; the raw record is expected at
# BASE_DIR/data/UF_data.csv and every output is written under BASE_DIR.
import os

BASE_DIR = os.environ.get('MBR_BASE_DIR', os.path.abspath('..'))
DATA_DIR = os.path.join(BASE_DIR, 'data')
os.makedirs(DATA_DIR, exist_ok=True)


# Data inventory

In [ ]:
import os
import json
import pandas as pd
from pathlib import Path
from datetime import datetime
import hashlib

print("=" * 80)
print(" " * 15 + "UF MEMBRANE/BIOLOGICAL PROJECT - WORKFLOW-AWARE INVENTORY")
print("=" * 80)

# ==================== CONFIGURATION ====================

# Define data directories with workflow context
DATA_DIRECTORIES = {
 'Step1_DataPreprocessing': {
 'path': BASE_DIR,
 'description': 'Data cleaning and noise removal (cleaning period detection)',
 'contains': ['Cleaned datasets', 'Outlier detection results', 'Preprocessing visualizations',
 'Step-by-step cleaning workflow figures']
 }
}

OUTPUT_DIR = BASE_DIR
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ==================== HELPER FUNCTIONS ====================

def get_file_size_readable(size_bytes):
 """Convert bytes to human-readable format."""
 for unit in ['B', 'KB', 'MB', 'GB']:
 if size_bytes < 1024.0:
 return f"{size_bytes:.2f} {unit}"
 size_bytes /= 1024.0
 return f"{size_bytes:.2f} TB"

def get_file_hash(filepath):
 """Generate MD5 hash for file verification."""
 try:
 hash_md5 = hashlib.md5()
 with open(filepath, "rb") as f:
 for chunk in iter(lambda: f.read(4096), b""):
 hash_md5.update(chunk)
 return hash_md5.hexdigest()[:8]
 except:
 return "N/A"

def categorize_file(filename, filepath=''):
 """Categorize file by extension, name patterns, and location."""
 ext = os.path.splitext(filename)[1].lower()
 name_lower = filename.lower()

    # Image/Figure extensions
 if ext in ['.png', '.jpg', '.jpeg', '.svg', '.pdf', '.eps', '.tiff']:
        # Further categorize figures by content
 if any(kw in name_lower for kw in ['shap', 'feature', 'importance', 'waterfall', 'beeswarm']):
 return 'Figure_SHAP'
 elif any(kw in name_lower for kw in ['step', 'cleaning', 'preprocess', 'outlier']):
 return 'Figure_Preprocessing'
 elif any(kw in name_lower for kw in ['optimal', 'feasible', 'sensitivity', 'correlation']):
 return 'Figure_OptimalConditions'
 elif any(kw in name_lower for kw in ['rul', 'warning', 'prediction', 'tcr', 'lifetime']):
 return 'Figure_EarlyWarning'
 elif any(kw in name_lower for kw in ['performance', 'model', 'comparison', 'prediction']):
 return 'Figure_ModelPerformance'
 else:
 return 'Figure_Other'

    # Data extensions
 elif ext in ['.csv', '.xlsx', '.xls']:
 if any(kw in name_lower for kw in ['raw', 'clean', 'preprocess']):
 return 'Data_Cleaned'
 elif any(kw in name_lower for kw in ['optimal', 'feasible', 'sensitivity']):
 return 'Data_OptimalConditions'
 elif any(kw in name_lower for kw in ['rul', 'warning', 'tcr']):
 return 'Data_EarlyWarning'
 elif any(kw in name_lower for kw in ['shap', 'feature']):
 return 'Data_SHAP'
 elif any(kw in name_lower for kw in ['result', 'performance', 'metric']):
 return 'Data_Results'
 else:
 return 'Data_Other'

    # Model files
 elif ext in ['.pkl', '.joblib', '.h5', '.pt', '.pth', '.ckpt']:
 return 'Model_TrainedModel'

    # Code files
 elif ext in ['.py', '.ipynb', '.r', '.m']:
 return 'Code'

    # Documents
 elif ext in ['.docx', '.doc', '.txt', '.md', '.pdf']:
 return 'Document'

    # Configuration
 elif ext in ['.json', '.yaml', '.yml', '.ini', '.cfg']:
 return 'Config'

 else:
 return 'Other'

def infer_subdirectory_purpose(dir_name):
 """Infer the purpose of a subdirectory from its name."""
 name_lower = dir_name.lower()

 purpose_map = {
        # Preprocessing subdirs
 'step': 'Preprocessing step visualization',
 'cleaning': 'Cleaning detection analysis',
 'outlier': 'Outlier removal process',
 'raw': 'Raw unprocessed data',
 'cleaned': 'Cleaned processed data',

        # ML/SHAP subdirs
 'shap': 'SHAP interpretability analysis',
 'feature': 'Feature importance analysis',
 'model': 'Trained models',
 'performance': 'Model performance metrics',
 'prediction': 'Prediction results',
 'comparison': 'Model comparison results',

        # Optimal conditions subdirs
 'optimal': 'Optimal operating conditions',
 'feasible': 'Feasible region analysis',
 'sensitivity': 'Sensitivity analysis',
 'correlation': 'Correlation analysis',
 'range': 'Operating range determination',
 'distribution': 'Parameter distributions',

        # Early warning subdirs
 'rul': 'Remaining Useful Life prediction',
 'warning': 'Early warning system',
 'tcr': 'Time to Cleaning Required',
 'lifetime': 'Membrane lifetime analysis',
 'cycle': 'Cleaning cycle prediction',
 'alert': 'Alert system performance'
 }

 for keyword, purpose in purpose_map.items():
 if keyword in name_lower:
 return purpose

 return None

def scan_directory(root_path, workflow_context='', max_depth=10):
 """
 Recursively scan directory with workflow awareness.
 """
 if not os.path.exists(root_path):
 return None

 dir_name = os.path.basename(root_path)
 purpose = infer_subdirectory_purpose(dir_name)

 inventory = {
 'path': root_path,
 'name': dir_name,
 'workflow_context': workflow_context,
 'purpose': purpose,
 'type': 'directory',
 'size': 0,
 'file_count': 0,
 'children': [],
 'files': [],
 'summary': {
 'figures': 0,
 'data_files': 0,
 'models': 0,
 'code': 0
 }
 }

 try:
 items = sorted(os.listdir(root_path))
 except PermissionError:
 inventory['error'] = 'Permission Denied'
 return inventory

 for item in items:
 item_path = os.path.join(root_path, item)

 if os.path.isdir(item_path):
 if max_depth > 0:
 subdir_info = scan_directory(item_path, workflow_context, max_depth - 1)
 if subdir_info:
 inventory['children'].append(subdir_info)
 inventory['size'] += subdir_info['size']
 inventory['file_count'] += subdir_info['file_count']
                    # Aggregate summaries
 for key in inventory['summary']:
 inventory['summary'][key] += subdir_info['summary'][key]

 elif os.path.isfile(item_path):
 try:
 file_stat = os.stat(item_path)
 category = categorize_file(item, item_path)

 file_info = {
 'name': item,
 'path': item_path,
 'size': file_stat.st_size,
 'size_readable': get_file_size_readable(file_stat.st_size),
 'modified': datetime.fromtimestamp(file_stat.st_mtime).strftime('%Y-%m-%d %H:%M:%S'),
 'extension': os.path.splitext(item)[1].lower(),
 'category': category,
 'workflow_context': workflow_context,
 'hash': get_file_hash(item_path)
 }

 inventory['files'].append(file_info)
 inventory['size'] += file_stat.st_size
 inventory['file_count'] += 1

                # Update summary counts
 if category.startswith('Figure'):
 inventory['summary']['figures'] += 1
 elif category.startswith('Data'):
 inventory['summary']['data_files'] += 1
 elif category.startswith('Model'):
 inventory['summary']['models'] += 1
 elif category == 'Code':
 inventory['summary']['code'] += 1

 except Exception as e:
 print(f" Error reading {item_path}: {e}")

 return inventory

# ==================== ENHANCED TREE VISUALIZATION ====================

def generate_tree_text(inventory, prefix="", is_last=True, show_files=True, show_summary=True):
 """Generate enhanced ASCII tree with workflow context."""
 if not inventory:
 return ""

 lines = []

    # Current item
 connector = "└── " if is_last else "├── "
 name = inventory['name']

    # Build directory line with enhanced info
 if inventory['type'] == 'directory':
 size_info = f"{get_file_size_readable(inventory['size'])}"
 count_info = f"{inventory['file_count']} files"

        # Add summary if significant
 summary_parts = []
 if show_summary and inventory['summary']['figures'] > 0:
 summary_parts.append(f" {inventory['summary']['figures']} figs")
 if show_summary and inventory['summary']['data_files'] > 0:
 summary_parts.append(f" {inventory['summary']['data_files']} data")
 if show_summary and inventory['summary']['models'] > 0:
 summary_parts.append(f" {inventory['summary']['models']} models")

 summary_str = ", ".join(summary_parts) if summary_parts else count_info

        # Add purpose annotation
 purpose_str = f" [{inventory['purpose']}]" if inventory['purpose'] else ""

 lines.append(f"{prefix}{connector} {name}/ ({summary_str}, {size_info}){purpose_str}")
 else:
 lines.append(f"{prefix}{connector}{name}")

    # Update prefix for children
 extension = " " if is_last else "│ "
 new_prefix = prefix + extension

    # Files in current directory - grouped by category
 if show_files and inventory.get('files'):
 files = inventory['files']

        # Group files by category
 by_category = {}
 for f in files:
 cat = f['category']
 if cat not in by_category:
 by_category[cat] = []
 by_category[cat].append(f)

        # Display grouped files
 categories_to_show = sorted(by_category.keys())
 for cat_idx, cat in enumerate(categories_to_show):
 cat_files = sorted(by_category[cat], key=lambda x: x['name'])

            # Show category header if multiple categories
 if len(categories_to_show) > 1:
 is_last_cat = (cat_idx == len(categories_to_show) - 1) and not inventory.get('children')
 cat_connector = "└── " if is_last_cat else "├── "

                # Choose emoji based on category
 cat_emoji = {
 'Figure_Preprocessing': '',
 'Figure_SHAP': '',
 'Figure_OptimalConditions': '',
 'Figure_EarlyWarning': '',
 'Figure_ModelPerformance': '',
 'Data_Cleaned': '',
 'Data_OptimalConditions': '',
 'Data_EarlyWarning': '',
 'Data_SHAP': '',
 'Model_TrainedModel': ''
 }.get(cat, '')

 cat_display = cat.replace('_', ' ')
 lines.append(f"{new_prefix}{cat_connector}{cat_emoji} {cat_display} ({len(cat_files)})")

 file_prefix = new_prefix + (" " if is_last_cat else "│ ")
 else:
 file_prefix = new_prefix

            # Show individual files
 for f_idx, file_info in enumerate(cat_files):
 is_last_file = (f_idx == len(cat_files) - 1)
 if len(categories_to_show) == 1:
 is_last_file = is_last_file and not inventory.get('children')

 file_connector = "└── " if is_last_file else "├── "
 file_line = f"{file_prefix}{file_connector}{file_info['name']} ({file_info['size_readable']})"
 lines.append(file_line)

    # Subdirectories
 children = inventory.get('children', [])
 for i, child in enumerate(children):
 is_last_child = (i == len(children) - 1)
 lines.append(generate_tree_text(child, new_prefix, is_last_child, show_files, show_summary))

 return "\n".join(filter(None, lines))

# ==================== WORKFLOW-AWARE STATISTICS ====================

def generate_workflow_statistics(inventories):
 """Generate statistics organized by research workflow."""

 stats = {
 'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
 'workflow_summary': {},
 'overall': {
 'total_files': 0,
 'total_size': 0,
 'by_category': {},
 'by_extension': {}
 }
 }

 def count_recursive(inv, workflow_step):
 stats['overall']['total_files'] += inv['file_count']
 stats['overall']['total_size'] += inv['size']

        # Initialize workflow summary
 if workflow_step not in stats['workflow_summary']:
 stats['workflow_summary'][workflow_step] = {
 'files': 0,
 'size': 0,
 'figures': 0,
 'data_files': 0,
 'models': 0,
 'subdirectories': []
 }

 step_stats = stats['workflow_summary'][workflow_step]
 step_stats['files'] += inv['file_count']
 step_stats['size'] += inv['size']
 step_stats['figures'] += inv['summary']['figures']
 step_stats['data_files'] += inv['summary']['data_files']
 step_stats['models'] += inv['summary']['models']

        # Track subdirectories with purpose
 if inv['name'] != workflow_step and inv['purpose']:
 step_stats['subdirectories'].append({
 'name': inv['name'],
 'purpose': inv['purpose'],
 'files': inv['file_count']
 })

        # Process files
 for file_info in inv.get('files', []):
            # By category
 category = file_info['category']
 if category not in stats['overall']['by_category']:
 stats['overall']['by_category'][category] = {'count': 0, 'size': 0}
 stats['overall']['by_category'][category]['count'] += 1
 stats['overall']['by_category'][category]['size'] += file_info['size']

            # By extension
 ext = file_info['extension']
 if ext not in stats['overall']['by_extension']:
 stats['overall']['by_extension'][ext] = {'count': 0, 'size': 0}
 stats['overall']['by_extension'][ext]['count'] += 1
 stats['overall']['by_extension'][ext]['size'] += file_info['size']

        # Process children
 for child in inv.get('children', []):
 count_recursive(child, workflow_step)

 for workflow_step, inv in inventories.items():
 if inv:
 count_recursive(inv, workflow_step)

 return stats

# ==================== MAIN EXECUTION ====================

def generate_complete_inventory():
 """Main function with workflow awareness."""

 print("\n" + "=" * 80)
 print("RESEARCH WORKFLOW")
 print("=" * 80)
 print("\n This project follows a 3-step workflow:\n")

 for step, info in DATA_DIRECTORIES.items():
 print(f"{step}:")
 print(f" Description: {info['description']}")
 print(f" Location: {info['path']}")
 print(f" Contains: {', '.join(info['contains'][:2])}...")
 print()

 print("\n" + "=" * 80)
 print("SCANNING DIRECTORIES")
 print("=" * 80)

 inventories = {}

 for step, info in DATA_DIRECTORIES.items():
 print(f"\n Scanning: {step}")
 print(f" Path: {info['path']}")

 inventory = scan_directory(info['path'], workflow_context=step)

 if inventory:
 inventories[step] = inventory
 print(f" {inventory['file_count']} files ({get_file_size_readable(inventory['size'])})")
 print(f" {inventory['summary']['figures']} figures, "
 f"{inventory['summary']['data_files']} data files, "
 f"{inventory['summary']['models']} models")
 else:
 print(f" Directory not found")
 inventories[step] = None

    # Generate workflow-aware statistics
 print("\n" + "=" * 80)
 print("GENERATING WORKFLOW STATISTICS")
 print("=" * 80)

 stats = generate_workflow_statistics(inventories)

 print(f"\n Overall Statistics:")
 print(f" Total Files: {stats['overall']['total_files']:,}")
 print(f" Total Size: {get_file_size_readable(stats['overall']['total_size'])}")

 print(f"\n By Workflow Step:")
 for step, step_stats in stats['workflow_summary'].items():
 print(f"\n {step}:")
 print(f" Files: {step_stats['files']:,}")
 print(f" Figures: {step_stats['figures']}, Data: {step_stats['data_files']}, Models: {step_stats['models']}")
 print(f" Size: {get_file_size_readable(step_stats['size'])}")

    # Generate enhanced tree visualizations
 print("\n" + "=" * 80)
 print("GENERATING ENHANCED TREES")
 print("=" * 80)

 trees = {}
 for step, inventory in inventories.items():
 if inventory:
 print(f"\n Creating tree for: {step}")
 trees[step] = generate_tree_text(inventory, show_files=True, show_summary=True)

    # Save outputs
 print("\n" + "=" * 80)
 print("SAVING INVENTORY FILES")
 print("=" * 80)

 timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

    # 1. Enhanced tree visualization
 tree_file = f"{OUTPUT_DIR}/DIRECTORY_TREE_{timestamp}.txt"
 with open(tree_file, 'w', encoding='utf-8') as f:
 f.write("=" * 80 + "\n")
 f.write("UF MEMBRANE/BIOLOGICAL PROJECT - WORKFLOW-AWARE DIRECTORY STRUCTURE\n")
 f.write(f"Generated: {stats['timestamp']}\n")
 f.write("=" * 80 + "\n\n")

 f.write("RESEARCH WORKFLOW:\n")
 f.write("-" * 80 + "\n\n")

 for step, info in DATA_DIRECTORIES.items():
 f.write(f"{step}\n")
 f.write(f" {info['description']}\n")
 f.write(f" {info['path']}\n")
 f.write(f" Contains: {', '.join(info['contains'])}\n\n")

 f.write("\n" + "=" * 80 + "\n")
 f.write("DIRECTORY TREES\n")
 f.write("=" * 80 + "\n\n")

 f.write("Legend:\n")
 f.write(" = Preprocessing figures\n")
 f.write(" = SHAP analysis\n")
 f.write(" = Optimal conditions\n")
 f.write(" = Early warning system\n")
 f.write(" = Model performance\n")
 f.write(" = Cleaned data\n")
 f.write(" = Trained models\n\n")

 for step, tree in trees.items():
 f.write(f"\n{'=' * 80}\n")
 f.write(f"{step}\n")
 f.write(f"{'=' * 80}\n\n")
 f.write(tree)
 f.write("\n\n")
 print(f" Tree: {tree_file}")

    # 2. Workflow summary
 summary_file = f"{OUTPUT_DIR}/WORKFLOW_SUMMARY_{timestamp}.txt"
 with open(summary_file, 'w') as f:
 f.write("=" * 80 + "\n")
 f.write("RO MEMBRANE PROJECT - WORKFLOW SUMMARY\n")
 f.write(f"Generated: {stats['timestamp']}\n")
 f.write("=" * 80 + "\n\n")

 for step, step_stats in stats['workflow_summary'].items():
 f.write(f"\n{'=' * 80}\n")
 f.write(f"{step}\n")
 f.write(f"{'=' * 80}\n\n")

 f.write(f"Description: {DATA_DIRECTORIES[step]['description']}\n\n")

 f.write(f"Statistics:\n")
 f.write(f" Total Files: {step_stats['files']:,}\n")
 f.write(f" Figures: {step_stats['figures']}\n")
 f.write(f" Data Files: {step_stats['data_files']}\n")
 f.write(f" Models: {step_stats['models']}\n")
 f.write(f" Total Size: {get_file_size_readable(step_stats['size'])}\n\n")

 if step_stats['subdirectories']:
 f.write(f"Key Subdirectories:\n")
 for subdir in step_stats['subdirectories']:
 f.write(f" • {subdir['name']}/\n")
 f.write(f" Purpose: {subdir['purpose']}\n")
 f.write(f" Files: {subdir['files']}\n")
 print(f" Summary: {summary_file}")

    # 3. Save complete JSON
 json_file = f"{OUTPUT_DIR}/inventory_complete_{timestamp}.json"
 with open(json_file, 'w') as f:
 json.dump({
 'metadata': {
 'timestamp': stats['timestamp'],
 'workflow': DATA_DIRECTORIES
 },
 'inventories': inventories,
 'statistics': stats
 }, f, indent=2)
 print(f" JSON: {json_file}")

    # 4. Extract all files
 all_files = []
 for step, inventory in inventories.items():
 if inventory:
 files = extract_all_files(inventory, step)
 all_files.extend(files)

    # 5. Save file list
 csv_file = f"{OUTPUT_DIR}/file_list_{timestamp}.csv"
 df_files = pd.DataFrame(all_files)
 if not df_files.empty:
 cols = ['workflow_context', 'category', 'name', 'extension', 'size_readable',
 'modified', 'hash', 'path']
 df_files = df_files[[c for c in cols if c in df_files.columns]]
 df_files.to_csv(csv_file, index=False)
 print(f" CSV: {csv_file}")

    # 6. Create MANUSCRIPT HELPER
 helper_file = f"{OUTPUT_DIR}/MANUSCRIPT_HELPER_{timestamp}.txt"
 create_manuscript_helper(helper_file, all_files, stats)
 print(f" Helper: {helper_file}")

    # Final summary
 print("\n" + "=" * 80)
 print("INVENTORY GENERATION COMPLETE")
 print("=" * 80)
 print(f"\n All files saved to: {OUTPUT_DIR}")
 print(f"\n KEY FILES:")
 print(f" 1. {tree_file}")
 print(f" → Enhanced directory tree with workflow context")
 print(f" 2. {helper_file}")
 print(f" → Quick reference for manuscript writing")
 print(f" 3. {summary_file}")
 print(f" → Workflow step summaries")
 print("\n" + "=" * 80)

 return {
 'inventories': inventories,
 'statistics': stats,
 'all_files': all_files,
 'key_files': {
 'tree': tree_file,
 'helper': helper_file,
 'summary': summary_file
 }
 }

def extract_all_files(inventory, workflow_step=''):
 """Extract flat list of all files."""
 files = []

 for file_info in inventory.get('files', []):
 file_entry = file_info.copy()
 file_entry['workflow_step'] = workflow_step
 files.append(file_entry)

 for child in inventory.get('children', []):
 files.extend(extract_all_files(child, workflow_step))

 return files

def create_manuscript_helper(filename, all_files, stats):
 """Create manuscript helper organized by paper sections."""

 with open(filename, 'w', encoding='utf-8') as f:
 f.write("=" * 80 + "\n")
 f.write("UF MEMBRANE/BIOLOGICAL PROJECT - MANUSCRIPT WRITING HELPER\n")
 f.write(f"Generated: {stats['timestamp']}\n")
 f.write("=" * 80 + "\n\n")

 f.write("This file organizes all figures and data by manuscript section.\n")
 f.write("Use this as a quick reference when writing each part of the paper.\n\n")

        # Section mapping aligned with workflow
 sections = {
 'SECTION 2.3: Data Preprocessing & Quality Control': {
 'workflow': 'Step1_DataPreprocessing',
 'categories': ['Figure_Preprocessing', 'Data_Cleaned'],
 'description': 'Cleaning period detection, outlier removal, data quality'
 },
 'SECTION 2.5-2.6: ML Model Development & Performance': {
 'workflow': 'Step2_MLPrediction',
 'categories': ['Figure_ModelPerformance', 'Data_Results', 'Model_TrainedModel'],
 'description': 'Model comparison, performance metrics, predictions'
 },
 'SECTION 2.7: SHAP Interpretability Analysis': {
 'workflow': 'Step2_MLPrediction',
 'categories': ['Figure_SHAP', 'Data_SHAP'],
 'description': 'Feature importance, SHAP values, waterfall plots'
 },
 'SECTION 2.8: Optimal Operating Conditions': {
 'workflow': 'Step2_MLPrediction',
 'categories': ['Figure_OptimalConditions', 'Data_OptimalConditions'],
 'description': 'Feasible regions, sensitivity analysis, operating windows'
 },
 'SECTION 2.9: Early Warning System': {
 'workflow': 'Step3_EarlyWarning',
 'categories': ['Figure_EarlyWarning', 'Data_EarlyWarning'],
 'description': 'RUL prediction, cleaning cycle forecasts, membrane lifetime'
 }
 }

 for section_title, section_info in sections.items():
 f.write(f"\n{'=' * 80}\n")
 f.write(f"{section_title}\n")
 f.write(f"{'=' * 80}\n\n")
 f.write(f"Description: {section_info['description']}\n")
 f.write(f"Workflow Step: {section_info['workflow']}\n\n")

            # Filter files for this section
 section_files = [
 file for file in all_files
 if file['workflow_context'] == section_info['workflow']
 and file['category'] in section_info['categories']
 ]

 if section_files:
                # Separate figures and data
 figures = [f for f in section_files if f['category'].startswith('Figure')]
 data_files = [f for f in section_files if f['category'].startswith('Data')]
 models = [f for f in section_files if f['category'].startswith('Model')]

 if figures:
 f.write(f"FIGURES ({len(figures)}):\n")
 f.write("-" * 80 + "\n")
 for fig in sorted(figures, key=lambda x: x['name']):
 f.write(f" {fig['name']}\n")
 f.write(f" Path: {fig['path']}\n")
 f.write(f" Size: {fig['size_readable']}\n")
 f.write(f" Modified: {fig['modified']}\n\n")

 if data_files:
 f.write(f"\nDATA FILES ({len(data_files)}):\n")
 f.write("-" * 80 + "\n")
 for data in sorted(data_files, key=lambda x: x['name']):
 f.write(f" {data['name']} ({data['size_readable']})\n")
 f.write(f" Path: {data['path']}\n")
 f.write(f" Modified: {data['modified']}\n\n")

 if models:
 f.write(f"\nTRAINED MODELS ({len(models)}):\n")
 f.write("-" * 80 + "\n")
 for model in sorted(models, key=lambda x: x['name']):
 f.write(f" {model['name']} ({model['size_readable']})\n")
 f.write(f" Path: {model['path']}\n\n")
 else:
 f.write(" (No files found for this section)\n")

# ==================== RUN ====================

if __name__ == "__main__":
 result = generate_complete_inventory()
 print("\n Workflow-aware inventory generation complete!")


In [ ]:
import os
import pandas as pd
from datetime import datetime
FOLDER = BASE_DIR
REPORT_PATH = os.path.join(FOLDER, 'CSV_data_inventory.txt')

lines = []  # collect all output lines

def log(text=""):
 print(text)
 lines.append(text)

# ── 1. Directory Tree ──────────────────────────────────────
log("=" * 70)
log(" DATA INVENTORY ,  UF MEMBRANE/BIOLOGICAL PROCESS")
log(f" Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
log("=" * 70)

all_files = []
for root, dirs, files in os.walk(FOLDER):
    # Skip hidden folders
 dirs[:] = [d for d in dirs if not d.startswith('.')]
 level = root.replace(FOLDER, '').count(os.sep)
 indent = '│ ' * level
 folder_name = os.path.basename(root)
 log(f"{indent} {folder_name}/")
 sub_indent = '│ ' * (level + 1)
 for f in sorted(files):
 if f.startswith('.') or f.startswith('_data_inventory'):
 continue
 fpath = os.path.join(root, f)
 size_kb = os.path.getsize(fpath) / 1024
 ext = os.path.splitext(f)[1].lower()
 icon = '' if ext == '.csv' else '' if ext == '.txt' else '' if ext == '.png' else ''
 log(f"{sub_indent}{icon} {f} ({size_kb:.1f} KB)")
 all_files.append({
 'filename': f,
 'path': fpath,
 'relative_path': os.path.relpath(fpath, FOLDER),
 'extension': ext,
 'size_kb': round(size_kb, 1)
 })

# ── 2. Summary counts ─────────────────────────────────────
log("\n" + "=" * 70)
log(" FILE SUMMARY")
log("=" * 70)
ext_counts = {}
for f in all_files:
 ext_counts[f['extension']] = ext_counts.get(f['extension'], 0) + 1
for ext, count in sorted(ext_counts.items()):
 log(f" {ext or '(no ext)':10s} : {count} files")
log(f" {'TOTAL':10s} : {len(all_files)} files")

# ── 3. CSV Details ─────────────────────────────────────────
csv_files = [f for f in all_files if f['extension'] == '.csv']

log("\n" + "=" * 70)
log(f" CSV FILE DETAILS ({len(csv_files)} files)")
log("=" * 70)

csv_summaries = []

for i, f in enumerate(csv_files, 1):
 log(f"\n{'─' * 60}")
 log(f" [{i}/{len(csv_files)}] {f['relative_path']}")
 log(f" Size: {f['size_kb']:.1f} KB")
 log(f" Path: {f['path']}")

 try:
 df = pd.read_csv(f['path'])
 log(f" Shape: {df.shape[0]} rows × {df.shape[1]} columns")
 log(f" Columns: {list(df.columns)}")
 log(f" Dtypes:")
 for col in df.columns:
 log(f" {col:25s} → {df[col].dtype} "
 f"(nulls: {df[col].isna().sum()})")

 log(f"\n Numeric statistics:")
 desc = df.describe().round(4)
 desc_str = desc.to_string()
 for line in desc_str.split('\n'):
 log(f" {line}")

        # First 3 rows
 log(f"\n First 3 rows:")
 for line in df.head(3).to_string(index=False).split('\n'):
 log(f" {line}")

 csv_summaries.append({
 'filename': f['filename'],
 'path': f['path'],
 'rows': df.shape[0],
 'cols': df.shape[1],
 'columns': list(df.columns),
 'size_kb': f['size_kb']
 })

 except Exception as e:
 log(f" Error reading: {e}")
 csv_summaries.append({
 'filename': f['filename'],
 'path': f['path'],
 'rows': None,
 'cols': None,
 'columns': None,
 'size_kb': f['size_kb'],
 'error': str(e)
 })

# ── 4. Quick reference table ──────────────────────────────
log("\n" + "=" * 70)
log(" QUICK REFERENCE TABLE")
log("=" * 70)
log(f" {'Filename':<45s} {'Rows':>6s} {'Cols':>5s} {'Columns'}")
log(f" {'─'*45} {'─'*6} {'─'*5} {'─'*40}")
for s in csv_summaries:
 if s.get('rows') is not None:
 cols_str = ', '.join(s['columns'][:5])
 if len(s['columns']) > 5:
 cols_str += f', ... (+{len(s["columns"])-5})'
 log(f" {s['filename']:<45s} {s['rows']:>6d} {s['cols']:>5d} {cols_str}")
 else:
 log(f" {s['filename']:<45s} ERROR")

# ── 5. Save report ─────────────────────────────────────────
with open(REPORT_PATH, 'w') as f:
 f.write('\n'.join(lines))

print(f"\n{'=' * 70}")
print(f" Report saved to: {REPORT_PATH}")
print(f" ({len(lines)} lines, {len(csv_files)} CSV files analyzed)")
print(f" Share this file with me for the next step.")


# 1. Set up and data loading

In [ ]:
%pip install tabulate


In [ ]:
# -*- coding: utf-8 -*-
"""UF_data Machine Learning Analysis

For Ultrafiltration (UF) membrane system

# STEP 1: IMPORT LIBRARIES AND SETUP
"""

# ==================== SECTION 1: IMPORT LIBRARY ====================

# Note: Run this first if tabulate is not installed
# pip install tabulate

# Core data manipulation and numerical computing
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# System utilities
import os
from datetime import datetime

# Statistical analysis
from scipy import stats

# Scikit-learn: Model selection and evaluation
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import (
 mean_squared_error,
 mean_absolute_error,
 r2_score,
 accuracy_score,
 precision_score,
 recall_score,
 f1_score,
 confusion_matrix,
 multilabel_confusion_matrix
)

# Scikit-learn: Regression models
from sklearn.linear_model import LinearRegression, ElasticNet, Lasso, Ridge
from sklearn.ensemble import (
 RandomForestRegressor,
 ExtraTreesRegressor,
 GradientBoostingRegressor,
 BaggingRegressor
)
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR

# Scikit-learn: Classification models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

# Scikit-learn: Feature importance
from sklearn.inspection import permutation_importance

# Deep learning with TensorFlow/Keras
import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Dense, Activation, BatchNormalization
from tensorflow.keras import layers, losses

# Model persistence
import joblib

# Table formatting
from tabulate import tabulate

# Google Colab utilities (if using Colab)
try:

 COLAB_AVAILABLE = True
except ImportError:
 COLAB_AVAILABLE = False
 print("Note: Google Colab not available. Skipping Colab-specific imports.")

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print("="*60)
print(" All libraries imported successfully")


# 1. Load data and data preprocessing

In [ ]:
print("data root set
")


In [ ]:
# -*- coding: utf-8 -*-
"""
UF Water Treatment ML Code - 
Step 1: Imports and Setup

UPDATED FEATURES:
Input features (7): glu, mlss, air, fm, cn, hrt, srt
Output features (3): tmp, flow, lv

Changes from original:
- Removed: lv from inputs (now an output), cn1, cn2 (merged to cn), hrt1, hrt2 (merged to hrt)
- Added: lv to outputs
- Total: 7 inputs → 3 outputs (was 10 inputs → 2 outputs)
"""

# ==================== STEP 1: IMPORTS AND SETUP ====================

print("\n" + "="*80)
print(" " * 25 + "STEP 1: IMPORTS AND SETUP")
print("="*80)

# Standard libraries
import os
import pickle
import warnings
warnings.filterwarnings('ignore')

# Data manipulation
import numpy as np
import pandas as pd

# Machine learning - scikit-learn
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# Machine learning models
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (RandomForestRegressor, ExtraTreesRegressor,
 GradientBoostingRegressor, AdaBoostRegressor,
 BaggingRegressor, HistGradientBoostingRegressor)
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor

# XGBoost and LightGBM
try:
 import xgboost as xgb
 HAS_XGBOOST = True
except ImportError:
 print(" XGBoost not installed. Install with: pip install xgboost")
 HAS_XGBOOST = False

try:
 import lightgbm as lgb
 HAS_LIGHTGBM = True
except ImportError:
 print(" LightGBM not installed. Install with: pip install lightgbm")
 HAS_LIGHTGBM = False

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Additional utilities
from scipy import stats
from datetime import datetime

print(" All required libraries imported successfully")

# ==================== CONFIGURATION ====================

# Set random seed for reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Feature definitions - UPDATED
INPUT_FEATURES = ['glu', 'mlss', 'air', 'fm', 'cn', 'hrt', 'srt']
OUTPUT_FEATURES = ['tmp', 'flow', 'lv']

N_INPUTS = len(INPUT_FEATURES)
N_OUTPUTS = len(OUTPUT_FEATURES)

print(f"\n Configuration set:")
print(f" • Random seed: {RANDOM_SEED}")
print(f" • Input features ({N_INPUTS}): {', '.join(INPUT_FEATURES)}")
print(f" • Output features ({N_OUTPUTS}): {', '.join(OUTPUT_FEATURES)}")

# Feature descriptions
FEATURE_DESCRIPTIONS = {
    # Input features
 'glu': 'Glucose concentration (g/L)',
 'mlss': 'Mixed Liquor Suspended Solids (mg/L)',
 'air': 'Air flow rate (L/min)',
 'fm': 'Flux rate (L/m²/h)',
 'cn': 'Concentration (combined cn1+cn2)',
 'hrt': 'Hydraulic Retention Time (combined hrt1+hrt2) (h)',
 'srt': 'Solids Retention Time (days)',

    # Output features
 'tmp': 'Transmembrane Pressure (bar)',
 'flow': 'Permeate Flow Rate (L/h)',
 'lv': 'Liquid Volume (L)'
}

print(f"\n Feature descriptions loaded")

# ==================== DIRECTORY SETUP ====================

# Detect environment (Google Colab or local)
try:

 IN_COLAB = True
 print("\n Running in Google Colab")
 print(" Mounting Google Drive...")

    # Set base directory
 BASE_DIR = BASE_DIR
 DATA_FILE = f'{BASE_DIR}/UF_data.csv'

except ImportError:
 IN_COLAB = False
 print("\n Running in local environment")

    # Set base directory for local
 BASE_DIR = './UF_data_results'
 DATA_FILE = './UF_data/UF_data.csv'

# Create output directories
RESULTS_DIR = f'{BASE_DIR}/Results'
MODELS_DIR = f'{BASE_DIR}/Trained_Models'
VISUALIZATIONS_DIR = f'{BASE_DIR}/Visualizations'

for directory in [RESULTS_DIR, MODELS_DIR, VISUALIZATIONS_DIR]:
 os.makedirs(directory, exist_ok=True)

print(f"\n Directories configured:")
print(f" • Base: {BASE_DIR}")
print(f" • Data file: {DATA_FILE}")
print(f" • Results: {RESULTS_DIR}")
print(f" • Models: {MODELS_DIR}")
print(f" • Visualizations: {VISUALIZATIONS_DIR}")

# ==================== VISUALIZATION SETTINGS ====================

# Set global plot style
plt.style.use('default')
sns.set_palette("husl")

# Disable grid by default (cleaner plots)
plt.rcParams['axes.grid'] = False

# Set default figure parameters
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['font.size'] = 11
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['xtick.labelsize'] = 10
plt.rcParams['ytick.labelsize'] = 10
plt.rcParams['legend.fontsize'] = 10

print(f"\n Visualization settings configured")

# ==================== MODEL CONFIGURATION ====================

# Define all models to be used
MODEL_CONFIGS = {
 'Linear Regression': {
 'model': LinearRegression(),
 'type': 'linear',
 'description': 'Basic linear regression'
 },
 'Ridge Regression': {
 'model': Ridge(alpha=1.0, random_state=RANDOM_SEED),
 'type': 'linear',
 'description': 'L2 regularized linear regression'
 },
 'Lasso Regression': {
 'model': Lasso(alpha=0.1, random_state=RANDOM_SEED, max_iter=10000),
 'type': 'linear',
 'description': 'L1 regularized linear regression'
 },
 'ElasticNet': {
 'model': ElasticNet(alpha=0.1, l1_ratio=0.5, random_state=RANDOM_SEED, max_iter=10000),
 'type': 'linear',
 'description': 'L1 + L2 regularized linear regression'
 },
 'Decision Tree': {
 'model': DecisionTreeRegressor(random_state=RANDOM_SEED, max_depth=10),
 'type': 'tree',
 'description': 'Single decision tree'
 },
 'Random Forest': {
 'model': RandomForestRegressor(n_estimators=100, random_state=RANDOM_SEED, n_jobs=-1),
 'type': 'ensemble',
 'description': 'Ensemble of decision trees'
 },
 'Extra Trees': {
 'model': ExtraTreesRegressor(n_estimators=100, random_state=RANDOM_SEED, n_jobs=-1),
 'type': 'ensemble',
 'description': 'Extremely randomized trees'
 },
 'Gradient Boosting': {
 'model': GradientBoostingRegressor(n_estimators=100, random_state=RANDOM_SEED),
 'type': 'boosting',
 'description': 'Gradient boosting trees'
 },
 'AdaBoost': {
 'model': AdaBoostRegressor(n_estimators=100, random_state=RANDOM_SEED),
 'type': 'boosting',
 'description': 'Adaptive boosting'
 },
 'Hist Gradient Boosting': {
 'model': HistGradientBoostingRegressor(random_state=RANDOM_SEED),
 'type': 'boosting',
 'description': 'Histogram-based gradient boosting'
 },
 'KNN': {
 'model': KNeighborsRegressor(n_neighbors=5, n_jobs=-1),
 'type': 'instance',
 'description': 'K-nearest neighbors'
 },
 'SVR': {
 'model': SVR(kernel='rbf', C=1.0),
 'type': 'kernel',
 'description': 'Support vector regression'
 },
 'Bagging': {
 'model': BaggingRegressor(n_estimators=100, random_state=RANDOM_SEED, n_jobs=-1),
 'type': 'ensemble',
 'description': 'Bootstrap aggregating'
 },
 'MLP Neural Network': {
 'model': MLPRegressor(hidden_layer_sizes=(100, 50), random_state=RANDOM_SEED,
 max_iter=1000, early_stopping=True),
 'type': 'neural',
 'description': 'Multi-layer perceptron'
 }
}

# Add XGBoost if available
if HAS_XGBOOST:
 MODEL_CONFIGS['XGBoost'] = {
 'model': xgb.XGBRegressor(n_estimators=100, random_state=RANDOM_SEED, n_jobs=-1),
 'type': 'boosting',
 'description': 'Extreme gradient boosting'
 }

# Add LightGBM if available
if HAS_LIGHTGBM:
 MODEL_CONFIGS['LightGBM'] = {
 'model': lgb.LGBMRegressor(n_estimators=100, random_state=RANDOM_SEED, n_jobs=-1, verbose=-1),
 'type': 'boosting',
 'description': 'Light gradient boosting machine'
 }

print(f"\n Model configurations loaded:")
print(f" • Total models: {len(MODEL_CONFIGS)}")
print(f" • Linear models: {sum(1 for m in MODEL_CONFIGS.values() if m['type'] == 'linear')}")
print(f" • Tree models: {sum(1 for m in MODEL_CONFIGS.values() if m['type'] == 'tree')}")
print(f" • Ensemble models: {sum(1 for m in MODEL_CONFIGS.values() if m['type'] == 'ensemble')}")
print(f" • Boosting models: {sum(1 for m in MODEL_CONFIGS.values() if m['type'] == 'boosting')}")

# ==================== UTILITY FUNCTIONS ====================

def print_section_header(title):
 """Print a formatted section header."""
 print("\n" + "="*80)
 print(f" {title}")
 print("="*80)

def save_results(data, filename, file_format='csv'):
 """Save results to file."""
 filepath = f'{RESULTS_DIR}/{filename}'

 if file_format == 'csv':
 if isinstance(data, pd.DataFrame):
 data.to_csv(filepath, index=False)
 else:
 pd.DataFrame(data).to_csv(filepath, index=False)
 elif file_format == 'pickle':
 with open(filepath, 'wb') as f:
 pickle.dump(data, f)

 print(f" Saved: {filename}")
 return filepath

def create_timestamp():
 """Create a timestamp string."""
 return datetime.now().strftime("%Y%m%d_%H%M%S")

print(f"\n Utility functions defined")

# ==================== SUMMARY ====================

print("\n" + "="*80)
print("STEP 1 COMPLETE - SYSTEM READY")
print("="*80)

print(f"\nSystem Configuration:")
print(f" • Python environment: {'Google Colab' if IN_COLAB else 'Local'}")
print(f" • Input features: {N_INPUTS}")
print(f" • Output features: {N_OUTPUTS}")
print(f" • Models configured: {len(MODEL_CONFIGS)}")
print(f" • Random seed: {RANDOM_SEED}")

print(f"\nNext Steps:")
print(f" → Step 2: Load and explore data")
print(f" → Step 3: Data preprocessing and standardization")
print(f" → Step 4: Exploratory data analysis")
print(f" → Step 5: Model training and evaluation")

print("\n" + "="*80)


# 2&3. Data separation and saved testing set and training set

In [ ]:
# -*- coding: utf-8 -*-
"""
UF Water Treatment ML Code - 
Steps 2 & 3: Data Loading, Splitting, and Standardization

UPDATED FEATURES:
Input features (7): glu, mlss, air, fm, cn, hrt, srt
Output features (3): tmp, flow, lv
"""

# ==================== STEP 2: DATA LOADING ====================

print("\n" + "="*80)
print(" " * 25 + "STEP 2: DATA LOADING")
print("="*80)

# Load the dataset
print(f"\nLoading data from: {DATA_FILE}")
try:
 df = pd.read_csv(DATA_FILE)
 print(f" Data loaded successfully")
 print(f" • Total samples: {len(df)}")
 print(f" • Total features: {len(df.columns)}")
except FileNotFoundError:
 print(f"\n ERROR: Data file not found at {DATA_FILE}")
 print("Please ensure the data file exists at the specified location.")
 raise

# Display first few rows
print("\n" + "-"*80)
print("DATA PREVIEW (First 5 rows)")
print("-"*80)
print(df.head())

# Display data information
print("\n" + "-"*80)
print("DATA INFORMATION")
print("-"*80)
print(df.info())

# Display basic statistics
print("\n" + "-"*80)
print("BASIC STATISTICS")
print("-"*80)
print(df.describe())

# Check for missing values
print("\n" + "-"*80)
print("MISSING VALUES CHECK")
print("-"*80)
missing = df.isnull().sum()
if missing.sum() == 0:
 print(" No missing values found")
else:
 print(" Missing values detected:")
 print(missing[missing > 0])

# Verify all required columns exist
print("\n" + "-"*80)
print("FEATURE VERIFICATION")
print("-"*80)

all_features = INPUT_FEATURES + OUTPUT_FEATURES
missing_features = [f for f in all_features if f not in df.columns]

if missing_features:
 print(f"\n ERROR: Missing required features: {missing_features}")
 print(f"Available columns: {list(df.columns)}")
 raise ValueError("Required features not found in dataset")
else:
 print(f" All required features found in dataset")
 print(f"\nInput features ({N_INPUTS}):")
 for i, feat in enumerate(INPUT_FEATURES, 1):
 print(f" {i}. {feat}: {FEATURE_DESCRIPTIONS[feat]}")
 print(f"\nOutput features ({N_OUTPUTS}):")
 for i, feat in enumerate(OUTPUT_FEATURES, 1):
 print(f" {i}. {feat}: {FEATURE_DESCRIPTIONS[feat]}")

# ==================== STEP 3: DATA SPLITTING AND STANDARDIZATION ====================

print("\n" + "="*80)
print(" " * 20 + "STEP 3: DATA SPLITTING AND STANDARDIZATION")
print("="*80)

# Extract input and output data
print("\nExtracting features...")

# Input features (7 features)
Input = df[INPUT_FEATURES].values
print(f" Input shape: {Input.shape} ({N_INPUTS} features)")

# Output features (3 features)
Output = df[OUTPUT_FEATURES].values
print(f" Output shape: {Output.shape} ({N_OUTPUTS} features)")

# Check for NaN values in outputs
print("\nChecking for NaN values in outputs...")
for i, feat in enumerate(OUTPUT_FEATURES):
 nan_count = np.isnan(Output[:, i]).sum()
 if nan_count > 0:
 print(f" {feat}: {nan_count} NaN values found ({nan_count/len(Output)*100:.2f}%)")
 else:
 print(f" {feat}: No NaN values")

# Handle NaN values - drop rows with any NaN in outputs
if np.isnan(Output).any():
 print("\n NaN values detected in output features. Removing affected rows...")
 valid_mask = ~np.isnan(Output).any(axis=1)
 Input = Input[valid_mask]
 Output = Output[valid_mask]
 print(f" Removed {(~valid_mask).sum()} rows with NaN values")
 print(f" New dataset size: {len(Input)} samples")
else:
 print("\n No NaN values in output features")

# Display value ranges before standardization
print("\n" + "-"*80)
print("VALUE RANGES (Before Standardization)")
print("-"*80)

print("\nInput Features:")
for i, feat in enumerate(INPUT_FEATURES):
 print(f" {feat:8s}: min={Input[:, i].min():10.4f}, max={Input[:, i].max():10.4f}, "
 f"mean={Input[:, i].mean():10.4f}, std={Input[:, i].std():10.4f}")

print("\nOutput Features:")
for i, feat in enumerate(OUTPUT_FEATURES):
 print(f" {feat:8s}: min={Output[:, i].min():10.4f}, max={Output[:, i].max():10.4f}, "
 f"mean={Output[:, i].mean():10.4f}, std={Output[:, i].std():10.4f}")

# ==================== STANDARDIZATION ====================

print("\n" + "-"*80)
print("STANDARDIZING DATA")
print("-"*80)

# Calculate statistics for inputs
Input_mean = np.mean(Input, axis=0)
Input_std = np.std(Input, axis=0)

# Prevent division by zero (if std is 0, set it to 1)
Input_std = np.where(Input_std == 0, 1, Input_std)

# Standardize inputs: (X - mean) / std
Input_std_data = (Input - Input_mean) / Input_std

print(f" Input data standardized")
print(f" • Mean: {Input_mean}")
print(f" • Std: {Input_std}")

# Calculate statistics for outputs
Output_mean = np.mean(Output, axis=0)
Output_std = np.std(Output, axis=0)

# Prevent division by zero
Output_std = np.where(Output_std == 0, 1, Output_std)

# Standardize outputs: (Y - mean) / std
Output_std_data = (Output - Output_mean) / Output_std

print(f"\n Output data standardized")
print(f" • Mean: {Output_mean}")
print(f" • Std: {Output_std}")

# Verify standardization (mean ≈ 0, std ≈ 1)
print("\n" + "-"*80)
print("STANDARDIZATION VERIFICATION")
print("-"*80)

print("\nStandardized Input Features:")
for i, feat in enumerate(INPUT_FEATURES):
 mean_check = Input_std_data[:, i].mean()
 std_check = Input_std_data[:, i].std()
 print(f" {feat:8s}: mean={mean_check:8.6f} (should be ≈0), std={std_check:8.6f} (should be ≈1)")

print("\nStandardized Output Features:")
for i, feat in enumerate(OUTPUT_FEATURES):
 mean_check = Output_std_data[:, i].mean()
 std_check = Output_std_data[:, i].std()
 print(f" {feat:8s}: mean={mean_check:8.6f} (should be ≈0), std={std_check:8.6f} (should be ≈1)")

# ==================== TRAIN-TEST SPLIT ====================

print("\n" + "-"*80)
print("TRAIN-TEST SPLIT")
print("-"*80)

# Split data into training and testing sets (70-30 split)
TEST_SIZE = 0.3
TRAIN_SIZE = 1 - TEST_SIZE

X_train, X_test, Y_train, Y_test = train_test_split(
 Input_std_data, Output_std_data,
 test_size=TEST_SIZE,
 random_state=RANDOM_SEED
)

print(f"\n Data split complete (random_state={RANDOM_SEED})")
print(f" • Train size: {TRAIN_SIZE*100:.0f}% ({len(X_train)} samples)")
print(f" • Test size: {TEST_SIZE*100:.0f}% ({len(X_test)} samples)")

print(f"\nTrain set shapes:")
print(f" • X_train: {X_train.shape} (standardized inputs)")
print(f" • Y_train: {Y_train.shape} (standardized outputs)")

print(f"\nTest set shapes:")
print(f" • X_test: {X_test.shape} (standardized inputs)")
print(f" • Y_test: {Y_test.shape} (standardized outputs)")

# ==================== SAVE FIXPOINT (STANDARDIZATION PARAMETERS) ====================

print("\n" + "-"*80)
print("SAVING STANDARDIZATION PARAMETERS (FIXPOINT)")
print("-"*80)

# Create fixpoint dictionary with all standardization parameters
fixpoint = {
 'Input_mean': Input_mean,
 'Input_std': Input_std,
 'Output_mean': Output_mean,
 'Output_std': Output_std,
 'Input_features': INPUT_FEATURES,
 'Output_features': OUTPUT_FEATURES,
 'n_inputs': N_INPUTS,
 'n_outputs': N_OUTPUTS,
 'random_seed': RANDOM_SEED,
 'test_size': TEST_SIZE,
 'n_samples_total': len(Input),
 'n_samples_train': len(X_train),
 'n_samples_test': len(X_test),
 'timestamp': create_timestamp()
}

# Save fixpoint for later use (denormalization, predictions, etc.)
fixpoint_file = f'{BASE_DIR}/fixpoint.pkl'
with open(fixpoint_file, 'wb') as f:
 pickle.dump(fixpoint, f)

print(f" Fixpoint saved to: {fixpoint_file}")
print(f"\nFixpoint contains:")
print(f" • Input statistics (mean, std) for {N_INPUTS} features")
print(f" • Output statistics (mean, std) for {N_OUTPUTS} features")
print(f" • Feature names and configuration")
print(f" • Split parameters")

# ==================== HELPER FUNCTIONS FOR DENORMALIZATION ====================

print("\n" + "-"*80)
print("DEFINING DENORMALIZATION FUNCTIONS")
print("-"*80)

def denormalize_input(X_std):
 """
 Convert standardized input back to original scale.

 Args:
 X_std: Standardized input data (N x 7)

 Returns:
 X_original: Original scale input data (N x 7)
 """
 return X_std * Input_std + Input_mean

def denormalize_output(Y_std):
 """
 Convert standardized output back to original scale.

 Args:
 Y_std: Standardized output data (N x 3)

 Returns:
 Y_original: Original scale output data (N x 3)
 """
 return Y_std * Output_std + Output_mean

def standardize_input(X):
 """
 Convert original input to standardized scale.

 Args:
 X: Original scale input data (N x 7)

 Returns:
 X_std: Standardized input data (N x 7)
 """
 return (X - Input_mean) / Input_std

def standardize_output(Y):
 """
 Convert original output to standardized scale.

 Args:
 Y: Original scale output data (N x 3)

 Returns:
 Y_std: Standardized output data (N x 3)
 """
 return (Y - Output_mean) / Output_std

print(" Denormalization/standardization functions defined:")
print(" • denormalize_input(X_std)")
print(" • denormalize_output(Y_std)")
print(" • standardize_input(X)")
print(" • standardize_output(Y)")

# ==================== VERIFICATION ====================

print("\n" + "-"*80)
print("VERIFICATION - DENORMALIZATION TEST")
print("-"*80)

# Test denormalization on a few samples
test_samples = X_train[:3]
test_denorm = denormalize_input(test_samples)

print("\nTesting denormalization on first 3 training samples:")
print("\nStandardized values (X_train[:3]):")
print(test_samples)

print("\nDenormalized values (original scale):")
print(test_denorm)

print("\nFeature ranges after denormalization:")
for i, feat in enumerate(INPUT_FEATURES):
 print(f" {feat:8s}: min={test_denorm[:, i].min():10.4f}, max={test_denorm[:, i].max():10.4f}")

# ==================== SUMMARY ====================

print("\n" + "="*80)
print("STEPS 2 & 3 COMPLETE - DATA READY FOR MODELING")
print("="*80)

print(f"\nData Summary:")
print(f" • Total samples: {len(Input)}")
print(f" • Input features: {N_INPUTS} ({', '.join(INPUT_FEATURES)})")
print(f" • Output features: {N_OUTPUTS} ({', '.join(OUTPUT_FEATURES)})")
print(f" • Training samples: {len(X_train)} ({TRAIN_SIZE*100:.0f}%)")
print(f" • Test samples: {len(X_test)} ({TEST_SIZE*100:.0f}%)")

print(f"\nStandardization:")
print(f" All features standardized (mean≈0, std≈1)")
print(f" Parameters saved to fixpoint.pkl")

print(f"\nData shapes:")
print(f" • X_train: {X_train.shape}")
print(f" • Y_train: {Y_train.shape}")
print(f" • X_test: {X_test.shape}")
print(f" • Y_test: {Y_test.shape}")

print(f"\nNext Steps:")
print(f" → Step 4: Exploratory data analysis")
print(f" → Step 5: Model training and evaluation")

print("\n" + "="*80)


# 4. Data visualization

In [ ]:
# -*- coding: utf-8 -*-
"""
UF Water Treatment ML Code - 
Step 4: Comprehensive Data Visualization and Analysis (Before & After Standardization)

UPDATED FEATURES:
Input features (7): glu, mlss, air, fm, cn, hrt, srt
Output features (3): tmp, flow, lv
Total: 10 features combined for clean 2x5 or 5x2 layouts

ALL visualizations saved to:
 ./Visualizations/EDA
"""

# ==================== STEP 4: COMPREHENSIVE DATA VISUALIZATION ====================

print("\n" + "="*80)
print(" " * 10 + "STEP 4: COMPREHENSIVE DATA VISUALIZATION AND ANALYSIS")
print("="*80)

# ── Save directory ──────────────────────────────────────────────────────────
VIZ_DIR = './Manual_Validation/EDA'
os.makedirs(VIZ_DIR, exist_ok=True)
print(f"\n Visualization directory: {VIZ_DIR}")

# ── Combined feature setup (10 features) ───────────────────────────────────
ALL_FEATURES = INPUT_FEATURES + OUTPUT_FEATURES  # 10 total
N_ALL = len(ALL_FEATURES)

# Original (before standardization)
All_Original = np.hstack([Input, Output])                  # (N, 10)
# Standardized (after standardization)
All_Standardized = np.hstack([Input_std_data, Output_std_data])  # (N, 10)

# DataFrames for convenience
df_orig = pd.DataFrame(All_Original, columns=ALL_FEATURES)
df_std = pd.DataFrame(All_Standardized, columns=ALL_FEATURES)

# Color coding: inputs = blue palette, outputs = red/orange palette
INPUT_COLORS = ['#03254c', '#03254c', '#03254c', '#03254c', '#03254c', '#03254c', '#03254c']
OUTPUT_COLORS = ['#03254c','#03254c','#03254c']
ALL_COLORS = INPUT_COLORS + OUTPUT_COLORS

# Feature type labels for annotations
FEAT_TYPE = ['Input']*N_INPUTS + ['Output']*N_OUTPUTS

print(f" Combined dataset: {All_Original.shape[0]} samples × {N_ALL} features")
print(f" Inputs ({N_INPUTS}): {', '.join(INPUT_FEATURES)}")
print(f" Outputs ({N_OUTPUTS}): {', '.join(OUTPUT_FEATURES)}")

# ════════════════════════════════════════════════════════════════════════════
# HELPER: consistent figure saving
# ════════════════════════════════════════════════════════════════════════════
def save_fig(fig, filename):
 filepath = f'{VIZ_DIR}/{filename}'
 fig.savefig(filepath, dpi=300, bbox_inches='tight', facecolor='white')
 plt.close(fig)
 print(f" Saved: {filename}")

plot_counter = [0]  # mutable counter

# ════════════════════════════════════════════════════════════════════════════
# 1. TIME SERIES – BEFORE vs AFTER (Side by Side)
# ════════════════════════════════════════════════════════════════════════════

print("\n" + "-"*80)
print("1. TIME SERIES PLOTS – Before vs After Standardization")
print("-"*80)

fig, axes = plt.subplots(N_ALL, 2, figsize=(22, 3*N_ALL))
fig.suptitle('Time Series: Before vs After Standardization (All 10 Features)',
 fontsize=28, fontweight='bold', y=1.005)

for idx, feat in enumerate(ALL_FEATURES):
 color = ALL_COLORS[idx]

    # ── Before ──
 ax_b = axes[idx, 0]
 data_b = All_Original[:, idx]
 ax_b.plot(data_b, linewidth=0.8, alpha=0.8, color=color)
 mean_b = np.mean(data_b)
 ax_b.axhline(mean_b, color='red', ls='--', lw=1, alpha=0.6)
 ax_b.set_ylabel(feat, fontsize=22, fontweight='bold')
 if idx == 0:
 ax_b.set_title('Before Standardization', fontsize=28, fontweight='bold')
 ax_b.tick_params(labelsize=18)
    # Stats box
 stats_t = f'μ={mean_b:.2f}\nσ={np.std(data_b):.2f}'
 ax_b.text(0.01, 0.95, stats_t, transform=ax_b.transAxes, fontsize=16,
 va='top', bbox=dict(boxstyle='round', fc='wheat', alpha=0.5))
 if idx < N_ALL - 1:
 ax_b.set_xticklabels([])

    # ── After ──
 ax_a = axes[idx, 1]
 data_a = All_Standardized[:, idx]
 ax_a.plot(data_a, linewidth=0.8, alpha=0.8, color=color)
 mean_a = np.mean(data_a)
 ax_a.axhline(mean_a, color='red', ls='--', lw=1, alpha=0.6)
 if idx == 0:
 ax_a.set_title('After Standardization', fontsize=28, fontweight='bold')
 ax_a.tick_params(labelsize=18)
 stats_t2 = f'μ={mean_a:.4f}\nσ={np.std(data_a):.4f}'
 ax_a.text(0.01, 0.95, stats_t2, transform=ax_a.transAxes, fontsize=16,
 va='top', bbox=dict(boxstyle='round', fc='lightcyan', alpha=0.5))
 if idx < N_ALL - 1:
 ax_a.set_xticklabels([])

    # Type badge
 """badge_color = '#1f77b4' if FEAT_TYPE[idx] == 'Input' else '#d62728'
 ax_b.text(-0.12, 0.5, FEAT_TYPE[idx], transform=ax_b.transAxes,
 fontsize=16, fontweight='bold', color='white', va='center', ha='center',
 bbox=dict(boxstyle='round,pad=0.3', fc=badge_color, alpha=0.8))"""

axes[-1, 0].set_xlabel('Sample Index', fontsize=22, fontweight='bold')
axes[-1, 1].set_xlabel('Sample Index', fontsize=22, fontweight='bold')

plt.tight_layout()
save_fig(fig, '01_TimeSeries_Before_vs_After.png')

# ════════════════════════════════════════════════════════════════════════════
# 2. DISTRIBUTION HISTOGRAMS – BEFORE vs AFTER (Overlaid)
# ════════════════════════════════════════════════════════════════════════════

print("\n" + "-"*80)
print("2. DISTRIBUTION HISTOGRAMS – Before vs After (Overlaid)")
print("-"*80)

fig, axes = plt.subplots(2, 5, figsize=(28, 10))
fig.suptitle('Distribution Comparison: Before (blue) vs After (orange) Standardization',
 fontsize=20, fontweight='bold', y=1.01)
axes_flat = axes.flatten()

for idx, feat in enumerate(ALL_FEATURES):
 ax = axes_flat[idx]

 data_b = All_Original[:, idx]
 data_a = All_Standardized[:, idx]

    # Before histogram (left y-axis)
 ax.hist(data_b, bins=50, alpha=0.55, color='steelblue', edgecolor='navy',
 linewidth=0.4, label='Before')
 ax.set_xlabel(feat, fontsize=12, fontweight='bold')
 ax.set_ylabel('Frequency', fontsize=10)
 ax.axvline(np.mean(data_b), color='navy', ls='--', lw=1.5, label=f'μ_b={np.mean(data_b):.2f}')

    # After histogram (secondary x-axis via twin)
 ax2 = ax.twiny()
 ax2.hist(data_a, bins=50, alpha=0.45, color='#780606', edgecolor='saddlebrown',
 linewidth=0.4, label='After')
 ax2.axvline(np.mean(data_a), color='saddlebrown', ls='--', lw=1.5)
 ax2.tick_params(labelsize=8, colors='#780606')

 ax.tick_params(labelsize=9)
 ax.set_title(f'{feat} ({FEAT_TYPE[idx]})', fontsize=12, fontweight='bold')

    # Stats annotation
 skew_b = stats.skew(data_b)
 skew_a = stats.skew(data_a)
 ax.text(0.97, 0.97,
 f'Before: μ={np.mean(data_b):.2f}, σ={np.std(data_b):.2f}\n'
 f'After: μ={np.mean(data_a):.4f}, σ={np.std(data_a):.4f}\n'
 f'Skew(B)={skew_b:.2f}',
 transform=ax.transAxes, fontsize=7.5, va='top', ha='right',
 bbox=dict(boxstyle='round', fc='lightyellow', alpha=0.7))

# Legend
from matplotlib.patches import Patch
legend_elements = [Patch(fc='steelblue', alpha=0.6, label='Before Std'),
 Patch(fc='#780606', alpha=0.5, label='After Std')]
fig.legend(handles=legend_elements, loc='upper right', fontsize=12, framealpha=0.9)

plt.tight_layout()
save_fig(fig, '02_Distribution_Before_vs_After_Overlaid.png')

# ════════════════════════════════════════════════════════════════════════════
# 3. SEPARATE DISTRIBUTION PANELS (Before | After)
# ════════════════════════════════════════════════════════════════════════════

print("\n" + "-"*80)
print("3. DISTRIBUTION PANELS – Separate Before & After")
print("-"*80)

for label, data_arr, cmap_name, tag in [
 ('Before Standardization', All_Original, 'Blues', 'Before'),
 ('After Standardization', All_Standardized, 'Oranges', 'After')
]:
 fig, axes = plt.subplots(2, 5, figsize=(28, 10))
 fig.suptitle(f'Distribution Analysis – {label}', fontsize=20, fontweight='bold', y=1.01)
 axes_flat = axes.flatten()

 for idx, feat in enumerate(ALL_FEATURES):
 ax = axes_flat[idx]
 feat_data = data_arr[:, idx]

 color = ALL_COLORS[idx]
 ax.hist(feat_data, bins=50, alpha=0.7, color=color, edgecolor='black', linewidth=0.4)

 mean_v = np.mean(feat_data)
 median_v = np.median(feat_data)
 ax.axvline(mean_v, color='red', ls='--', lw=2, label=f'Mean={mean_v:.2f}')
 ax.axvline(median_v, color='green', ls='--', lw=2, label=f'Median={median_v:.2f}')

 ax.set_xlabel(feat, fontsize=11, fontweight='bold')
 ax.set_ylabel('Frequency', fontsize=10)
 ax.set_title(f'{feat} ({FEAT_TYPE[idx]})', fontsize=12, fontweight='bold')
 ax.legend(fontsize=8, loc='upper right')

 skew_v = stats.skew(feat_data)
 kurt_v = stats.kurtosis(feat_data)
 ax.text(0.97, 0.72,
 f'σ={np.std(feat_data):.2f}\nSkew={skew_v:.2f}\nKurt={kurt_v:.2f}',
 transform=ax.transAxes, fontsize=8, va='top', ha='right',
 bbox=dict(boxstyle='round', fc='wheat', alpha=0.5))

 plt.tight_layout()
 save_fig(fig, f'03_Distribution_{tag}.png')

# ════════════════════════════════════════════════════════════════════════════
# 4. KDE (Kernel Density) COMPARISON – Before vs After
# ════════════════════════════════════════════════════════════════════════════

print("\n" + "-"*80)
print("4. KDE DENSITY PLOTS – Before vs After")
print("-"*80)

fig, axes = plt.subplots(2, 5, figsize=(28, 10))
fig.suptitle('Kernel Density Estimation: Before vs After Standardization',
 fontsize=20, fontweight='bold', y=1.01)
axes_flat = axes.flatten()

for idx, feat in enumerate(ALL_FEATURES):
 ax = axes_flat[idx]
 data_b = All_Original[:, idx]
 data_a = All_Standardized[:, idx]

    # KDE for Before (on primary x-axis)
 sns.kdeplot(data_b, ax=ax, color='steelblue', fill=True, alpha=0.35,
 linewidth=2, label='Before')
 ax.set_xlabel(f'{feat} (Before)', fontsize=10, fontweight='bold')
 ax.set_ylabel('Density', fontsize=10)

    # KDE for After (on twin x-axis)
 ax2 = ax.twiny()
 sns.kdeplot(data_a, ax=ax2, color='#780606', fill=True, alpha=0.3,
 linewidth=2, label='After')
 ax2.set_xlabel(f'{feat} (After)', fontsize=9, color='#780606')
 ax2.tick_params(labelsize=8, colors='#780606')

 ax.set_title(f'{feat} ({FEAT_TYPE[idx]})', fontsize=12, fontweight='bold')
 ax.tick_params(labelsize=9)

    # Combined legend
 from matplotlib.lines import Line2D
 handles = [Line2D([0],[0], color='steelblue', lw=2, label='Before'),
 Line2D([0],[0], color='#780606', lw=2, label='After')]
 ax.legend(handles=handles, fontsize=8, loc='upper right')

plt.tight_layout()
save_fig(fig, '04_KDE_Before_vs_After.png')

# ════════════════════════════════════════════════════════════════════════════
# 5. BOX PLOTS – BEFORE vs AFTER (Side by Side per Feature)
# ════════════════════════════════════════════════════════════════════════════

print("\n" + "-"*80)
print("5. BOX PLOTS – Before vs After")
print("-"*80)

fig, axes = plt.subplots(2, 5, figsize=(28, 10))
fig.suptitle('Box Plots: Before (blue) vs After (orange) Standardization',
 fontsize=20, fontweight='bold', y=1.01)
axes_flat = axes.flatten()

for idx, feat in enumerate(ALL_FEATURES):
 ax = axes_flat[idx]
 data_b = All_Original[:, idx]
 data_a = All_Standardized[:, idx]

 bp = ax.boxplot([data_b, data_a],
 labels=['Before', 'After'],
 patch_artist=True, notch=True, showmeans=True,
 meanprops=dict(marker='D', markerfacecolor='green', markersize=6))

 bp['boxes'][0].set_facecolor('steelblue')
 bp['boxes'][0].set_alpha(0.6)
 bp['boxes'][1].set_facecolor('#780606')
 bp['boxes'][1].set_alpha(0.6)

 for median in bp['medians']:
 median.set_color('red')
 median.set_linewidth(2)

 ax.set_title(f'{feat} ({FEAT_TYPE[idx]})', fontsize=12, fontweight='bold')
 ax.tick_params(labelsize=10)
 ax.grid(True, alpha=0.3, axis='y')

    # Outlier count
 q1_b, q3_b = np.percentile(data_b, [25, 75])
 iqr_b = q3_b - q1_b
 n_out_b = ((data_b < q1_b - 1.5*iqr_b) | (data_b > q3_b + 1.5*iqr_b)).sum()
 ax.text(0.5, 0.02, f'Outliers(B): {n_out_b}', transform=ax.transAxes,
 fontsize=8, ha='center', bbox=dict(fc='lightyellow', alpha=0.7))

plt.tight_layout()
save_fig(fig, '05_BoxPlots_Before_vs_After.png')

# ════════════════════════════════════════════════════════════════════════════
# 6. VIOLIN PLOTS – BEFORE vs AFTER
# ════════════════════════════════════════════════════════════════════════════

print("\n" + "-"*80)
print("6. VIOLIN PLOTS – Before vs After")
print("-"*80)

fig, axes = plt.subplots(2, 5, figsize=(28, 10))
fig.suptitle('Violin Plots: Before vs After Standardization',
 fontsize=20, fontweight='bold', y=1.01)
axes_flat = axes.flatten()

for idx, feat in enumerate(ALL_FEATURES):
 ax = axes_flat[idx]

 parts_b = ax.violinplot([All_Original[:, idx]], positions=[0], showmeans=True,
 showmedians=True, showextrema=True)
 for pc in parts_b['bodies']:
 pc.set_facecolor('steelblue')
 pc.set_alpha(0.5)

 parts_a = ax.violinplot([All_Standardized[:, idx]], positions=[1], showmeans=True,
 showmedians=True, showextrema=True)
 for pc in parts_a['bodies']:
 pc.set_facecolor('#780606')
 pc.set_alpha(0.5)

 ax.set_xticks([0, 1])
 ax.set_xticklabels(['Before', 'After'], fontsize=10)
 ax.set_title(f'{feat} ({FEAT_TYPE[idx]})', fontsize=12, fontweight='bold')
 ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
save_fig(fig, '06_ViolinPlots_Before_vs_After.png')

# ════════════════════════════════════════════════════════════════════════════
# 7. CORRELATION HEATMAPS – BEFORE vs AFTER (Side by Side)
# ════════════════════════════════════════════════════════════════════════════

print("\n" + "-"*80)
print("7. CORRELATION HEATMAPS – Before vs After")
print("-"*80)

corr_orig = df_orig.corr()
corr_std = df_std.corr()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(26, 11))
fig.suptitle('Correlation Matrix: Before vs After Standardization',
 fontsize=20, fontweight='bold', y=1.01)

# Before
sns.heatmap(corr_orig, annot=True, fmt='.2f', cmap='coolwarm', center=0,
 square=True, linewidths=0.5, vmin=-1, vmax=1, ax=ax1,
 cbar_kws={'label': 'Correlation', 'shrink': 0.8})
ax1.set_title('Before Standardization', fontsize=16, fontweight='bold')

# After
sns.heatmap(corr_std, annot=True, fmt='.2f', cmap='coolwarm', center=0,
 square=True, linewidths=0.5, vmin=-1, vmax=1, ax=ax2,
 cbar_kws={'label': 'Correlation', 'shrink': 0.8})
ax2.set_title('After Standardization', fontsize=16, fontweight='bold')

# Add separator lines between inputs and outputs
for ax in [ax1, ax2]:
 ax.axhline(y=N_INPUTS, color='black', linewidth=2)
 ax.axvline(x=N_INPUTS, color='black', linewidth=2)

plt.tight_layout()
save_fig(fig, '07_Correlation_Heatmap_Before_vs_After.png')

# ════════════════════════════════════════════════════════════════════════════
# 8. CORRELATION HEATMAP – FULL (Before Only, Large & Detailed)
# ════════════════════════════════════════════════════════════════════════════

print("\n" + "-"*80)
print("8. CORRELATION HEATMAP – Full Detail (Before Standardization)")
print("-"*80)

fig, ax = plt.subplots(figsize=(14, 12))
mask = np.triu(np.ones_like(corr_orig, dtype=bool), k=1)
sns.heatmap(corr_orig, mask=mask, annot=True, fmt='.3f', cmap='RdBu_r', center=0,
 square=True, linewidths=0.8, vmin=-1, vmax=1, ax=ax,
 cbar_kws={'label': 'Pearson Correlation', 'shrink': 0.85},
 annot_kws={'fontsize': 11, 'fontweight': 'bold'})
ax.set_title('Correlation Matrix – Lower Triangle (Before Standardization)\n'
 '(Black lines separate Inputs from Outputs)',
 fontsize=16, fontweight='bold')
ax.axhline(y=N_INPUTS, color='black', linewidth=2.5)
ax.axvline(x=N_INPUTS, color='black', linewidth=2.5)

plt.tight_layout()
save_fig(fig, '08_Correlation_Heatmap_Full_LowerTriangle.png')

# ════════════════════════════════════════════════════════════════════════════
# 9. CORRELATION HEATMAP – After Standardization (Lower Triangle)
# ════════════════════════════════════════════════════════════════════════════

print("\n" + "-"*80)
print("9. CORRELATION HEATMAP – After Standardization (Lower Triangle)")
print("-"*80)

fig, ax = plt.subplots(figsize=(14, 12))
mask = np.triu(np.ones_like(corr_std, dtype=bool), k=1)
sns.heatmap(corr_std, mask=mask, annot=True, fmt='.3f', cmap='RdBu_r', center=0,
 square=True, linewidths=0.8, vmin=-1, vmax=1, ax=ax,
 cbar_kws={'label': 'Pearson Correlation', 'shrink': 0.85},
 annot_kws={'fontsize': 11, 'fontweight': 'bold'})
ax.set_title('Correlation Matrix – Lower Triangle (After Standardization)\n'
 '(Black lines separate Inputs from Outputs)',
 fontsize=16, fontweight='bold')
ax.axhline(y=N_INPUTS, color='black', linewidth=2.5)
ax.axvline(x=N_INPUTS, color='black', linewidth=2.5)

plt.tight_layout()
save_fig(fig, '09_Correlation_Heatmap_After_LowerTriangle.png')

# ════════════════════════════════════════════════════════════════════════════
# 10. INPUT-OUTPUT SCATTER RELATIONSHIPS (Before Standardization)
# ════════════════════════════════════════════════════════════════════════════

print("\n" + "-"*80)
print("10. INPUT-OUTPUT SCATTER RELATIONSHIPS")
print("-"*80)

fig, axes = plt.subplots(N_OUTPUTS, N_INPUTS, figsize=(4*N_INPUTS, 4.5*N_OUTPUTS))
fig.suptitle('Input → Output Relationships (Before Standardization)',
 fontsize=20, fontweight='bold', y=1.01)

for out_idx, out_feat in enumerate(OUTPUT_FEATURES):
 for in_idx, in_feat in enumerate(INPUT_FEATURES):
 ax = axes[out_idx, in_idx]
 x_data = Input[:, in_idx]
 y_data = Output[:, out_idx]

 ax.scatter(x_data, y_data, alpha=0.4, s=12, c=ALL_COLORS[in_idx], edgecolors='none')

        # Correlation & trend line
 corr = np.corrcoef(x_data, y_data)[0, 1]
 z = np.polyfit(x_data, y_data, 1)
 p = np.poly1d(z)
 x_trend = np.linspace(x_data.min(), x_data.max(), 100)
 ax.plot(x_trend, p(x_trend), 'r--', lw=2, alpha=0.8, label=f'r={corr:.3f}')

 ax.set_xlabel(in_feat, fontsize=10, fontweight='bold')
 ax.set_ylabel(out_feat, fontsize=10, fontweight='bold')
 ax.set_title(f'{in_feat} → {out_feat}', fontsize=11, fontweight='bold')
 ax.legend(fontsize=8, loc='best')
 ax.grid(True, alpha=0.3)

plt.tight_layout()
save_fig(fig, '10_InputOutput_Scatter_Relationships.png')

# ════════════════════════════════════════════════════════════════════════════
# 11. PAIRPLOT / SCATTER MATRIX (Sampled, Before Standardization)
# ════════════════════════════════════════════════════════════════════════════

print("\n" + "-"*80)
print("11. PAIRPLOT / SCATTER MATRIX (Before Standardization)")
print("-"*80)

# Sample for performance if dataset is large
n_sample = min(500, len(df_orig))
df_sample = df_orig.sample(n=n_sample, random_state=RANDOM_SEED).copy()
df_sample['Type'] = ['Input-dom']*len(df_sample)  # placeholder for hue

g = sns.pairplot(df_sample[ALL_FEATURES], diag_kind='kde',
 plot_kws={'alpha': 0.3, 's': 8, 'color': 'steelblue'},
 diag_kws={'color': 'steelblue', 'fill': True, 'alpha': 0.4})
g.figure.suptitle('Pairwise Scatter Matrix – Before Standardization (All 10 Features)',
 fontsize=18, fontweight='bold', y=1.01)
save_fig(g.figure, '11_Pairplot_Before.png')

# ════════════════════════════════════════════════════════════════════════════
# 12. PAIRPLOT / SCATTER MATRIX (Sampled, After Standardization)
# ════════════════════════════════════════════════════════════════════════════

print("\n" + "-"*80)
print("12. PAIRPLOT / SCATTER MATRIX (After Standardization)")
print("-"*80)

df_std_sample = df_std.sample(n=n_sample, random_state=RANDOM_SEED)
g2 = sns.pairplot(df_std_sample[ALL_FEATURES], diag_kind='kde',
 plot_kws={'alpha': 0.3, 's': 8, 'color': '#780606'},
 diag_kws={'color': '#780606', 'fill': True, 'alpha': 0.4})
g2.figure.suptitle('Pairwise Scatter Matrix – After Standardization (All 10 Features)',
 fontsize=18, fontweight='bold', y=1.01)
save_fig(g2.figure, '12_Pairplot_After.png')

# ════════════════════════════════════════════════════════════════════════════
# 13. RANGE COMPARISON BAR CHART
# ════════════════════════════════════════════════════════════════════════════

print("\n" + "-"*80)
print("13. FEATURE RANGE COMPARISON (Before vs After)")
print("-"*80)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(24, 8))
fig.suptitle('Feature Range Comparison: Before vs After Standardization',
 fontsize=20, fontweight='bold', y=1.01)

x_pos = np.arange(N_ALL)
width = 0.35

# Before
means_b = np.mean(All_Original, axis=0)
stds_b = np.std(All_Original, axis=0)
mins_b = np.min(All_Original, axis=0)
maxs_b = np.max(All_Original, axis=0)

ax1.bar(x_pos, means_b, width, yerr=stds_b, capsize=4, color=ALL_COLORS, alpha=0.7,
 edgecolor='black', linewidth=0.5)
ax1.errorbar(x_pos, means_b, yerr=[means_b - mins_b, maxs_b - means_b],
 fmt='none', ecolor='red', capsize=6, capthick=1.5, label='Min–Max range')
ax1.set_xticks(x_pos)
ax1.set_xticklabels(ALL_FEATURES, rotation=45, ha='right', fontsize=11, fontweight='bold')
ax1.set_title('Before Standardization', fontsize=14, fontweight='bold')
ax1.set_ylabel('Value', fontsize=12, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3, axis='y')

# After
means_a = np.mean(All_Standardized, axis=0)
stds_a = np.std(All_Standardized, axis=0)
mins_a = np.min(All_Standardized, axis=0)
maxs_a = np.max(All_Standardized, axis=0)

ax2.bar(x_pos, means_a, width, yerr=stds_a, capsize=4, color=ALL_COLORS, alpha=0.7,
 edgecolor='black', linewidth=0.5)
ax2.errorbar(x_pos, means_a, yerr=[means_a - mins_a, maxs_a - means_a],
 fmt='none', ecolor='red', capsize=6, capthick=1.5, label='Min–Max range')
ax2.set_xticks(x_pos)
ax2.set_xticklabels(ALL_FEATURES, rotation=45, ha='right', fontsize=11, fontweight='bold')
ax2.set_title('After Standardization', fontsize=14, fontweight='bold')
ax2.set_ylabel('Standardized Value', fontsize=12, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
save_fig(fig, '13_Range_Comparison_Before_vs_After.png')

# ════════════════════════════════════════════════════════════════════════════
# 14. MEAN & STD COMPARISON (Grouped Bar)
# ════════════════════════════════════════════════════════════════════════════

print("\n" + "-"*80)
print("14. MEAN & STD GROUPED BAR – Before vs After")
print("-"*80)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(24, 8))
fig.suptitle('Mean & Std Comparison: Before vs After Standardization',
 fontsize=20, fontweight='bold', y=1.01)

x = np.arange(N_ALL)
w = 0.35

# Mean comparison
ax1.bar(x - w/2, means_b, w, label='Before', color='steelblue', alpha=0.7, edgecolor='black')
ax1.bar(x + w/2, means_a, w, label='After', color='#780606', alpha=0.7, edgecolor='black')
ax1.set_xticks(x)
ax1.set_xticklabels(ALL_FEATURES, rotation=45, ha='right', fontsize=11, fontweight='bold')
ax1.set_title('Mean Comparison', fontsize=14, fontweight='bold')
ax1.set_ylabel('Mean Value', fontsize=12, fontweight='bold')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3, axis='y')

# Std comparison
ax2.bar(x - w/2, stds_b, w, label='Before', color='steelblue', alpha=0.7, edgecolor='black')
ax2.bar(x + w/2, stds_a, w, label='After', color='#780606', alpha=0.7, edgecolor='black')
ax2.set_xticks(x)
ax2.set_xticklabels(ALL_FEATURES, rotation=45, ha='right', fontsize=11, fontweight='bold')
ax2.set_title('Std Deviation Comparison', fontsize=14, fontweight='bold')
ax2.set_ylabel('Std Value', fontsize=12, fontweight='bold')
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
save_fig(fig, '14_Mean_Std_Comparison.png')

# ════════════════════════════════════════════════════════════════════════════
# 15. SKEWNESS & KURTOSIS COMPARISON
# ════════════════════════════════════════════════════════════════════════════

print("\n" + "-"*80)
print("15. SKEWNESS & KURTOSIS – Before vs After")
print("-"*80)

skew_b = [stats.skew(All_Original[:, i]) for i in range(N_ALL)]
skew_a = [stats.skew(All_Standardized[:, i]) for i in range(N_ALL)]
kurt_b = [stats.kurtosis(All_Original[:, i]) for i in range(N_ALL)]
kurt_a = [stats.kurtosis(All_Standardized[:, i]) for i in range(N_ALL)]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(24, 8))
fig.suptitle('Skewness & Kurtosis: Before vs After Standardization',
 fontsize=20, fontweight='bold', y=1.01)

ax1.bar(x - w/2, skew_b, w, label='Before', color='steelblue', alpha=0.7, edgecolor='black')
ax1.bar(x + w/2, skew_a, w, label='After', color='#780606', alpha=0.7, edgecolor='black')
ax1.axhline(0, color='black', ls='-', lw=0.8)
ax1.set_xticks(x)
ax1.set_xticklabels(ALL_FEATURES, rotation=45, ha='right', fontsize=11, fontweight='bold')
ax1.set_title('Skewness Comparison', fontsize=14, fontweight='bold')
ax1.set_ylabel('Skewness', fontsize=12, fontweight='bold')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3, axis='y')

ax2.bar(x - w/2, kurt_b, w, label='Before', color='steelblue', alpha=0.7, edgecolor='black')
ax2.bar(x + w/2, kurt_a, w, label='After', color='#780606', alpha=0.7, edgecolor='black')
ax2.axhline(0, color='black', ls='-', lw=0.8)
ax2.set_xticks(x)
ax2.set_xticklabels(ALL_FEATURES, rotation=45, ha='right', fontsize=11, fontweight='bold')
ax2.set_title('Kurtosis Comparison', fontsize=14, fontweight='bold')
ax2.set_ylabel('Excess Kurtosis', fontsize=12, fontweight='bold')
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
save_fig(fig, '15_Skewness_Kurtosis_Comparison.png')

# ════════════════════════════════════════════════════════════════════════════
# 16. Q-Q PLOTS – Before vs After
# ════════════════════════════════════════════════════════════════════════════

print("\n" + "-"*80)
print("16. Q-Q PLOTS – Before vs After")
print("-"*80)

fig, axes = plt.subplots(2, 5, figsize=(28, 10))
fig.suptitle('Q-Q Plots (Normality Check): Before vs After Standardization',
 fontsize=20, fontweight='bold', y=1.01)
axes_flat = axes.flatten()

for idx, feat in enumerate(ALL_FEATURES):
 ax = axes_flat[idx]
 data_a = All_Standardized[:, idx]

    # Q-Q plot for standardized data
 (osm, osr), (slope, intercept, r) = stats.probplot(data_a, dist='norm')
 ax.scatter(osm, osr, alpha=0.4, s=10, color=ALL_COLORS[idx], edgecolors='none')
 ax.plot(osm, slope*np.array(osm) + intercept, 'r-', lw=2,
 label=f'R²={r**2:.4f}')

 ax.set_xlabel('Theoretical Quantiles', fontsize=9)
 ax.set_ylabel('Sample Quantiles', fontsize=9)
 ax.set_title(f'{feat} ({FEAT_TYPE[idx]})', fontsize=12, fontweight='bold')
 ax.legend(fontsize=9, loc='upper left')
 ax.grid(True, alpha=0.3)

plt.tight_layout()
save_fig(fig, '16_QQ_Plots_After_Standardization.png')

# ════════════════════════════════════════════════════════════════════════════
# 17. OUTLIER COUNT COMPARISON (IQR Method)
# ════════════════════════════════════════════════════════════════════════════

print("\n" + "-"*80)
print("17. OUTLIER COUNT COMPARISON")
print("-"*80)

def count_outliers(data):
 q1 = np.percentile(data, 25)
 q3 = np.percentile(data, 75)
 iqr = q3 - q1
 return ((data < q1 - 1.5*iqr) | (data > q3 + 1.5*iqr)).sum()

outliers_b = [count_outliers(All_Original[:, i]) for i in range(N_ALL)]
outliers_a = [count_outliers(All_Standardized[:, i]) for i in range(N_ALL)]

fig, ax = plt.subplots(figsize=(16, 8))
ax.bar(x - w/2, outliers_b, w, label='Before', color='steelblue', alpha=0.7, edgecolor='black')
ax.bar(x + w/2, outliers_a, w, label='After', color='#780606', alpha=0.7, edgecolor='black')
ax.set_xticks(x)
ax.set_xticklabels(ALL_FEATURES, rotation=45, ha='right', fontsize=12, fontweight='bold')
ax.set_title('Outlier Count per Feature (IQR Method): Before vs After',
 fontsize=16, fontweight='bold')
ax.set_ylabel('Number of Outliers', fontsize=13, fontweight='bold')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3, axis='y')

for i in range(N_ALL):
 ax.text(i - w/2, outliers_b[i] + 0.5, str(outliers_b[i]), ha='center', fontsize=9, fontweight='bold')
 ax.text(i + w/2, outliers_a[i] + 0.5, str(outliers_a[i]), ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
save_fig(fig, '17_Outlier_Count_Comparison.png')

# ════════════════════════════════════════════════════════════════════════════
# 18. HEATMAP – ABSOLUTE CORRELATION (Input→Output Only)
# ════════════════════════════════════════════════════════════════════════════

print("\n" + "-"*80)
print("18. INPUT→OUTPUT ABSOLUTE CORRELATION HEATMAP")
print("-"*80)

# Cross-correlation (7 inputs x 3 outputs)
cross_corr = np.zeros((N_INPUTS, N_OUTPUTS))
for i in range(N_INPUTS):
 for j in range(N_OUTPUTS):
 cross_corr[i, j] = np.corrcoef(Input[:, i], Output[:, j])[0, 1]

cross_corr_df = pd.DataFrame(cross_corr, index=INPUT_FEATURES, columns=OUTPUT_FEATURES)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8))
fig.suptitle('Input → Output Correlation (Before Standardization)',
 fontsize=18, fontweight='bold', y=1.01)

sns.heatmap(cross_corr_df, annot=True, fmt='.3f', cmap='coolwarm', center=0,
 linewidths=0.8, vmin=-1, vmax=1, ax=ax1,
 cbar_kws={'label': 'Pearson r'},
 annot_kws={'fontsize': 12, 'fontweight': 'bold'})
ax1.set_title('Signed Correlation', fontsize=14, fontweight='bold')

sns.heatmap(cross_corr_df.abs(), annot=True, fmt='.3f', cmap='YlOrRd',
 linewidths=0.8, vmin=0, vmax=1, ax=ax2,
 cbar_kws={'label': '|Pearson r|'},
 annot_kws={'fontsize': 12, 'fontweight': 'bold'})
ax2.set_title('Absolute Correlation', fontsize=14, fontweight='bold')

plt.tight_layout()
save_fig(fig, '18_InputOutput_Correlation_Heatmap.png')

# ════════════════════════════════════════════════════════════════════════════
# 19. PARALLEL COORDINATES PLOT
# ════════════════════════════════════════════════════════════════════════════

print("\n" + "-"*80)
print("19. PARALLEL COORDINATES PLOT (After Standardization)")
print("-"*80)

fig, ax = plt.subplots(figsize=(20, 8))

# Sample for readability
n_lines = min(200, len(df_std))
sample_idx = np.random.choice(len(df_std), n_lines, replace=False)

for i in sample_idx:
 vals = All_Standardized[i, :]
 ax.plot(range(N_ALL), vals, alpha=0.15, linewidth=0.8, color='steelblue')

# Mean line
ax.plot(range(N_ALL), np.mean(All_Standardized, axis=0), 'r-', linewidth=3,
 label='Mean', zorder=10)
# +/- 1 std
ax.fill_between(range(N_ALL),
 np.mean(All_Standardized, axis=0) - np.std(All_Standardized, axis=0),
 np.mean(All_Standardized, axis=0) + np.std(All_Standardized, axis=0),
 alpha=0.2, color='red', label='±1 Std')

ax.set_xticks(range(N_ALL))
ax.set_xticklabels(ALL_FEATURES, rotation=45, ha='right', fontsize=12, fontweight='bold')
ax.set_title('Parallel Coordinates Plot (After Standardization)', fontsize=16, fontweight='bold')
ax.set_ylabel('Standardized Value', fontsize=13, fontweight='bold')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)

# Separator line between inputs and outputs
ax.axvline(x=N_INPUTS - 0.5, color='black', ls='--', lw=2, alpha=0.7)
ax.text(N_INPUTS - 0.5, ax.get_ylim()[1], '← Inputs | Outputs →',
 ha='center', va='bottom', fontsize=11, fontweight='bold',
 bbox=dict(fc='yellow', alpha=0.7))

plt.tight_layout()
save_fig(fig, '19_Parallel_Coordinates.png')

# ════════════════════════════════════════════════════════════════════════════
# 20. STATISTICAL SUMMARY TABLE (Saved as CSV)
# ════════════════════════════════════════════════════════════════════════════

print("\n" + "-"*80)
print("20. STATISTICAL SUMMARY TABLES")
print("-"*80)

def build_stats_table(data, features, label):
 rows = []
 for idx, feat in enumerate(features):
 d = data[:, idx]
 rows.append({
 'Feature': feat,
 'Type': FEAT_TYPE[idx] if features == ALL_FEATURES else ('Input' if feat in INPUT_FEATURES else 'Output'),
 'Mean': d.mean(),
 'Std': d.std(),
 'Min': d.min(),
 'Q1': np.percentile(d, 25),
 'Median': np.median(d),
 'Q3': np.percentile(d, 75),
 'Max': d.max(),
 'Skewness': stats.skew(d),
 'Kurtosis': stats.kurtosis(d),
 'Outliers_IQR': count_outliers(d)
 })
 df_stats = pd.DataFrame(rows)
 filepath = f'{VIZ_DIR}/Stats_{label}.csv'
 df_stats.to_csv(filepath, index=False)
 print(f" Saved: Stats_{label}.csv")
 return df_stats

stats_before = build_stats_table(All_Original, ALL_FEATURES, 'Before_Standardization')
stats_after = build_stats_table(All_Standardized, ALL_FEATURES, 'After_Standardization')

# Correlation analysis CSV
corr_results = []
for out_idx, out_feat in enumerate(OUTPUT_FEATURES):
 for in_idx, in_feat in enumerate(INPUT_FEATURES):
 corr = np.corrcoef(Input[:, in_idx], Output[:, out_idx])[0, 1]
 corr_results.append({
 'Input': in_feat,
 'Output': out_feat,
 'Correlation': corr,
 'Abs_Correlation': abs(corr)
 })
corr_df = pd.DataFrame(corr_results).sort_values('Abs_Correlation', ascending=False)
corr_df.to_csv(f'{VIZ_DIR}/Correlation_InputOutput.csv', index=False)
print(f" Saved: Correlation_InputOutput.csv")

# Print summary
print("\nBefore Standardization:")
print(stats_before.to_string(index=False))
print("\nAfter Standardization:")
print(stats_after.to_string(index=False))

# ════════════════════════════════════════════════════════════════════════════
# 21. SUMMARY DASHBOARD – Single Page Overview
# ════════════════════════════════════════════════════════════════════════════

print("\n" + "-"*80)
print("21. SUMMARY DASHBOARD")
print("-"*80)

fig = plt.figure(figsize=(30, 20))
fig.suptitle('EDA Summary Dashboard – UF Water Treatment (MBR Process)\n'
 '7 Inputs + 3 Outputs = 10 Features',
 fontsize=22, fontweight='bold', y=0.995)

gs = fig.add_gridspec(3, 4, hspace=0.35, wspace=0.3)

# (0,0-1) Correlation heatmap (before)
ax1 = fig.add_subplot(gs[0, 0:2])
sns.heatmap(corr_orig, annot=True, fmt='.1f', cmap='coolwarm', center=0,
 square=True, linewidths=0.3, vmin=-1, vmax=1, ax=ax1,
 cbar_kws={'shrink': 0.6}, annot_kws={'fontsize': 7})
ax1.set_title('Correlation (Before)', fontsize=13, fontweight='bold')
ax1.axhline(y=N_INPUTS, color='black', linewidth=1.5)
ax1.axvline(x=N_INPUTS, color='black', linewidth=1.5)
ax1.tick_params(labelsize=8)

# (0,2-3) Correlation heatmap (after)
ax2 = fig.add_subplot(gs[0, 2:4])
sns.heatmap(corr_std, annot=True, fmt='.1f', cmap='coolwarm', center=0,
 square=True, linewidths=0.3, vmin=-1, vmax=1, ax=ax2,
 cbar_kws={'shrink': 0.6}, annot_kws={'fontsize': 7})
ax2.set_title('Correlation (After)', fontsize=13, fontweight='bold')
ax2.axhline(y=N_INPUTS, color='black', linewidth=1.5)
ax2.axvline(x=N_INPUTS, color='black', linewidth=1.5)
ax2.tick_params(labelsize=8)

# (1,0-1) Box plots before
ax3 = fig.add_subplot(gs[1, 0:2])
bp3 = ax3.boxplot([All_Original[:, i] for i in range(N_ALL)],
 labels=ALL_FEATURES, patch_artist=True, showmeans=True,
 meanprops=dict(marker='D', markerfacecolor='green', markersize=4))
for i, patch in enumerate(bp3['boxes']):
 patch.set_facecolor(ALL_COLORS[i])
 patch.set_alpha(0.6)
ax3.set_title('Box Plots (Before)', fontsize=13, fontweight='bold')
ax3.tick_params(axis='x', rotation=45, labelsize=9)
ax3.grid(True, alpha=0.3, axis='y')

# (1,2-3) Box plots after
ax4 = fig.add_subplot(gs[1, 2:4])
bp4 = ax4.boxplot([All_Standardized[:, i] for i in range(N_ALL)],
 labels=ALL_FEATURES, patch_artist=True, showmeans=True,
 meanprops=dict(marker='D', markerfacecolor='green', markersize=4))
for i, patch in enumerate(bp4['boxes']):
 patch.set_facecolor(ALL_COLORS[i])
 patch.set_alpha(0.6)
ax4.set_title('Box Plots (After)', fontsize=13, fontweight='bold')
ax4.tick_params(axis='x', rotation=45, labelsize=9)
ax4.grid(True, alpha=0.3, axis='y')

# (2,0) Skewness
ax5 = fig.add_subplot(gs[2, 0])
ax5.barh(ALL_FEATURES, skew_b, color=ALL_COLORS, alpha=0.7, edgecolor='black')
ax5.axvline(0, color='black', lw=0.8)
ax5.set_title('Skewness (Before)', fontsize=13, fontweight='bold')
ax5.tick_params(labelsize=9)

# (2,1) Skewness After
ax6 = fig.add_subplot(gs[2, 1])
ax6.barh(ALL_FEATURES, skew_a, color=ALL_COLORS, alpha=0.7, edgecolor='black')
ax6.axvline(0, color='black', lw=0.8)
ax6.set_title('Skewness (After)', fontsize=13, fontweight='bold')
ax6.tick_params(labelsize=9)

# (2,2) Kurtosis
ax7 = fig.add_subplot(gs[2, 2])
ax7.barh(ALL_FEATURES, kurt_b, color=ALL_COLORS, alpha=0.7, edgecolor='black')
ax7.axvline(0, color='black', lw=0.8)
ax7.set_title('Kurtosis (Before)', fontsize=13, fontweight='bold')
ax7.tick_params(labelsize=9)

# (2,3) Kurtosis After
ax8 = fig.add_subplot(gs[2, 3])
ax8.barh(ALL_FEATURES, kurt_a, color=ALL_COLORS, alpha=0.7, edgecolor='black')
ax8.axvline(0, color='black', lw=0.8)
ax8.set_title('Kurtosis (After)', fontsize=13, fontweight='bold')
ax8.tick_params(labelsize=9)

save_fig(fig, '21_Summary_Dashboard.png')

# ════════════════════════════════════════════════════════════════════════════
# FINAL SUMMARY
# ════════════════════════════════════════════════════════════════════════════

print("\n" + "="*80)
print("STEP 4 COMPLETE – COMPREHENSIVE EDA VISUALIZATION FINISHED")
print("="*80)

print(f"\n All files saved to: {VIZ_DIR}")
print(f"\n Visualizations Created (21 figures + 3 CSVs):")
print(f" 01. Time Series – Before vs After Standardization")
print(f" 02. Distribution Histograms – Overlaid Before vs After")
print(f" 03. Distribution Histograms – Separate Before & After (2 files)")
print(f" 04. KDE Density Plots – Before vs After")
print(f" 05. Box Plots – Before vs After")
print(f" 06. Violin Plots – Before vs After")
print(f" 07. Correlation Heatmaps – Side by Side (Before vs After)")
print(f" 08. Correlation Heatmap – Lower Triangle (Before)")
print(f" 09. Correlation Heatmap – Lower Triangle (After)")
print(f" 10. Input→Output Scatter Relationships")
print(f" 11. Pairplot / Scatter Matrix (Before)")
print(f" 12. Pairplot / Scatter Matrix (After)")
print(f" 13. Feature Range Comparison Bar Chart")
print(f" 14. Mean & Std Grouped Bar Comparison")
print(f" 15. Skewness & Kurtosis Comparison")
print(f" 16. Q-Q Plots (Normality Check)")
print(f" 17. Outlier Count Comparison")
print(f" 18. Input→Output Correlation Heatmap (Signed + Absolute)")
print(f" 19. Parallel Coordinates Plot")
print(f" 20. Statistical Summary CSVs (3 files)")
print(f" 21. Summary Dashboard (Single Page Overview)")

print(f"\n Key Design Decisions:")
print(f" • All 10 features combined → clean 2×5 layouts (no empty spaces)")
print(f" • Every plot compares Before vs After standardization")
print(f" • Input/Output features color-coded (blue=input, red/orange=output)")
print(f" • Black separator lines on heatmaps mark Input|Output boundary")

print(f"\n Next: Step 5 – Model Training and Evaluation")
print("="*80)


# 5. Define ML models for separate output prediction



In [ ]:
# -*- coding: utf-8 -*-
"""
UF Water Treatment ML Code - 
Step 5: Traditional ML Models (Separate Output Training)

UPDATED FEATURES:
Input features (7): glu, mlss, air, fm, cn, hrt, srt
Output features (3): tmp, flow, lv

Each output is trained separately with independent models.
"""

# ==================== STEP 5: TRADITIONAL ML MODELS ====================

print("\n" + "="*80)
print(" " * 15 + "STEP 5: TRADITIONAL ML MODELS (SEPARATE OUTPUT TRAINING)")
print("="*80)

print(f"\nTraining Strategy:")
print(f" • Input features: {N_INPUTS} ({', '.join(INPUT_FEATURES)})")
print(f" • Output features: {N_OUTPUTS} ({', '.join(OUTPUT_FEATURES)})")
print(f" • Each output trained independently")
print(f" • Total models to train: {len(MODEL_CONFIGS)} × {N_OUTPUTS} = {len(MODEL_CONFIGS) * N_OUTPUTS}")

# ==================== TRAINING FUNCTIONS ====================

def train_model_for_output(model, X_train, y_train, X_test, y_test, model_name, output_name):
 """
 Train a model for a single output and evaluate performance.

 Args:
 model: ML model instance
 X_train: Training inputs (N, 7)
 y_train: Training output for specific feature (N,)
 X_test: Test inputs (M, 7)
 y_test: Test output for specific feature (M,)
 model_name: Name of the model
 output_name: Name of the output feature

 Returns:
 dict: Results including model, predictions, and metrics
 """

    # Train the model
 model.fit(X_train, y_train)

    # Make predictions
 y_train_pred = model.predict(X_train)
 y_test_pred = model.predict(X_test)

    # Calculate metrics
    # Training metrics
 train_mse = mean_squared_error(y_train, y_train_pred)
 train_rmse = np.sqrt(train_mse)
 train_mae = mean_absolute_error(y_train, y_train_pred)
 train_r2 = r2_score(y_train, y_train_pred)

    # Test metrics
 test_mse = mean_squared_error(y_test, y_test_pred)
 test_rmse = np.sqrt(test_mse)
 test_mae = mean_absolute_error(y_test, y_test_pred)
 test_r2 = r2_score(y_test, y_test_pred)

    # Store results
 results = {
 'model': model,
 'model_name': model_name,
 'output_name': output_name,
 'y_train_pred': y_train_pred,
 'y_test_pred': y_test_pred,
 'train_mse': train_mse,
 'train_rmse': train_rmse,
 'train_mae': train_mae,
 'train_r2': train_r2,
 'test_mse': test_mse,
 'test_rmse': test_rmse,
 'test_mae': test_mae,
 'test_r2': test_r2
 }

 return results

# ==================== CROSS-VALIDATION FUNCTION ====================

def cross_validate_model(model, X, y, model_name, output_name, n_splits=5):
 """
 Perform k-fold cross-validation.

 Args:
 model: ML model instance
 X: Input features (N, 7)
 y: Output feature (N,)
 model_name: Name of the model
 output_name: Name of the output feature
 n_splits: Number of folds

 Returns:
 dict: Cross-validation results
 """

 kfold = KFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_SEED)

 cv_r2_scores = []
 cv_rmse_scores = []

 for train_idx, val_idx in kfold.split(X):
 X_fold_train, X_fold_val = X[train_idx], X[val_idx]
 y_fold_train, y_fold_val = y[train_idx], y[val_idx]

        # Train model
 model.fit(X_fold_train, y_fold_train)

        # Predict
 y_fold_pred = model.predict(X_fold_val)

        # Calculate metrics
 r2 = r2_score(y_fold_val, y_fold_pred)
 rmse = np.sqrt(mean_squared_error(y_fold_val, y_fold_pred))

 cv_r2_scores.append(r2)
 cv_rmse_scores.append(rmse)

 results = {
 'cv_r2_mean': np.mean(cv_r2_scores),
 'cv_r2_std': np.std(cv_r2_scores),
 'cv_rmse_mean': np.mean(cv_rmse_scores),
 'cv_rmse_std': np.std(cv_rmse_scores),
 'cv_r2_scores': cv_r2_scores,
 'cv_rmse_scores': cv_rmse_scores
 }

 return results

# ==================== MAIN TRAINING LOOP ====================

print("\n" + "-"*80)
print("TRAINING MODELS")
print("-"*80)

# Dictionary to store all trained models and results
ml_trained_models = {}
all_results = []

# Train models for each output
for output_idx, output_name in enumerate(OUTPUT_FEATURES):

 print(f"\n{'='*80}")
 print(f"TRAINING MODELS FOR OUTPUT: {output_name}")
 print(f"{'='*80}")

    # Extract single output column
 y_train_single = Y_train[:, output_idx]
 y_test_single = Y_test[:, output_idx]

 print(f" • Training samples: {len(y_train_single)}")
 print(f" • Test samples: {len(y_test_single)}")
 print(f" • Models to train: {len(MODEL_CONFIGS)}")

    # Train each model
 for model_idx, (model_name, model_config) in enumerate(MODEL_CONFIGS.items(), 1):

 print(f"\n [{model_idx}/{len(MODEL_CONFIGS)}] Training {model_name} for {output_name}...")

 try:
            # Create fresh model instance
 from sklearn.base import clone
 model = clone(model_config['model'])

            # Train and evaluate
 results = train_model_for_output(
 model, X_train, y_train_single, X_test, y_test_single,
 model_name, output_name
 )

            # Perform cross-validation
 cv_results = cross_validate_model(
 clone(model_config['model']), X_train, y_train_single,
 model_name, output_name, n_splits=5
 )

            # Merge results
 results.update(cv_results)
 results['model_type'] = model_config['type']
 results['description'] = model_config['description']

            # Store model
 if model_name not in ml_trained_models:
 ml_trained_models[model_name] = {}

 ml_trained_models[model_name][output_name] = {
 'model': results['model'],
 'results': results
 }

            # Store for summary
 all_results.append(results)

            # Print results
 print(f" Train R²: {results['train_r2']:.4f} | Test R²: {results['test_r2']:.4f}")
 print(f" CV R²: {results['cv_r2_mean']:.4f} ± {results['cv_r2_std']:.4f}")
 print(f" Success")

 except Exception as e:
 print(f" Failed: {str(e)}")
 continue

print(f"\n{'='*80}")
print("TRAINING COMPLETE")
print(f"{'='*80}")

# ==================== SAVE TRAINED MODELS ====================

print("\n" + "-"*80)
print("SAVING TRAINED MODELS")
print("-"*80)

# Save all models as individual files
print("\nSaving individual model files...")
for model_name in ml_trained_models.keys():
 for output_name in OUTPUT_FEATURES:
 if output_name in ml_trained_models[model_name]:
 model_obj = ml_trained_models[model_name][output_name]['model']
 filename = f"{model_name.replace(' ', '_')}_{output_name}.pkl"
 filepath = f"{MODELS_DIR}/{filename}"

 try:
 import joblib
 joblib.dump(model_obj, filepath)
 print(f" {filename}")
 except Exception:
 with open(filepath, 'wb') as f:
 pickle.dump(model_obj, f)
 print(f" {filename}")

# Save complete results dictionary
results_file = f"{MODELS_DIR}/ml_trained_models.pkl"
with open(results_file, 'wb') as f:
 pickle.dump(ml_trained_models, f)
print(f"\n Complete results saved to: ml_trained_models.pkl")

print(f"\n Total models saved: {len(ml_trained_models) * N_OUTPUTS}")

# ==================== RESULTS SUMMARY ====================

print("\n" + "-"*80)
print("RESULTS SUMMARY")
print("-"*80)

# Create results DataFrame
results_data = []
for result in all_results:
 results_data.append({
 'Model': result['model_name'],
 'Output': result['output_name'],
 'Type': result['model_type'],
 'Train_R2': result['train_r2'],
 'Test_R2': result['test_r2'],
 'CV_R2_Mean': result['cv_r2_mean'],
 'CV_R2_Std': result['cv_r2_std'],
 'Train_RMSE': result['train_rmse'],
 'Test_RMSE': result['test_rmse'],
 'CV_RMSE_Mean': result['cv_rmse_mean'],
 'Test_MAE': result['test_mae']
 })

results_df = pd.DataFrame(results_data)

# Display summary for each output
for output_name in OUTPUT_FEATURES:
 print(f"\n{'='*80}")
 print(f"RESULTS FOR OUTPUT: {output_name}")
 print(f"{'='*80}")

 output_results = results_df[results_df['Output'] == output_name].copy()
 output_results = output_results.sort_values('Test_R2', ascending=False)

 print("\nTop 5 Models (by Test R²):")
 print("-"*80)
 top5 = output_results.head(5)[['Model', 'Train_R2', 'Test_R2', 'CV_R2_Mean', 'Test_RMSE']]
 print(top5.to_string(index=False))

 if len(output_results) > 0:
 print(f"\nBest Model: {output_results.iloc[0]['Model']}")
 print(f" • Test R²: {output_results.iloc[0]['Test_R2']:.4f}")
 print(f" • CV R² Mean: {output_results.iloc[0]['CV_R2_Mean']:.4f} ± {output_results.iloc[0]['CV_R2_Std']:.4f}")
 print(f" • Test RMSE: {output_results.iloc[0]['Test_RMSE']:.4f}")
 else:
 print(f"\n No models successfully trained for {output_name}")

# Save complete results to CSV
results_csv = f"{RESULTS_DIR}/Model_Training_Results_Step5.csv"
results_df.to_csv(results_csv, index=False)
print(f"\n Complete results saved to: {results_csv}")

# ==================== PERFORMANCE COMPARISON ====================

print("\n" + "-"*80)
print("MODEL PERFORMANCE COMPARISON")
print("-"*80)

# Calculate overall statistics
print("\nOverall Statistics:")
print("-"*80)

for output_name in OUTPUT_FEATURES:
 output_results = results_df[results_df['Output'] == output_name]

 print(f"\n{output_name}:")
 print(f" • Mean Test R²: {output_results['Test_R2'].mean():.4f}")
 print(f" • Max Test R²: {output_results['Test_R2'].max():.4f}")
 print(f" • Min Test R²: {output_results['Test_R2'].min():.4f}")
 print(f" • Models with R² > 0.9: {(output_results['Test_R2'] > 0.90).sum()}/{len(output_results)}")
 print(f" • Models with R² > 0.95: {(output_results['Test_R2'] > 0.95).sum()}/{len(output_results)}")

# ==================== FINAL SUMMARY ====================

print("\n" + "="*80)
print("STEP 5 COMPLETE - MODEL TRAINING FINISHED")
print("="*80)

print(f"\nTraining Summary:")
print(f" • Models trained: {len(ml_trained_models)}")
print(f" • Outputs predicted: {N_OUTPUTS} ({', '.join(OUTPUT_FEATURES)})")
print(f" • Total model-output combos: {sum(len(ml_trained_models[m]) for m in ml_trained_models)}")

print(f"\nBest Models:")
for output_name in OUTPUT_FEATURES:
 output_results = results_df[results_df['Output'] == output_name]
 if len(output_results) > 0:
 best_model = output_results.loc[output_results['Test_R2'].idxmax(), 'Model']
 best_r2 = output_results['Test_R2'].max()
 print(f" • {output_name}: {best_model} (R²={best_r2:.4f})")
 else:
 print(f" • {output_name}: No models trained successfully")

print(f"\nFiles Saved:")
print(f" • Individual models: {MODELS_DIR}/ ({sum(len(ml_trained_models[m]) for m in ml_trained_models)} files)")
print(f" • Complete results: ml_trained_models.pkl")
print(f" • Results CSV: Model_Training_Results_Step5.csv")

print(f"\nNext Steps:")
print(f" → Step 6: Prediction visualizations and overfitting analysis")
print(f" → Step 7: Data cleaning (if needed)")
print(f" → Step 8: Feature importance analysis (SHAP)")
print(f" → Step 9: Optimal operating conditions")

print("\n" + "="*80)

# ==================== BRIDGE: Reformat Step 5 outputs for Step 6 ====================
# Step 6 expects:
# test_predictions[output_name][model_name] → np.array of test predictions
# test_metrics[output_name][model_name] → dict with keys R2, RMSE, MAE
# train_metrics[output_name][model_name] → dict with keys R2, RMSE, MAE
# best_models[output_name] → string (best model name)

print("\n" + "-"*80)
print("BRIDGE: Preparing variables for Step 6 …")
print("-"*80)

test_predictions = {out: {} for out in OUTPUT_FEATURES}
test_metrics = {out: {} for out in OUTPUT_FEATURES}
train_metrics = {out: {} for out in OUTPUT_FEATURES}
best_models = {}

for model_name, outputs in ml_trained_models.items():
 for output_name, data in outputs.items():
 r = data['results']

        # Predictions
 test_predictions[output_name][model_name] = r['y_test_pred']

        # Test metrics
 test_metrics[output_name][model_name] = {
 'R2': r['test_r2'],
 'RMSE': r['test_rmse'],
 'MAE': r['test_mae']
 }

        # Train metrics
 train_metrics[output_name][model_name] = {
 'R2': r['train_r2'],
 'RMSE': r['train_rmse'],
 'MAE': r['train_mae']
 }

# Populate best_models from results_df
for output_name in OUTPUT_FEATURES:
 subset = results_df[results_df['Output'] == output_name]
 if len(subset) > 0:
 best_models[output_name] = subset.loc[subset['Test_R2'].idxmax(), 'Model']
 else:
 best_models[output_name] = None

print("\n Bridge variables created successfully!")
print("\n Variables now available for Step 6:")
print(f" • test_predictions – {sum(len(v) for v in test_predictions.values())} model-output entries")
print(f" • test_metrics – {sum(len(v) for v in test_metrics.values())} model-output entries")
print(f" • train_metrics – {sum(len(v) for v in train_metrics.values())} model-output entries")
print(f" • best_models – {best_models}")
print("\n→ Ready to run Step 6!\n")
print("="*80)


# 6.1. Prediction visualization and Data saving

In [ ]:
# -*- coding: utf-8 -*-
"""
UF Water Treatment ML Code - 
Step 6: Prediction Visualizations with Overfitting Analysis

UPDATED FEATURES:
- 5% and 10% tolerance bands around perfect prediction line
- RdBu_r color theme throughout
- Additional comparison plots:
 (A) Scatter plots (Predicted vs Actual) - all models, per output
 (B) R² bar chart comparison across all models
 (C) RMSE bar chart comparison across all models
 (D) Residual plots (Error vs Actual) - all models
 (E) Train vs Test R² bubble chart (overfitting overview)
 (F) Heatmap: R² across all models × outputs
"""
import os

# ==================== DIRECTORY SETUP ====================
# Adjust the base path to match your project structure

BASE_DIR = '.'                          # or your project root path
MODELS_DIR = f'{BASE_DIR}/models'
RESULTS_DIR = f'{BASE_DIR}/results'
VIZ_PLOTS_DIR = f'{BASE_DIR}/visualizations'

# Create directories if they don't exist
for d in [MODELS_DIR, RESULTS_DIR, VIZ_PLOTS_DIR]:
 os.makedirs(d, exist_ok=True)

print(f" VIZ_PLOTS_DIR set to: {VIZ_PLOTS_DIR}")

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.cm as cm
from matplotlib.patches import Patch
import pandas as pd
import os

# ──────────────────────────────────────────────────────────────
# COLOR THEME SETUP (RdBu_r)
# ──────────────────────────────────────────────────────────────

CMAP = plt.get_cmap('RdBu_r')
COLOR_BLUE = CMAP(0.85)   # deep blue – good predictions / train
COLOR_RED = CMAP(0.15)   # deep red – bad / overfit / test
COLOR_MID = CMAP(0.50)   # neutral white-ish
COLOR_LBLUE = CMAP(0.72)   # light blue scatter fill
COLOR_LRED = CMAP(0.72)   # light blue scatter fill (overfit)

BAND5_COLOR = CMAP(0.70)   # 5 % band fill
BAND10_COLOR = CMAP(0.60)   # 10 % band fill

FIGBG = '#F7F7F7'
AXBG = '#FFFFFF'

def rdbu_discrete(n):
 """Return n evenly spaced colours from RdBu_r."""
 return [CMAP(i / (n - 1)) for i in range(n)]

# ──────────────────────────────────────────────────────────────
# HELPER – tolerance bands
# ──────────────────────────────────────────────────────────────

def add_tolerance_bands(ax, min_val, max_val):
 """
 Draw 5 % and 10 % tolerance bands around the perfect-prediction line
 and the line itself.
 """
 x = np.linspace(min_val, max_val, 300)

    # 10 % band (wider, lighter)
 ax.fill_between(x, x * 0.90, x * 1.10,
 color=BAND10_COLOR, alpha=0.18, label='±10 % band',
 zorder=1)
    # 5 % band (narrower, slightly darker)
 ax.fill_between(x, x * 0.95, x * 1.05,
 color=BAND5_COLOR, alpha=0.30, label='±5 % band',
 zorder=2)
    # Perfect prediction line
 ax.plot(x, x, color=COLOR_RED, linewidth=1.8, linestyle='--',
 label='Perfect prediction', zorder=3)

# ──────────────────────────────────────────────────────────────
# PLOT A – SCATTER (Predicted vs Actual) – all models, per output
# ──────────────────────────────────────────────────────────────

print("\n" + "="*60)
print("CREATING VISUALIZATIONS")
print("="*60)

has_predictions = any(len(test_predictions[out]) > 0 for out in OUTPUT_FEATURES)

if not has_predictions:
 print("\n No predictions available – skipping visualizations")
else:
 print("\nA. Scatter plots with ±5 % / ±10 % tolerance bands …")

 for output_name in OUTPUT_FEATURES:
 if len(test_predictions[output_name]) == 0:
 continue

 output_idx = OUTPUT_FEATURES.index(output_name)
 all_model_names = sorted(test_predictions[output_name].keys())
 n_models = len(all_model_names)

 n_cols = 4
 n_rows = int(np.ceil(n_models / n_cols))

 fig, axes = plt.subplots(n_rows, n_cols,
 figsize=(20, 5.5 * n_rows),
 facecolor=FIGBG)
 if n_rows == 1:
 axes = axes.reshape(1, -1)
 axes_flat = axes.flatten()

 Y_test_orig = Y_test * Output_std + Output_mean
 y_test_actual = Y_test_orig[:, output_idx]

 colors = rdbu_discrete(max(n_models, 2))

 for idx, model_name in enumerate(all_model_names):
 ax = axes_flat[idx]
 ax.set_facecolor(AXBG)

 y_test_pred = test_predictions[output_name][model_name]

            # ── axis range with 5 % padding
 pad = 0.05
 combined = np.concatenate([y_test_actual, y_test_pred])
 lo = combined.min()
 hi = combined.max()
 rng = hi - lo
 lo -= rng * pad
 hi += rng * pad

            # Tolerance bands + perfect line
 add_tolerance_bands(ax, lo, hi)

            # Colour scatter by residual magnitude (RdBu_r)
 residuals = y_test_pred - y_test_actual
 abs_res_norm = np.abs(residuals) / (np.abs(y_test_actual).mean() + 1e-9)
 scatter_colors = [CMAP(min(v * 2, 1.0)) for v in abs_res_norm]

 ax.scatter(y_test_actual, y_test_pred,
 c=scatter_colors, alpha=0.75, s=40,
 edgecolors='none', zorder=4)

            # Metrics
 test_m = test_metrics[output_name][model_name]
 train_m = train_metrics[output_name][model_name]
 r2_gap = train_m['R2'] - test_m['R2']

            # % within bands
 pct5 = np.mean(np.abs(residuals) <= 0.05 * np.abs(y_test_actual)) * 100
 pct10 = np.mean(np.abs(residuals) <= 0.10 * np.abs(y_test_actual)) * 100

 textstr = (f"R²={test_m['R2']:.3f} ΔR²={r2_gap:.3f}\n"
 f"RMSE={test_m['RMSE']:.3f} MAE={test_m['MAE']:.3f}\n"
 f"Within ±5%: {pct5:.1f}%\n"
 f"Within ±10%: {pct10:.1f}%")

 box_fc = BAND10_COLOR if r2_gap > 0.15 else BAND10_COLOR
 props = dict(boxstyle='round,pad=0.4', facecolor=box_fc,
 alpha=0.85, edgecolor='none')
 ax.text(0.04, 0.97, textstr, transform=ax.transAxes, fontsize=7.5,
 verticalalignment='top', bbox=props, fontweight='bold',
 color='black')

 ax.set_xlim(lo, hi)
 ax.set_ylim(lo, hi)
 ax.set_xlabel('Actual', fontsize=9, fontweight='bold')
 ax.set_ylabel('Predicted', fontsize=9, fontweight='bold')
 ax.set_title(model_name, fontsize=9, fontweight='bold', pad=6,
 color='black')
 ax.legend(fontsize=6, loc='lower right', framealpha=0.7)
 ax.set_aspect('equal', adjustable='box')

 for idx in range(n_models, len(axes_flat)):
 axes_flat[idx].axis('off')

 fig.suptitle(
 f'{output_name} – All Models: Test Predictions '
 f'[scatter colour = residual magnitude | title colour: blue=OK, red=overfit]',
 fontsize=13, fontweight='bold', y=0.998, color='#222222')
 plt.tight_layout(rect=[0, 0, 1, 0.990], h_pad=3.0, w_pad=2.0)

 out_path = f'{VIZ_PLOTS_DIR}/A_Scatter_{output_name}.png'
 plt.savefig(out_path, dpi=300, bbox_inches='tight', facecolor=FIGBG)
 plt.close()
 print(f" A_Scatter_{output_name}.png ({n_models} models)")

    # ──────────────────────────────────────────────────────────
    # PLOT B – R² Bar Chart comparison (train vs test, per output)
    # ──────────────────────────────────────────────────────────

 print("\nB. R² bar charts (train vs test) …")

 for output_name in OUTPUT_FEATURES:
 if len(test_metrics[output_name]) == 0:
 continue

 model_names = sorted(test_metrics[output_name].keys())
 n_m = len(model_names)

 train_r2s = [train_metrics[output_name][m]['R2'] for m in model_names]
 test_r2s = [test_metrics[output_name][m]['R2'] for m in model_names]

 x = np.arange(n_m)
 width = 0.38

 fig, ax = plt.subplots(figsize=(max(10, n_m * 1.2), 6), facecolor=FIGBG)
 ax.set_facecolor(AXBG)

 bars_train = ax.bar(x - width/2, train_r2s, width,
 color=COLOR_BLUE, label='Train R²',
 alpha=0.88, edgecolor='white', linewidth=0.7)
 bars_test = ax.bar(x + width/2, test_r2s, width,
 color=COLOR_RED, label='Test R²',
 alpha=0.88, edgecolor='white', linewidth=0.7)

        # Value labels
 for bar in list(bars_train) + list(bars_test):
 h = bar.get_height()
 ax.text(bar.get_x() + bar.get_width() / 2., h + 0.005,
 f'{h:.3f}', ha='center', va='bottom', fontsize=7.5,
 fontweight='bold', color='#333333')

        # Overfit annotation arrow
 for i, m in enumerate(model_names):
 gap = train_r2s[i] - test_r2s[i]
 if gap > 0.15:
 ax.annotate(' overfit',
 xy=(i + width/2, test_r2s[i]),
 xytext=(i + width/2, test_r2s[i] + 0.07),
 ha='center', fontsize=7, color=COLOR_RED,
 arrowprops=dict(arrowstyle='->', color=COLOR_RED,
 lw=1.2))

 ax.set_xticks(x)
 ax.set_xticklabels(model_names, rotation=35, ha='right', fontsize=9)
 ax.set_ylabel('R²', fontsize=11, fontweight='bold')
 ax.set_ylim(0, min(1.12, max(train_r2s + test_r2s) + 0.15))
 ax.set_title(f'{output_name} – R² Comparison: Train vs Test',
 fontsize=13, fontweight='bold', color='#222222')
 ax.legend(fontsize=10)
 ax.axhline(1.0, color='#999999', linewidth=0.8, linestyle=':')

 plt.tight_layout()
 out_path = f'{VIZ_PLOTS_DIR}/B_R2_Bar_{output_name}.png'
 plt.savefig(out_path, dpi=300, bbox_inches='tight', facecolor=FIGBG)
 plt.close()
 print(f" B_R2_Bar_{output_name}.png")

    # ──────────────────────────────────────────────────────────
    # PLOT C – RMSE Bar Chart comparison
    # ──────────────────────────────────────────────────────────

 print("\nC. RMSE bar charts (train vs test) …")

 for output_name in OUTPUT_FEATURES:
 if len(test_metrics[output_name]) == 0:
 continue

 model_names = sorted(test_metrics[output_name].keys())
 n_m = len(model_names)

 train_rmses = [train_metrics[output_name][m]['RMSE'] for m in model_names]
 test_rmses = [test_metrics[output_name][m]['RMSE'] for m in model_names]

 x = np.arange(n_m)
 width = 0.38

 fig, ax = plt.subplots(figsize=(max(10, n_m * 1.2), 6), facecolor=FIGBG)
 ax.set_facecolor(AXBG)

 norm_vals = test_rmses
 cval = (np.array(norm_vals) - min(norm_vals)) / (max(norm_vals) - min(norm_vals) + 1e-9)

 bars_train = ax.bar(x - width/2, train_rmses, width,
 color=COLOR_BLUE, label='Train RMSE',
 alpha=0.88, edgecolor='white', linewidth=0.7)
 bars_test = ax.bar(x + width/2, test_rmses, width,
 color=[CMAP(0.15 + 0.5 * c) for c in cval],
 label='Test RMSE',
 alpha=0.88, edgecolor='white', linewidth=0.7)

 for bar in list(bars_train) + list(bars_test):
 h = bar.get_height()
 ax.text(bar.get_x() + bar.get_width() / 2., h * 1.005,
 f'{h:.3f}', ha='center', va='bottom', fontsize=7.5,
 fontweight='bold', color='#333333')

 ax.set_xticks(x)
 ax.set_xticklabels(model_names, rotation=35, ha='right', fontsize=9)
 ax.set_ylabel('RMSE', fontsize=11, fontweight='bold')
 ax.set_title(f'{output_name} – RMSE Comparison: Train vs Test',
 fontsize=13, fontweight='bold', color='#222222')
 ax.legend(fontsize=10)

 plt.tight_layout()
 out_path = f'{VIZ_PLOTS_DIR}/C_RMSE_Bar_{output_name}.png'
 plt.savefig(out_path, dpi=300, bbox_inches='tight', facecolor=FIGBG)
 plt.close()
 print(f" C_RMSE_Bar_{output_name}.png")

    # ──────────────────────────────────────────────────────────
    # PLOT D – RESIDUAL PLOTS (Error vs Actual) – all models
    # ──────────────────────────────────────────────────────────

 print("\nD. Residual plots (Error vs Actual) …")

 for output_name in OUTPUT_FEATURES:
 if len(test_predictions[output_name]) == 0:
 continue

 output_idx = OUTPUT_FEATURES.index(output_name)
 all_model_names = sorted(test_predictions[output_name].keys())
 n_models = len(all_model_names)

 n_cols = 4
 n_rows = int(np.ceil(n_models / n_cols))

 fig, axes = plt.subplots(n_rows, n_cols,
 figsize=(20, 5 * n_rows),
 facecolor=FIGBG)
 if n_rows == 1:
 axes = axes.reshape(1, -1)
 axes_flat = axes.flatten()

 Y_test_orig = Y_test * Output_std + Output_mean
 y_test_actual = Y_test_orig[:, output_idx]

 for idx, model_name in enumerate(all_model_names):
 ax = axes_flat[idx]
 ax.set_facecolor(AXBG)

 y_test_pred = test_predictions[output_name][model_name]
 residuals = y_test_pred - y_test_actual

            # Colour by residual sign/magnitude
 res_norm = residuals / (np.abs(residuals).max() + 1e-9)  # –1 … +1
 col_vals = (res_norm + 1) / 2                             # 0 … 1
 scat_colors = [CMAP(v) for v in col_vals]

 ax.scatter(y_test_actual, residuals,
 c=scat_colors, s=35, alpha=0.78,
 edgecolors='none', zorder=3)

 ax.axhline(0, color='#333333', linewidth=1.2, linestyle='-', zorder=4)
 ax.axhline( 0.05 * y_test_actual.mean(), color=BAND5_COLOR,
 linewidth=1.2, linestyle='--', alpha=0.8,
 label='±5 % mean', zorder=4)
 ax.axhline(-0.05 * y_test_actual.mean(), color=BAND5_COLOR,
 linewidth=1.2, linestyle='--', alpha=0.8, zorder=4)
 ax.axhline( 0.10 * y_test_actual.mean(), color=BAND10_COLOR,
 linewidth=1.2, linestyle=':', alpha=0.8,
 label='±10 % mean', zorder=4)
 ax.axhline(-0.10 * y_test_actual.mean(), color=BAND10_COLOR,
 linewidth=1.2, linestyle=':', alpha=0.8, zorder=4)

 test_m = test_metrics[output_name][model_name]
 train_m = train_metrics[output_name][model_name]
 r2_gap = train_m['R2'] - test_m['R2']

 ax.set_xlabel('Actual', fontsize=9, fontweight='bold')
 ax.set_ylabel('Residual (Pred − Actual)', fontsize=9, fontweight='bold')
 ax.set_title(model_name, fontsize=9, fontweight='bold', pad=6,
 color=COLOR_RED if r2_gap > 0.15 else COLOR_BLUE)
 ax.legend(fontsize=6.5, loc='upper right', framealpha=0.7)

 for idx in range(n_models, len(axes_flat)):
 axes_flat[idx].axis('off')

 fig.suptitle(f'{output_name} – Residual Plots '
 f'[colour = residual sign: blue=under-predict, red=over-predict]',
 fontsize=13, fontweight='bold', y=0.999, color='#222222')
 plt.tight_layout(rect=[0, 0, 1, 0.990], h_pad=3.0, w_pad=2.0)

 out_path = f'{VIZ_PLOTS_DIR}/D_Residuals_{output_name}.png'
 plt.savefig(out_path, dpi=300, bbox_inches='tight', facecolor=FIGBG)
 plt.close()
 print(f" D_Residuals_{output_name}.png ({n_models} models)")

    # ──────────────────────────────────────────────────────────
    # PLOT E – Train vs Test R² Bubble Chart (Overfitting Overview)
    # ──────────────────────────────────────────────────────────

 print("\nE. Train vs Test R² bubble chart (overfitting overview) …")

 for output_name in OUTPUT_FEATURES:
 if len(test_metrics[output_name]) == 0:
 continue

 model_names = sorted(test_metrics[output_name].keys())
 n_m = len(model_names)

 train_r2s = np.array([train_metrics[output_name][m]['R2'] for m in model_names])
 test_r2s = np.array([test_metrics[output_name][m]['R2'] for m in model_names])
 gaps = train_r2s - test_r2s

        # Bubble size ∝ test RMSE
 test_rmses = np.array([test_metrics[output_name][m]['RMSE'] for m in model_names])
 bubble_sz = 200 + 800 * (test_rmses / (test_rmses.max() + 1e-9))

        # Colour ∝ R² gap (blue=no overfit, red=overfit)
 gap_norm = (gaps - gaps.min()) / (gaps.max() - gaps.min() + 1e-9)
 colors = [CMAP(g) for g in gap_norm]

 fig, ax = plt.subplots(figsize=(8, 7), facecolor=FIGBG)
 ax.set_facecolor(AXBG)

 sc = ax.scatter(train_r2s, test_r2s,
 s=bubble_sz, c=gap_norm, cmap='RdBu_r',
 alpha=0.85, edgecolors='white', linewidth=1.2,
 vmin=0, vmax=1, zorder=3)

        # Diagonal (perfect generalisation)
 lo = min(train_r2s.min(), test_r2s.min()) - 0.05
 hi = max(train_r2s.max(), test_r2s.max()) + 0.05
 ax.plot([lo, hi], [lo, hi], color='#555555', linewidth=1.2,
 linestyle='--', label='Perfect generalisation', zorder=2)

        # Overfit threshold line (gap = 0.15)
 x_line = np.linspace(0.15, hi, 100)
 ax.plot(x_line, x_line - 0.15, color=COLOR_RED, linewidth=1.2,
 linestyle=':', alpha=0.7, label='Gap = 0.15 (overfit threshold)', zorder=2)

        # Model labels
 for i, m in enumerate(model_names):
 ax.annotate(m, (train_r2s[i], test_r2s[i]),
 textcoords='offset points', xytext=(6, 4),
 fontsize=7.5, color='#333333')

 cbar = plt.colorbar(sc, ax=ax, pad=0.02)
 cbar.set_label('Normalised R² gap (blue=low, red=high)', fontsize=9)

 ax.set_xlabel('Train R²', fontsize=11, fontweight='bold')
 ax.set_ylabel('Test R²', fontsize=11, fontweight='bold')
 ax.set_title(f'{output_name} – Train vs Test R²\n'
 f'(bubble size ∝ test RMSE)',
 fontsize=12, fontweight='bold', color='#222222')
 ax.legend(fontsize=8, loc='lower right')

 plt.tight_layout()
 out_path = f'{VIZ_PLOTS_DIR}/E_TrainTest_Bubble_{output_name}.png'
 plt.savefig(out_path, dpi=300, bbox_inches='tight', facecolor=FIGBG)
 plt.close()
 print(f" E_TrainTest_Bubble_{output_name}.png")

    # ──────────────────────────────────────────────────────────
    # PLOT F – Heatmap: Test R² across ALL models × ALL outputs
    # ──────────────────────────────────────────────────────────

 print("\nF. Heatmap: R² across all models × all outputs …")

 all_model_names = sorted(ml_trained_models.keys())
 r2_matrix = np.full((len(all_model_names), len(OUTPUT_FEATURES)), np.nan)

 for j, output_name in enumerate(OUTPUT_FEATURES):
 for i, model_name in enumerate(all_model_names):
 if model_name in test_metrics[output_name]:
 r2_matrix[i, j] = test_metrics[output_name][model_name]['R2']

 fig, ax = plt.subplots(figsize=(max(6, len(OUTPUT_FEATURES) * 2.0),
 max(5, len(all_model_names) * 0.55 + 1.5)),
 facecolor=FIGBG)
 ax.set_facecolor(AXBG)

 im = ax.imshow(r2_matrix, cmap='RdBu_r', aspect='auto',
 vmin=max(0, np.nanmin(r2_matrix) - 0.05),
 vmax=1.0)

 ax.set_xticks(np.arange(len(OUTPUT_FEATURES)))
 ax.set_xticklabels(OUTPUT_FEATURES, fontsize=11, fontweight='bold')
 ax.set_yticks(np.arange(len(all_model_names)))
 ax.set_yticklabels(all_model_names, fontsize=9)

    # Annotate cells
 for i in range(len(all_model_names)):
 for j in range(len(OUTPUT_FEATURES)):
 v = r2_matrix[i, j]
 if not np.isnan(v):
 text_color = 'white' if (v < 0.45 or v > 0.85) else '#222222'
 ax.text(j, i, f'{v:.3f}', ha='center', va='center',
 fontsize=8.5, fontweight='bold', color=text_color)

 cbar = plt.colorbar(im, ax=ax, pad=0.02, shrink=0.85)
 cbar.set_label('Test R²', fontsize=10)

 ax.set_title('Test R² Heatmap – All Models × All Outputs\n'
 '(blue = high R², red = low R²)',
 fontsize=12, fontweight='bold', color='#222222', pad=12)

 plt.tight_layout()
 out_path = f'{VIZ_PLOTS_DIR}/F_R2_Heatmap_AllModels.png'
 plt.savefig(out_path, dpi=300, bbox_inches='tight', facecolor=FIGBG)
 plt.close()
 print(f" F_R2_Heatmap_AllModels.png")

# ──────────────────────────────────────────────────────────────
# SUMMARY
# ──────────────────────────────────────────────────────────────

print("\n" + "="*80)
print("STEP 6 COMPLETE – ALL VISUALIZATIONS SAVED")
print("="*80)

plot_catalogue = [
 ("A", "Scatter (Predicted vs Actual) + ±5/10% tolerance bands", "per output"),
 ("B", "R² Bar Chart – Train vs Test", "per output"),
 ("C", "RMSE Bar Chart – Train vs Test", "per output"),
 ("D", "Residual Plots (Error vs Actual)", "per output"),
 ("E", "Train vs Test R² Bubble Chart (overfit overview)", "per output"),
 ("F", "R² Heatmap – All models × all outputs", "one file"),
]

print(f"\nPlot catalogue → {VIZ_PLOTS_DIR}/")
for code, desc, scope in plot_catalogue:
 print(f" [{code}] {desc} ({scope})")

print(f"\nBest Test-R² models:")
for output_name in OUTPUT_FEATURES:
 print(f" • {output_name}: {best_models.get(output_name, 'N/A')}")

print("\n" + "="*80)


# 7. SHAP analysis

In [ ]:
# -*- coding: utf-8 -*-
"""
UF Water Treatment ML Code - COMPREHENSIVE SHAP ANALYSIS
Step 8: SHAP Feature Importance Analysis

COMPLETE FEATURE SET:
 1. SHAP feature importance rankings (mean |SHAP|)
 2. SHAP beeswarm plots
 3. Multi-model SHAP comparison (KEY NOVELTY)
 4. SHAP dependence plots
 5. SHAP interaction plots (Nice to have)
 6. TreeSHAP vs KernelSHAP comparison (Nice to have)

CONSOLIDATED CSV OUTPUTS:
 All models' feature importance → 1 CSV file
 All models' beeswarm data → 1 CSV file
 All models' dependence data → 1 CSV file
 All models' interaction data → 1 CSV file
 Multi-model ranking comparison → 1 CSV file
 Explainer type comparison → 1 CSV file
"""

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.cm as cm
from matplotlib.patches import Patch, Rectangle
import pandas as pd
import os
from datetime import datetime
from scipy.stats import spearmanr, kendalltau
import seaborn as sns

# SHAP imports
try:
 import shap
 print(" SHAP library imported")
except ImportError:
 print(" ERROR: SHAP library not found")
 print("Install with: pip install shap")
 raise

# ==================== STEP 8: SHAP ANALYSIS ====================

print("\n" + "="*80)
print(" " * 15 + "STEP 8: COMPREHENSIVE SHAP FEATURE IMPORTANCE ANALYSIS")
print(" " * 20 + "With Multi-Model Comparison & Consolidated CSVs")
print("="*80)

# ──────────────────────────────────────────────────────────────
# COLOR THEME SETUP (RdBu_r)
# ──────────────────────────────────────────────────────────────

CMAP = plt.get_cmap('RdBu_r')
COLOR_BLUE = CMAP(0.85)
COLOR_RED = CMAP(0.15)
COLOR_MID = CMAP(0.50)

FIGBG = '#F7F7F7'
AXBG = '#FFFFFF'

def rdbu_gradient(n, reverse=False):
 """Return n colours from blue to red (or reversed)."""
 vals = np.linspace(0.15, 0.85, n)
 if reverse:
 vals = vals[::-1]
 return [CMAP(v) for v in vals]

# ==================== SETUP ====================

if 'BASE_DIR' not in globals():
 print("\n Running Step 8 standalone - setting up directories...")

 try:

 BASE_DIR = BASE_DIR
 except:
 BASE_DIR = './UF_data_results'

 RESULTS_DIR = f'{BASE_DIR}/Results'
 MODELS_DIR = f'{BASE_DIR}/Trained_Models'
 VISUALIZATIONS_DIR = f'{BASE_DIR}/Visualizations'

 fixpoint_file = f'{BASE_DIR}/fixpoint.pkl'
 if not os.path.exists(fixpoint_file):
 print(f" ERROR: fixpoint.pkl not found at {fixpoint_file}")
 raise FileNotFoundError("fixpoint.pkl not found")

 with open(fixpoint_file, 'rb') as f:
 fixpoint = pickle.load(f)

 INPUT_FEATURES = fixpoint['Input_features']
 OUTPUT_FEATURES = fixpoint['Output_features']
 N_INPUTS = fixpoint['n_inputs']
 N_OUTPUTS = fixpoint['n_outputs']
 Input_mean = fixpoint['Input_mean']
 Input_std = fixpoint['Input_std']
 Output_mean = fixpoint['Output_mean']
 Output_std = fixpoint['Output_std']
 RANDOM_SEED = fixpoint['random_seed']

 print(f" Loaded configuration from fixpoint")
else:
 print("\n Running after previous steps - using existing variables")

# ==================== CONFIGURATION ====================

MODELS_TO_ANALYZE = None
SHAP_SAMPLE_SIZE = 1000
INTERACTION_SAMPLE_SIZE = 200  # Smaller for computational efficiency

print(f"\nConfiguration:")
print(f" • Input features ({N_INPUTS}): {', '.join(INPUT_FEATURES)}")
print(f" • Output features ({N_OUTPUTS}): {', '.join(OUTPUT_FEATURES)}")
print(f" • SHAP sample size: {SHAP_SAMPLE_SIZE}")
print(f" • Interaction sample size: {INTERACTION_SAMPLE_SIZE}")

# Create directory structure
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
ANALYSIS_DIR = f'{BASE_DIR}/Feature_Importance_{timestamp}'
VIZ_DIR = f'{ANALYSIS_DIR}/Visualizations'
VIZ_INDIVIDUAL = f'{VIZ_DIR}/Individual_Models'
VIZ_COMPARISON = f'{VIZ_DIR}/Multi_Model_Comparisons'
CSV_DIR = f'{ANALYSIS_DIR}/CSV_Results'
CSV_CONSOLIDATED = f'{CSV_DIR}/Consolidated'

for d in [ANALYSIS_DIR, VIZ_DIR, VIZ_INDIVIDUAL, VIZ_COMPARISON, CSV_DIR, CSV_CONSOLIDATED]:
 os.makedirs(d, exist_ok=True)

print(f"\nOutput structure:")
print(f" • Individual model plots: {VIZ_INDIVIDUAL}/")
print(f" • Multi-model comparisons: {VIZ_COMPARISON}/")
print(f" • Consolidated CSVs: {CSV_CONSOLIDATED}/")

plt.rcParams['axes.grid'] = False

# ==================== LOAD DATA ====================

print("\n" + "="*60)
print("LOADING DATA")
print("="*60)

if 'X_train' not in globals():
 DATA_FILE = f'{BASE_DIR}/UF_data.csv'
 CLEANED_DATA_FILE = f'{BASE_DIR}/UF_data_CLEANED.csv'

 if os.path.exists(CLEANED_DATA_FILE):
 DATA_FILE = CLEANED_DATA_FILE
 print(f" Using cleaned data")

 df = pd.read_csv(DATA_FILE)
 print(f" Loaded {len(df)} samples")

 Input = df[INPUT_FEATURES].values
 Output = df[OUTPUT_FEATURES].values

 if np.isnan(Output).any():
 valid_mask = ~np.isnan(Output).any(axis=1)
 Input = Input[valid_mask]
 Output = Output[valid_mask]
 print(f" Removed {(~valid_mask).sum()} rows with NaN")

 Input_std_data = (Input - Input_mean) / Input_std
 Output_std_data = (Output - Output_mean) / Output_std

 TEST_SIZE = fixpoint['test_size']
 X_train, X_test, Y_train, Y_test = train_test_split(
 Input_std_data, Output_std_data,
 test_size=TEST_SIZE,
 random_state=RANDOM_SEED
 )

 print(f" Data split: {len(X_train)} train, {len(X_test)} test")
else:
 print(" Using existing data splits")

# Sample for SHAP
if len(X_test) > SHAP_SAMPLE_SIZE:
 np.random.seed(RANDOM_SEED)
 sample_indices = np.random.choice(len(X_test), SHAP_SAMPLE_SIZE, replace=False)
 X_shap = X_test[sample_indices]
 X_shap_interaction = X_test[sample_indices[:INTERACTION_SAMPLE_SIZE]]
else:
 X_shap = X_test
 X_shap_interaction = X_test[:INTERACTION_SAMPLE_SIZE]

print(f" SHAP samples: {len(X_shap)} main, {len(X_shap_interaction)} interactions")

# ==================== LOAD MODELS ====================

print("\n" + "="*60)
print("LOADING TRAINED MODELS")
print("="*60)

if 'ml_trained_models' not in globals() or not ml_trained_models:
 model_files = [f for f in os.listdir(MODELS_DIR)
 if f.endswith('.pkl') and f not in ['fixpoint.pkl', 'ml_trained_models.pkl']]

 ml_trained_models = {}

 for model_file in sorted(model_files):
 basename = model_file.replace('.pkl', '')
 parts = basename.rsplit('_', 1)

 if len(parts) != 2:
 continue

 model_name_raw, output_name = parts
 model_name = model_name_raw.replace('_', ' ')

 if output_name not in OUTPUT_FEATURES:
 continue

 try:
 import joblib
 model_data = joblib.load(f"{MODELS_DIR}/{model_file}")
 except:
 try:
 with open(f"{MODELS_DIR}/{model_file}", 'rb') as f:
 model_data = pickle.load(f)
 except:
 continue

 if model_name not in ml_trained_models:
 ml_trained_models[model_name] = {}

 ml_trained_models[model_name][output_name] = {'model': model_data}

 print(f" Loaded {len(ml_trained_models)} models")

all_model_names = sorted(ml_trained_models.keys())

if MODELS_TO_ANALYZE is not None:
 model_names = [m for m in MODELS_TO_ANALYZE if m in all_model_names]
else:
 model_names = all_model_names

print(f" Analyzing {len(model_names)} models")

# ==================== SHAP ANALYSIS FUNCTIONS ====================

def get_shap_explainer(model, model_name, X_background):
 """Get appropriate SHAP explainer and return type info."""

    # Try TreeExplainer first for tree-based models
 if any(keyword in model_name for keyword in ['Tree', 'Forest', 'Boost', 'Bagging']):
 try:
 explainer = shap.TreeExplainer(model)
 return explainer, 'TreeSHAP'
 except:
 pass

    # Fall back to KernelExplainer
 try:
 if len(X_background) > 100:
 bg_indices = np.random.choice(len(X_background), 100, replace=False)
 X_bg = X_background[bg_indices]
 else:
 X_bg = X_background

 explainer = shap.KernelExplainer(model.predict, X_bg)
 return explainer, 'KernelSHAP'
 except Exception as e:
 print(f" Failed to create explainer: {str(e)}")
 return None, None

def analyze_model_shap_full(model_name, output_name, model, X_train, X_shap, X_interaction):
 """
 Comprehensive SHAP analysis including:
 - Main effects (SHAP values)
 - Interaction effects (for tree models)
 """
 print(f" Analyzing {model_name} for {output_name}...")

 try:
 explainer, explainer_type = get_shap_explainer(model, model_name, X_train)

 if explainer is None:
 return None, None, None, None

 print(f" Using {explainer_type}...")

        # Main SHAP values
 shap_values = explainer.shap_values(X_shap)
 if isinstance(shap_values, list):
 shap_values = shap_values[0]

 print(f" Main SHAP values calculated")

        # Interaction values (only for TreeSHAP and if data not too large)
 shap_interaction_values = None
 if explainer_type == 'TreeSHAP' and len(X_interaction) <= INTERACTION_SAMPLE_SIZE:
 try:
 explainer_tree = shap.TreeExplainer(model)
 shap_interaction_values = explainer_tree.shap_interaction_values(X_interaction)
 if isinstance(shap_interaction_values, list):
 shap_interaction_values = shap_interaction_values[0]
 print(f" Interaction values calculated")
 except Exception as e:
 print(f" Interaction values failed: {str(e)}")

 return shap_values, shap_interaction_values, explainer, explainer_type

 except Exception as e:
 print(f" Error: {str(e)}")
 return None, None, None, None

# ==================== VISUALIZATION FUNCTIONS ====================

def create_shap_beeswarm(shap_values, X_shap, feature_names, model_name, output_name, viz_dir):
 """Create SHAP beeswarm plot with RdBu_r theme."""
 fig, ax = plt.subplots(figsize=(12, 8), facecolor=FIGBG)

 shap.summary_plot(shap_values, X_shap, feature_names=feature_names,
 show=False, plot_type='dot', cmap=CMAP)

 ax = plt.gca()
 ax.set_facecolor(AXBG)
 ax.set_title(f'SHAP Beeswarm: {model_name} - {output_name}',
 fontsize=14, fontweight='bold', pad=15, color='#222222')

 plt.tight_layout()

 filename = f'{viz_dir}/Beeswarm_{model_name.replace(" ", "_")}_{output_name}.png'
 plt.savefig(filename, dpi=300, bbox_inches='tight', facecolor=FIGBG)
 plt.close()

 return filename

def create_shap_importance(shap_values, feature_names, model_name, output_name, viz_dir):
 """Create feature importance bar plot."""
 importance = np.abs(shap_values).mean(axis=0)

 importance_df = pd.DataFrame({
 'Feature': feature_names,
 'SHAP_Importance': importance
 }).sort_values('SHAP_Importance', ascending=False)

 fig, ax = plt.subplots(figsize=(10, 8), facecolor=FIGBG)
 ax.set_facecolor(AXBG)

 n_feats = len(importance_df)
 colors = rdbu_gradient(n_feats, reverse=True)

 bars = ax.barh(range(n_feats), importance_df['SHAP_Importance'],
 color=colors, edgecolor='white', linewidth=1.2)

 ax.set_yticks(range(n_feats))
 ax.set_yticklabels(importance_df['Feature'], fontsize=11, fontweight='bold')
 ax.set_xlabel('Mean |SHAP Value|', fontsize=12, fontweight='bold')
 ax.set_title(f'Feature Importance: {model_name} - {output_name}',
 fontsize=13, fontweight='bold', pad=12, color='#222222')
 ax.invert_yaxis()

 for bar, val in zip(bars, importance_df['SHAP_Importance']):
 ax.text(val * 1.02, bar.get_y() + bar.get_height()/2,
 f'{val:.4f}', va='center', fontsize=9, fontweight='bold')

 plt.tight_layout()

 filename = f'{viz_dir}/Importance_{model_name.replace(" ", "_")}_{output_name}.png'
 plt.savefig(filename, dpi=300, bbox_inches='tight', facecolor=FIGBG)
 plt.close()

 return filename, importance_df

def create_shap_dependence(shap_values, X_shap, feature_names, model_name, output_name, viz_dir):
 """Create dependence plots for all features."""
 importance = np.abs(shap_values).mean(axis=0)
 sorted_indices = np.argsort(importance)[::-1]

 n_features = len(feature_names)
 X_original = X_shap * Input_std + Input_mean

 n_cols = 5
 n_rows = int(np.ceil(n_features / n_cols))

 fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 4 * n_rows), facecolor=FIGBG)
 fig.suptitle(f'SHAP Dependence: {model_name} - {output_name}',
 fontsize=15, fontweight='bold', y=0.998, color='#222222')

 if n_rows == 1:
 axes = axes.reshape(1, -1)
 axes_flat = axes.flatten()

 for plot_idx, feat_idx in enumerate(sorted_indices):
 ax = axes_flat[plot_idx]
 ax.set_facecolor(AXBG)

 feature_name = feature_names[feat_idx]
 x_values = X_original[:, feat_idx]

 scatter = ax.scatter(x_values, shap_values[:, feat_idx],
 c=x_values, cmap='RdBu_r',
 alpha=0.7, s=35, edgecolors='none')

 ax.set_xlabel(f'{feature_name}', fontsize=9.5, fontweight='bold')
 ax.set_ylabel('SHAP value', fontsize=9.5, fontweight='bold')
 ax.set_title(f'{feature_name} (#{plot_idx+1})',
 fontsize=10, fontweight='bold', color='#222222')

 cbar = plt.colorbar(scatter, ax=ax)
 cbar.set_label('Value', fontsize=7.5)
 cbar.ax.tick_params(labelsize=7)

 ax.axhline(0, color='#555555', linestyle='--', linewidth=0.8, alpha=0.5)

 for idx in range(n_features, len(axes_flat)):
 axes_flat[idx].axis('off')

 plt.tight_layout()

 filename = f'{viz_dir}/Dependence_{model_name.replace(" ", "_")}_{output_name}.png'
 plt.savefig(filename, dpi=300, bbox_inches='tight', facecolor=FIGBG)
 plt.close()

 return filename

def create_shap_interactions(shap_interaction_values, X_interaction, feature_names,
 model_name, output_name, viz_dir):
 """Create SHAP interaction heatmap."""
 if shap_interaction_values is None:
 return None

    # Average absolute interaction effects
 interaction_matrix = np.abs(shap_interaction_values).mean(axis=0)

 fig, ax = plt.subplots(figsize=(10, 9), facecolor=FIGBG)
 ax.set_facecolor(AXBG)

 im = ax.imshow(interaction_matrix, cmap='RdBu_r', aspect='auto')

 ax.set_xticks(np.arange(len(feature_names)))
 ax.set_yticks(np.arange(len(feature_names)))
 ax.set_xticklabels(feature_names, rotation=45, ha='right', fontsize=10)
 ax.set_yticklabels(feature_names, fontsize=10)

    # Annotate cells
 for i in range(len(feature_names)):
 for j in range(len(feature_names)):
 v = interaction_matrix[i, j]
 text_color = 'white' if (v < interaction_matrix.max() * 0.3 or
 v > interaction_matrix.max() * 0.7) else '#222222'
 ax.text(j, i, f'{v:.3f}', ha='center', va='center',
 fontsize=7.5, fontweight='bold', color=text_color)

 cbar = plt.colorbar(im, ax=ax)
 cbar.set_label('Mean |SHAP Interaction|', fontsize=10)

 ax.set_title(f'SHAP Interaction Effects: {model_name} - {output_name}\n'
 f'(diagonal = main effects, off-diagonal = interactions)',
 fontsize=12, fontweight='bold', color='#222222', pad=12)

 plt.tight_layout()

 filename = f'{viz_dir}/Interactions_{model_name.replace(" ", "_")}_{output_name}.png'
 plt.savefig(filename, dpi=300, bbox_inches='tight', facecolor=FIGBG)
 plt.close()

 return filename

# ==================== MAIN ANALYSIS ====================

print("\n" + "="*80)
print("PERFORMING COMPREHENSIVE SHAP ANALYSIS")
print("="*80)

# Storage for consolidated data
all_importance_data = []
all_beeswarm_data = []
all_dependence_data = []
all_interaction_data = []
all_explainer_info = []

# Per-output storage
all_importance = {output_name: {} for output_name in OUTPUT_FEATURES}
all_shap_values = {output_name: {} for output_name in OUTPUT_FEATURES}
successful_analyses = {output_name: [] for output_name in OUTPUT_FEATURES}

for output_name in OUTPUT_FEATURES:
 print(f"\n{'='*60}")
 print(f"ANALYZING OUTPUT: {output_name}")
 print(f"{'='*60}")

 for model_name in model_names:
 if output_name not in ml_trained_models[model_name]:
 continue

 model_data = ml_trained_models[model_name][output_name]

 if isinstance(model_data, dict) and 'model' in model_data:
 model = model_data['model']
 else:
 model = model_data

        # Full SHAP analysis
 shap_values, shap_inter, explainer, explainer_type = analyze_model_shap_full(
 model_name, output_name, model, X_train, X_shap, X_shap_interaction
 )

 if shap_values is None:
 continue

        # Record explainer type
 all_explainer_info.append({
 'Model': model_name,
 'Output': output_name,
 'Explainer_Type': explainer_type
 })

 print(f" Creating visualizations...")

        # 1. Beeswarm plot
 beeswarm_file = create_shap_beeswarm(
 shap_values, X_shap, INPUT_FEATURES, model_name, output_name, VIZ_INDIVIDUAL
 )

        # Store beeswarm data
 for sample_idx in range(len(X_shap)):
 for feat_idx, feat_name in enumerate(INPUT_FEATURES):
 all_beeswarm_data.append({
 'Model': model_name,
 'Output': output_name,
 'Explainer_Type': explainer_type,
 'Sample_Index': sample_idx,
 'Feature': feat_name,
 'Feature_Value_Normalized': X_shap[sample_idx, feat_idx],
 'Feature_Value_Original': (X_shap[sample_idx, feat_idx] * Input_std[feat_idx] +
 Input_mean[feat_idx]),
 'SHAP_Value': shap_values[sample_idx, feat_idx]
 })

 print(f" Beeswarm plot")

        # 2. Feature importance
 importance_file, importance_df = create_shap_importance(
 shap_values, INPUT_FEATURES, model_name, output_name, VIZ_INDIVIDUAL
 )

        # Store importance data with ranking
 importance_df['Rank'] = range(1, len(importance_df) + 1)
 importance_df['Model'] = model_name
 importance_df['Output'] = output_name
 importance_df['Explainer_Type'] = explainer_type
 all_importance_data.append(importance_df)

 print(f" Feature importance")

        # 3. Dependence plots
 dependence_file = create_shap_dependence(
 shap_values, X_shap, INPUT_FEATURES, model_name, output_name, VIZ_INDIVIDUAL
 )

        # Store dependence data
 X_original = X_shap * Input_std + Input_mean
 for feat_idx, feat_name in enumerate(INPUT_FEATURES):
 for sample_idx in range(len(X_shap)):
 all_dependence_data.append({
 'Model': model_name,
 'Output': output_name,
 'Explainer_Type': explainer_type,
 'Sample_Index': sample_idx,
 'Feature': feat_name,
 'Feature_Value_Original': X_original[sample_idx, feat_idx],
 'Feature_Value_Normalized': X_shap[sample_idx, feat_idx],
 'SHAP_Value': shap_values[sample_idx, feat_idx]
 })

 print(f" Dependence plots")

        # 4. Interaction plots (if available)
 if shap_inter is not None:
 interaction_file = create_shap_interactions(
 shap_inter, X_shap_interaction, INPUT_FEATURES,
 model_name, output_name, VIZ_INDIVIDUAL
 )

            # Store interaction data
 interaction_matrix = np.abs(shap_inter).mean(axis=0)
 for i, feat_i in enumerate(INPUT_FEATURES):
 for j, feat_j in enumerate(INPUT_FEATURES):
 all_interaction_data.append({
 'Model': model_name,
 'Output': output_name,
 'Feature_1': feat_i,
 'Feature_2': feat_j,
 'Mean_Abs_Interaction': interaction_matrix[i, j],
 'Is_Main_Effect': (i == j)
 })

 print(f" Interaction plots")

        # Store for comparisons
 all_importance[output_name][model_name] = importance_df
 all_shap_values[output_name][model_name] = shap_values
 successful_analyses[output_name].append(model_name)

 print(f" Complete")

# ==================== SAVE CONSOLIDATED CSVs ====================

print("\n" + "="*60)
print("SAVING CONSOLIDATED CSV FILES")
print("="*60)

# 1. ALL MODELS FEATURE IMPORTANCE - CONSOLIDATED
if all_importance_data:
 consolidated_importance = pd.concat(all_importance_data, ignore_index=True)
 consolidated_importance = consolidated_importance[['Model', 'Output', 'Explainer_Type',
 'Feature', 'SHAP_Importance', 'Rank']]
 importance_path = f'{CSV_CONSOLIDATED}/All_Models_Feature_Importance.csv'
 consolidated_importance.to_csv(importance_path, index=False)
 print(f" Saved: All_Models_Feature_Importance.csv ({len(consolidated_importance)} rows)")

# 2. ALL MODELS BEESWARM DATA - CONSOLIDATED
if all_beeswarm_data:
 beeswarm_df = pd.DataFrame(all_beeswarm_data)
 beeswarm_path = f'{CSV_CONSOLIDATED}/All_Models_Beeswarm_Data.csv'
 beeswarm_df.to_csv(beeswarm_path, index=False)
 print(f" Saved: All_Models_Beeswarm_Data.csv ({len(beeswarm_df)} rows)")

# 3. ALL MODELS DEPENDENCE DATA - CONSOLIDATED
if all_dependence_data:
 dependence_df = pd.DataFrame(all_dependence_data)
 dependence_path = f'{CSV_CONSOLIDATED}/All_Models_Dependence_Data.csv'
 dependence_df.to_csv(dependence_path, index=False)
 print(f" Saved: All_Models_Dependence_Data.csv ({len(dependence_df)} rows)")

# 4. ALL MODELS INTERACTION DATA - CONSOLIDATED
if all_interaction_data:
 interaction_df = pd.DataFrame(all_interaction_data)
 interaction_path = f'{CSV_CONSOLIDATED}/All_Models_Interaction_Data.csv'
 interaction_df.to_csv(interaction_path, index=False)
 print(f" Saved: All_Models_Interaction_Data.csv ({len(interaction_df)} rows)")

# 5. EXPLAINER TYPE SUMMARY
if all_explainer_info:
 explainer_df = pd.DataFrame(all_explainer_info)
 explainer_path = f'{CSV_CONSOLIDATED}/Explainer_Type_Summary.csv'
 explainer_df.to_csv(explainer_path, index=False)
 print(f" Saved: Explainer_Type_Summary.csv ({len(explainer_df)} rows)")

# 6. MULTI-MODEL RANKING COMPARISON (KEY FOR PAPER)
print("\nCreating multi-model ranking comparison...")
ranking_comparison = []

for output_name in OUTPUT_FEATURES:
 if len(successful_analyses[output_name]) == 0:
 continue

 models = successful_analyses[output_name]

    # Create ranking matrix
 for feat in INPUT_FEATURES:
 feat_data = {'Feature': feat, 'Output': output_name}

 for model_name in models:
 imp_df = all_importance[output_name][model_name]
 rank = imp_df[imp_df['Feature'] == feat]['Rank'].values[0]
 importance = imp_df[imp_df['Feature'] == feat]['SHAP_Importance'].values[0]

 feat_data[f'{model_name}_Rank'] = rank
 feat_data[f'{model_name}_Importance'] = importance

        # Calculate statistics across models
 ranks = [feat_data[f'{m}_Rank'] for m in models]
 importances = [feat_data[f'{m}_Importance'] for m in models]

 feat_data['Mean_Rank'] = np.mean(ranks)
 feat_data['Std_Rank'] = np.std(ranks)
 feat_data['Mean_Importance'] = np.mean(importances)
 feat_data['Std_Importance'] = np.std(importances)
 feat_data['Rank_Range'] = max(ranks) - min(ranks)
 feat_data['Consistency_Score'] = 1 / (1 + feat_data['Std_Rank'])  # Higher = more consistent

 ranking_comparison.append(feat_data)

if ranking_comparison:
 ranking_df = pd.DataFrame(ranking_comparison)
 ranking_path = f'{CSV_CONSOLIDATED}/Multi_Model_Ranking_Comparison.csv'
 ranking_df.to_csv(ranking_path, index=False)
 print(f" Saved: Multi_Model_Ranking_Comparison.csv")

# ──────────────────────────────────────────────────────────────
# MULTI-MODEL COMPARISON PLOTS
# ──────────────────────────────────────────────────────────────

print("\n" + "="*80)
print("CREATING MULTI-MODEL COMPARISON VISUALIZATIONS")
print("="*80)

# PLOT 1: Feature Importance Heatmap (Models × Features)
print("\n1. Feature importance heatmap...")

for output_name in OUTPUT_FEATURES:
 if len(successful_analyses[output_name]) < 2:
 continue

 models = successful_analyses[output_name]

    # Build importance matrix
 imp_matrix = np.zeros((len(models), len(INPUT_FEATURES)))

 for i, model_name in enumerate(models):
 imp_df = all_importance[output_name][model_name]
 for j, feat in enumerate(INPUT_FEATURES):
 val = imp_df[imp_df['Feature'] == feat]['SHAP_Importance'].values[0]
 imp_matrix[i, j] = val

    # Normalize by row for comparison
 imp_matrix_norm = imp_matrix / (imp_matrix.max(axis=1, keepdims=True) + 1e-9)

 fig, ax = plt.subplots(figsize=(14, max(8, len(models) * 0.5 + 2)), facecolor=FIGBG)
 ax.set_facecolor(AXBG)

 im = ax.imshow(imp_matrix_norm, cmap='RdBu_r', aspect='auto', vmin=0, vmax=1)

 ax.set_xticks(np.arange(len(INPUT_FEATURES)))
 ax.set_xticklabels(INPUT_FEATURES, fontsize=11, fontweight='bold', rotation=0)
 ax.set_yticks(np.arange(len(models)))
 ax.set_yticklabels(models, fontsize=9)

    # Annotate
 for i in range(len(models)):
 for j in range(len(INPUT_FEATURES)):
 v = imp_matrix_norm[i, j]
 text_color = 'white' if (v < 0.35 or v > 0.80) else '#222222'
 ax.text(j, i, f'{v:.2f}', ha='center', va='center',
 fontsize=8, fontweight='bold', color=text_color)

 cbar = plt.colorbar(im, ax=ax, pad=0.02)
 cbar.set_label('Normalized Importance', fontsize=10)

 ax.set_title(f'{output_name} – Feature Importance Across All Models\n'
 f'(row-normalized: blue=high, red=low)',
 fontsize=14, fontweight='bold', color='#222222', pad=15)

 plt.tight_layout()
 plt.savefig(f'{VIZ_COMPARISON}/Importance_Heatmap_{output_name}.png',
 dpi=300, bbox_inches='tight', facecolor=FIGBG)
 plt.close()
 print(f" Importance_Heatmap_{output_name}.png")

# PLOT 2: Ranking Comparison with Consistency
print("\n2. Feature ranking comparison...")

for output_name in OUTPUT_FEATURES:
 if len(successful_analyses[output_name]) < 2:
 continue

 models = successful_analyses[output_name]

    # Build ranking matrix
 rank_matrix = np.zeros((len(models), len(INPUT_FEATURES)))

 for i, model_name in enumerate(models):
 imp_df = all_importance[output_name][model_name]
 for j, feat in enumerate(INPUT_FEATURES):
 rank = imp_df[imp_df['Feature'] == feat]['Rank'].values[0]
 rank_matrix[i, j] = rank

    # Calculate mean rank and std for each feature
 mean_ranks = rank_matrix.mean(axis=0)
 std_ranks = rank_matrix.std(axis=0)

    # Sort by mean rank
 sorted_indices = np.argsort(mean_ranks)

 fig, ax = plt.subplots(figsize=(12, 8), facecolor=FIGBG)
 ax.set_facecolor(AXBG)

 x = np.arange(len(INPUT_FEATURES))
 colors = rdbu_gradient(len(INPUT_FEATURES), reverse=True)

 bars = ax.barh(x, mean_ranks[sorted_indices],
 xerr=std_ranks[sorted_indices],
 color=[colors[i] for i in sorted_indices],
 edgecolor='white', linewidth=1.2,
 error_kw={'linewidth': 2, 'ecolor': '#555555'})

 ax.set_yticks(x)
 ax.set_yticklabels([INPUT_FEATURES[i] for i in sorted_indices],
 fontsize=11, fontweight='bold')
 ax.set_xlabel('Mean Rank Across Models (±Std)', fontsize=12, fontweight='bold')
 ax.set_xlim(0, len(INPUT_FEATURES) + 1)
 ax.invert_yaxis()
 ax.invert_xaxis()  # Lower rank = better = leftward

 ax.set_title(f'{output_name} – Feature Importance Ranking Across Models\n'
 f'(blue=consistently important, red=less important, error bars=variability)',
 fontsize=13, fontweight='bold', color='#222222', pad=15)

    # Add vertical lines for rank positions
 for rank in range(1, len(INPUT_FEATURES) + 1):
 ax.axvline(rank, color='#DDDDDD', linestyle=':', linewidth=0.8, alpha=0.5)

 plt.tight_layout()
 plt.savefig(f'{VIZ_COMPARISON}/Ranking_Comparison_{output_name}.png',
 dpi=300, bbox_inches='tight', facecolor=FIGBG)
 plt.close()
 print(f" Ranking_Comparison_{output_name}.png")

# PLOT 3: Model Agreement Matrix (Spearman Correlation)
print("\n3. Model agreement matrix...")

for output_name in OUTPUT_FEATURES:
 if len(successful_analyses[output_name]) < 2:
 continue

 models = successful_analyses[output_name]

    # Build ranking matrix
 rank_matrix = np.zeros((len(models), len(INPUT_FEATURES)))

 for i, model_name in enumerate(models):
 imp_df = all_importance[output_name][model_name]
 for j, feat in enumerate(INPUT_FEATURES):
 rank = imp_df[imp_df['Feature'] == feat]['Rank'].values[0]
 rank_matrix[i, j] = rank

    # Compute correlation matrix
 corr_matrix = np.zeros((len(models), len(models)))
 for i in range(len(models)):
 for j in range(len(models)):
 corr, _ = spearmanr(rank_matrix[i], rank_matrix[j])
 corr_matrix[i, j] = corr

 fig, ax = plt.subplots(figsize=(max(10, len(models) * 0.7),
 max(9, len(models) * 0.65)),
 facecolor=FIGBG)
 ax.set_facecolor(AXBG)

 im = ax.imshow(corr_matrix, cmap='RdBu_r', aspect='auto', vmin=-1, vmax=1)

 ax.set_xticks(np.arange(len(models)))
 ax.set_xticklabels(models, rotation=45, ha='right', fontsize=9)
 ax.set_yticks(np.arange(len(models)))
 ax.set_yticklabels(models, fontsize=9)

 for i in range(len(models)):
 for j in range(len(models)):
 v = corr_matrix[i, j]
 text_color = 'white' if abs(v) > 0.6 else '#222222'
 ax.text(j, i, f'{v:.2f}', ha='center', va='center',
 fontsize=8, fontweight='bold', color=text_color)

 cbar = plt.colorbar(im, ax=ax, pad=0.02)
 cbar.set_label('Spearman ρ', fontsize=10)

 ax.set_title(f'{output_name} – Model Agreement on Feature Rankings\n'
 f'(Spearman correlation: blue=high agreement, red=disagreement)',
 fontsize=13, fontweight='bold', color='#222222', pad=15)

 plt.tight_layout()
 plt.savefig(f'{VIZ_COMPARISON}/Agreement_Matrix_{output_name}.png',
 dpi=300, bbox_inches='tight', facecolor=FIGBG)
 plt.close()
 print(f" Agreement_Matrix_{output_name}.png")

# PLOT 4: TreeSHAP vs KernelSHAP Comparison (if both exist)
print("\n4. TreeSHAP vs KernelSHAP comparison...")

explainer_df = pd.DataFrame(all_explainer_info)

for output_name in OUTPUT_FEATURES:
 output_explainers = explainer_df[explainer_df['Output'] == output_name]

 tree_models = output_explainers[output_explainers['Explainer_Type'] == 'TreeSHAP']['Model'].tolist()
 kernel_models = output_explainers[output_explainers['Explainer_Type'] == 'KernelSHAP']['Model'].tolist()

 if not tree_models or not kernel_models:
 print(f" Skipping {output_name} (need both TreeSHAP and KernelSHAP)")
 continue

    # Average importance across explainer types
 tree_avg = {}
 kernel_avg = {}

 for feat in INPUT_FEATURES:
 tree_vals = []
 kernel_vals = []

 for model in tree_models:
 if model in all_importance[output_name]:
 imp_df = all_importance[output_name][model]
 val = imp_df[imp_df['Feature'] == feat]['SHAP_Importance'].values[0]
 tree_vals.append(val)

 for model in kernel_models:
 if model in all_importance[output_name]:
 imp_df = all_importance[output_name][model]
 val = imp_df[imp_df['Feature'] == feat]['SHAP_Importance'].values[0]
 kernel_vals.append(val)

 tree_avg[feat] = np.mean(tree_vals) if tree_vals else 0
 kernel_avg[feat] = np.mean(kernel_vals) if kernel_vals else 0

    # Plot comparison
 fig, ax = plt.subplots(figsize=(12, 8), facecolor=FIGBG)
 ax.set_facecolor(AXBG)

 x = np.arange(len(INPUT_FEATURES))
 width = 0.35

 bars1 = ax.bar(x - width/2, [tree_avg[f] for f in INPUT_FEATURES],
 width, label=f'TreeSHAP (n={len(tree_models)})',
 color=COLOR_BLUE, alpha=0.85, edgecolor='white', linewidth=1.2)
 bars2 = ax.bar(x + width/2, [kernel_avg[f] for f in INPUT_FEATURES],
 width, label=f'KernelSHAP (n={len(kernel_models)})',
 color=COLOR_RED, alpha=0.85, edgecolor='white', linewidth=1.2)

 ax.set_xticks(x)
 ax.set_xticklabels(INPUT_FEATURES, fontsize=11, fontweight='bold')
 ax.set_ylabel('Average |SHAP|', fontsize=12, fontweight='bold')
 ax.set_title(f'{output_name} – TreeSHAP vs KernelSHAP Comparison\n'
 f'(average feature importance by explainer type)',
 fontsize=13, fontweight='bold', color='#222222', pad=15)
 ax.legend(fontsize=11)

 plt.tight_layout()
 plt.savefig(f'{VIZ_COMPARISON}/TreeVsKernel_{output_name}.png',
 dpi=300, bbox_inches='tight', facecolor=FIGBG)
 plt.close()
 print(f" TreeVsKernel_{output_name}.png")

# ==================== SUMMARY ====================

print("\n" + "="*80)
print("STEP 8 COMPLETE – COMPREHENSIVE SHAP ANALYSIS FINISHED")
print("="*80)

print(f"\n ANALYSIS SUMMARY:")
print(f"{'='*60}")

for output_name in OUTPUT_FEATURES:
 n_models = len(successful_analyses[output_name])
 tree_count = len([m for m in successful_analyses[output_name]
 if any(info['Model'] == m and info['Explainer_Type'] == 'TreeSHAP'
 for info in all_explainer_info)])
 kernel_count = n_models - tree_count

 print(f"\n{output_name}:")
 print(f" • Total models analyzed: {n_models}")
 print(f" • TreeSHAP: {tree_count} models")
 print(f" • KernelSHAP: {kernel_count} models")

print(f"\n\n CONSOLIDATED CSV FILES ({CSV_CONSOLIDATED}/):")
print(f"{'='*60}")
print(f" All_Models_Feature_Importance.csv")
print(f" → Feature importance rankings for ALL models")
print(f" All_Models_Beeswarm_Data.csv")
print(f" → Complete beeswarm plot data for ALL models")
print(f" All_Models_Dependence_Data.csv")
print(f" → Dependence plot data for ALL models")
print(f" All_Models_Interaction_Data.csv")
print(f" → Interaction effects (TreeSHAP models only)")
print(f" Multi_Model_Ranking_Comparison.csv")
print(f" → KEY: Cross-model ranking comparison & consistency metrics")
print(f" Explainer_Type_Summary.csv")
print(f" → TreeSHAP vs KernelSHAP breakdown")

print(f"\n\n VISUALIZATIONS:")
print(f"{'='*60}")
print(f"\nIndividual Models ({VIZ_INDIVIDUAL}/):")
print(f" • Beeswarm plots")
print(f" • Feature importance bars")
print(f" • Dependence plots (all features)")
print(f" • Interaction heatmaps (TreeSHAP only)")

print(f"\nMulti-Model Comparisons ({VIZ_COMPARISON}/):")
print(f" • Importance_Heatmap_[output].png")
print(f" • Ranking_Comparison_[output].png")
print(f" • Agreement_Matrix_[output].png")
print(f" • TreeVsKernel_[output].png (if applicable)")

print(f"\n\n KEY FEATURES FOR YOUR PAPER:")
print(f"{'='*60}")
print(f" 1. SHAP feature importance rankings (mean |SHAP|)")
print(f" 2. SHAP beeswarm plots")
print(f" 3. Multi-model SHAP comparison ⭐ (MAIN CONTRIBUTION)")
print(f" 4. SHAP dependence plots")
print(f" 5. SHAP interaction analysis")
print(f" 6. TreeSHAP vs KernelSHAP comparison")

print(f"\n\n RECOMMENDED FOR PAPER:")
print(f"{'='*60}")
print(f" Use: Multi_Model_Ranking_Comparison.csv")
print(f" → Shows feature importance consistency across 16 models")
print(f" Use: Importance_Heatmap_[output].png")
print(f" → Visual comparison of all models")
print(f" Use: Agreement_Matrix_[output].png")
print(f" → Shows which models agree on feature importance")

print("\n" + "="*80)


# 6.2. n-fold prediction

In [ ]:
#  WORKFLOW: MULTIPLE RANDOM SPLITS FOR ROBUSTNESS ANALYSIS
# ==================== IMPORTS ====================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from datetime import datetime
from tabulate import tabulate
import warnings
warnings.filterwarnings('ignore')

# Scikit-learn imports
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import (RandomForestRegressor, ExtraTreesRegressor,
 GradientBoostingRegressor, BaggingRegressor,
 AdaBoostRegressor, HistGradientBoostingRegressor)
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
from sklearn.base import clone

# Check if running in Colab
try:

 COLAB_AVAILABLE = True

except ImportError:
 COLAB_AVAILABLE = False

print("\n" + "="*80)
print(" " * 15 + " WORKFLOW: MULTIPLE RANDOM SPLITS ANALYSIS")
print("="*80)

# ==================== CONFIGURATION ====================

# Define column names
input_columns = ['glu', 'mlss', 'air', 'fm', 'cn', 'hrt', 'srt']
output_columns = ['tmp', 'flow', 'lv']

# Paths
if COLAB_AVAILABLE:
 OUTPUT_DIR = BASE_DIR
else:
 OUTPUT_DIR = './UF_data_results'

DATA_PATH = os.path.join(OUTPUT_DIR, 'UF_data.csv')

# ===== USER INPUT: NUMBER OF SPLITS =====
print("\n" + "="*60)
print("CONFIGURATION")
print("="*60)

while True:
 try:
 N_SPLITS = int(input("\nHow many random splits would you like to perform? (Recommended: 5-20): "))
 if N_SPLITS < 1:
 print(" Please enter a number greater than 0")
 continue
 elif N_SPLITS > 50:
 confirm = input(f" {N_SPLITS} splits is quite high and may take a long time. Continue? (yes/no): ").strip().lower()
 if confirm not in ['yes', 'y']:
 continue
 break
 except ValueError:
 print(" Please enter a valid integer")

TEST_SIZE = 0.3

# Generate random seeds
np.random.seed(42)
RANDOM_SEEDS = np.random.randint(1, 1000000, size=N_SPLITS).tolist()

# Create main results directory
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
RESULTS_DIR = os.path.join(OUTPUT_DIR, f'Multi_Split_Analysis_{N_SPLITS}splits_{timestamp}')
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f"\nConfiguration:")
print(f" Data path: {DATA_PATH}")
print(f" Results directory: {RESULTS_DIR}")
print(f" Number of splits: {N_SPLITS}")
print(f" Test size: {TEST_SIZE*100}%")
print(f" Random seeds: {RANDOM_SEEDS[:5]}{'...' if N_SPLITS > 5 else ''}")

# ==================== LOAD DATA ====================

print("\n" + "="*60)
print("LOADING DATA")
print("="*60)

df = pd.read_csv(DATA_PATH)
print(f" Loaded data: {df.shape}")

# Verify columns
expected_columns = input_columns + output_columns
missing_cols = [col for col in expected_columns if col not in df.columns]
if missing_cols:
 raise ValueError(f"Missing columns: {missing_cols}")

# Extract arrays
Input = df[input_columns].values
Output = df[output_columns].values

print(f" Input shape: {Input.shape}")
print(f" Output shape: {Output.shape}")

# ==================== DATA PREPROCESSING FUNCTIONS ====================

def data_preprocessing(Input, Output):
 """Standardize input and output data using z-score normalization."""
 Input_mean = np.mean(Input, axis=0)
 Input_std = np.std(Input, axis=0)
 Input_std = np.where(Input_std == 0, 1, Input_std)
 Input_standardized = (Input - Input_mean) / Input_std

 Output_mean = np.mean(Output, axis=0)
 Output_std = np.std(Output, axis=0)
 Output_std = np.where(Output_std == 0, 1, Output_std)
 Output_standardized = (Output - Output_mean) / Output_std

 fixpoint = {
 'Input_mean': Input_mean,
 'Input_std': Input_std,
 'Output_mean': Output_mean,
 'Output_std': Output_std
 }

 return Input_standardized, Output_standardized, fixpoint

# ==================== DEFINE ML MODELS ====================

print("\n" + "="*60)
print("DEFINING ML MODELS")
print("="*60)

# Import advanced models
try:
 from xgboost import XGBRegressor
 XGBOOST_AVAILABLE = True
 print(" XGBoost available")
except ImportError:
 XGBOOST_AVAILABLE = False
 print(" XGBoost not available")

try:
 from lightgbm import LGBMRegressor
 LIGHTGBM_AVAILABLE = True
 print(" LightGBM available")
except ImportError:
 LIGHTGBM_AVAILABLE = False
 print(" LightGBM not available")

try:
 from catboost import CatBoostRegressor
 CATBOOST_AVAILABLE = True
 print(" CatBoost available")
except ImportError:
 CATBOOST_AVAILABLE = False
 print(" CatBoost not available")

# Define base models
ml_models_base = {
 'Linear Regression': LinearRegression(),
 'Ridge Regression': Ridge(alpha=1.0, random_state=42),
 'Lasso Regression': Lasso(alpha=0.1, random_state=42),
 'ElasticNet': ElasticNet(alpha=0.1, l1_ratio=0.5, random_state=42),
 'Decision Tree': DecisionTreeRegressor(max_depth=10, random_state=42),
 'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
 'Extra Trees': ExtraTreesRegressor(n_estimators=100, random_state=42),
 'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
 'AdaBoost': AdaBoostRegressor(n_estimators=100, random_state=42),
 'Hist Gradient Boosting': HistGradientBoostingRegressor(max_iter=100, random_state=42),
 'KNN': KNeighborsRegressor(n_neighbors=5),
 'SVR': SVR(kernel='rbf', C=1.0, epsilon=0.1),
 'Bagging': BaggingRegressor(n_estimators=100, random_state=42),
 'MLP Neural Network': MLPRegressor(hidden_layer_sizes=(100, 50), max_iter=500,
 random_state=42, early_stopping=True)
}

if XGBOOST_AVAILABLE:
 ml_models_base['XGBoost'] = XGBRegressor(n_estimators=100, random_state=42, verbosity=0)

if LIGHTGBM_AVAILABLE:
 ml_models_base['LightGBM'] = LGBMRegressor(n_estimators=100, random_state=42, verbose=-1)

if CATBOOST_AVAILABLE:
 ml_models_base['CatBoost'] = CatBoostRegressor(iterations=100, random_state=42, verbose=0)

print(f"\n Defined {len(ml_models_base)} models")

# Confirm to proceed
print("\n" + "="*60)
print(f"Ready to train {len(ml_models_base)} models on {N_SPLITS} different splits")
print(f"Total trainings: {N_SPLITS * len(ml_models_base) * len(output_columns)}")
print("="*60)

proceed = input("\nProceed with training? (yes/no): ").strip().lower()
if proceed not in ['yes', 'y']:
 print("Training cancelled.")
 exit()

# ==================== MAIN LOOP: TRAIN ON MULTIPLE SPLITS ====================

print("\n" + "="*80)
print("STARTING MULTI-SPLIT ANALYSIS")
print("="*80)

# Storage for all results
all_splits_results = {
 'split_info': [],
 'train_metrics': [],
 'test_metrics': [],
 'predictions': {}
}

# For each split
for split_idx, random_seed in enumerate(RANDOM_SEEDS, 1):
 print(f"\n{'='*80}")
 print(f"SPLIT {split_idx}/{N_SPLITS} (Random Seed: {random_seed})")
 print(f"{'='*80}")

    # Create split directory
 split_dir = os.path.join(RESULTS_DIR, f'Split_{split_idx:02d}_seed_{random_seed}')
 split_csv_dir = os.path.join(split_dir, 'CSV')
 split_viz_dir = os.path.join(split_dir, 'Visualizations')
 os.makedirs(split_csv_dir, exist_ok=True)
 os.makedirs(split_viz_dir, exist_ok=True)

    # ===== SPLIT DATA =====
 print(f"\nSplitting data with seed {random_seed}...")
 X_train, X_test, Y_train, Y_test = train_test_split(
 Input, Output,
 test_size=TEST_SIZE,
 random_state=random_seed,
 shuffle=True
 )

 print(f" Train: {X_train.shape[0]} samples")
 print(f" Test: {X_test.shape[0]} samples")

    # Store split info
 all_splits_results['split_info'].append({
 'Split': split_idx,
 'Random_Seed': random_seed,
 'Train_Size': len(X_train),
 'Test_Size': len(X_test)
 })

    # ===== STANDARDIZE DATA =====
 print(f"Standardizing data...")
 X_train_std, Y_train_std, fixpoint = data_preprocessing(X_train, Y_train)
 X_test_std = (X_test - fixpoint['Input_mean']) / fixpoint['Input_std']
 Y_test_std = (Y_test - fixpoint['Output_mean']) / fixpoint['Output_std']

    # ===== VISUALIZE SPLIT DISTRIBUTION =====
 print(f"Creating split distribution visualization...")

 fig, axes = plt.subplots(2, len(output_columns), figsize=(14, 10))

 for idx, output_name in enumerate(output_columns):
 output_idx = output_columns.index(output_name)

        # Original scale
 ax = axes[0, idx]
 ax.hist(Y_train[:, output_idx], bins=30, alpha=0.6, label='Train',
 color='blue', edgecolor='darkblue')
 ax.hist(Y_test[:, output_idx], bins=30, alpha=0.6, label='Test',
 color='green', edgecolor='darkgreen')
 ax.set_xlabel(output_name, fontsize=11, fontweight='bold')
 ax.set_ylabel('Frequency', fontsize=11, fontweight='bold')
 ax.set_title(f'{output_name} - Original Scale', fontsize=12, fontweight='bold')
 ax.legend()
 ax.grid(True, alpha=0.3)

        # Standardized scale
 ax = axes[1, idx]
 ax.hist(Y_train_std[:, output_idx], bins=30, alpha=0.6, label='Train',
 color='blue', edgecolor='darkblue')
 ax.hist(Y_test_std[:, output_idx], bins=30, alpha=0.6, label='Test',
 color='green', edgecolor='darkgreen')
 ax.set_xlabel(output_name, fontsize=11, fontweight='bold')
 ax.set_ylabel('Frequency', fontsize=11, fontweight='bold')
 ax.set_title(f'{output_name} - Standardized Scale', fontsize=12, fontweight='bold')
 ax.legend()
 ax.grid(True, alpha=0.3)

 plt.suptitle(f'Split {split_idx}: Train/Test Distribution\n(Seed: {random_seed})',
 fontsize=16, fontweight='bold')
 plt.tight_layout()

 dist_file = os.path.join(split_viz_dir, 'split_distribution.png')
 plt.savefig(dist_file, dpi=300, bbox_inches='tight')
 plt.close()

    # ===== TRAIN MODELS =====
 print(f"\nTraining models...")

 split_predictions = {'train': {}, 'test': {}}

 for output_name in output_columns:
 print(f"\n {output_name}:")
 output_idx = output_columns.index(output_name)

 split_predictions['train'][output_name] = {}
 split_predictions['test'][output_name] = {}

 for model_name, base_model in ml_models_base.items():
 try:
                # Clone model
 model = clone(base_model)

                # Train
 y_train_std = Y_train_std[:, output_idx]
 y_test_std = Y_test_std[:, output_idx]

 model.fit(X_train_std, y_train_std)

                # Predict on train
 y_train_pred_std = model.predict(X_train_std)
 y_train_pred = (y_train_pred_std * fixpoint['Output_std'][output_idx] +
 fixpoint['Output_mean'][output_idx])
 split_predictions['train'][output_name][model_name] = y_train_pred

                # Predict on test
 y_test_pred_std = model.predict(X_test_std)
 y_test_pred = (y_test_pred_std * fixpoint['Output_std'][output_idx] +
 fixpoint['Output_mean'][output_idx])
 split_predictions['test'][output_name][model_name] = y_test_pred

                # Calculate metrics - TRAIN
 y_train_actual = Y_train[:, output_idx]
 train_mse = mean_squared_error(y_train_actual, y_train_pred)
 train_rmse = np.sqrt(train_mse)
 train_mae = mean_absolute_error(y_train_actual, y_train_pred)
 train_r2 = r2_score(y_train_actual, y_train_pred)

 all_splits_results['train_metrics'].append({
 'Split': split_idx,
 'Random_Seed': random_seed,
 'Output': output_name,
 'Model': model_name,
 'MSE': train_mse,
 'RMSE': train_rmse,
 'MAE': train_mae,
 'R2': train_r2
 })

                # Calculate metrics - TEST
 y_test_actual = Y_test[:, output_idx]
 test_mse = mean_squared_error(y_test_actual, y_test_pred)
 test_rmse = np.sqrt(test_mse)
 test_mae = mean_absolute_error(y_test_actual, y_test_pred)
 test_r2 = r2_score(y_test_actual, y_test_pred)

 all_splits_results['test_metrics'].append({
 'Split': split_idx,
 'Random_Seed': random_seed,
 'Output': output_name,
 'Model': model_name,
 'MSE': test_mse,
 'RMSE': test_rmse,
 'MAE': test_mae,
 'R2': test_r2
 })

 print(f" {model_name:25s}: Train R²={train_r2:.4f}, Test R²={test_r2:.4f}")

 except Exception as e:
 print(f" {model_name}: Error - {str(e)}")

    # ===== SAVE PREDICTIONS FOR THIS SPLIT =====
 print(f"\nSaving predictions for Split {split_idx}...")

 for output_name in output_columns:
        # Train predictions
 train_df = pd.DataFrame(X_train, columns=input_columns)
 train_df['Actual'] = Y_train[:, output_columns.index(output_name)]
 for model_name in split_predictions['train'][output_name].keys():
 train_df[f'Pred_{model_name.replace(" ", "_")}'] = split_predictions['train'][output_name][model_name]

 train_csv = os.path.join(split_csv_dir, f'train_predictions_{output_name}.csv')
 train_df.to_csv(train_csv, index=False)

        # Test predictions
 test_df = pd.DataFrame(X_test, columns=input_columns)
 test_df['Actual'] = Y_test[:, output_columns.index(output_name)]
 for model_name in split_predictions['test'][output_name].keys():
 test_df[f'Pred_{model_name.replace(" ", "_")}'] = split_predictions['test'][output_name][model_name]

 test_csv = os.path.join(split_csv_dir, f'test_predictions_{output_name}.csv')
 test_df.to_csv(test_csv, index=False)

    # ===== CREATE VISUALIZATION FOR THIS SPLIT =====
 print(f"Creating prediction visualizations for Split {split_idx}...")

    # Find best model for this split
 split_test_df = pd.DataFrame(all_splits_results['test_metrics'])
 split_test_df = split_test_df[split_test_df['Split'] == split_idx]

 for output_name in output_columns:
 output_metrics = split_test_df[split_test_df['Output'] == output_name]
 best_model = output_metrics.loc[output_metrics['R2'].idxmax(), 'Model']

 fig, axes = plt.subplots(1, 2, figsize=(16, 6))

 output_idx = output_columns.index(output_name)

        # Train scatter
 ax = axes[0]
 y_train_actual = Y_train[:, output_idx]
 y_train_pred = split_predictions['train'][output_name][best_model]

 ax.scatter(y_train_actual, y_train_pred, alpha=0.6, s=40,
 c='blue', edgecolors='darkblue', linewidth=0.5)

 min_val = min(y_train_actual.min(), y_train_pred.min())
 max_val = max(y_train_actual.max(), y_train_pred.max())
 ax.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2)

 train_r2 = r2_score(y_train_actual, y_train_pred)
 train_rmse = np.sqrt(mean_squared_error(y_train_actual, y_train_pred))
 train_mae = mean_absolute_error(y_train_actual, y_train_pred)

 ax.text(0.05, 0.95, f'R² = {train_r2:.4f}\nRMSE = {train_rmse:.4f}\nMAE = {train_mae:.4f}',
 transform=ax.transAxes, fontsize=11, verticalalignment='top',
 bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.8))

 ax.set_xlabel('Actual', fontsize=12, fontweight='bold')
 ax.set_ylabel('Predicted', fontsize=12, fontweight='bold')
 ax.set_title(f'Training Set', fontsize=13, fontweight='bold')
 ax.grid(True, alpha=0.3)
 ax.set_aspect('equal', adjustable='box')

        # Test scatter
 ax = axes[1]
 y_test_actual = Y_test[:, output_idx]
 y_test_pred = split_predictions['test'][output_name][best_model]

 ax.scatter(y_test_actual, y_test_pred, alpha=0.6, s=40,
 c='green', edgecolors='darkgreen', linewidth=0.5)

 min_val = min(y_test_actual.min(), y_test_pred.min())
 max_val = max(y_test_actual.max(), y_test_pred.max())
 ax.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2)

 test_r2 = r2_score(y_test_actual, y_test_pred)
 test_rmse = np.sqrt(mean_squared_error(y_test_actual, y_test_pred))
 test_mae = mean_absolute_error(y_test_actual, y_test_pred)

 ax.text(0.05, 0.95, f'R² = {test_r2:.4f}\nRMSE = {test_rmse:.4f}\nMAE = {test_mae:.4f}',
 transform=ax.transAxes, fontsize=11, verticalalignment='top',
 bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.8))

 ax.set_xlabel('Actual', fontsize=12, fontweight='bold')
 ax.set_ylabel('Predicted', fontsize=12, fontweight='bold')
 ax.set_title(f'Test Set', fontsize=13, fontweight='bold')
 ax.grid(True, alpha=0.3)
 ax.set_aspect('equal', adjustable='box')

 plt.suptitle(f'Split {split_idx}: {output_name} - {best_model}\n(Seed: {random_seed})',
 fontsize=15, fontweight='bold')
 plt.tight_layout()

 scatter_file = os.path.join(split_viz_dir, f'predictions_{output_name}.png')
 plt.savefig(scatter_file, dpi=300, bbox_inches='tight')
 plt.close()

 print(f"\n Split {split_idx} complete")

# ==================== CONSOLIDATE AND ANALYZE RESULTS ====================

print("\n" + "="*80)
print("CONSOLIDATING RESULTS ACROSS ALL SPLITS")
print("="*80)

# Convert to DataFrames
split_info_df = pd.DataFrame(all_splits_results['split_info'])
train_metrics_df = pd.DataFrame(all_splits_results['train_metrics'])
test_metrics_df = pd.DataFrame(all_splits_results['test_metrics'])

# Save consolidated results
consolidated_dir = os.path.join(RESULTS_DIR, 'Consolidated_Results')
os.makedirs(consolidated_dir, exist_ok=True)

split_info_df.to_csv(os.path.join(consolidated_dir, 'split_info.csv'), index=False)
train_metrics_df.to_csv(os.path.join(consolidated_dir, 'train_metrics_all_splits.csv'), index=False)
test_metrics_df.to_csv(os.path.join(consolidated_dir, 'test_metrics_all_splits.csv'), index=False)

print(f" Saved consolidated results to: {consolidated_dir}")

# ==================== CONSISTENCY ANALYSIS ====================

print("\n" + "="*80)
print("ANALYZING CONSISTENCY ACROSS SPLITS")
print("="*80)

# Calculate statistics across splits for ALL metrics
consistency_results = []

for output_name in output_columns:
 for model_name in ml_models_base.keys():
        # Filter for this output and model
 model_metrics = test_metrics_df[
 (test_metrics_df['Output'] == output_name) &
 (test_metrics_df['Model'] == model_name)
 ]

 if len(model_metrics) > 0:
 consistency_results.append({
 'Output': output_name,
 'Model': model_name,
                # R2 statistics
 'Mean_R2': model_metrics['R2'].mean(),
 'Std_R2': model_metrics['R2'].std(),
 'Min_R2': model_metrics['R2'].min(),
 'Max_R2': model_metrics['R2'].max(),
 'CV_R2_%': (model_metrics['R2'].std() / model_metrics['R2'].mean()) * 100 if model_metrics['R2'].mean() != 0 else 0,
                # RMSE statistics
 'Mean_RMSE': model_metrics['RMSE'].mean(),
 'Std_RMSE': model_metrics['RMSE'].std(),
 'Min_RMSE': model_metrics['RMSE'].min(),
 'Max_RMSE': model_metrics['RMSE'].max(),
 'CV_RMSE_%': (model_metrics['RMSE'].std() / model_metrics['RMSE'].mean()) * 100 if model_metrics['RMSE'].mean() != 0 else 0,
                # MAE statistics
 'Mean_MAE': model_metrics['MAE'].mean(),
 'Std_MAE': model_metrics['MAE'].std(),
 'Min_MAE': model_metrics['MAE'].min(),
 'Max_MAE': model_metrics['MAE'].max(),
 'CV_MAE_%': (model_metrics['MAE'].std() / model_metrics['MAE'].mean()) * 100 if model_metrics['MAE'].mean() != 0 else 0,
                # MSE statistics
 'Mean_MSE': model_metrics['MSE'].mean(),
 'Std_MSE': model_metrics['MSE'].std(),
 'Min_MSE': model_metrics['MSE'].min(),
 'Max_MSE': model_metrics['MSE'].max(),
 })

consistency_df = pd.DataFrame(consistency_results)
consistency_csv = os.path.join(consolidated_dir, 'consistency_analysis.csv')
consistency_df.to_csv(consistency_csv, index=False)
print(f" Saved consistency analysis")

# Display top models by different metrics
print("\n" + "="*60)
print("TOP MODELS BY DIFFERENT METRICS")
print("="*60)

for output_name in output_columns:
 print(f"\n{'='*60}")
 print(f"{output_name.upper()}")
 print(f"{'='*60}")

 output_consistency = consistency_df[consistency_df['Output'] == output_name].copy()

    # Best R²
 print(f"\n1. Best R² (Higher is Better):")
 print("-" * 60)
 top_r2 = output_consistency.nlargest(5, 'Mean_R2')
 print(tabulate(top_r2[['Model', 'Mean_R2', 'Std_R2', 'CV_R2_%']],
 headers='keys', tablefmt='grid', showindex=False, floatfmt='.4f'))

    # Best RMSE
 print(f"\n2. Best RMSE (Lower is Better):")
 print("-" * 60)
 top_rmse = output_consistency.nsmallest(5, 'Mean_RMSE')
 print(tabulate(top_rmse[['Model', 'Mean_RMSE', 'Std_RMSE', 'CV_RMSE_%']],
 headers='keys', tablefmt='grid', showindex=False, floatfmt='.4f'))

    # Best MAE
 print(f"\n3. Best MAE (Lower is Better):")
 print("-" * 60)
 top_mae = output_consistency.nsmallest(5, 'Mean_MAE')
 print(tabulate(top_mae[['Model', 'Mean_MAE', 'Std_MAE', 'CV_MAE_%']],
 headers='keys', tablefmt='grid', showindex=False, floatfmt='.4f'))

    # Most Consistent (lowest CV for R²)
 print(f"\n4. Most Consistent R² (Lowest CV):")
 print("-" * 60)
 most_consistent = output_consistency.nsmallest(5, 'CV_R2_%')
 print(tabulate(most_consistent[['Model', 'Mean_R2', 'Std_R2', 'CV_R2_%']],
 headers='keys', tablefmt='grid', showindex=False, floatfmt='.4f'))

# ==================== VISUALIZATION: CONSISTENCY ANALYSIS ====================

print("\n" + "="*80)
print("CREATING CONSISTENCY VISUALIZATIONS")
print("="*80)

viz_dir = os.path.join(consolidated_dir, 'Visualizations')
os.makedirs(viz_dir, exist_ok=True)

# Metrics to visualize
metrics_info = [
 ('R2', 'R² Score', 'higher', 0, 1),
 ('RMSE', 'RMSE', 'lower', None, None),
 ('MAE', 'MAE', 'lower', None, None)
]

# 1. Box plots for each metric
print("\n1. Creating box plots for R², RMSE, and MAE...")

for metric_name, metric_label, direction, vmin, vmax in metrics_info:
 for output_name in output_columns:
 output_metrics = test_metrics_df[test_metrics_df['Output'] == output_name]

        # Get top 10 models by mean metric
 if direction == 'higher':
 top_models = consistency_df[consistency_df['Output'] == output_name].nlargest(10, f'Mean_{metric_name}')['Model'].values
 else:
 top_models = consistency_df[consistency_df['Output'] == output_name].nsmallest(10, f'Mean_{metric_name}')['Model'].values

 fig, ax = plt.subplots(figsize=(14, 8))

 data_to_plot = []
 labels = []

 for model in top_models:
 model_data = output_metrics[output_metrics['Model'] == model][metric_name].values
 data_to_plot.append(model_data)
 labels.append(model)

 bp = ax.boxplot(data_to_plot, labels=labels, patch_artist=True, showfliers=True)

        # Color boxes
 for patch in bp['boxes']:
 patch.set_facecolor('lightblue')
 patch.set_edgecolor('darkblue')
 patch.set_linewidth(1.5)

 for element in ['whiskers', 'fliers', 'means', 'medians', 'caps']:
 plt.setp(bp[element], color='darkblue', linewidth=1.5)

 ax.set_xlabel('Model', fontsize=12, fontweight='bold')
 ax.set_ylabel(metric_label, fontsize=12, fontweight='bold')
 ax.set_title(f'{output_name} - {metric_label} Consistency Across {N_SPLITS} Splits\n(Top 10 Models by Mean {metric_label})',
 fontsize=14, fontweight='bold')
 ax.grid(True, alpha=0.3, axis='y')
 plt.xticks(rotation=45, ha='right')

 if vmin is not None and vmax is not None:
 ax.set_ylim([vmin, vmax])

 plt.tight_layout()

 boxplot_file = os.path.join(viz_dir, f'{metric_name}_consistency_boxplot_{output_name}.png')
 plt.savefig(boxplot_file, dpi=300, bbox_inches='tight')
 plt.close()
 print(f" Saved: {metric_name}_consistency_boxplot_{output_name}.png")

# 2. Line plots showing metric trajectory across splits
print("\n2. Creating trajectory plots for R², RMSE, and MAE...")

for metric_name, metric_label, direction, vmin, vmax in metrics_info:
 for output_name in output_columns:
 output_metrics = test_metrics_df[test_metrics_df['Output'] == output_name]

 if direction == 'higher':
 top_models = consistency_df[consistency_df['Output'] == output_name].nlargest(5, f'Mean_{metric_name}')['Model'].values
 else:
 top_models = consistency_df[consistency_df['Output'] == output_name].nsmallest(5, f'Mean_{metric_name}')['Model'].values

 fig, ax = plt.subplots(figsize=(14, 8))

 for model in top_models:
 model_data = output_metrics[output_metrics['Model'] == model].sort_values('Split')
 ax.plot(model_data['Split'], model_data[metric_name], 'o-', linewidth=2,
 markersize=8, label=model, alpha=0.8)

 ax.set_xlabel('Split Number', fontsize=12, fontweight='bold')
 ax.set_ylabel(metric_label, fontsize=12, fontweight='bold')
 ax.set_title(f'{output_name} - {metric_label} Across {N_SPLITS} Splits\n(Top 5 Models)',
 fontsize=14, fontweight='bold')
 ax.legend(fontsize=10, loc='best')
 ax.grid(True, alpha=0.3)
 ax.set_xticks(range(1, N_SPLITS + 1))

 if vmin is not None and vmax is not None:
 ax.set_ylim([vmin, vmax])

 plt.tight_layout()

 line_file = os.path.join(viz_dir, f'{metric_name}_trajectory_{output_name}.png')
 plt.savefig(line_file, dpi=300, bbox_inches='tight')
 plt.close()
 print(f" Saved: {metric_name}_trajectory_{output_name}.png")

# 3. Heatmaps for each metric
print("\n3. Creating heatmaps for R², RMSE, and MAE...")

for metric_name, metric_label, direction, vmin_val, vmax_val in metrics_info:
 for output_name in output_columns:
 output_metrics = test_metrics_df[test_metrics_df['Output'] == output_name]

        # Pivot to create matrix
 pivot = output_metrics.pivot(index='Model', columns='Split', values=metric_name)

        # Sort by mean metric
 if direction == 'higher':
 pivot = pivot.loc[pivot.mean(axis=1).sort_values(ascending=False).index]
 else:
 pivot = pivot.loc[pivot.mean(axis=1).sort_values(ascending=True).index]

        # ========== ADAPTIVE SIZING AND ANNOTATION ==========
 n_splits = len(pivot.columns)
 n_models = len(pivot.index)

        # Dynamic figure size based on data dimensions
        # Width: 0.5 inches per split, constrained between 12-35 inches
 width = max(12, min(35, n_splits * 0.5))
        # Height: 0.6 inches per model, constrained between 8-20 inches
 height = max(8, min(20, n_models * 0.6))

 fig, ax = plt.subplots(figsize=(width, height))

        # Choose colormap
 if direction == 'higher':
 cmap = 'RdYlGn'
 else:
 cmap = 'RdYlGn_r'

        # ========== ADAPTIVE ANNOTATION STRATEGY ==========
        # Determine whether to show annotations and what font size
 if n_splits <= 15:
 show_annot = True
 annot_fontsize = 9
 linewidths = 0.5
 elif n_splits <= 25:
 show_annot = True
 annot_fontsize = 7
 linewidths = 0.3
 elif n_splits <= 40:
 show_annot = True
 annot_fontsize = 5
 linewidths = 0.2
 elif n_splits <= 60:
 show_annot = True
 annot_fontsize = 4
 linewidths = 0.1
 else:
            # Too many splits - don't show text annotations
 show_annot = False
 linewidths = 0.05
 print(f" Note: Hiding cell annotations for {output_name} {metric_label} ({n_splits} splits - values would overlap)")

        # Create heatmap with adaptive settings
 im = sns.heatmap(pivot,
 annot=show_annot,
 fmt='.3f' if show_annot else None,
 cmap=cmap,
 vmin=vmin_val,
 vmax=vmax_val,
 ax=ax,
 cbar_kws={'label': metric_label},
 annot_kws={'fontsize': annot_fontsize} if show_annot else None,
 linewidths=linewidths,
 linecolor='white')

        # Labels
 ax.set_xlabel('Split Number', fontsize=13, fontweight='bold')
 ax.set_ylabel('Model', fontsize=13, fontweight='bold')

        # Title with info about number of splits
 title = f'{output_name} - {metric_label} Heatmap Across All Splits\n'
 title += f'({N_SPLITS} splits × {n_models} models)'
 ax.set_title(title, fontsize=15, fontweight='bold', pad=15)

        # Adjust tick label sizes based on number of splits
 if n_splits <= 20:
 x_labelsize = 10
 y_labelsize = 10
 elif n_splits <= 40:
 x_labelsize = 8
 y_labelsize = 9
 else:
 x_labelsize = 6
 y_labelsize = 8

 ax.tick_params(axis='x', labelsize=x_labelsize)
 ax.tick_params(axis='y', labelsize=y_labelsize)

        # Rotate x-axis labels if many splits
 if n_splits > 25:
 plt.setp(ax.get_xticklabels(), rotation=90, ha='center')

 plt.tight_layout()

 heatmap_file = os.path.join(viz_dir, f'{metric_name}_heatmap_{output_name}.png')
 plt.savefig(heatmap_file, dpi=300, bbox_inches='tight')
 plt.close()
 print(f" Saved: {metric_name}_heatmap_{output_name}.png")

# 4. Error bars plots (Mean ± Std) for each metric
print("\n4. Creating error bar plots for R², RMSE, and MAE...")

for metric_name, metric_label, direction, vmin, vmax in metrics_info:
 for output_name in output_columns:
 output_consistency = consistency_df[consistency_df['Output'] == output_name].copy()

 if direction == 'higher':
 output_consistency = output_consistency.sort_values(f'Mean_{metric_name}', ascending=False).head(10)
 else:
 output_consistency = output_consistency.sort_values(f'Mean_{metric_name}', ascending=True).head(10)

 fig, ax = plt.subplots(figsize=(12, 8))

 models = output_consistency['Model'].values
 means = output_consistency[f'Mean_{metric_name}'].values
 stds = output_consistency[f'Std_{metric_name}'].values

 x = np.arange(len(models))
 bars = ax.bar(x, means, yerr=stds, capsize=5, alpha=0.7,
 color='skyblue', edgecolor='darkblue', linewidth=1.5,
 error_kw={'linewidth': 2, 'ecolor': 'red'})

 ax.set_xlabel('Model', fontsize=12, fontweight='bold')
 ax.set_ylabel(f'{metric_label} (Mean ± Std)', fontsize=12, fontweight='bold')
 ax.set_title(f'{output_name} - {metric_label} with Uncertainty\n(Top 10 Models)',
 fontsize=14, fontweight='bold')
 ax.set_xticks(x)
 ax.set_xticklabels(models, rotation=45, ha='right')
 ax.grid(True, alpha=0.3, axis='y')

 if vmin is not None and vmax is not None:
 ax.set_ylim([vmin, vmax])

        # Add value labels
 for bar, mean, std in zip(bars, means, stds):
 height = bar.get_height()
 ax.text(bar.get_x() + bar.get_width()/2., height,
 f'{mean:.3f}±{std:.3f}',
 ha='center', va='bottom', fontsize=8, fontweight='bold')

 plt.tight_layout()

 errorbar_file = os.path.join(viz_dir, f'{metric_name}_errorbar_{output_name}.png')
 plt.savefig(errorbar_file, dpi=300, bbox_inches='tight')
 plt.close()
 print(f" Saved: {metric_name}_errorbar_{output_name}.png")

# 5. Coefficient of Variation comparison for all metrics
print("\n5. Creating coefficient of variation comparison...")

fig, axes = plt.subplots(len(output_columns), 3, figsize=(20, 7*len(output_columns)))
if len(output_columns) == 1:
 axes = axes.reshape(1, -1)

for out_idx, output_name in enumerate(output_columns):
 output_consistency = consistency_df[consistency_df['Output'] == output_name].copy()

 for metric_idx, (metric_name, metric_label, _, _, _) in enumerate(metrics_info):
 ax = axes[out_idx, metric_idx]

        # Sort by CV
 output_sorted = output_consistency.sort_values(f'CV_{metric_name}_%').head(10)

 models = output_sorted['Model'].values
 cv = output_sorted[f'CV_{metric_name}_%'].values

 colors = ['green' if x < 5 else 'orange' if x < 10 else 'red' for x in cv]

 bars = ax.barh(range(len(models)), cv, color=colors,
 edgecolor='black', linewidth=1.5)

 ax.set_yticks(range(len(models)))
 ax.set_yticklabels(models, fontsize=10)
 ax.set_xlabel('Coefficient of Variation (%)', fontsize=11, fontweight='bold')
 ax.set_title(f'{output_name} - {metric_label} CV\n(Lower = More Consistent)',
 fontsize=12, fontweight='bold')
 ax.axvline(x=5, color='green', linestyle='--', linewidth=1.5, alpha=0.5)
 ax.axvline(x=10, color='orange', linestyle='--', linewidth=1.5, alpha=0.5)
 ax.grid(True, alpha=0.3, axis='x')
 ax.invert_yaxis()

        # Add value labels
 for bar, val in zip(bars, cv):
 width = bar.get_width()
 ax.text(width + 0.5, bar.get_y() + bar.get_height()/2., f'{val:.2f}%',
 ha='left', va='center', fontsize=9, fontweight='bold')

plt.suptitle(f'Model Consistency Analysis: Coefficient of Variation ({N_SPLITS} Splits)',
 fontsize=16, fontweight='bold')
plt.tight_layout()

cv_file = os.path.join(viz_dir, 'CV_Comparison_All_Metrics.png')
plt.savefig(cv_file, dpi=300, bbox_inches='tight')
plt.close()
print(f" Saved: CV_Comparison_All_Metrics.png")

# 6. Comprehensive Summary Dashboard
print("\n6. Creating comprehensive summary dashboard...")

fig = plt.figure(figsize=(22, 16))
gs = fig.add_gridspec(4, 3, hspace=0.4, wspace=0.35)

fig.suptitle(f'Multi-Split Analysis: Comprehensive Summary ({N_SPLITS} Random Splits)',
 fontsize=20, fontweight='bold', y=0.98)

plot_idx = 0
for out_idx, output_name in enumerate(output_columns):
 for metric_idx, (metric_name, metric_label, direction, _, _) in enumerate(metrics_info):
 row = plot_idx // 3
 col = plot_idx % 3

 ax = fig.add_subplot(gs[row, col])

 output_consistency = consistency_df[consistency_df['Output'] == output_name].copy()

 if direction == 'higher':
 output_sorted = output_consistency.sort_values(f'Mean_{metric_name}', ascending=False).head(8)
 else:
 output_sorted = output_consistency.sort_values(f'Mean_{metric_name}', ascending=True).head(8)

 models = output_sorted['Model'].values
 means = output_sorted[f'Mean_{metric_name}'].values
 stds = output_sorted[f'Std_{metric_name}'].values

 x = np.arange(len(models))
 ax.bar(x, means, yerr=stds, capsize=3, alpha=0.7,
 color='lightblue', edgecolor='darkblue', linewidth=1.2)

 ax.set_ylabel(metric_label, fontsize=10, fontweight='bold')
 ax.set_title(f'{output_name} - {metric_label}\n(Top 8 by Mean)',
 fontsize=11, fontweight='bold')
 ax.set_xticks(x)
 ax.set_xticklabels(models, rotation=45, ha='right', fontsize=8)
 ax.grid(True, alpha=0.3, axis='y')

 plot_idx += 1

# Statistics table
ax_table = fig.add_subplot(gs[3, :])
ax_table.axis('off')

stats_data = [['Metric', 'Value']]
stats_data.append(['Number of Splits', str(N_SPLITS)])
stats_data.append(['Test Size per Split', f'{TEST_SIZE*100:.0f}%'])
stats_data.append(['Total Models Tested', str(len(ml_models_base))])
stats_data.append(['Total Trainings', str(N_SPLITS * len(ml_models_base) * len(output_columns))])

for output_name in output_columns:
 best_r2 = consistency_df[consistency_df['Output'] == output_name].nlargest(1, 'Mean_R2').iloc[0]
 best_rmse = consistency_df[consistency_df['Output'] == output_name].nsmallest(1, 'Mean_RMSE').iloc[0]
 best_mae = consistency_df[consistency_df['Output'] == output_name].nsmallest(1, 'Mean_MAE').iloc[0]

 stats_data.append([f'{output_name} - Best R²', f"{best_r2['Model']} ({best_r2['Mean_R2']:.4f})"])
 stats_data.append([f'{output_name} - Best RMSE', f"{best_rmse['Model']} ({best_rmse['Mean_RMSE']:.4f})"])
 stats_data.append([f'{output_name} - Best MAE', f"{best_mae['Model']} ({best_mae['Mean_MAE']:.4f})"])

table = ax_table.table(cellText=stats_data, cellLoc='left', loc='center',
 colWidths=[0.35, 0.65])
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 2.2)

for i in range(2):
 table[(0, i)].set_facecolor('#4A90E2')
 table[(0, i)].set_text_props(weight='bold', color='white')

for i in range(1, len(stats_data)):
 for j in range(2):
 if i % 2 == 0:
 table[(i, j)].set_facecolor('#F0F0F0')
 table[(i, j)].set_text_props(weight='bold', fontsize=9)

ax_table.set_title('Analysis Summary & Best Models', fontsize=13, fontweight='bold', pad=15)

plt.tight_layout()
dashboard_file = os.path.join(viz_dir, 'Comprehensive_Summary_Dashboard.png')
plt.savefig(dashboard_file, dpi=300, bbox_inches='tight')
plt.close()
print(f" Saved: Comprehensive_Summary_Dashboard.png")

# ==================== FINAL SUMMARY ====================

print("\n" + "="*80)
print(" " * 25 + "ANALYSIS COMPLETE")
print("="*80)

print(f"\n Main Results Directory: {RESULTS_DIR}")
print(f"\n Individual Split Results:")
print(f" {N_SPLITS} directories (Split_01 to Split_{N_SPLITS:02d})")

print(f"\n Consolidated Results: {consolidated_dir}")
print(f" • split_info.csv")
print(f" • train_metrics_all_splits.csv")
print(f" • test_metrics_all_splits.csv")
print(f" • consistency_analysis.csv (with R², RMSE, MAE statistics)")

print(f"\n Consistency Visualizations: {viz_dir}")
print(f" For each metric (R², RMSE, MAE):")
print(f" • Box plots")
print(f" • Trajectory plots")
print(f" • Heatmaps")
print(f" • Error bar plots")
print(f" Plus:")
print(f" • CV_Comparison_All_Metrics.png")
print(f" • Comprehensive_Summary_Dashboard.png")

print("\n" + "="*60)
print("KEY FINDINGS")
print("="*60)

for output_name in output_columns:
 print(f"\n{output_name.upper()}:")
 print("-" * 60)

 output_data = consistency_df[consistency_df['Output'] == output_name]

    # Best by R²
 best_r2 = output_data.nlargest(1, 'Mean_R2').iloc[0]
 print(f"\nBest R²: {best_r2['Model']}")
 print(f" Mean R²: {best_r2['Mean_R2']:.4f} ± {best_r2['Std_R2']:.4f}")
 print(f" Range: [{best_r2['Min_R2']:.4f}, {best_r2['Max_R2']:.4f}]")
 print(f" CV: {best_r2['CV_R2_%']:.2f}%")

    # Best by RMSE
 best_rmse = output_data.nsmallest(1, 'Mean_RMSE').iloc[0]
 print(f"\nBest RMSE: {best_rmse['Model']}")
 print(f" Mean RMSE: {best_rmse['Mean_RMSE']:.4f} ± {best_rmse['Std_RMSE']:.4f}")
 print(f" Range: [{best_rmse['Min_RMSE']:.4f}, {best_rmse['Max_RMSE']:.4f}]")
 print(f" CV: {best_rmse['CV_RMSE_%']:.2f}%")

    # Best by MAE
 best_mae = output_data.nsmallest(1, 'Mean_MAE').iloc[0]
 print(f"\nBest MAE: {best_mae['Model']}")
 print(f" Mean MAE: {best_mae['Mean_MAE']:.4f} ± {best_mae['Std_MAE']:.4f}")
 print(f" Range: [{best_mae['Min_MAE']:.4f}, {best_mae['Max_MAE']:.4f}]")
 print(f" CV: {best_mae['CV_MAE_%']:.2f}%")

    # Most Consistent
 most_consistent = output_data.nsmallest(1, 'CV_R2_%').iloc[0]
 print(f"\nMost Consistent (by R² CV): {most_consistent['Model']}")
 print(f" Mean R²: {most_consistent['Mean_R2']:.4f} ± {most_consistent['Std_R2']:.4f}")
 print(f" CV: {most_consistent['CV_R2_%']:.2f}%")

print("\n" + "="*80)
print(" All results saved successfully!")
print("="*80)


# 8. Validation

In [ ]:
# -*- coding: utf-8 -*-
"""
UF Water Treatment ML Code - Interactive Manual Validation ()
WITH CREATIVE VISUALIZATIONS and Beautiful Blue-to-Red Color Palette

Updated for: 7 inputs (glu, mlss, air, fm, cn, hrt, srt) and 3 outputs (tmp, flow, lv)
"""

# ==================== SECTION 8B: INTERACTIVE MANUAL VALIDATION ====================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, Wedge, Rectangle, FancyBboxPatch, PathPatch
from matplotlib.collections import PatchCollection
import matplotlib.patches as mpatches
from matplotlib.path import Path
import seaborn as sns
from matplotlib import cm
from scipy import stats
import joblib
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from tabulate import tabulate
from datetime import datetime

print("\n" + "="*80)
print(" " * 15 + "SECTION 8B: INTERACTIVE MANUAL VALIDATION")
print(" " * 20 + "UF Membrane System")
print("="*80)

# Disable grids globally
plt.rcParams['axes.grid'] = False

# ==================== BEAUTIFUL BLUE-TO-RED COLOR CONFIGURATION ====================

COLORS = {
    # Primary gradients: Beautiful blues
 'primary': ['#4facfe', '#00f2fe'],  # Light blue gradient
 'primary_dark': ['#667eea', '#764ba2'],  # Deep blue-purple

    # Success: Blue-cyan range
 'success': ['#00d2ff', '#3a7bd5'],  # Ocean blue gradient

    # Error/Warning: Red-orange range
 'error': ['#eb3349', '#f45c43'],  # Warm red gradient
 'warning': ['#ff6a88', '#ff99ac'],  # Soft pink-red

    # Info: Turquoise-blue range
 'info': ['#0ac8b9', '#00b4db'],  # Turquoise gradient

    # Accent: Purple-blue range
 'accent': ['#a8c0ff', '#3f2b96'],  # Purple-blue gradient

    # Neutrals
 'dark': '#2c3e50',
 'medium': '#34495e',
 'light': '#ecf0f1',
 'background': '#f8f9fa',

    # Blue-Red Spectrum (for heatmaps and gradients)
 'spectrum_blue': '#0066cc',  # Deep blue
 'spectrum_cyan': '#00b4d8',  # Cyan
 'spectrum_white': '#ffffff',  # White (neutral)
 'spectrum_orange': '#ff6b35',  # Orange
 'spectrum_red': '#e63946',  # Deep red
}

# Beautiful colormaps for UF system
BLUE_RED_CMAP = 'RdYlBu_r'  # Red-Yellow-Blue reversed (blue=good, red=bad)
DENSITY_CMAP = 'Blues'  # Blue density gradient (legacy)
PERFORMANCE_CMAP = 'coolwarm'  # Blue to red through white

# Note: For hexbin plots, we use 'YlGnBu_r' for better visibility
# This provides darker blues at low densities that are easier to see

# ==================== CONFIGURATION ====================

# Define column names for UF data
input_cols = ['glu', 'mlss', 'air', 'fm', 'cn', 'hrt', 'srt']
output_cols = ['tmp', 'flow', 'lv']

# Check if running in Colab
try:

 COLAB_AVAILABLE = True
except ImportError:
 COLAB_AVAILABLE = False

# Define paths
if COLAB_AVAILABLE:
 OUTPUT_DIR = BASE_DIR
else:
 OUTPUT_DIR = './UF_data_results'

VALIDATION_DIR = os.path.join(OUTPUT_DIR, 'Manual_Validation')
os.makedirs(VALIDATION_DIR, exist_ok=True)

print(f"\nConfiguration:")
print(f" Input features (7): {input_cols}")
print(f" Output targets (3): {output_cols}")
print(f" Output directory: {OUTPUT_DIR}")
print(f" Validation directory: {VALIDATION_DIR}")

# ==================== LOAD TRAINED MODELS AND FIXPOINT ====================

print("\n" + "="*60)
print("Loading Trained Models and Preprocessing Parameters")
print("="*60)

# Load fixpoint parameters
fixpoint_path = os.path.join(OUTPUT_DIR, 'fixpoint.pkl')
if not os.path.exists(fixpoint_path):
 print(f" ERROR: Fixpoint file not found: {fixpoint_path}")
 raise FileNotFoundError(f"Fixpoint file not found: {fixpoint_path}")

fix_point = joblib.load(fixpoint_path)
print(f" Loaded fixpoint parameters from: {fixpoint_path}")

# Load trained models
models_dir = os.path.join(OUTPUT_DIR, 'Trained_Models')
if not os.path.exists(models_dir):
 print(f" ERROR: Trained models directory not found: {models_dir}")
 raise FileNotFoundError(f"Trained models directory not found: {models_dir}")

# Load all trained models for each output
trained_models = {}

for output_name in output_cols:
 trained_models[output_name] = {}

    # List all model files for this output
 model_files = [f for f in os.listdir(models_dir) if f.endswith(f'_{output_name}.pkl')]

 for model_file in model_files:
        # Extract model name from filename
 model_name = model_file.replace(f'_{output_name}.pkl', '').replace('_', ' ')
 model_path = os.path.join(models_dir, model_file)

 try:
 trained_models[output_name][model_name] = joblib.load(model_path)
 except Exception as e:
 print(f" Error loading {model_file}: {str(e)}")

print(f"\n Loaded models:")
for output_name in output_cols:
 print(f" {output_name}: {len(trained_models[output_name])} models")

# ==================== INPUT METHODS ====================

def get_input_method():
 """Ask user how they want to provide input data."""

 print("\n" + "="*60)
 print("Input Method Selection")
 print("="*60)
 print("\nHow would you like to provide input data?")
 print(" 1. Single point (manual entry)")
 print(" 2. Multiple points (manual entry)")
 print(" 3. Load from CSV file")
 print("-" * 60)

 while True:
 choice = input("Enter your choice (1/2/3): ").strip()
 if choice == '1':
 return 'single'
 elif choice == '2':
 return 'manual_batch'
 elif choice == '3':
 return 'csv'
 else:
 print(" Invalid choice. Please enter 1, 2, or 3.")

def get_single_input(input_columns):
 """Get single manual input from user."""

 print("\n" + "="*60)
 print("Enter Input Values")
 print("="*60)

 input_data = {}

 print("\nPlease enter values for each input feature:")
 print("-" * 60)

 for col in input_columns:
 while True:
 try:
 value = float(input(f" {col}: "))
 input_data[col] = value
 break
 except ValueError:
 print(f" Invalid input. Please enter a numeric value.")

 input_df = pd.DataFrame([input_data])

 print("\n" + "="*60)
 print("Input Summary")
 print("="*60)
 print(tabulate(input_df, headers='keys', tablefmt='grid', showindex=False, floatfmt='.4f'))

 return input_df

def get_batch_manual_input(input_columns):
 """Get multiple manual inputs from user."""

 print("\n" + "="*60)
 print("Enter Multiple Input Points")
 print("="*60)

 while True:
 try:
 n_points = int(input("\nHow many data points do you want to enter? "))
 if n_points > 0:
 break
 else:
 print(" Please enter a positive number.")
 except ValueError:
 print(" Invalid input. Please enter a whole number.")

 all_inputs = []

 for i in range(n_points):
 print(f"\n{'='*60}")
 print(f"Data Point {i+1} of {n_points}")
 print('='*60)

 input_data = {}

 for col in input_columns:
 while True:
 try:
 value = float(input(f" {col}: "))
 input_data[col] = value
 break
 except ValueError:
 print(f" Invalid input. Please enter a numeric value.")

 all_inputs.append(input_data)

 input_df = pd.DataFrame(all_inputs)

 print("\n" + "="*60)
 print("Input Summary - All Data Points")
 print("="*60)
 print(tabulate(input_df, headers='keys', tablefmt='grid', showindex=True, floatfmt='.4f'))

 return input_df

def get_csv_input(input_columns):
 """Load input data from CSV file."""

 print("\n" + "="*60)
 print("Load Input Data from CSV")
 print("="*60)

 while True:
 file_path = input("\nEnter the full path to your CSV file: ").strip()

 if not os.path.exists(file_path):
 print(f" File not found: {file_path}")
 retry = input(" Try again? (yes/no): ").strip().lower()
 if retry not in ['yes', 'y']:
 return None
 continue

 try:
 df = pd.read_csv(file_path)

            # Check if all required columns exist
 missing_cols = [col for col in input_columns if col not in df.columns]
 if missing_cols:
 print(f" Missing required columns: {missing_cols}")
 print(f" Available columns: {df.columns.tolist()}")
 retry = input(" Try again? (yes/no): ").strip().lower()
 if retry not in ['yes', 'y']:
 return None
 continue

            # Extract only the required input columns
 input_df = df[input_columns].copy()

 print(f"\n Loaded {len(input_df)} data points from CSV")
 print("\n" + "="*60)
 print("Input Data Preview (first 10 rows)")
 print("="*60)
 print(tabulate(input_df.head(10), headers='keys', tablefmt='grid',
 showindex=True, floatfmt='.4f'))

 if len(input_df) > 10:
 print(f"\n ... and {len(input_df) - 10} more rows")

 return input_df

 except Exception as e:
 print(f" Error reading CSV: {str(e)}")
 retry = input(" Try again? (yes/no): ").strip().lower()
 if retry not in ['yes', 'y']:
 return None

# ==================== OUTPUT METHODS ====================

def get_batch_output(output_columns, n_points):
 """Get multiple actual output values from user."""

 print("\n" + "="*60)
 print("Enter Actual Output Values")
 print("="*60)

 all_outputs = []

 for i in range(n_points):
 print(f"\n{'='*60}")
 print(f"Actual Output for Data Point {i+1} of {n_points}")
 print('='*60)

 output_data = {}

 for col in output_columns:
 while True:
 try:
 value = float(input(f" {col}: "))
 output_data[col] = value
 break
 except ValueError:
 print(f" Invalid input. Please enter a numeric value.")

 all_outputs.append(output_data)

 output_df = pd.DataFrame(all_outputs)

 print("\n" + "="*60)
 print("Actual Output Summary - All Data Points")
 print("="*60)
 print(tabulate(output_df, headers='keys', tablefmt='grid', showindex=True, floatfmt='.4f'))

 return output_df

def get_csv_output(output_columns, n_points):
 """Load actual output data from CSV file or manual entry."""

 print("\n" + "="*60)
 print("Provide Actual Output Values")
 print("="*60)
 print("\nHow would you like to provide actual output values?")
 print(" 1. Enter manually")
 print(" 2. Load from CSV file")
 print("-" * 60)

 while True:
 choice = input("Enter your choice (1/2): ").strip()
 if choice == '1':
 return get_batch_output(output_columns, n_points)
 elif choice == '2':
 break
 else:
 print(" Invalid choice. Please enter 1 or 2.")

 while True:
 file_path = input("\nEnter the full path to your CSV file with actual outputs: ").strip()

 if not os.path.exists(file_path):
 print(f" File not found: {file_path}")
 retry = input(" Try again? (yes/no): ").strip().lower()
 if retry not in ['yes', 'y']:
 return None
 continue

 try:
 df = pd.read_csv(file_path)

            # Check if all required columns exist
 missing_cols = [col for col in output_columns if col not in df.columns]
 if missing_cols:
 print(f" Missing required columns: {missing_cols}")
 print(f" Available columns: {df.columns.tolist()}")
 retry = input(" Try again? (yes/no): ").strip().lower()
 if retry not in ['yes', 'y']:
 return None
 continue

            # Extract only the required output columns
 output_df = df[output_columns].copy()

            # Check if number of rows matches
 if len(output_df) != n_points:
 print(f" Row count mismatch: Expected {n_points}, got {len(output_df)}")
 use_anyway = input(f" Use first {n_points} rows anyway? (yes/no): ").strip().lower()
 if use_anyway in ['yes', 'y']:
 output_df = output_df.head(n_points)
 else:
 retry = input(" Try again? (yes/no): ").strip().lower()
 if retry not in ['yes', 'y']:
 return None
 continue

 print(f"\n Loaded {len(output_df)} actual output values from CSV")
 print("\n" + "="*60)
 print("Actual Output Preview")
 print("="*60)
 print(tabulate(output_df.head(10), headers='keys', tablefmt='grid',
 showindex=True, floatfmt='.4f'))

 return output_df

 except Exception as e:
 print(f" Error reading CSV: {str(e)}")
 retry = input(" Try again? (yes/no): ").strip().lower()
 if retry not in ['yes', 'y']:
 return None

# ==================== PREDICTION FUNCTIONS ====================

def predict_batch(input_df, trained_models, fix_point, input_columns, output_columns):
 """Make predictions for batch of inputs using all trained models."""

 print("\n" + "="*60)
 print("Generating Predictions from All Models")
 print("="*60)

    # Convert input to array
 X_input = input_df[input_columns].values
 n_points = len(X_input)

    # Standardize input
 X_input_scaled = (X_input - fix_point['Input_mean']) / fix_point['Input_std']

 predictions_dict = {}

 for output_idx, output_name in enumerate(output_columns):
 print(f"\n Processing {output_name}...")

 predictions_dict[output_name] = {}

 model_count = 0
 for model_name, model in trained_models[output_name].items():
 try:
                # Make predictions (on scaled data)
 pred_scaled = model.predict(X_input_scaled)

                # Transform back to original scale
 pred_original = (pred_scaled * fix_point['Output_std'][output_idx] +
 fix_point['Output_mean'][output_idx])

 predictions_dict[output_name][model_name] = pred_original
 model_count += 1

 except Exception as e:
 print(f" Error with {model_name}: {str(e)}")
 predictions_dict[output_name][model_name] = None

 print(f" Generated predictions from {model_count} models for {n_points} points")

 print("\n All predictions generated")

 return predictions_dict

def calculate_batch_errors(predictions_dict, actual_df, output_columns):
 """Calculate errors for batch predictions."""

 print("\n" + "="*60)
 print("Calculating Prediction Errors")
 print("="*60)

 error_results = {}
 all_summary_stats = []

 for output_name in output_columns:
 print(f"\n Calculating errors for {output_name}...")

 actual_values = actual_df[output_name].values
 error_results[output_name] = {}

 for model_name, pred_values in predictions_dict[output_name].items():
 if pred_values is not None:
 errors = pred_values - actual_values
 abs_errors = np.abs(errors)
 percent_errors = (abs_errors / (np.abs(actual_values) + 1e-10)) * 100

                # Calculate aggregate metrics
 mae = mean_absolute_error(actual_values, pred_values)
 rmse = np.sqrt(mean_squared_error(actual_values, pred_values))
 r2 = r2_score(actual_values, pred_values)

 error_results[output_name][model_name] = {
 'predictions': pred_values,
 'errors': errors,
 'abs_errors': abs_errors,
 'percent_errors': percent_errors,
 'MAE': mae,
 'RMSE': rmse,
 'R²': r2,
 'mean_error': errors.mean(),
 'std_error': errors.std()
 }

 all_summary_stats.append({
 'Output': output_name,
 'Model': model_name,
 'MAE': mae,
 'RMSE': rmse,
 'R²': r2,
 'Mean_Error': errors.mean(),
 'Std_Error': errors.std()
 })

 print(f" {model_name}: MAE={mae:.4f}, RMSE={rmse:.4f}, R²={r2:.4f}")

 summary_df = pd.DataFrame(all_summary_stats)

 print("\n All errors calculated")

 return error_results, summary_df

def create_batch_summary_tables(summary_df, output_columns):
 """Create and display summary tables for batch prediction errors."""

 print("\n" + "="*60)
 print("Prediction Error Summary")
 print("="*60)

 for output_name in output_columns:
 output_summary = summary_df[summary_df['Output'] == output_name].copy()
 output_summary = output_summary.sort_values(by='MAE')

 print(f"\n{output_name}:")
 print("-" * 60)
 print(tabulate(output_summary[['Model', 'MAE', 'RMSE', 'R²', 'Mean_Error', 'Std_Error']],
 headers='keys', tablefmt='grid', showindex=False, floatfmt='.4f'))

        # Identify best model
 best_model = output_summary.iloc[0]
 print(f"\n Best Model: {best_model['Model']}")
 print(f" MAE: {best_model['MAE']:.4f}")
 print(f" RMSE: {best_model['RMSE']:.4f}")
 print(f" R²: {best_model['R²']:.4f}")

# ==================== CREATIVE VISUALIZATION FUNCTIONS ====================

def visualize_batch_results(input_df, predictions_dict, actual_df, error_results,
 output_columns, summary_df, save_dir):
 """Create comprehensive visualizations for batch predictions - CREATIVE VERSION with Blue-Red Palette."""

 print("\n" + "="*60)
 print(" Creating Creative Visualizations (Blue-Red Palette)")
 print("="*60)

    # Get best model for each output
 best_models = {}
 for output_name in output_columns:
 output_summary = summary_df[summary_df['Output'] == output_name]
 best_model_name = output_summary.loc[output_summary['MAE'].idxmin(), 'Model']
 best_models[output_name] = best_model_name

    # ========================================================================
    # 1. SCATTER PLOT with ERROR TOLERANCE BANDS and METRICS BELOW
    # ========================================================================
 print("\n Creating scatter plots with error tolerance bands...")

    # Create figure with all plots in one row
 fig, axes = plt.subplots(1, len(output_columns), figsize=(7*len(output_columns), 7))
 if len(output_columns) == 1:
 axes = [axes]

 fig.suptitle(' Model Performance Dashboard - UF System\nPrediction Quality Analysis with Error Tolerance Bands',
 fontsize=20, fontweight='bold', y=0.98, color=COLORS['dark'])

 for idx, output_name in enumerate(output_columns):
 ax = axes[idx]

 best_model = best_models[output_name]
 actual = actual_df[output_name].values
 predicted = predictions_dict[output_name][best_model]

        # Get metrics for this output
 metrics = error_results[output_name][best_model]
 r2_value = metrics["R²"]
 mae_value = metrics["MAE"]
 rmse_value = metrics["RMSE"]

        # Calculate proper axis limits (same for both x and y to maintain equal aspect)
 min_val = min(actual.min(), predicted.min())
 max_val = max(actual.max(), predicted.max())
 data_range = max_val - min_val
 margin = data_range * 0.05

 plot_min = min_val - margin
 plot_max = max_val + margin

        # Create x values for bands using the PLOT range (not data range)
 x_line = np.linspace(plot_min, plot_max, 100)

        # ±20% Error Band (outer, lighter)
 y_upper_20 = x_line * 1.20
 y_lower_20 = x_line * 0.80
 ax.fill_between(x_line, y_lower_20, y_upper_20,
 color='#ffcdd2', alpha=0.3,
 label='±20% Error', zorder=1)

        # ±10% Error Band (inner, darker)
 y_upper_10 = x_line * 1.10
 y_lower_10 = x_line * 0.90
 ax.fill_between(x_line, y_lower_10, y_upper_10,
 color='#ef9a9a', alpha=0.4,
 label='±10% Error', zorder=2)

        # Perfect prediction line
 ax.plot(x_line, x_line,
 color='#2c3e50', linestyle='-', linewidth=3, alpha=0.9,
 zorder=3, label='Perfect Prediction')

        # Scatter plot with uniform blue color (no error coloring)
 scatter = ax.scatter(actual, predicted,
 c='#1976d2', s=80, alpha=0.6,
 edgecolors='white', linewidth=0.8,
 zorder=4)

 clean_name = output_name.replace('_', ' ').title()
 ax.set_title(f'{clean_name} - {best_model}',
 fontsize=14, fontweight='bold', color=COLORS['dark'], pad=15)
 ax.set_xlabel('Actual Values', fontsize=12, fontweight='bold', color=COLORS['dark'])
 ax.set_ylabel('Predicted Values', fontsize=12, fontweight='bold', color=COLORS['dark'])

        # CRITICAL: Set same limits for both axes
 ax.set_xlim(plot_min, plot_max)
 ax.set_ylim(plot_min, plot_max)

        # Set equal aspect ratio
 ax.set_aspect('equal', adjustable='box')
 ax.grid(True, alpha=0.3, linestyle=':', color='gray')
 ax.set_facecolor('#f8f9fa')
 ax.legend(loc='upper left', fontsize=9, framealpha=0.95,
 edgecolor='gray', fancybox=True)

        # Count points within error bands
 percent_errors = np.abs((predicted - actual) / (np.abs(actual) + 1e-10)) * 100
 within_10 = np.sum(percent_errors <= 10) / len(percent_errors) * 100
 within_20 = np.sum(percent_errors <= 20) / len(percent_errors) * 100

        # Add metrics text box below the plot with error band statistics
 metrics_text = (f'R² = {r2_value:.4f} | MAE = {mae_value:.4f} | RMSE = {rmse_value:.4f}\n'
 f'Within ±10%: {within_10:.1f}% | Within ±20%: {within_20:.1f}%')

        # Position text box at bottom center of plot
 ax.text(0.5, -0.18, metrics_text,
 transform=ax.transAxes,
 fontsize=11, fontweight='bold',
 ha='center', va='top',
 bbox=dict(boxstyle='round,pad=0.8',
 facecolor='#e3f2fd',
 edgecolor='#1976d2',
 linewidth=2.5,
 alpha=0.95),
 color=COLORS['dark'],
 zorder=10)

    # Adjust layout to accommodate two-line metrics text below plots
 plt.tight_layout(rect=[0, 0.08, 1, 0.96])
 plt.savefig(f'{save_dir}/Best_Models_Scatter.png', dpi=300, bbox_inches='tight', facecolor='white')
 plt.close()
 print(" Saved: Best_Models_Scatter.png")

    # ========================================================================
    # 2. RADIAL BAR CHART
    # ========================================================================
 print("\n Creating radial bar chart for model comparison...")

 fig, axes = plt.subplots(1, 2, figsize=(18, 9), subplot_kw=dict(projection='polar'))
 fig.suptitle(' Model Performance Comparison - Radial Visualization\nUF Membrane System',
 fontsize=22, fontweight='bold', y=0.98, color=COLORS['dark'])

 for metric_idx, (metric, ax) in enumerate([('MAE', axes[0]), ('R²', axes[1])]):
 metric_data = []
 for output_name in output_columns:
 output_summary = summary_df[summary_df['Output'] == output_name].copy()
 output_summary = output_summary.sort_values(by=metric, ascending=(metric != 'R²'))
 metric_data.append(output_summary.head(8))

 all_models = pd.concat(metric_data)['Model'].unique()[:10]
 n_models = len(all_models)
 angles = np.linspace(0, 2 * np.pi, n_models, endpoint=False).tolist()
 bar_width = (2 * np.pi) / n_models * 0.25

        # Beautiful blue gradient colors for outputs
 colors_output = ['#4facfe', '#00f2fe', '#0ac8b9'][:len(output_columns)]

 for out_idx, output_name in enumerate(output_columns):
 values = []
 for model in all_models:
 model_data = summary_df[(summary_df['Output'] == output_name) &
 (summary_df['Model'] == model)]
 if len(model_data) > 0:
 val = model_data[metric].values[0]
 if metric == 'R²':
 values.append(val)
 else:
 max_val = summary_df[summary_df['Output'] == output_name][metric].max()
 values.append(1 - (val / max_val))
 else:
 values.append(0)

 bar_angles = [a + out_idx * bar_width for a in angles]
 ax.bar(bar_angles, values, width=bar_width, color=colors_output[out_idx],
 alpha=0.8, edgecolor='white', linewidth=2,
 label=output_name.replace('_', ' ').title())

 ax.set_theta_zero_location('N')
 ax.set_theta_direction(-1)
 ax.set_xticks(angles)
 ax.set_xticklabels(all_models, fontsize=9)
 ax.set_ylim(0, 1)
 ax.set_yticks([0.25, 0.5, 0.75, 1.0])
 ax.set_yticklabels(['0.25', '0.5', '0.75', '1.0'], fontsize=8, color=COLORS['medium'])
 ax.grid(True, linestyle=':', alpha=0.5, color='#b2bec3')
 ax.set_facecolor('#fafafa')

 metric_name = 'Mean Absolute Error (Inverted)' if metric == 'MAE' else 'R² Score'
 ax.set_title(f'{metric_name}', fontsize=14, fontweight='bold', color=COLORS['dark'], pad=20)
 ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=10, framealpha=0.9)

 plt.tight_layout()
 plt.savefig(f'{save_dir}/Performance_Heatmap.png', dpi=300, bbox_inches='tight', facecolor='white')
 plt.close()
 print(" Saved: Performance_Heatmap.png")

    # ========================================================================
    # 3. VIOLIN PLOT + SWARM PLOT
    # ========================================================================
 print("\n Creating violin + swarm plot for error distribution...")

 fig, axes = plt.subplots(1, len(output_columns), figsize=(6*len(output_columns), 8))
 if len(output_columns) == 1:
 axes = [axes]

 fig.suptitle(' Prediction Error Distribution Analysis - UF System',
 fontsize=22, fontweight='bold', y=0.98, color=COLORS['dark'])

 for idx, output_name in enumerate(output_columns):
 ax = axes[idx]
 best_model = best_models[output_name]
 errors = error_results[output_name][best_model]['errors']

 parts = ax.violinplot([errors], positions=[0], widths=0.7,
 showmeans=True, showextrema=True, showmedians=True)

        # Blue violin body
 for pc in parts['bodies']:
 pc.set_facecolor('#74b9ff')
 pc.set_edgecolor('#0984e3')
 pc.set_alpha(0.7)
 pc.set_linewidth(2)

 for partname in ('cbars', 'cmins', 'cmaxes', 'cmedians', 'cmeans'):
 if partname in parts:
 parts[partname].set_edgecolor(COLORS['dark'])
 parts[partname].set_linewidth(2)

 np.random.seed(42)
 y_jitter = np.random.normal(0, 0.04, size=len(errors))

        # Blue-to-red color mapping for errors
 scatter = ax.scatter(y_jitter, errors, c=np.abs(errors), cmap=BLUE_RED_CMAP,
 s=50, alpha=0.6, edgecolors='white', linewidth=0.5, zorder=3)

 ax.axhline(y=0, color='#0984e3', linestyle='--', linewidth=3, alpha=0.8,
 label='Zero Error Target', zorder=1)
 ax.axhline(y=errors.mean(), color='#00b894', linestyle='-', linewidth=2.5,
 alpha=0.8, label=f'Mean = {errors.mean():.4f}', zorder=1)

 ci_95 = 1.96 * errors.std()
 ax.axhspan(-ci_95, ci_95, alpha=0.15, color='#74b9ff', zorder=0, label='95% CI')

 clean_name = output_name.replace('_', ' ').title()
 ax.set_title(f'{clean_name}\n{best_model}', fontsize=14, fontweight='bold',
 color=COLORS['dark'], pad=15)
 ax.set_ylabel('Prediction Error', fontsize=12, fontweight='bold', color=COLORS['dark'])
 ax.set_xticks([])
 ax.set_xlim(-0.5, 0.5)
 ax.grid(True, axis='y', alpha=0.3, linestyle=':', color='#b2bec3')
 ax.set_axisbelow(True)
 ax.set_facecolor('#fafafa')
 ax.legend(loc='upper right', fontsize=9, framealpha=0.9)

 stats_text = (f'Statistics:\n'
 f'Mean: {errors.mean():.4f}\n'
 f'Std: {errors.std():.4f}\n'
 f'Median: {np.median(errors):.4f}\n'
 f'IQR: {np.percentile(errors, 75) - np.percentile(errors, 25):.4f}\n'
 f'Skewness: {stats.skew(errors):.3f}')

 ax.text(0.02, 0.98, stats_text, transform=ax.transAxes, fontsize=9,
 verticalalignment='top',
 bbox=dict(boxstyle='round,pad=0.6', facecolor='#dfe6e9',
 edgecolor=COLORS['medium'], linewidth=2, alpha=0.9),
 fontfamily='monospace', color=COLORS['dark'])

 cbar = plt.colorbar(scatter, ax=ax, pad=0.02, aspect=30)
 cbar.set_label('Absolute Error', fontsize=9, fontweight='bold', rotation=270, labelpad=15)

 plt.tight_layout()
 plt.savefig(f'{save_dir}/Error_Distribution.png', dpi=300, bbox_inches='tight', facecolor='white')
 plt.close()
 print(" Saved: Error_Distribution.png")

    # ========================================================================
    # 4. 3D PERFORMANCE LANDSCAPE
    # ========================================================================
 print("\n Creating 3D performance landscape...")

 from mpl_toolkits.mplot3d import Axes3D

    # Adjust layout based on number of outputs
 if len(output_columns) == 3:
 fig = plt.figure(figsize=(18, 12))
 ax1 = fig.add_subplot(231, projection='3d')
 ax2 = fig.add_subplot(232, projection='3d')
 ax3 = fig.add_subplot(233, projection='3d')
 ax4 = fig.add_subplot(234)
 ax5 = fig.add_subplot(235)
 ax6 = fig.add_subplot(236)
 residual_axes = [ax4, ax5, ax6]
 else:
 fig = plt.figure(figsize=(16, 10))
 ax1 = fig.add_subplot(221, projection='3d')
 ax2 = fig.add_subplot(222, projection='3d')
 ax3 = fig.add_subplot(223)
 ax4 = fig.add_subplot(224)
 residual_axes = [ax3, ax4]

 fig.suptitle(' 3D Performance Landscape & Prediction Quality\nUF Membrane System',
 fontsize=20, fontweight='bold', y=0.98, color=COLORS['dark'])

 for idx, output_name in enumerate(output_columns):
 if idx < 3:
 ax_3d = [ax1, ax2, ax3][idx]

 best_model = best_models[output_name]
 actual = actual_df[output_name].values
 predicted = predictions_dict[output_name][best_model]
 errors = error_results[output_name][best_model]['errors']

 scatter = ax_3d.scatter(actual, predicted, np.abs(errors), c=np.abs(errors),
 cmap=BLUE_RED_CMAP, s=100, alpha=0.7,
 edgecolors='white', linewidth=0.5)

 x_range = np.linspace(actual.min(), actual.max(), 10)
 y_range = np.linspace(predicted.min(), predicted.max(), 10)
 X, Y = np.meshgrid(x_range, y_range)
 Z = np.zeros_like(X)
 ax_3d.plot_surface(X, Y, Z, alpha=0.2, color='#0984e3')

 clean_name = output_name.replace('_', ' ').title()
 ax_3d.set_title(f'{clean_name}\n{best_model}', fontsize=12,
 fontweight='bold', color=COLORS['dark'], pad=10)
 ax_3d.set_xlabel('Actual', fontsize=10, fontweight='bold')
 ax_3d.set_ylabel('Predicted', fontsize=10, fontweight='bold')
 ax_3d.set_zlabel('|Error|', fontsize=10, fontweight='bold')
 ax_3d.view_init(elev=20, azim=45)

 cbar = plt.colorbar(scatter, ax=ax_3d, pad=0.1, shrink=0.8)
 cbar.set_label('Absolute Error', fontsize=9, rotation=270, labelpad=15)

        # Residual plot
 if idx < len(residual_axes):
 ax_2d = residual_axes[idx]
 best_model = best_models[output_name]
 predicted = predictions_dict[output_name][best_model]
 errors = error_results[output_name][best_model]['errors']

 scatter_2d = ax_2d.scatter(predicted, errors, c=np.abs(errors), cmap=BLUE_RED_CMAP,
 s=80, alpha=0.7, edgecolors='white', linewidth=1)

 ax_2d.axhline(y=0, color='#0984e3', linestyle='--', linewidth=2, alpha=0.8)

 std_error = errors.std()
 ax_2d.axhspan(-std_error, std_error, alpha=0.15, color='#74b9ff', label='±1σ Std Dev')
 ax_2d.axhspan(-2*std_error, 2*std_error, alpha=0.08, color='#74b9ff', label='±2σ Std Dev')

 z = np.polyfit(predicted, errors, 1)
 p = np.poly1d(z)
 ax_2d.plot(predicted, p(predicted), color=COLORS['error'][0], alpha=0.5, linewidth=2,
 label=f'Trend (slope={z[0]:.4f})')

 clean_name = output_name.replace('_', ' ').title()
 ax_2d.set_title(f'Residual Plot - {clean_name}', fontsize=12,
 fontweight='bold', color=COLORS['dark'], pad=10)
 ax_2d.set_xlabel('Predicted Values', fontsize=10, fontweight='bold')
 ax_2d.set_ylabel('Residuals', fontsize=10, fontweight='bold')
 ax_2d.grid(True, alpha=0.3, linestyle=':', color='#b2bec3')
 ax_2d.set_facecolor('#fafafa')
 ax_2d.legend(fontsize=8, loc='best', framealpha=0.9)

 cbar_2d = plt.colorbar(scatter_2d, ax=ax_2d, pad=0.02)
 cbar_2d.set_label('|Error|', fontsize=9, rotation=270, labelpad=15)

 plt.tight_layout()
 plt.savefig(f'{save_dir}/Model_Ranking.png', dpi=300, bbox_inches='tight', facecolor='white')
 plt.close()
 print(" Saved: Model_Ranking.png")

 print("\n All visualizations created successfully!")

def save_batch_results(input_df, predictions_dict, actual_df, error_results,
 summary_df, input_columns, output_columns, save_dir):
 """Save batch prediction results to CSV files."""

 print("\n" + "="*60)
 print("Saving Results")
 print("="*60)

 timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    # Save detailed predictions for each output
 for output_name in output_columns:
 results_df = input_df.copy()

 if actual_df is not None:
 results_df[f'Actual_{output_name}'] = actual_df[output_name]

 for model_name, predictions in predictions_dict[output_name].items():
 if predictions is not None:
 safe_model = model_name.replace(' ', '_').replace('(', '').replace(')', '')
 results_df[f'Pred_{safe_model}'] = predictions

 if actual_df is not None:
 results_df[f'Error_{safe_model}'] = (predictions -
 actual_df[output_name].values)

 safe_name = output_name.replace('.', '_')
 filename = f'{save_dir}/Predictions_{safe_name}_{timestamp}.csv'
 results_df.to_csv(filename, index=False)
 print(f" Saved: Predictions_{safe_name}_{timestamp}.csv")

    # Save summary statistics
 if summary_df is not None:
 summary_file = f'{save_dir}/Summary_{timestamp}.csv'
 summary_df.to_csv(summary_file, index=False)
 print(f" Saved: Summary_{timestamp}.csv")

 print("\n All results saved")

# ==================== MAIN INTERACTIVE LOOP ====================

def run_interactive_validation():
 """Run the interactive manual validation system."""

 while True:
 print("\n" + "="*80)
 print(" " * 25 + "NEW PREDICTION SESSION")
 print(" " * 28 + "UF System")
 print("="*80)

        # Step 1: Get input method
 method = get_input_method()

        # Step 2: Get input data
 if method == 'single':
 input_df = get_single_input(input_cols)
 elif method == 'manual_batch':
 input_df = get_batch_manual_input(input_cols)
 elif method == 'csv':
 input_df = get_csv_input(input_cols)

 if input_df is None or len(input_df) == 0:
 print("\n No input data provided. Exiting...")
 break

        # Step 3: Generate predictions
 predictions_dict = predict_batch(
 input_df, trained_models, fix_point, input_cols, output_cols
 )

        # Step 4: Ask if user wants to compare with actual values
 print("\n" + "="*60)
 while True:
 compare = input("Do you want to enter actual output values for comparison? (yes/no): ").strip().lower()
 if compare in ['yes', 'y', 'no', 'n']:
 break
 print(" Please enter 'yes' or 'no'")

 if compare in ['yes', 'y']:
            # Step 5: Get actual output values
 actual_df = get_csv_output(output_cols, len(input_df))

 if actual_df is not None:
                # Step 6: Calculate errors
 error_results, summary_df = calculate_batch_errors(
 predictions_dict, actual_df, output_cols
 )

                # Step 7: Display summary
 create_batch_summary_tables(summary_df, output_cols)

                # Step 8: Create visualizations
 visualize_batch_results(input_df, predictions_dict, actual_df,
 error_results, output_cols, summary_df, VALIDATION_DIR)

                # Step 9: Save results
 save_batch_results(input_df, predictions_dict, actual_df,
 error_results, summary_df, input_cols, output_cols, VALIDATION_DIR)
 else:
 print("\n No actual output data provided. Skipping comparison...")
 save_batch_results(input_df, predictions_dict, None, None, None,
 input_cols, output_cols, VALIDATION_DIR)
 else:
            # Save predictions without comparison
 save_batch_results(input_df, predictions_dict, None, None, None,
 input_cols, output_cols, VALIDATION_DIR)

        # Ask if user wants to make another prediction
 print("\n" + "="*60)
 while True:
 another = input("Do you want to make another prediction? (yes/no): ").strip().lower()
 if another in ['yes', 'y', 'no', 'n']:
 break
 print(" Please enter 'yes' or 'no'")

 if another in ['no', 'n']:
 break

 print("\n" + "="*80)
 print(" " * 20 + "SECTION 8B COMPLETE")
 print("="*80)
 print("\n Interactive validation session ended")
 print(f" All results saved in: {VALIDATION_DIR}")

# ==================== EXECUTE ====================

print("\n" + "="*60)
print("Starting Interactive Validation System - UF Membrane")
print("="*60)
print("\nThis system allows you to:")
print(" • Enter single or multiple data points manually")
print(" • Load data from CSV files")
print(" • Get predictions from all trained models")
print(" • Compare with actual values (optional)")
print(" • View CREATIVE performance metrics with Blue-Red palette")
print("\n Color Scheme: Blue = Good Performance, Red = Poor Performance")

run_interactive_validation()


# 10. Optimal conditions

In [ ]:
# -*- coding: utf-8 -*-
"""
UF Water Treatment ML Code - Optimal Operating Conditions 

Key Improvements:
 1. Input bounds constrained to OBSERVED data range (no extrapolation)
 2. SHAP analysis computed WITHIN the feasible region
 3. ALE (Accumulated Local Effects) plots for handling correlated features
 4. Side-by-side comparison: Pearson vs SHAP vs ALE sensitivity rankings
 5. Enhanced 2D feasible region plots with DENSITY visualization

UF System:
 Input features (7): glu, mlss, air, fm, cn, hrt, srt
 Output features (3): tmp, flow, lv

Author:
Date: 2026-02-15
"""

# ==================== IMPORTS ====================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
from scipy.stats import qmc
from scipy.ndimage import gaussian_filter
from tabulate import tabulate
from itertools import combinations
from tqdm import tqdm
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print("\n" + "="*80)
print(" " * 10 + "SECTION 9 (): OPTIMAL OPERATING CONDITIONS")
print(" " * 20 + "UF Membrane System")
print("="*80)

plt.rcParams['axes.grid'] = False

# ==================== DIRECTORY SETUP ====================

def create_optimal_directories(base_dir):
 """Create organized directory structure."""
 dirs = {
 'base': base_dir,
 'ranges': f'{base_dir}/1_Input_Ranges',
 'sensitivity': f'{base_dir}/2_Sensitivity_Analysis',
 'feasible_regions': f'{base_dir}/3_Feasible_Regions',
 'distributions': f'{base_dir}/4_Distributions',
 'heatmaps': f'{base_dir}/5_Correlation_Heatmaps',
 'csv': f'{base_dir}/6_CSV_Data',
 'shap_feasible': f'{base_dir}/7_SHAP_Feasible_Region',
 'ale': f'{base_dir}/8_ALE_Plots',
 'comparison': f'{base_dir}/9_Method_Comparison',
 }
 for dir_path in dirs.values():
 os.makedirs(dir_path, exist_ok=True)
 return dirs

# ==================== FIX 1: OBSERVED DATA BOUNDS ====================

def get_observed_data_bounds(input_cols, fix_point, X_train_standardized):
 """
 FIX 1: Use OBSERVED data range instead of mean ± N*std.

 This ensures all sampled points fall within the model's validated
 operating envelope, eliminating extrapolation artifacts.

 Parameters:
 input_cols: list of input feature names
 fix_point: dict with 'Input_mean' and 'Input_std' arrays
 X_train_standardized: the standardized training data (n_samples, n_features)

 Returns:
 bounds: list of (min, max) tuples in ORIGINAL scale
 bounds_info: DataFrame with bound details for reporting
 """
 print("\n" + "="*60)
 print("FIX 1: OBSERVED DATA BOUNDS (No Extrapolation)")
 print("="*60)

    # Inverse-standardize to get original-scale training data
 X_original = X_train_standardized * fix_point['Input_std'] + fix_point['Input_mean']

 bounds = []
 rows = []

 for i, col in enumerate(input_cols):
 obs_min = X_original[:, i].min()
 obs_max = X_original[:, i].max()
 obs_mean = X_original[:, i].mean()
 obs_std = X_original[:, i].std()

        # Use observed min/max as bounds
 bounds.append((obs_min, obs_max))

        # Also compute what the old method would have used (for comparison)
 old_lo = max(0.0, obs_mean - 3 * obs_std)
 old_hi = obs_mean + 3 * obs_std

 rows.append({
 'Feature': col,
 'Observed_Min': obs_min,
 'Observed_Max': obs_max,
 'Old_Lo (mean-3σ)': old_lo,
 'Old_Hi (mean+3σ)': old_hi,
 'Extrapolation_Lo': 'YES' if old_lo < obs_min else 'no',
 'Extrapolation_Hi': 'YES' if old_hi > obs_max else 'no',
 })

 print(f" {col:<15}: [{obs_min:.3f}, {obs_max:.3f}]"
 f" (old: [{old_lo:.3f}, {old_hi:.3f}]"
 f"{' EXTRAP' if old_lo < obs_min or old_hi > obs_max else ''})")

 bounds_info = pd.DataFrame(rows)

    # Count extrapolation issues
 n_extrap = sum(1 for r in rows
 if r['Extrapolation_Lo'] == 'YES' or r['Extrapolation_Hi'] == 'YES')
 print(f"\n {n_extrap} features had extrapolation in old method → now fixed")
 print(f" All bounds constrained to observed training data range")

 return bounds, bounds_info

# ==================== USER INPUT FUNCTIONS ====================

def get_target_output_ranges(output_cols):
 """Get target output ranges from user."""
 print("\n" + "="*60)
 print("DEFINE TARGET OUTPUT RANGES")
 print("="*60)
 print("\nFor UF membrane system:")
 print(" tmp: Transmembrane Pressure")
 print(" flow: Permeate Flow Rate")
 print(" lv: Level/Volume")

 target_ranges = {}
 for out in output_cols:
 while True:
 try:
 print(f"\n{out}:")
 mn = float(input(" Minimum value: "))
 mx = float(input(" Maximum value: "))
 if mn >= mx:
 print(" min must be < max")
 continue
 target_ranges[out] = {'min': mn, 'max': mx}
 break
 except ValueError:
 print(" Invalid number")

 print("\n Target output ranges:")
 for k, v in target_ranges.items():
 print(f" {k}: [{v['min']}, {v['max']}]")

 return target_ranges

def select_models(trained_models, output_cols):
 """
 Allow user to select which model to use for each output.
 Handles the structure: trained_models[model_name][output_name] = model
 """
 print("\n" + "="*60)
 print("MODEL SELECTION")
 print("="*60)

    # Reorganize by output
 models_by_output = {}
 for output_name in output_cols:
 models_by_output[output_name] = []
 for model_name in trained_models.keys():
 if output_name in trained_models[model_name]:
 models_by_output[output_name].append(model_name)

 selected = {}
 for out in output_cols:
 models = models_by_output[out]

 if not models:
 print(f"\n ERROR: No trained models found for {out}")
 raise ValueError(f"No models available for {out}")

 print(f"\n{out} available models:")
 for i, m in enumerate(models, 1):
 print(f" {i}. {m}")

 while True:
 try:
 idx = int(input("Select model number: ")) - 1
 if 0 <= idx < len(models):
 selected[out] = models[idx]
 break
 except:
 pass
 print(" Invalid selection")

 print("\n Selected models:")
 for k, v in selected.items():
 print(f" {k}: {v}")

 return selected

# ==================== FEASIBILITY SAMPLER ====================

def sample_feasible_space(
 trained_models, selected_models, target_ranges,
 input_bounds, fix_point, input_cols, output_cols,
 n_samples=500_000
):
 """
 Sample the input space using Latin Hypercube Sampling
 and find feasible regions satisfying target constraints.
 """
 print("\n" + "="*60)
 print("SEARCHING FEASIBLE INPUT SPACE")
 print("="*60)
 print(f"Sampling {n_samples:,} points within observed data bounds...")

    # Latin Hypercube Sampling
 sampler = qmc.LatinHypercube(d=len(input_cols))
 U = sampler.random(n_samples)

    # Scale to input bounds (observed range only)
 X = np.array([
 input_bounds[i][0] + U[:, i] * (input_bounds[i][1] - input_bounds[i][0])
 for i in range(len(input_cols))
 ]).T

 feasible_X, feasible_Y = [], []
 feasible_X_std = []  # Also store standardized version for SHAP

 for x in tqdm(X, desc="Feasibility check", unit="samples"):
        # Standardize
 xs = (x - fix_point['Input_mean']) / fix_point['Input_std']
 xs = xs.reshape(1, -1)

 ys = []
 valid = True
 for j, out in enumerate(output_cols):
 model_name = selected_models[out]
 model_data = trained_models[model_name][out]

            # Handle dict wrapper
 if isinstance(model_data, dict) and 'model' in model_data:
 model = model_data['model']
 else:
 model = model_data

 y = model.predict(xs)[0]
 y = y * fix_point['Output_std'][j] + fix_point['Output_mean'][j]

 if not (target_ranges[out]['min'] <= y <= target_ranges[out]['max']):
 valid = False
 break
 ys.append(y)

 if valid:
 feasible_X.append(x)
 feasible_Y.append(ys)
 feasible_X_std.append(xs.flatten())

 feasible_X = np.array(feasible_X) if feasible_X else np.array([]).reshape(0, len(input_cols))
 feasible_Y = np.array(feasible_Y) if feasible_Y else np.array([]).reshape(0, len(output_cols))
 feasible_X_std = np.array(feasible_X_std) if feasible_X_std else np.array([]).reshape(0, len(input_cols))

 print(f"\n Feasible samples found: {len(feasible_X):,} / {n_samples:,}"
 f" ({100*len(feasible_X)/n_samples:.2f}%)")

 if len(feasible_X) == 0:
 print("\n No feasible points found!")
 print(" Suggestions:")
 print(" • Widen the target ranges")
 print(" • The current operating conditions may not achieve all targets simultaneously")

 return feasible_X, feasible_Y, feasible_X_std

# ==================== ANALYSIS FUNCTIONS ====================

def compute_ranges(feasible_X, input_cols):
 """Compute statistical ranges for feasible inputs."""
 rows = []
 for i, col in enumerate(input_cols):
 rows.append({
 'Input': col,
 'Min': feasible_X[:, i].min(),
 'P05': np.percentile(feasible_X[:, i], 5),
 'Mean': feasible_X[:, i].mean(),
 'Median': np.median(feasible_X[:, i]),
 'P95': np.percentile(feasible_X[:, i], 95),
 'Max': feasible_X[:, i].max(),
 'Std': feasible_X[:, i].std()
 })
 return pd.DataFrame(rows)

def pearson_sensitivity(feasible_X, feasible_Y, input_cols, output_cols):
 """Original Pearson correlation-based sensitivity (kept for comparison)."""
 rows = []
 for i, col in enumerate(input_cols):
 for j, out in enumerate(output_cols):
 corr = np.corrcoef(feasible_X[:, i], feasible_Y[:, j])[0, 1]
 rows.append({
 'Input': col, 'Output': out,
 'Correlation': corr, 'Abs_Correlation': abs(corr)
 })
 df = pd.DataFrame(rows)
 return df.sort_values('Abs_Correlation', ascending=False)

def extract_boundary_points(feasible_X, input_cols, k=3):
 """Extract extreme boundary points."""
 rows = []
 for i, col in enumerate(input_cols):
 idx_min = np.argsort(feasible_X[:, i])[:k]
 idx_max = np.argsort(feasible_X[:, i])[-k:]
 for idx in idx_min:
 rows.append({'Input': col, 'Type': 'Minimum', 'Value': feasible_X[idx, i]})
 for idx in idx_max:
 rows.append({'Input': col, 'Type': 'Maximum', 'Value': feasible_X[idx, i]})
 return pd.DataFrame(rows)

# ==================== FIX 2: SHAP WITHIN FEASIBLE REGION ====================

def shap_within_feasible_region(
 trained_models, selected_models, feasible_X_std,
 input_cols, output_cols, fix_point, dirs,
 max_shap_samples=2000
):
 """
 FIX 2: Compute SHAP values WITHIN the feasible region only.
 """
 print("\n" + "="*60)
 print("FIX 2: SHAP ANALYSIS WITHIN FEASIBLE REGION")
 print("="*60)

 if len(feasible_X_std) == 0:
 print(" No feasible points ,  skipping SHAP analysis")
 return None

    # Subsample if too many points
 if len(feasible_X_std) > max_shap_samples:
 np.random.seed(42)
 idx = np.random.choice(len(feasible_X_std), max_shap_samples, replace=False)
 X_shap = feasible_X_std[idx]
 print(f" Subsampled {max_shap_samples} from {len(feasible_X_std)} feasible points")
 else:
 X_shap = feasible_X_std
 print(f" Using all {len(X_shap)} feasible points")

 shap_results = {}

 for out in output_cols:
 print(f"\n Computing SHAP for {out}...")
 model_name = selected_models[out]
 model_data = trained_models[model_name][out]

        # Handle dict wrapper
 if isinstance(model_data, dict) and 'model' in model_data:
 model = model_data['model']
 else:
 model = model_data

        # Create SHAP explainer
 try:
 explainer = shap.TreeExplainer(model)
 shap_values = explainer.shap_values(X_shap)
 print(f" Used TreeExplainer (fast)")
 except Exception as e:
 print(f" TreeExplainer failed ({e}), using KernelExplainer...")
 bg = shap.kmeans(X_shap, min(50, len(X_shap)))
 explainer = shap.KernelExplainer(model.predict, bg)
 shap_values = explainer.shap_values(X_shap, nsamples=100)
 print(f" Used KernelExplainer")

        # Compute mean |SHAP| importance
 mean_abs_shap = np.abs(shap_values).mean(axis=0)
 importance_df = pd.DataFrame({
 'Feature': input_cols,
 'Mean_Abs_SHAP_Feasible': mean_abs_shap
 }).sort_values('Mean_Abs_SHAP_Feasible', ascending=False)

 shap_results[out] = {
 'shap_values': shap_values,
 'X_shap': X_shap,
 'importance': importance_df,
 'explainer': explainer,
 }

 print(f" SHAP Feature Importance ({out}, feasible region):")
 for _, row in importance_df.iterrows():
 print(f" {row['Feature']:<15}: {row['Mean_Abs_SHAP_Feasible']:.6f}")

        # --- SHAP Beeswarm Plot ---
 fig, ax = plt.subplots(figsize=(12, 6))
 shap.summary_plot(
 shap_values, X_shap,
 feature_names=input_cols,
 show=False, plot_size=None
 )
 plt.title(f'SHAP Beeswarm (Feasible Region)\n{model_name} - {out}',
 fontsize=14, fontweight='bold')
 plt.tight_layout()
 plt.savefig(f"{dirs['shap_feasible']}/{out}_Beeswarm_Feasible.png",
 dpi=300, bbox_inches='tight')
 plt.close()
 print(f" Saved: {out}_Beeswarm_Feasible.png")

        # --- SHAP Feature Importance Bar ---
 fig, ax = plt.subplots(figsize=(10, 6))
 shap.summary_plot(
 shap_values, X_shap,
 feature_names=input_cols,
 plot_type='bar',
 show=False, plot_size=None
 )
 plt.title(f'SHAP Feature Importance (Feasible Region)\n{model_name} - {out}',
 fontsize=14, fontweight='bold')
 plt.tight_layout()
 plt.savefig(f"{dirs['shap_feasible']}/{out}_FeatureImportance_Feasible.png",
 dpi=300, bbox_inches='tight')
 plt.close()
 print(f" Saved: {out}_FeatureImportance_Feasible.png")

        # --- SHAP Partial Dependence Plots ---
 n_features = len(input_cols)
 n_cols_plot = 4
 n_rows_plot = (n_features + n_cols_plot - 1) // n_cols_plot

 fig, axes = plt.subplots(n_rows_plot, n_cols_plot, figsize=(20, 5 * n_rows_plot))
 axes = axes.flatten()

        # Destandardize X_shap for plotting
 X_shap_original = X_shap * fix_point['Input_std'] + fix_point['Input_mean']

        # Sort features by importance
 sorted_features = importance_df['Feature'].tolist()
 sorted_indices = [input_cols.index(f) for f in sorted_features]

 for plot_idx, feat_idx in enumerate(sorted_indices):
 ax = axes[plot_idx]
 feat_name = input_cols[feat_idx]
 rank = plot_idx + 1
 mean_shap_val = mean_abs_shap[feat_idx]

 ax.scatter(
 X_shap_original[:, feat_idx],
 shap_values[:, feat_idx],
 alpha=0.4, s=15, c='blue', edgecolors='none'
 )
 ax.axhline(y=0, color='red', linestyle='--', linewidth=1.5, alpha=0.7)
 ax.set_xlabel(f'{feat_name} (Original Scale)', fontsize=10, fontweight='bold')
 ax.set_ylabel('SHAP Value', fontsize=10, fontweight='bold')
 ax.set_title(f'{feat_name}\n(Rank #{rank}, Mean |SHAP|: {mean_shap_val:.4f})',
 fontsize=11, fontweight='bold')
 ax.tick_params(labelsize=9)

        # Hide unused axes
 for idx in range(len(sorted_indices), len(axes)):
 axes[idx].set_visible(False)

 plt.suptitle(f'SHAP Partial Dependence (Feasible Region): {model_name} - {out}',
 fontsize=16, fontweight='bold', y=1.01)
 plt.tight_layout()
 plt.savefig(f"{dirs['shap_feasible']}/{out}_PartialDependence_Feasible.png",
 dpi=300, bbox_inches='tight')
 plt.close()
 print(f" Saved: {out}_PartialDependence_Feasible.png")

 return shap_results

# ==================== FIX 3: ALE PLOTS ====================

def compute_ale(model, X, feature_idx, n_bins=40):
 """
 Compute Accumulated Local Effects for a single feature.
 """
 feature_values = X[:, feature_idx]

    # Create bin edges using quantiles
 percentiles = np.linspace(0, 100, n_bins + 1)
 bin_edges = np.unique(np.percentile(feature_values, percentiles))

 if len(bin_edges) < 3:
 return np.array([feature_values.mean()]), np.array([0.0])

 local_effects = []
 bin_counts = []
 bin_centers = []

 for i in range(len(bin_edges) - 1):
 lo, hi = bin_edges[i], bin_edges[i + 1]

 if i == len(bin_edges) - 2:
 mask = (feature_values >= lo) & (feature_values <= hi)
 else:
 mask = (feature_values >= lo) & (feature_values < hi)

 if mask.sum() == 0:
 continue

 X_bin = X[mask].copy()

 X_upper = X_bin.copy()
 X_upper[:, feature_idx] = hi
 pred_upper = model.predict(X_upper)

 X_lower = X_bin.copy()
 X_lower[:, feature_idx] = lo
 pred_lower = model.predict(X_lower)

 local_effect = np.mean(pred_upper - pred_lower)
 local_effects.append(local_effect)
 bin_counts.append(mask.sum())
 bin_centers.append((lo + hi) / 2)

 if len(local_effects) == 0:
 return np.array([feature_values.mean()]), np.array([0.0])

 ale_values = np.cumsum(local_effects)

    # Center
 total_samples = sum(bin_counts)
 weighted_mean = sum(a * c for a, c in zip(ale_values, bin_counts)) / total_samples
 ale_values = ale_values - weighted_mean

 return np.array(bin_centers), ale_values

def plot_ale_all_features(
 trained_models, selected_models, feasible_X_std,
 input_cols, output_cols, fix_point, dirs,
 n_bins=40
):
 """
 FIX 3: Create ALE plots for all features within the feasible region.
 """
 print("\n" + "="*60)
 print("FIX 3: ALE (ACCUMULATED LOCAL EFFECTS) ANALYSIS")
 print("="*60)

 if len(feasible_X_std) == 0:
 print(" No feasible points ,  skipping ALE analysis")
 return None

 ale_results = {}

 for out in output_cols:
 print(f"\n Computing ALE for {out}...")
 model_name = selected_models[out]
 model_data = trained_models[model_name][out]

        # Handle dict wrapper
 if isinstance(model_data, dict) and 'model' in model_data:
 model = model_data['model']
 else:
 model = model_data

 ale_data = {}
 ale_importances = {}

 for i, col in enumerate(input_cols):
 bin_centers_std, ale_vals = compute_ale(model, feasible_X_std, i, n_bins)

            # Convert to original scale
 bin_centers_orig = bin_centers_std * fix_point['Input_std'][i] + fix_point['Input_mean'][i]

 j = output_cols.index(out)
 ale_vals_orig = ale_vals * fix_point['Output_std'][j]

 ale_data[col] = {
 'bin_centers': bin_centers_orig,
 'ale_values': ale_vals_orig,
 }

 ale_importances[col] = np.std(ale_vals_orig) if len(ale_vals_orig) > 1 else 0.0

 sorted_features = sorted(ale_importances.keys(),
 key=lambda x: ale_importances[x], reverse=True)

        # --- Plot ALE for all features ---
 n_features = len(input_cols)
 n_cols_plot = 4
 n_rows_plot = (n_features + n_cols_plot - 1) // n_cols_plot

 fig, axes = plt.subplots(n_rows_plot, n_cols_plot, figsize=(20, 5 * n_rows_plot))
 axes = axes.flatten()

 for plot_idx, feat_name in enumerate(sorted_features):
 ax = axes[plot_idx]
 d = ale_data[feat_name]
 importance = ale_importances[feat_name]

 ax.plot(d['bin_centers'], d['ale_values'],
 'b-', linewidth=2.5, alpha=0.8)
 ax.fill_between(d['bin_centers'], 0, d['ale_values'],
 alpha=0.15, color='blue')
 ax.axhline(y=0, color='red', linestyle='--', linewidth=1.5, alpha=0.7)

 ax.set_xlabel(f'{feat_name} (Original Scale)', fontsize=10, fontweight='bold')
 ax.set_ylabel(f'ALE ({out})', fontsize=10, fontweight='bold')
 ax.set_title(f'{feat_name}\n(Rank #{plot_idx+1}, ALE Importance: {importance:.4f})',
 fontsize=11, fontweight='bold')
 ax.tick_params(labelsize=9)

 for idx in range(len(sorted_features), len(axes)):
 axes[idx].set_visible(False)

 plt.suptitle(f'ALE Plots (Feasible Region): {model_name} - {out}\n'
 f'Accumulated Local Effects handle correlated features',
 fontsize=15, fontweight='bold', y=1.02)
 plt.tight_layout()
 plt.savefig(f"{dirs['ale']}/{out}_ALE_Feasible.png",
 dpi=300, bbox_inches='tight')
 plt.close()
 print(f" Saved: {out}_ALE_Feasible.png")

        # --- ALE Feature Importance Bar Chart ---
 fig, ax = plt.subplots(figsize=(10, 6))
 sorted_imp = sorted(ale_importances.items(), key=lambda x: x[1], reverse=True)
 names = [x[0] for x in sorted_imp]
 values = [x[1] for x in sorted_imp]

 ax.barh(names[::-1], values[::-1], color='steelblue',
 edgecolor='black', linewidth=1.5, alpha=0.7)
 ax.set_xlabel('ALE Importance (Std of ALE values)', fontsize=12, fontweight='bold')
 ax.set_title(f'ALE Feature Importance (Feasible Region)\n{model_name} - {out}',
 fontsize=14, fontweight='bold')
 ax.tick_params(labelsize=11)
 plt.tight_layout()
 plt.savefig(f"{dirs['ale']}/{out}_ALE_Importance_Feasible.png",
 dpi=300, bbox_inches='tight')
 plt.close()
 print(f" Saved: {out}_ALE_Importance_Feasible.png")

 ale_results[out] = {
 'ale_data': ale_data,
 'importances': ale_importances,
 'sorted_features': sorted_features,
 }

 return ale_results

# ==================== METHOD COMPARISON ====================

def compare_all_methods(
 pearson_df, shap_results, ale_results,
 input_cols, output_cols, dirs
):
 """
 Create side-by-side comparison of all three sensitivity methods.
 """
 print("\n" + "="*60)
 print("METHOD COMPARISON: Pearson vs SHAP vs ALE")
 print("="*60)

 comparison_dfs = {}

 for out in output_cols:
 print(f"\n {out}:")

 rows = []
 for col in input_cols:
            # Pearson
 pearson_row = pearson_df[
 (pearson_df['Input'] == col) & (pearson_df['Output'] == out)
 ]
 pearson_corr = pearson_row['Correlation'].values[0] if len(pearson_row) > 0 else 0
 pearson_abs = abs(pearson_corr)

            # SHAP feasible
 if shap_results and out in shap_results:
 imp_df = shap_results[out]['importance']
 shap_row = imp_df[imp_df['Feature'] == col]
 shap_val = shap_row['Mean_Abs_SHAP_Feasible'].values[0] if len(shap_row) > 0 else 0
 else:
 shap_val = 0

            # ALE
 if ale_results and out in ale_results:
 ale_val = ale_results[out]['importances'].get(col, 0)
 else:
 ale_val = 0

 rows.append({
 'Feature': col,
 'Pearson_Corr': pearson_corr,
 'Pearson_Abs': pearson_abs,
 'SHAP_Feasible': shap_val,
 'ALE_Importance': ale_val,
 })

 comp_df = pd.DataFrame(rows)

        # Add ranks
 comp_df['Pearson_Rank'] = comp_df['Pearson_Abs'].rank(ascending=False).astype(int)
 comp_df['SHAP_Rank'] = comp_df['SHAP_Feasible'].rank(ascending=False).astype(int)
 comp_df['ALE_Rank'] = comp_df['ALE_Importance'].rank(ascending=False).astype(int)

 comp_df['Rank_Variance'] = comp_df[['Pearson_Rank', 'SHAP_Rank', 'ALE_Rank']].var(axis=1)
 comp_df['Agreement'] = comp_df['Rank_Variance'].apply(
 lambda v: ' Strong' if v < 1.0 else ('~ Partial' if v < 3.0 else ' Divergent')
 )

 comp_df = comp_df.sort_values('SHAP_Rank')
 comparison_dfs[out] = comp_df

 print(tabulate(
 comp_df[['Feature', 'Pearson_Rank', 'SHAP_Rank', 'ALE_Rank',
 'Pearson_Corr', 'SHAP_Feasible', 'ALE_Importance', 'Agreement']],
 headers='keys', tablefmt='grid', showindex=False, floatfmt=".4f"
 ))

    # --- Comparison Visualization ---
 fig, axes = plt.subplots(1, len(output_cols), figsize=(10 * len(output_cols), 8))
 if len(output_cols) == 1:
 axes = [axes]

 for idx, out in enumerate(output_cols):
 ax = axes[idx]
 comp_df = comparison_dfs[out].sort_values('SHAP_Rank')

 features = comp_df['Feature'].values
 x = np.arange(len(features))
 width = 0.25

 p_vals = comp_df['Pearson_Abs'].values
 s_vals = comp_df['SHAP_Feasible'].values
 a_vals = comp_df['ALE_Importance'].values

 p_norm = p_vals / p_vals.max() if p_vals.max() > 0 else p_vals
 s_norm = s_vals / s_vals.max() if s_vals.max() > 0 else s_vals
 a_norm = a_vals / a_vals.max() if a_vals.max() > 0 else a_vals

 bars1 = ax.bar(x - width, p_norm, width, label='Pearson |Corr|',
 color='coral', edgecolor='black', linewidth=1.5, alpha=0.7)
 bars2 = ax.bar(x, s_norm, width, label='SHAP (Feasible)',
 color='steelblue', edgecolor='black', linewidth=1.5, alpha=0.7)
 bars3 = ax.bar(x + width, a_norm, width, label='ALE Importance',
 color='seagreen', edgecolor='black', linewidth=1.5, alpha=0.7)

 ax.set_xlabel('Feature', fontsize=13, fontweight='bold')
 ax.set_ylabel('Normalized Importance (0-1)', fontsize=13, fontweight='bold')
 ax.set_title(f'{out}: Method Comparison\n(Pearson vs SHAP vs ALE)',
 fontsize=14, fontweight='bold')
 ax.set_xticks(x)
 ax.set_xticklabels(features, rotation=45, ha='right', fontsize=11)
 ax.legend(fontsize=11)
 ax.tick_params(labelsize=11)

 plt.tight_layout()
 plt.savefig(f"{dirs['comparison']}/Method_Comparison_Bar.png",
 dpi=300, bbox_inches='tight')
 plt.close()
 print(f"\n Saved: Method_Comparison_Bar.png")

    # --- Rank Comparison Heatmap ---
 for out in output_cols:
 comp_df = comparison_dfs[out].sort_values('SHAP_Rank')

 fig, ax = plt.subplots(figsize=(8, 6))

 rank_matrix = comp_df[['Pearson_Rank', 'SHAP_Rank', 'ALE_Rank']].values.T
 features = comp_df['Feature'].values

 im = ax.imshow(rank_matrix, cmap='YlOrRd', aspect='auto')

 ax.set_xticks(np.arange(len(features)))
 ax.set_yticks(np.arange(3))
 ax.set_xticklabels(features, rotation=45, ha='right', fontsize=11)
 ax.set_yticklabels(['Pearson', 'SHAP', 'ALE'], fontsize=12, fontweight='bold')

 for i in range(3):
 for j in range(len(features)):
 ax.text(j, i, f'{int(rank_matrix[i, j])}',
 ha='center', va='center', fontsize=12, fontweight='bold',
 color='white' if rank_matrix[i, j] > 4 else 'black')

 ax.set_title(f'{out}: Feature Rank Comparison\n(1=Most Important, {len(features)}=Least)',
 fontsize=14, fontweight='bold')
 plt.colorbar(im, ax=ax, label='Rank')
 plt.tight_layout()
 plt.savefig(f"{dirs['comparison']}/{out}_Rank_Heatmap.png",
 dpi=300, bbox_inches='tight')
 plt.close()
 print(f" Saved: {out}_Rank_heatmap.png")

 return comparison_dfs

# ==================== ENHANCED 2D FEASIBLE REGIONS WITH DENSITY ====================

def plot_2d_feasible_regions_with_density(feasible_X, input_cols, dirs):
 """
 : Create 2D scatter plots with DENSITY visualization.

 This shows both:
 1. Individual points (the individual observations)
 2. Density gradient (so high-density regions are clearly visible)

 Uses hexbin with alpha blending for best visualization.
 """
 print("\n Creating ENHANCED 2D feasible region plots with density...")

 pairs = list(combinations(range(len(input_cols)), 2))
 n_pairs = len(pairs)

 print(f" Generating {n_pairs} pairwise density plots...")

 n_cols = 5
 n_rows = (n_pairs + n_cols - 1) // n_cols

 fig = plt.figure(figsize=(20, 4 * n_rows))
 fig.suptitle(f'2D Feasible Regions with Density - All Pairwise Combinations ({n_pairs} pairs)\n'
 f'UF System - Color intensity shows point concentration',
 fontsize=20, fontweight='bold', y=0.998)

 for plot_idx, (i, j) in enumerate(pairs):
 ax = plt.subplot(n_rows, n_cols, plot_idx + 1)

        # METHOD 1: Hexbin (most effective for showing density)
        # Gridsize controls resolution - higher = more detail
 hb = ax.hexbin(feasible_X[:, i], feasible_X[:, j],
 gridsize=30, cmap='Blues', mincnt=1,
 edgecolors='none', alpha=0.8, linewidths=0.2)

        # Add colorbar for this subplot to show density scale
 cb = plt.colorbar(hb, ax=ax)
 cb.set_label('Point Density', fontsize=9)
 cb.ax.tick_params(labelsize=8)

        # Optional: Overlay scatter plot with very low alpha to show individual points
        # This makes it clear these are real data points
 ax.scatter(feasible_X[:, i], feasible_X[:, j],
 s=0.5, alpha=0.1, c='darkblue', edgecolors='none')

 ax.set_xlabel(input_cols[i], fontsize=11, fontweight='bold')
 ax.set_ylabel(input_cols[j], fontsize=11, fontweight='bold')
 ax.set_title(f'{input_cols[i]} vs {input_cols[j]}',
 fontsize=12, fontweight='bold')
 ax.tick_params(axis='both', labelsize=10)

 plt.tight_layout()
 plt.savefig(f"{dirs['feasible_regions']}/Feasible_Regions_2D_Density.png",
 dpi=300, bbox_inches='tight')
 plt.close()
 print(" Saved: Feasible_Regions_2D_Density.png")
 print(" Note: Color intensity shows where feasible points concentrate")

# ==================== ORIGINAL VISUALIZATION FUNCTIONS ====================

def plot_input_range_bars(ranges_df, dirs):
 """Create bar charts showing feasible ranges for each input."""
 fig, axes = plt.subplots(2, 4, figsize=(20, 10))
 axes = axes.flatten()
 for idx, row in ranges_df.iterrows():
 ax = axes[idx]
 ax.barh([0], [row['Max'] - row['Min']], left=row['Min'],
 height=0.3, color='lightblue', edgecolor='darkblue', linewidth=2)
 ax.axvline(row['P05'], color='orange', linewidth=2, linestyle='--', label='P05')
 ax.axvline(row['P95'], color='red', linewidth=2, linestyle='--', label='P95')
 ax.axvline(row['Mean'], color='green', linewidth=2, linestyle='-', label='Mean')
 ax.set_xlabel('Value', fontsize=11, fontweight='bold')
 ax.set_title(f"{row['Input']}", fontsize=13, fontweight='bold')
 ax.set_yticks([])
 ax.legend(fontsize=9, loc='upper right')
 ax.tick_params(axis='x', labelsize=10)
 axes[-1].axis('off')  # Hide last empty subplot
 plt.suptitle('Feasible Input Ranges - UF System', fontsize=18, fontweight='bold', y=0.995)
 plt.tight_layout()
 plt.savefig(f"{dirs['ranges']}/Input_Ranges_Bars.png", dpi=300, bbox_inches='tight')
 plt.close()

def plot_sensitivity_bars(sens_df, output_cols, dirs):
 """Create sensitivity bar charts."""
 fig, axes = plt.subplots(1, len(output_cols), figsize=(10*len(output_cols), 8))
 if len(output_cols) == 1: axes = [axes]
 fig.suptitle('Sensitivity Analysis (Correlation-Based) - UF System',
 fontsize=18, fontweight='bold')
 for idx, out in enumerate(output_cols):
 ax = axes[idx]
 out_data = sens_df[sens_df['Output'] == out].sort_values('Abs_Correlation', ascending=True)
 colors = ['green' if c > 0 else 'red' for c in out_data['Correlation']]
 ax.barh(out_data['Input'], out_data['Correlation'], color=colors,
 edgecolor='black', linewidth=1.5, alpha=0.7)
 ax.axvline(0, color='black', linewidth=2)
 ax.set_xlabel('Correlation', fontsize=13, fontweight='bold')
 ax.set_title(f'{out}', fontsize=15, fontweight='bold')
 ax.tick_params(axis='both', labelsize=11)
 plt.tight_layout()
 plt.savefig(f"{dirs['sensitivity']}/Sensitivity_Bars.png", dpi=300, bbox_inches='tight')
 plt.close()

def plot_input_distributions(feasible_X, input_cols, dirs):
 """Create distribution plots."""
 fig, axes = plt.subplots(2, 4, figsize=(20, 10))
 axes = axes.flatten()
 for idx, col in enumerate(input_cols):
 ax = axes[idx]
 data = feasible_X[:, idx]
 ax.hist(data, bins=50, color='skyblue', edgecolor='darkblue', linewidth=1, alpha=0.7)
 ax.axvline(data.mean(), color='red', linewidth=2.5, linestyle='--',
 label=f'Mean: {data.mean():.2f}')
 ax.axvline(np.median(data), color='green', linewidth=2.5, linestyle='--',
 label=f'Median: {np.median(data):.2f}')
 ax.set_xlabel('Value', fontsize=11, fontweight='bold')
 ax.set_ylabel('Frequency', fontsize=11, fontweight='bold')
 ax.set_title(col, fontsize=13, fontweight='bold')
 ax.legend(fontsize=9)
 ax.tick_params(axis='both', labelsize=10)
 axes[-1].axis('off')
 plt.suptitle('Distribution of Feasible Input Values - UF System',
 fontsize=18, fontweight='bold', y=0.995)
 plt.tight_layout()
 plt.savefig(f"{dirs['distributions']}/Input_Distributions.png",
 dpi=300, bbox_inches='tight')
 plt.close()

def plot_output_distributions(feasible_Y, output_cols, target_ranges, dirs):
 """Create output distribution plots."""
 fig, axes = plt.subplots(1, len(output_cols), figsize=(10*len(output_cols), 6))
 if len(output_cols) == 1: axes = [axes]
 fig.suptitle('Distribution of Predicted Outputs in Feasible Region - UF System',
 fontsize=18, fontweight='bold')
 for idx, col in enumerate(output_cols):
 ax = axes[idx]
 data = feasible_Y[:, idx]
 target = target_ranges[col]
 ax.hist(data, bins=50, color='lightgreen', edgecolor='darkgreen', linewidth=1, alpha=0.7)
 ax.axvline(target['min'], color='red', linewidth=3, linestyle='--',
 label=f"Target Min: {target['min']}")
 ax.axvline(target['max'], color='red', linewidth=3, linestyle='--',
 label=f"Target Max: {target['max']}")
 ax.axvline(data.mean(), color='blue', linewidth=2.5, linestyle='-',
 label=f'Mean: {data.mean():.2f}')
 ax.set_xlabel('Value', fontsize=13, fontweight='bold')
 ax.set_ylabel('Frequency', fontsize=13, fontweight='bold')
 ax.set_title(col, fontsize=15, fontweight='bold')
 ax.legend(fontsize=10)
 ax.tick_params(axis='both', labelsize=11)
 plt.tight_layout()
 plt.savefig(f"{dirs['distributions']}/Output_Distributions.png",
 dpi=300, bbox_inches='tight')
 plt.close()

def plot_correlation_heatmap(feasible_X, feasible_Y, input_cols, output_cols, dirs):
 """Create correlation heatmap."""
 n_inputs, n_outputs = len(input_cols), len(output_cols)
 corr_matrix = np.zeros((n_inputs, n_outputs))
 for i in range(n_inputs):
 for j in range(n_outputs):
 corr_matrix[i, j] = np.corrcoef(feasible_X[:, i], feasible_Y[:, j])[0, 1]
 fig, ax = plt.subplots(figsize=(10, 8))
 im = ax.imshow(corr_matrix, cmap='RdBu_r', aspect='auto', vmin=-1, vmax=1)
 ax.set_xticks(np.arange(n_outputs))
 ax.set_yticks(np.arange(n_inputs))
 ax.set_xticklabels(output_cols, fontsize=12, fontweight='bold')
 ax.set_yticklabels(input_cols, fontsize=11)
 plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")
 cbar = plt.colorbar(im, ax=ax)
 cbar.set_label('Correlation', fontsize=13, fontweight='bold', rotation=270, labelpad=20)
 for i in range(n_inputs):
 for j in range(n_outputs):
 ax.text(j, i, f'{corr_matrix[i, j]:.3f}', ha="center", va="center",
 color="black", fontsize=10, fontweight='bold')
 ax.set_title('Input-Output Correlation Heatmap\n(Feasible Region - UF System)',
 fontsize=16, fontweight='bold', pad=15)
 plt.tight_layout()
 plt.savefig(f"{dirs['heatmaps']}/Correlation_Heatmap.png", dpi=300, bbox_inches='tight')
 plt.close()

def plot_3d_feasible_space(feasible_X, input_cols, dirs):
 """Create 3D scatter plot."""
 if len(input_cols) >= 3:
 fig = plt.figure(figsize=(12, 10))
 ax = fig.add_subplot(111, projection='3d')
 ax.scatter(feasible_X[:, 0], feasible_X[:, 1], feasible_X[:, 2],
 c='blue', marker='o', s=1, alpha=0.3)
 ax.set_xlabel(input_cols[0], fontsize=12, fontweight='bold')
 ax.set_ylabel(input_cols[1], fontsize=12, fontweight='bold')
 ax.set_zlabel(input_cols[2], fontsize=12, fontweight='bold')
 ax.set_title('3D Feasible Space - UF System', fontsize=16, fontweight='bold')
 plt.tight_layout()
 plt.savefig(f"{dirs['feasible_regions']}/Feasible_Space_3D.png",
 dpi=300, bbox_inches='tight')
 plt.close()

# ==================== CSV EXPORT ====================

def save_all_results(
 ranges_df, pearson_df, boundary_df, bounds_info,
 feasible_X, feasible_Y, comparison_dfs,
 shap_results, ale_results,
 input_cols, output_cols,
 target_ranges, selected_models, dirs
):
 """Save all results to CSV."""
 print("\n Saving results to CSV...")
 timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

 ranges_df.to_csv(f"{dirs['csv']}/Input_Ranges_{timestamp}.csv", index=False)
 pearson_df.to_csv(f"{dirs['csv']}/Sensitivity_Pearson_{timestamp}.csv", index=False)
 bounds_info.to_csv(f"{dirs['csv']}/Bounds_Comparison_{timestamp}.csv", index=False)
 boundary_df.to_csv(f"{dirs['csv']}/Boundary_Points_{timestamp}.csv", index=False)

 feasible_df = pd.DataFrame(feasible_X, columns=input_cols)
 for idx, col in enumerate(output_cols):
 feasible_df[f'Predicted_{col}'] = feasible_Y[:, idx]
 feasible_df.to_csv(f"{dirs['csv']}/Feasible_Points_{timestamp}.csv", index=False)

 if comparison_dfs:
 for out, comp_df in comparison_dfs.items():
 comp_df.to_csv(f"{dirs['csv']}/Method_Comparison_{out}_{timestamp}.csv", index=False)

 if shap_results:
 for out in output_cols:
 if out in shap_results:
 shap_results[out]['importance'].to_csv(
 f"{dirs['csv']}/SHAP_Feasible_{out}_{timestamp}.csv", index=False)

 if ale_results:
 for out in output_cols:
 if out in ale_results:
 ale_df = pd.DataFrame([
 {'Feature': k, 'ALE_Importance': v}
 for k, v in ale_results[out]['importances'].items()
 ]).sort_values('ALE_Importance', ascending=False)
 ale_df.to_csv(f"{dirs['csv']}/ALE_Importance_{out}_{timestamp}.csv", index=False)

 config_df = pd.DataFrame([
 {'Parameter': 'Target Ranges', 'Value': str(target_ranges)},
 {'Parameter': 'Selected Models', 'Value': str(selected_models)},
 {'Parameter': 'Feasible Points Found', 'Value': len(feasible_X)},
 {'Parameter': 'Bounds Method', 'Value': 'Observed Data Range (no extrapolation)'},
 {'Parameter': 'Sensitivity Methods', 'Value': 'Pearson + SHAP + ALE'},
 {'Parameter': 'System Type', 'Value': 'UF Membrane'},
 {'Parameter': 'Timestamp', 'Value': timestamp},
 ])
 config_df.to_csv(f"{dirs['csv']}/Configuration_{timestamp}.csv", index=False)

 print(f" All CSV files saved with timestamp {timestamp}")

# ==================== MAIN EXECUTION ====================

def run_improved_feasible_analysis():
 """
 Master function: Improved optimal conditions analysis for UF system.
 """

    # UF System parameters
 input_cols = ['glu', 'mlss', 'air', 'fm', 'cn', 'hrt', 'srt']
 output_cols = ['tmp', 'flow', 'lv']

    # Set base directory
 try:

 base_dir = './Optimal_Conditions_v2'
 except:
 base_dir = './UF_data_results/Optimal_Conditions_v2'

 dirs = create_optimal_directories(base_dir)
 print(" Directory structure created")

    # Check prerequisites
 if 'ml_trained_models' not in globals():
 print("\n WARNING: ml_trained_models not found!")
 print(" Please run Step 5 (Model Training) first.")
 return

 if 'fixpoint' not in globals():
 print("\n WARNING: fixpoint not found!")
 print(" Please run Step 3 (Data Standardization) first.")
 return

    # User inputs
 target_ranges = get_target_output_ranges(output_cols)
 selected_models = select_models(ml_trained_models, output_cols)

    # FIX 1: Observed bounds
 try:
        # Try to get X_train from memory or load from file
 try:
 X_train_test = X_train  # Check if exists
 bounds, bounds_info = get_observed_data_bounds(input_cols, fixpoint, X_train)
 except NameError:
 print("\n X_train not found, attempting to load training data...")
 try:
                # Try to load cleaned data
 train_df = pd.read_csv(os.path.join(DATA_DIR, 'step7_cleaned_data.csv'))
 X_train_loaded = train_df[input_cols].values
 X_train_std = (X_train_loaded - fixpoint['Input_mean']) / fixpoint['Input_std']
 bounds, bounds_info = get_observed_data_bounds(input_cols, fixpoint, X_train_std)
 except Exception as e:
 print(f" Could not load training data: {e}")
 print(" Using fallback: mean ± 3σ bounds")
 bounds = []
 rows = []
 for i, col in enumerate(input_cols):
 mean = fixpoint['Input_mean'][i]
 std = fixpoint['Input_std'][i]
 lo = max(0.0, mean - 3 * std)
 hi = mean + 3 * std
 bounds.append((lo, hi))
 rows.append({
 'Feature': col,
 'Observed_Min': lo,
 'Observed_Max': hi,
 'Old_Lo (mean-3σ)': lo,
 'Old_Hi (mean+3σ)': hi,
 'Extrapolation_Lo': 'N/A',
 'Extrapolation_Hi': 'N/A',
 })
 bounds_info = pd.DataFrame(rows)
 except Exception as e:
 print(f"\n Error in bounds calculation: {e}")
 return

    # Sample feasible space
 Xf, Yf, Xf_std = sample_feasible_space(
 ml_trained_models, selected_models, target_ranges,
 bounds, fixpoint, input_cols, output_cols,
 n_samples=500000
 )

 if len(Xf) == 0:
 print("\n" + "="*60)
 print(" NO FEASIBLE POINTS FOUND")
 print("="*60)
 print("\nSuggestions:")
 print(" 1. Widen target ranges")
 print(" 2. This finding is important for the paper!")

 config_df = pd.DataFrame([{
 'Parameter': 'Result',
 'Value': 'No feasible points found with observed bounds'
 }, {
 'Parameter': 'Target Ranges',
 'Value': str(target_ranges)
 }])
 timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
 config_df.to_csv(f"{dirs['csv']}/NoFeasible_Config_{timestamp}.csv", index=False)
 bounds_info.to_csv(f"{dirs['csv']}/Bounds_Comparison_{timestamp}.csv", index=False)
 return

    # Core analyses
 print("\n" + "="*60)
 print("Computing Analyses")
 print("="*60)

 ranges_df = compute_ranges(Xf, input_cols)
 pearson_df = pearson_sensitivity(Xf, Yf, input_cols, output_cols)
 boundary_df = extract_boundary_points(Xf, input_cols)

 print("\n FEASIBLE INPUT RANGES:")
 print(tabulate(ranges_df, headers='keys', tablefmt='grid',
 showindex=False, floatfmt=".4f"))

 print("\n PEARSON SENSITIVITY (Top 10):")
 print(tabulate(pearson_df.head(10), headers='keys', tablefmt='grid',
 showindex=False, floatfmt=".4f"))

    # FIX 2: SHAP within feasible region
 shap_results = shap_within_feasible_region(
 ml_trained_models, selected_models, Xf_std,
 input_cols, output_cols, fixpoint, dirs,
 max_shap_samples=2000
 )

    # FIX 3: ALE analysis
 ale_results = plot_ale_all_features(
 ml_trained_models, selected_models, Xf_std,
 input_cols, output_cols, fixpoint, dirs,
 n_bins=40
 )

    # Method comparison
 comparison_dfs = compare_all_methods(
 pearson_df, shap_results, ale_results,
 input_cols, output_cols, dirs
 )

    # Visualizations
 print("\n" + "="*60)
 print("Creating Visualizations")
 print("="*60)

 plot_input_range_bars(ranges_df, dirs)
 print(" Input_Ranges_Bars.png")
 plot_sensitivity_bars(pearson_df, output_cols, dirs)
 print(" Sensitivity_Bars.png")

    # : Density visualization
 plot_2d_feasible_regions_with_density(Xf, input_cols, dirs)
 print(" Feasible_Regions_2D_Density.png (ENHANCED)")

 plot_input_distributions(Xf, input_cols, dirs)
 print(" Input_Distributions.png")
 plot_output_distributions(Yf, output_cols, target_ranges, dirs)
 print(" Output_Distributions.png")
 plot_correlation_heatmap(Xf, Yf, input_cols, output_cols, dirs)
 print(" Correlation_Heatmap.png")
 plot_3d_feasible_space(Xf, input_cols, dirs)
 print(" Feasible_Space_3D.png")

    # Save everything
 print("\n" + "="*60)
 print("Saving All Results")
 print("="*60)

 save_all_results(
 ranges_df, pearson_df, boundary_df, bounds_info,
 Xf, Yf, comparison_dfs,
 shap_results, ale_results,
 input_cols, output_cols,
 target_ranges, selected_models, dirs
 )

    # Final summary
 print("\n" + "="*80)
 print(" " * 10 + "SECTION 9 () COMPLETE - UF SYSTEM")
 print("="*80)
 print(f"\n Results saved to: {base_dir}/")
 print(f" ├── 1_Input_Ranges/")
 print(f" ├── 2_Sensitivity_Analysis/")
 print(f" ├── 3_Feasible_Regions/ ←  with density visualization")
 print(f" ├── 4_Distributions/")
 print(f" ├── 5_Correlation_Heatmaps/")
 print(f" ├── 6_CSV_Data/")
 print(f" ├── 7_SHAP_Feasible_Region/ ← NEW")
 print(f" ├── 8_ALE_Plots/ ← NEW")
 print(f" └── 9_Method_Comparison/ ← NEW")
 print(f"\n Feasible samples: {len(Xf):,}")
 print(f" No extrapolation (observed bounds only)")
 print(f" Three sensitivity methods computed")
 print(f" Enhanced 2D density visualization")
 print("="*80)

# ==================== EXECUTE ====================
print("\n" + "="*60)
print("Starting  Optimal Conditions Analysis - UF System")
print("="*60)
print("\nImprovements in this version:")
print(" 1. Input bounds = observed data range (no extrapolation)")
print(" 2. SHAP computed within feasible region")
print(" 3. ALE plots for correlated feature handling")
print(" 4. Three-method comparison (Pearson vs SHAP vs ALE)")
print(" 5. ENHANCED 2D plots with density visualization")
print("\nSystem: UF Membrane")
print(" Inputs (7): glu, mlss, air, fm, cn, hrt, srt")
print(" Outputs (3): tmp, flow, lv")
print()

run_improved_feasible_analysis()


# Optimal condition summary visualization

In [ ]:
# -*- coding: utf-8 -*-
"""
 Input Range Visualizations for UF System
Balanced, luminous palette ,  rich but not dark, vibrant but not garish.

Design philosophy:
 • Fills: medium-saturation cornflower blue (#5B8DD9) at moderate alpha → luminous
 • Outlines: deep slate (#1C3F6E) for crispness without blackness
 • Accent: warm terracotta (#C94F2C) for mean lines ,  pops beautifully
 • Reference: vivid teal (#0E9AA7) for P05/P95 ,  distinct from fills
 • Backgrounds: pure white ,  let colors breathe
 • Band fills: very light tint, never opaque
"""

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats as scipy_stats
from matplotlib.lines import Line2D

plt.rcParams['axes.grid'] = False
plt.rcParams['font.family'] = 'DejaVu Sans'

# ══════════════════════════════════════════════════════════════
#  COLOR PALETTE ,  luminous, medium saturation
# ══════════════════════════════════════════════════════════════

# Primary blue ,  cornflower/slate family (NOT dark navy)
BLUE_FILL = '#5B8DD9'   # main fill: violin, bars, hist → medium-sat, bright
BLUE_DEEP = '#2E5FAA'   # box fill, strip scatter → slightly deeper
BLUE_EDGE = '#1C3F6E'   # edges, outlines → dark but not black
BLUE_LIGHT = '#C5D9F5'   # very light: background tints → barely-there

# Warm terracotta accent (mean / highlight)
TERRA = '#C94F2C'   # bold terracotta ,  mean lines, median
TERRA_LIGHT = '#FAE2DB'   # very pale salmon ,  wide band fills

# Vivid teal (P05/P95 reference)
TEAL = '#0E9AA7'   # bright teal ,  distinct, crisp
TEAL_LIGHT = '#D6F2F5'   # very pale teal ,  narrow band fills

# Annotation
ANNO_BG = '#FDF6F0'   # warm off-white annotation boxes
ANNO_BORDER = '#C94F2C'   # terracotta border on boxes

# Neutrals
BLACK = '#1A1A2E'   # near-black for text
WHITE = '#FFFFFF'
FIGBG = '#FFFFFF'   # pure white figure background
PANEL_BG = '#F8FAFD'   # barely-blue panel background

# Method comparison ,  3 visually distinct
M_P_FILL = '#B8D4F5'   # Pearson ,  sky blue
M_P_EDGE = '#2E5FAA'
M_S_FILL = '#5B8DD9'   # SHAP ,  cornflower
M_S_EDGE = '#1C3F6E'
M_A_FILL = '#52C9A0'   # ALE ,  mint green
M_A_EDGE = '#1A8A6A'

# ══════════════════════════════════════════════════════════════
# DATA LOADING
# ══════════════════════════════════════════════════════════════

def load_feasible_data(csv_dir=None):
 if csv_dir is None:
 try:

 csv_dir = './Optimal_Conditions_v2/6_CSV_Data'
 except:
 csv_dir = './UF_data_results/Optimal_Conditions_v2/6_CSV_Data'

 if not os.path.exists(csv_dir):
 print(f" Not found: {csv_dir}"); return None,None,None,None,None

 files = sorted([f for f in os.listdir(csv_dir) if f.startswith('Feasible_Points_')], reverse=True)
 if not files: return None,None,None,None,None

 df = pd.read_csv(f"{csv_dir}/{files[0]}")
 ic = [c for c in df.columns if not c.startswith('Predicted_')]
 oc = [c.replace('Predicted_','') for c in df.columns if c.startswith('Predicted_')]
 X = df[ic].values
 Y = df[[f'Predicted_{c}' for c in oc]].values

 rfile = [f for f in os.listdir(csv_dir) if f.startswith('Input_Ranges_')]
 config = {'ranges': pd.read_csv(f"{csv_dir}/{rfile[0]}")} if rfile else None
 print(f" {len(X):,} pts | {len(ic)} inputs")
 return X, Y, ic, oc, config

# helper
def _style_ax(ax, spine_color=BLUE_EDGE, lw=1.2, bg=PANEL_BG):
 ax.set_facecolor(bg)
 for sp in ax.spines.values():
 sp.set_color(spine_color); sp.set_linewidth(lw)

def _legend_lines(ax, loc='upper left'):
 ax.legend(handles=[
 Line2D([0],[0], color=TEAL, ls='--', lw=2.2, label='P05 / P95'),
 Line2D([0],[0], color=TERRA, ls='-', lw=2.5, label='Mean'),
 ], loc=loc, fontsize=9, framealpha=0.96,
 edgecolor=BLUE_EDGE, facecolor=ANNO_BG)

# ══════════════════════════════════════════════════════════════
# VIZ 1 ,  ENHANCED VIOLIN + BOX
# ══════════════════════════════════════════════════════════════

def plot_enhanced_input_ranges(X, input_cols, save_path):
 print("\n1. Enhanced violin + box …")
 fig, axes = plt.subplots(2, 4, figsize=(26, 13), facecolor=FIGBG)
 axes = axes.flatten()

 for idx, col in enumerate(input_cols):
 ax = axes[idx]
 _style_ax(ax)

 data = X[:, idx]
 s = {
 'min': data.min(),
 'p05': np.percentile(data, 5),
 'p25': np.percentile(data, 25),
 'median': np.median(data),
 'p75': np.percentile(data, 75),
 'p95': np.percentile(data, 95),
 'max': data.max(),
 'mean': data.mean(),
 }

        # ── Violin ,  medium blue, semi-transparent → luminous
 parts = ax.violinplot([data], positions=[0], widths=0.72,
 showmeans=False, showextrema=False, showmedians=False)
 for pc in parts['bodies']:
 pc.set_facecolor(BLUE_FILL)
 pc.set_edgecolor(BLUE_EDGE)
 pc.set_alpha(0.55)           # key: 0.55 alpha = luminous, not heavy
 pc.set_linewidth(2)

        # ── Box ,  slightly deeper blue, clear contrast with violin
 ax.boxplot([data], positions=[0], widths=0.28,
 patch_artist=True,
 boxprops=dict(facecolor=BLUE_DEEP, alpha=0.75,
 linewidth=1.8, edgecolor=BLUE_EDGE),
 medianprops=dict(color=TERRA, linewidth=3.5, solid_capstyle='round'),
 whiskerprops=dict(linewidth=1.8, color=BLUE_EDGE),
 capprops=dict(linewidth=2.2, color=BLUE_EDGE),
 flierprops=dict(marker='o', markersize=4,
 markerfacecolor=TERRA, alpha=0.5,
 markeredgecolor='none'))

        # ── P05/P95 band ,  very pale teal
 ax.axhspan(s['p05'], s['p95'], alpha=0.12, color=TEAL_LIGHT, zorder=0)
 ax.axhline(s['p05'], color=TEAL, ls='--', lw=2, alpha=0.85, zorder=3)
 ax.axhline(s['p95'], color=TEAL, ls='--', lw=2, alpha=0.85, zorder=3)
 ax.axhline(s['mean'], color=TERRA, ls='-', lw=2.8, alpha=0.95, zorder=4)

        # ── Percentile markers ,  outside violin, clear
 ax.scatter([0.44], [s['p05']], s=160, c=TEAL, marker='v',
 edgecolors=BLUE_EDGE, lw=1.3, zorder=8)
 ax.scatter([0.44], [s['p95']], s=160, c=TEAL, marker='^',
 edgecolors=BLUE_EDGE, lw=1.3, zorder=8)
 ax.scatter([0.44], [s['mean']], s=180, c=TERRA, marker='D',
 edgecolors=BLUE_EDGE, lw=1.3, zorder=8)

        # ── Annotation box ,  warm, unobtrusive
 txt = f"Range: [{s['min']:.3g} – {s['max']:.3g}]\n"
 txt += f"P05–P95: [{s['p05']:.3g}, {s['p95']:.3g}]\n"
 txt += f"Mean: {s['mean']:.3g}"
 ax.text(0.97, 0.03, txt, transform=ax.transAxes,
 fontsize=9, va='bottom', ha='right',
 bbox=dict(boxstyle='round,pad=0.45', facecolor=ANNO_BG,
 edgecolor=ANNO_BORDER, lw=1.5, alpha=0.97),
 color=BLACK, family='monospace')

 ax.set_xlim(-0.52, 0.72)
 ax.set_xticks([])
 ax.set_ylabel('Value', fontsize=11, fontweight='bold', color=BLACK)
 ax.set_title(col, fontsize=15, fontweight='bold', pad=8, color=BLACK)
 ax.tick_params(axis='y', labelsize=10, colors=BLACK)
 if idx == 0: _legend_lines(ax)

 axes[-1].axis('off')
 plt.suptitle('Feasible Input Ranges ,  UF Operating Parameters\n(Violin + Box + Percentiles)',
 fontsize=21, fontweight='bold', y=0.999, color=BLACK)
 plt.tight_layout()
 plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor=FIGBG)
 plt.close(); print(f" {save_path}")

# ══════════════════════════════════════════════════════════════
# VIZ 2 ,  RAINCLOUD
# ══════════════════════════════════════════════════════════════

def plot_input_ranges_raincloud(X, input_cols, save_path):
 print("\n2. Raincloud plots …")
 fig, axes = plt.subplots(2, 4, figsize=(26, 14), facecolor=FIGBG)
 axes = axes.flatten()

 for idx, col in enumerate(input_cols):
 ax = axes[idx]
 _style_ax(ax)
 data = X[:, idx]
 p05 = np.percentile(data, 5); p95 = np.percentile(data, 95)
 mean = data.mean(); med = np.median(data)

        # ── Half-violin ,  cornflower, luminous
 parts = ax.violinplot([data], positions=[1.2], widths=0.65,
 showmeans=False, showextrema=False, showmedians=False)
 for pc in parts['bodies']:
 v = pc.get_paths()[0].vertices
 v[v[:,0] < 1.2, 0] = 1.2   # clip to right half
 pc.get_paths()[0].vertices = v
 pc.set_facecolor(BLUE_FILL)
 pc.set_edgecolor(BLUE_EDGE)
 pc.set_alpha(0.60)
 pc.set_linewidth(2)

        # ── Strip ,  lighter blue, fine scatter
 np.random.seed(42 + idx)
 n_pts = min(300, len(data))
 sidx = np.random.choice(len(data), n_pts, replace=False)
 jit = np.random.normal(0.28, 0.05, n_pts)
 ax.scatter(jit, data[sidx], alpha=0.35, s=12,
 c=BLUE_FILL, edgecolors='none')

        # ── Box
 ax.boxplot([data], positions=[0.74], widths=0.22,
 patch_artist=True, vert=True,
 boxprops=dict(facecolor=BLUE_DEEP, alpha=0.72,
 linewidth=1.8, edgecolor=BLUE_EDGE),
 medianprops=dict(color=TERRA, linewidth=3.5, solid_capstyle='round'),
 whiskerprops=dict(linewidth=1.8, color=BLUE_EDGE),
 capprops=dict(linewidth=2.2, color=BLUE_EDGE),
 flierprops=dict(marker='o', markersize=4,
 markerfacecolor=TERRA, alpha=0.5,
 markeredgecolor='none'))

        # ── Reference
 ax.axhspan(p05, p95, alpha=0.10, color=TEAL_LIGHT, zorder=0)
 ax.axhline(p05, color=TEAL, ls='--', lw=2, alpha=0.85)
 ax.axhline(p95, color=TEAL, ls='--', lw=2, alpha=0.85)
 ax.axhline(mean, color=TERRA, ls='-', lw=2.8, alpha=0.95)

 txt = f"Min {data.min():.3g} P05 {p05:.3g}\n"
 txt += f"Mean {mean:.3g} Med {med:.3g}\n"
 txt += f"P95 {p95:.3g} Max {data.max():.3g}"
 ax.text(0.98, 0.97, txt, transform=ax.transAxes,
 fontsize=9, va='top', ha='right',
 bbox=dict(boxstyle='round,pad=0.45', facecolor=ANNO_BG,
 edgecolor=ANNO_BORDER, lw=1.5, alpha=0.97),
 color=BLACK, family='monospace')

 ax.set_xlim(0.02, 1.65)
 ax.set_xticks([0.28, 0.74, 1.2])
 ax.set_xticklabels(['Strip','Box','Violin'], fontsize=10, fontweight='bold', color=BLACK)
 ax.set_ylabel('Value', fontsize=11, fontweight='bold', color=BLACK)
 ax.set_title(col, fontsize=15, fontweight='bold', pad=8, color=BLACK)
 ax.tick_params(axis='y', labelsize=10, colors=BLACK)
 if idx == 0: _legend_lines(ax)

 axes[-1].axis('off')
 plt.suptitle('Raincloud Plots: Feasible Input Ranges ,  UF System\n(Strip + Box + Half-Violin)',
 fontsize=21, fontweight='bold', y=0.999, color=BLACK)
 plt.tight_layout()
 plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor=FIGBG)
 plt.close(); print(f" {save_path}")

# ══════════════════════════════════════════════════════════════
# VIZ 3 ,  RIDGELINE (key fix: translucent fills, NOT opaque)
# ══════════════════════════════════════════════════════════════

def plot_input_ranges_ridgeline(X, input_cols, save_path):
 print("\n3. Ridgeline plots …")
 fig, ax = plt.subplots(figsize=(15, 11), facecolor=FIGBG)
 ax.set_facecolor(FIGBG)

 y_offset = 0
 y_spacing = 1.35
 n = len(input_cols)

 for idx, col in enumerate(input_cols):
 data = X[:, idx]
 data_norm = (data - data.min()) / (data.max() - data.min())
 kde = scipy_stats.gaussian_kde(data_norm)
 xr = np.linspace(-0.01, 1.01, 400)
 yd = kde(xr) * 0.95 + y_offset

        # ── Outer fill ,  very light teal wash (barely there)
 ax.fill_between(xr, y_offset, yd,
 alpha=0.18, color=BLUE_LIGHT, zorder=idx*2)
        # ── Main fill ,  cornflower blue, moderate alpha = luminous
 ax.fill_between(xr, y_offset, yd,
 alpha=0.62, color=BLUE_FILL, zorder=idx*2+1)
        # ── Crisp outline ,  deep slate
 ax.plot(xr, yd, color=BLUE_EDGE, lw=1.8, zorder=idx*2+2)
        # ── White baseline
 ax.plot(xr, [y_offset]*len(xr), color='white', lw=1.5, zorder=idx*2+3)

        # Percentile lines
 total = (n + 1) * y_spacing
 p05n = (np.percentile(data, 5) - data.min()) / (data.max() - data.min())
 p95n = (np.percentile(data, 95) - data.min()) / (data.max() - data.min())
 mn = (data.mean() - data.min()) / (data.max() - data.min())

 lo = y_offset / total; hi = (y_offset + 0.95) / total
 ax.axvline(p05n, ymin=lo, ymax=hi, color=TEAL, ls='--', lw=2.2, alpha=0.9, zorder=idx*2+4)
 ax.axvline(p95n, ymin=lo, ymax=hi, color=TEAL, ls='--', lw=2.2, alpha=0.9, zorder=idx*2+4)
 ax.axvline(mn, ymin=lo, ymax=hi, color=TERRA, ls='-', lw=3, alpha=1.0, zorder=idx*2+5)

        # Labels
 ax.text(-0.055, y_offset + 0.47, col, fontsize=13, fontweight='bold',
 va='center', ha='right', color=BLACK)
 ax.text(1.055, y_offset + 0.47,
 f"[{data.min():.3g} – {data.max():.3g}]",
 fontsize=9.5, va='center', ha='left', color=BLUE_DEEP)

 y_offset += y_spacing

 ax.set_xlim(-0.16, 1.20)
 ax.set_ylim(-0.4, y_offset)
 ax.set_yticks([])
 ax.set_xlabel('Normalized Value (0 = Min, 1 = Max)',
 fontsize=13, fontweight='bold', color=BLACK)
 ax.set_title('Ridgeline Plots: Feasible Input Distributions ,  UF System\n'
 '(Overlapping Density Curves)',
 fontsize=19, fontweight='bold', pad=18, color=BLACK)
 ax.tick_params(axis='x', labelsize=11, colors=BLACK)
 for s in ['top','right','left']: ax.spines[s].set_visible(False)
 ax.spines['bottom'].set_color(BLUE_EDGE); ax.spines['bottom'].set_linewidth(1.5)

 ax.legend(handles=[
 Line2D([0],[0], color=TEAL, ls='--', lw=2.2, label='P05 / P95'),
 Line2D([0],[0], color=TERRA, ls='-', lw=3, label='Mean'),
 ], loc='upper right', fontsize=11, framealpha=0.96,
 edgecolor=BLUE_EDGE, facecolor=ANNO_BG)

 plt.tight_layout()
 plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor=FIGBG)
 plt.close(); print(f" {save_path}")

# ══════════════════════════════════════════════════════════════
# VIZ 4 ,  CIRCULAR
# ══════════════════════════════════════════════════════════════

def plot_input_ranges_circular(X, input_cols, save_path):
 print("\n4. Circular radar …")
 fig = plt.figure(figsize=(13, 13), facecolor=FIGBG)
 ax = fig.add_subplot(111, projection='polar', facecolor='#F0F5FF')

 n = len(input_cols)
 angles = np.linspace(0, 2*np.pi, n, endpoint=False).tolist()

 p05s, means, p95s = [], [], []
 for i in range(n):
 d = X[:, i]
 dn = (d - d.min()) / (d.max() - d.min()) * 100
 p05s.append(np.percentile(dn, 5))
 means.append(dn.mean())
 p95s.append(np.percentile(dn, 95))

 ac = angles + angles[:1]
 p05c = p05s + p05s[:1]
 mc = means + means[:1]
 p95c = p95s + p95s[:1]
 maxc = [100] * (n+1)
 minc = [0] * (n+1)

    # Full range ,  very light salmon wash
 ax.fill_between(ac, minc, maxc, alpha=0.10, color=TERRA_LIGHT)
    # Robust P05–P95 ,  cornflower at moderate alpha
 ax.fill_between(ac, p05c, p95c, alpha=0.40, color=BLUE_FILL)

    # Lines
 ax.plot(ac, mc, 'o-', lw=2.8, color=TERRA, ms=9, zorder=6, label='Mean')
 ax.plot(ac, p05c, 's--', lw=2, color=TEAL, ms=7, zorder=5, alpha=0.9, label='P05')
 ax.plot(ac, p95c, '^--', lw=2, color=TEAL, ms=7, zorder=5, alpha=0.9, label='P95')

 ax.set_xticks(angles)
 ax.set_xticklabels(input_cols, fontsize=13, fontweight='bold', color=BLACK)
 ax.set_ylim(0, 100)
 ax.set_yticks([0, 25, 50, 75, 100])
 ax.set_yticklabels(['Min','25%','50%','75%','Max'], fontsize=10, color=BLUE_DEEP)
 ax.tick_params(pad=12)
 ax.grid(True, ls='--', alpha=0.3, color=BLUE_EDGE)
 ax.spines['polar'].set_color(BLUE_EDGE); ax.spines['polar'].set_linewidth(1.8)
 ax.set_title('Circular View: Feasible Input Ranges ,  UF System\n'
 '(Normalized 0–100)',
 fontsize=18, fontweight='bold', pad=32, y=1.07, color=BLACK)
 ax.legend(loc='upper right', bbox_to_anchor=(1.30, 1.07), fontsize=12,
 framealpha=0.96, edgecolor=BLUE_EDGE, facecolor=ANNO_BG)

 plt.tight_layout()
 plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor=FIGBG)
 plt.close(); print(f" {save_path}")

# ══════════════════════════════════════════════════════════════
# VIZ 5 ,  SUMMARY PANEL
# ══════════════════════════════════════════════════════════════

def plot_input_ranges_summary_panel(X, input_cols, save_path):
 print("\n5. Summary panel …")
 fig = plt.figure(figsize=(28, 24), facecolor=FIGBG)
 gs = fig.add_gridspec(4, 4, hspace=0.54, wspace=0.38)

 for idx, col in enumerate(input_cols):
 row = (idx*2)//4; cs = (idx*2)%4
 data = X[:, idx]
 s = {
 'min': data.min(),
 'p05': np.percentile(data, 5),
 'p25': np.percentile(data, 25),
 'median': np.median(data),
 'mean': data.mean(),
 'p75': np.percentile(data, 75),
 'p95': np.percentile(data, 95),
 'max': data.max(),
 'std': data.std(),
 }

        # ── LEFT: Distribution
 al = fig.add_subplot(gs[row, cs])
 _style_ax(al)
 al.hist(data, bins=40, alpha=0.62, color=BLUE_FILL,
 edgecolor=BLUE_EDGE, lw=0.7, density=True)
 kde = scipy_stats.gaussian_kde(data)
 xr = np.linspace(data.min(), data.max(), 250)
 al.plot(xr, kde(xr), color=BLUE_EDGE, lw=2.5, label='KDE')
 al.axvline(s['p05'], color=TEAL, ls='--', lw=2, alpha=0.85, label='P05/P95')
 al.axvline(s['p95'], color=TEAL, ls='--', lw=2, alpha=0.85)
 al.axvline(s['mean'], color=TERRA, ls='-', lw=2.8, alpha=0.95, label='Mean')
 al.set_xlabel('Value', fontsize=10, fontweight='bold', color=BLACK)
 al.set_ylabel('Density', fontsize=10, fontweight='bold', color=BLACK)
 al.set_title(f'{col} ,  Distribution', fontsize=12, fontweight='bold', color=BLACK)
 al.legend(fontsize=8, loc='upper right', framealpha=0.96,
 edgecolor=BLUE_EDGE, facecolor=ANNO_BG)
 al.tick_params(labelsize=9, colors=BLACK)

        # ── RIGHT: Range bars
 ar = fig.add_subplot(gs[row, cs+1])
 _style_ax(ar)
 yp = 0.5

        # Full range ,  very light salmon
 ar.barh([yp], [s['max']-s['min']], left=s['min'],
 height=0.20, color=TERRA_LIGHT, edgecolor=TERRA, lw=1.8,
 alpha=0.80, label='Full Range')
        # P05–P95 ,  cornflower
 ar.barh([yp], [s['p95']-s['p05']], left=s['p05'],
 height=0.15, color=BLUE_FILL, edgecolor=BLUE_EDGE, lw=1.8,
 alpha=0.72, label='P05–P95')
        # IQR ,  deeper blue
 ar.barh([yp], [s['p75']-s['p25']], left=s['p25'],
 height=0.10, color=BLUE_DEEP, edgecolor=BLUE_EDGE, lw=1.8,
 alpha=0.78, label='IQR')

 mks = [
 (s['min'], 'Min', TEAL, 'v', -0.29),
 (s['p05'], 'P05', TEAL, '<', -0.22),
 (s['p25'], 'P25', BLUE_DEEP, 's', -0.15),
 (s['median'], 'Med', TERRA, 'D', 0.22),
 (s['p75'], 'P75', BLUE_DEEP, 's', 0.15),
 (s['p95'], 'P95', TEAL, '>', 0.22),
 (s['max'], 'Max', TEAL, '^', 0.29),
 ]
 for val, lbl, c, mk, yo in mks:
 ar.scatter([val], [yp], s=150, c=[c], marker=mk,
 edgecolors=BLUE_EDGE, lw=1.3, zorder=10)
 ar.text(val, yp+yo, lbl, ha='center', va='center',
 fontsize=7, fontweight='bold',
 bbox=dict(boxstyle='round,pad=0.28', facecolor=ANNO_BG,
 edgecolor=c, lw=1.4, alpha=0.97), color=BLACK)

 ar.set_ylim(0.05, 1.25); ar.set_yticks([])
 ar.set_xlabel('Value', fontsize=10, fontweight='bold', color=BLACK)
 ar.set_title(f'{col} ,  Ranges', fontsize=12, fontweight='bold', color=BLACK)
 ar.tick_params(labelsize=9, colors=BLACK)
 stxt = (f"Min: {s['min']:.3g} Max: {s['max']:.3g}\n"
 f"P05: {s['p05']:.3g} P95: {s['p95']:.3g}\n"
 f"P25: {s['p25']:.3g} P75: {s['p75']:.3g}\n"
 f"Med: {s['median']:.3g} Mean: {s['mean']:.3g}\n"
 f"Std: {s['std']:.3g}")
 ar.text(0.02, 0.98, stxt, transform=ar.transAxes,
 fontsize=7.5, va='top',
 bbox=dict(boxstyle='round', facecolor=ANNO_BG,
 edgecolor=ANNO_BORDER, lw=1.6, alpha=0.97),
 family='monospace', color=BLACK)

 plt.suptitle('Comprehensive Summary: Feasible Input Ranges ,  UF System\n'
 '(Distribution + Range Visualization)',
 fontsize=23, fontweight='bold', y=0.999, color=BLACK)
 plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor=FIGBG)
 plt.close(); print(f" {save_path}")

# ══════════════════════════════════════════════════════════════
# MAIN
# ══════════════════════════════════════════════════════════════

def create_all_visualizations():
 X, Y, input_cols, output_cols, config = load_feasible_data()
 if X is None: return

 try:

 base = './Optimal_Conditions_v2'
 except:
 base = './UF_data_results/Optimal_Conditions_v2'

 out = f'{base}/Elegant_Input_Ranges'
 os.makedirs(out, exist_ok=True)

 plot_enhanced_input_ranges(X, input_cols, f'{out}/Input_Ranges_Enhanced.png')
 plot_input_ranges_raincloud(X, input_cols, f'{out}/Input_Ranges_Raincloud.png')
 plot_input_ranges_ridgeline(X, input_cols, f'{out}/Input_Ranges_Ridgeline.png')
 plot_input_ranges_circular(X, input_cols, f'{out}/Input_Ranges_Circular.png')
 plot_input_ranges_summary_panel(X, input_cols, f'{out}/Input_Ranges_Summary_Panel.png')

 print(f"\n All saved → {out}/")
 print(f"\n Elegant palette:")
 print(f" BLUE_FILL (violin/hist/bar fills) {BLUE_FILL} alpha ≈ 0.55–0.65")
 print(f" BLUE_DEEP (box/strip fills) {BLUE_DEEP} alpha ≈ 0.72–0.78")
 print(f" BLUE_EDGE (all outlines) {BLUE_EDGE}")
 print(f" TERRA (mean / accent) {TERRA}")
 print(f" TEAL (P05/P95 ref lines) {TEAL}")
 print(f" BLUE_LIGHT (background washes) {BLUE_LIGHT} alpha ≈ 0.10–0.18")

if __name__ == "__main__":
 create_all_visualizations()


# Optimal condition member visualization

In [ ]:
# -*- coding: utf-8 -*-
"""
 Main Visualization Script for UF Optimal Conditions
Balanced, luminous palette ,  rich but not dark, vibrant but not garish.
"""

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats as scipy_stats
from itertools import combinations
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.lines import Line2D

plt.rcParams['axes.grid'] = False
plt.rcParams['font.family'] = 'DejaVu Sans'

# ══════════════════════════════════════════════════════════════
#  COLOR PALETTE
# ══════════════════════════════════════════════════════════════

BLUE_FILL = '#5B8DD9'   # cornflower ,  main fills (alpha 0.55–0.65)
BLUE_DEEP = '#2E5FAA'   # deeper blue ,  box fills, strip
BLUE_EDGE = '#1C3F6E'   # dark slate ,  all edges/outlines
BLUE_LIGHT = '#C5D9F5'   # pale blue ,  background tints

TERRA = '#C94F2C'   # terracotta ,  mean / accent
TERRA_LIGHT = '#FAE2DB'   # pale salmon ,  wide band fills

TEAL = '#0E9AA7'   # vivid teal ,  P05/P95 lines
TEAL_LIGHT = '#D6F2F5'   # pale teal ,  narrow band fills

ANNO_BG = '#FDF6F0'   # warm off-white for annotation boxes
ANNO_BORDER = '#C94F2C'

PANEL_BG = '#F8FAFD'   # barely-blue subplot backgrounds
BLACK = '#1A1A2E'
WHITE = '#FFFFFF'
FIGBG = '#FFFFFF'

# Method comparison ,  3 distinct
M_P_FILL = '#B8D4F5'; M_P_EDGE = '#2E5FAA'   # Pearson ,  sky
M_S_FILL = '#5B8DD9'; M_S_EDGE = '#1C3F6E'   # SHAP ,  cornflower
M_A_FILL = '#52C9A0'; M_A_EDGE = '#1A8A6A'   # ALE ,  mint

# ── helpers ──────────────────────────────────────────────────

def _style_ax(ax):
 ax.set_facecolor(PANEL_BG)
 for sp in ax.spines.values():
 sp.set_color(BLUE_EDGE); sp.set_linewidth(1.2)

# ══════════════════════════════════════════════════════════════
# DATA LOADING
# ══════════════════════════════════════════════════════════════

def load_all_data(base_dir=None):
 if base_dir is None:
 try:

 base_dir = './Optimal_Conditions_v2'
 except:
 base_dir = './UF_data_results/Optimal_Conditions_v2'

 csv_dir = f'{base_dir}/6_CSV_Data'
 if not os.path.exists(csv_dir):
 print(f" Not found: {csv_dir}"); return None

 files = os.listdir(csv_dir)
 def latest(pfx): return sorted([f for f in files if f.startswith(pfx)], reverse=True)

 feas = latest('Feasible_Points_')
 if not feas: return None

 df = pd.read_csv(f"{csv_dir}/{feas[0]}")
 ic = [c for c in df.columns if not c.startswith('Predicted_')]
 oc = [c.replace('Predicted_','') for c in df.columns if c.startswith('Predicted_')]
 data = {
 'feasible_X': df[ic].values,
 'feasible_Y': df[[f'Predicted_{c}' for c in oc]].values,
 'input_cols': ic, 'output_cols': oc,
 }
 print(f" {len(data['feasible_X']):,} pts | {len(ic)} inputs")

 for pfx, key in [('Input_Ranges_','ranges_df'), ('Sensitivity_Pearson_','sensitivity_df')]:
 fs = latest(pfx)
 if fs: data[key] = pd.read_csv(f"{csv_dir}/{fs[0]}")

 for pfx, key in [('Method_Comparison_','method_comparison'),
 ('SHAP_Feasible_','shap_importance'),
 ('ALE_Importance_','ale_importance')]:
 fs = latest(pfx)
 if fs:
 data[key] = {}
 for f in fs:
 nm = f.replace(pfx,'').replace('.csv','').split('_')
 if nm: data[key][nm[0]] = pd.read_csv(f"{csv_dir}/{f}")
 return data

# ══════════════════════════════════════════════════════════════
# 1. INPUT RANGE BARS
# ══════════════════════════════════════════════════════════════

def plot_input_range_bars(data, save_dir):
 print("\n1. Input range bars …")
 rdf = data.get('ranges_df')
 if rdf is None: return

 n = len(rdf); nc = 4; nr = (n+nc-1)//nc
 fig, axes = plt.subplots(nr, nc, figsize=(22, nr*5+1.5), facecolor=FIGBG)
 axes = axes.flatten()

 for idx, row in rdf.iterrows():
 ax = axes[idx]; _style_ax(ax)

        # Full range ,  pale salmon band
 ax.barh([0.55], [row['Max']-row['Min']], left=row['Min'],
 height=0.28, color=TERRA_LIGHT, edgecolor=TERRA, lw=1.8,
 alpha=0.85, zorder=2, label='Full Range')
        # P05–P95 ,  cornflower
 ax.barh([0.55], [row['P95']-row['P05']], left=row['P05'],
 height=0.20, color=BLUE_FILL, edgecolor=BLUE_EDGE, lw=1.8,
 alpha=0.68, zorder=3, label='P05–P95')

 ax.axvline(row['P05'], color=TEAL, lw=2.5, ls='--', alpha=0.90, zorder=4, label='P05')
 ax.axvline(row['P95'], color=TEAL, lw=2.5, ls='--', alpha=0.90, zorder=4, label='P95')
 ax.axvline(row['Mean'], color=TERRA, lw=3.2, ls='-', alpha=0.95, zorder=5, label='Mean')

 ax.set_xlabel('Value', fontsize=11, fontweight='bold', color=BLACK)
 ax.set_title(str(row['Input']), fontsize=14, fontweight='bold', color=BLACK)
 ax.set_yticks([]); ax.set_ylim(0.20, 0.95)
 ax.legend(fontsize=9, loc='upper right', framealpha=0.96,
 edgecolor=BLUE_EDGE, facecolor=ANNO_BG)
 ax.tick_params(axis='x', labelsize=10, colors=BLACK)

 for i in range(n, len(axes)): axes[i].axis('off')
 plt.suptitle('Feasible Input Ranges ,  UF System',
 fontsize=20, fontweight='bold', y=0.999, color=BLACK)
 plt.tight_layout()
 plt.savefig(f"{save_dir}/Input_Ranges_Bars.png", dpi=300,
 bbox_inches='tight', facecolor=FIGBG)
 plt.close(); print(" Input_Ranges_Bars.png")

# ══════════════════════════════════════════════════════════════
# 2. SENSITIVITY BARS
# ══════════════════════════════════════════════════════════════

def plot_sensitivity_bars(data, save_dir):
 print("\n2. Sensitivity bars …")
 sdf = data.get('sensitivity_df')
 if sdf is None: return
 oc = data['output_cols']

 fig, axes = plt.subplots(1, len(oc), figsize=(11*len(oc), 9), facecolor=FIGBG)
 if len(oc)==1: axes=[axes]
 fig.suptitle('Sensitivity Analysis (Pearson Correlation) ,  UF System',
 fontsize=20, fontweight='bold', color=BLACK)

 for idx, out in enumerate(oc):
 ax = axes[idx]; _style_ax(ax)
 od = sdf[sdf['Output']==out].sort_values('Abs_Correlation', ascending=True)
 cols= [BLUE_DEEP if c > 0 else TERRA for c in od['Correlation']]
 bars= ax.barh(od['Input'], od['Correlation'],
 color=cols, edgecolor=BLUE_EDGE, lw=1.5, alpha=0.82)

 for bar, val in zip(bars, od['Correlation']):
 offset = 0.003 if val >= 0 else -0.003
 ax.text(val+offset, bar.get_y()+bar.get_height()/2,
 f'{val:.3f}', va='center',
 ha='left' if val>=0 else 'right',
 fontsize=9, fontweight='bold', color=BLACK)

 ax.axvline(0, color=BLUE_EDGE, lw=2)
 ax.set_xlabel('Correlation', fontsize=13, fontweight='bold', color=BLACK)
 ax.set_title(out, fontsize=16, fontweight='bold', color=BLACK)
 ax.tick_params(labelsize=11, colors=BLACK)
 ax.set_axisbelow(True)
 ax.xaxis.grid(True, color=BLUE_EDGE, alpha=0.12, lw=0.8)

 plt.tight_layout()
 plt.savefig(f"{save_dir}/Sensitivity_Bars.png", dpi=300,
 bbox_inches='tight', facecolor=FIGBG)
 plt.close(); print(" Sensitivity_Bars.png")

# ══════════════════════════════════════════════════════════════
# 3. 2D FEASIBLE REGIONS
# ══════════════════════════════════════════════════════════════

def plot_2d_feasible_regions_density(data, save_dir):
 print("\n3. 2D feasible regions …")
 X = data['feasible_X']; ic = data['input_cols']
 pairs = list(combinations(range(len(ic)), 2))
 nc = 5; nr = (len(pairs)+nc-1)//nc

 fig = plt.figure(figsize=(21, 4.8*nr), facecolor=FIGBG)
 fig.suptitle(f'2D Feasible Regions ,  UF System ({len(pairs)} pairs)',
 fontsize=20, fontweight='bold', y=0.999, color=BLACK)

 for pi, (i, j) in enumerate(pairs):
 ax = plt.subplot(nr, nc, pi+1); _style_ax(ax)
 hb = ax.hexbin(X[:,i], X[:,j], gridsize=28,
 cmap='Blues', mincnt=1,
 edgecolors='white', alpha=0.90, linewidths=0.1)
 cb = plt.colorbar(hb, ax=ax)
 cb.set_label('Count', fontsize=8, color=BLACK)
 cb.ax.tick_params(labelsize=7, colors=BLACK)
 ax.set_xlabel(ic[i], fontsize=11, fontweight='bold', color=BLACK)
 ax.set_ylabel(ic[j], fontsize=11, fontweight='bold', color=BLACK)
 ax.set_title(f'{ic[i]} vs {ic[j]}', fontsize=11, fontweight='bold', color=BLACK)
 ax.tick_params(labelsize=9, colors=BLACK)

 plt.tight_layout()
 plt.savefig(f"{save_dir}/Feasible_Regions_2D_Density.png",
 dpi=300, bbox_inches='tight', facecolor=FIGBG)
 plt.close(); print(" Feasible_Regions_2D_Density.png")

# ══════════════════════════════════════════════════════════════
# 4. INPUT DISTRIBUTIONS
# ══════════════════════════════════════════════════════════════

def plot_input_distributions(data, save_dir):
 print("\n4. Input distributions …")
 X = data['feasible_X']; ic = data['input_cols']
 fig, axes = plt.subplots(2, 4, figsize=(22, 11), facecolor=FIGBG)
 axes = axes.flatten()

 for idx, col in enumerate(ic):
 ax = axes[idx]; _style_ax(ax)
 d = X[:, idx]

        # Histogram ,  cornflower, moderate alpha
 n_vals, bins, patches = ax.hist(d, bins=50, color=BLUE_FILL,
 edgecolor=BLUE_EDGE, lw=0.6, alpha=0.65)

        # KDE twin axis overlay
 ax2 = ax.twinx()
 kde = scipy_stats.gaussian_kde(d)
 xr = np.linspace(d.min(), d.max(), 300)
 ax2.plot(xr, kde(xr), color=BLUE_EDGE, lw=2.5, alpha=0.9)
 ax2.set_yticks([])
 ax2.spines['right'].set_visible(False)
 ax2.spines['top'].set_visible(False)

 ax.axvline(d.mean(), color=TERRA, lw=3, ls='-', label=f'Mean {d.mean():.3g}')
 ax.axvline(np.median(d), color=TEAL, lw=2.5, ls='--', label=f'Median {np.median(d):.3g}')

 ax.set_xlabel('Value', fontsize=11, fontweight='bold', color=BLACK)
 ax.set_ylabel('Frequency', fontsize=11, fontweight='bold', color=BLACK)
 ax.set_title(col, fontsize=14, fontweight='bold', color=BLACK)
 ax.legend(fontsize=9, framealpha=0.96, edgecolor=BLUE_EDGE, facecolor=ANNO_BG)
 ax.tick_params(labelsize=10, colors=BLACK)

 axes[-1].axis('off')
 plt.suptitle('Distribution of Feasible Input Values ,  UF System',
 fontsize=20, fontweight='bold', y=0.999, color=BLACK)
 plt.tight_layout()
 plt.savefig(f"{save_dir}/Input_Distributions.png",
 dpi=300, bbox_inches='tight', facecolor=FIGBG)
 plt.close(); print(" Input_Distributions.png")

# ══════════════════════════════════════════════════════════════
# 5. OUTPUT DISTRIBUTIONS
# ══════════════════════════════════════════════════════════════

def plot_output_distributions(data, save_dir):
 print("\n5. Output distributions …")
 Y = data['feasible_Y']; oc = data['output_cols']
 fig, axes = plt.subplots(1, len(oc), figsize=(11*len(oc), 7), facecolor=FIGBG)
 if len(oc)==1: axes=[axes]
 fig.suptitle('Distribution of Predicted Outputs in Feasible Region ,  UF System',
 fontsize=20, fontweight='bold', color=BLACK)

 for idx, col in enumerate(oc):
 ax = axes[idx]; _style_ax(ax)
 d = Y[:, idx]
 ax.hist(d, bins=55, color=BLUE_FILL, edgecolor=BLUE_EDGE, lw=0.6, alpha=0.65)
 ax.axvline(d.mean(), color=TERRA, lw=3, ls='-',
 label=f'Mean: {d.mean():.3g}')
 ax.set_xlabel('Value', fontsize=13, fontweight='bold', color=BLACK)
 ax.set_ylabel('Frequency', fontsize=13, fontweight='bold', color=BLACK)
 ax.set_title(col, fontsize=16, fontweight='bold', color=BLACK)
 ax.legend(fontsize=11, framealpha=0.96, edgecolor=BLUE_EDGE, facecolor=ANNO_BG)
 ax.tick_params(labelsize=11, colors=BLACK)

 plt.tight_layout()
 plt.savefig(f"{save_dir}/Output_Distributions.png",
 dpi=300, bbox_inches='tight', facecolor=FIGBG)
 plt.close(); print(" Output_Distributions.png")

# ══════════════════════════════════════════════════════════════
# 6. CORRELATION HEATMAP
# ══════════════════════════════════════════════════════════════

def plot_correlation_heatmap(data, save_dir):
 print("\n6. Correlation heatmap …")
 X = data['feasible_X']; Y = data['feasible_Y']
 ic = data['input_cols']; oc = data['output_cols']
 corr = np.array([[np.corrcoef(X[:,i], Y[:,j])[0,1]
 for j in range(len(oc))] for i in range(len(ic))])

 fig, ax = plt.subplots(figsize=(11, 9), facecolor=FIGBG)
 _style_ax(ax)
 im = ax.imshow(corr, cmap='RdBu_r', aspect='auto', vmin=-1, vmax=1)
 ax.set_xticks(range(len(oc))); ax.set_yticks(range(len(ic)))
 ax.set_xticklabels(oc, fontsize=13, fontweight='bold', color=BLACK)
 ax.set_yticklabels(ic, fontsize=12, color=BLACK)
 plt.setp(ax.get_xticklabels(), rotation=45, ha='right', rotation_mode='anchor')

 cb = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
 cb.set_label('Pearson Correlation', fontsize=13, fontweight='bold',
 rotation=270, labelpad=22, color=BLACK)
 cb.ax.tick_params(colors=BLACK, labelsize=11)

 for i in range(len(ic)):
 for j in range(len(oc)):
 v = corr[i, j]
 tc = WHITE if abs(v) > 0.5 else BLACK
 ax.text(j, i, f'{v:.3f}', ha='center', va='center',
 color=tc, fontsize=11, fontweight='bold')

 ax.set_title('Input–Output Correlation Heatmap\n(Feasible Region ,  UF System)',
 fontsize=17, fontweight='bold', pad=16, color=BLACK)
 plt.tight_layout()
 plt.savefig(f"{save_dir}/Correlation_Heatmap.png",
 dpi=300, bbox_inches='tight', facecolor=FIGBG)
 plt.close(); print(" Correlation_Heatmap.png")

# ══════════════════════════════════════════════════════════════
# 7. 3D FEASIBLE SPACE
# ══════════════════════════════════════════════════════════════

def plot_3d_feasible_space(data, save_dir):
 print("\n7. 3D feasible space …")
 X = data['feasible_X']; ic = data['input_cols']
 if len(ic) < 3: return

 fig = plt.figure(figsize=(13, 11), facecolor=FIGBG)
 ax = fig.add_subplot(111, projection='3d')
 ax.set_facecolor('#EEF4FF')

 n_show = min(25000, len(X))
 idx = np.random.choice(len(X), n_show, replace=False)
 ax.scatter(X[idx,0], X[idx,1], X[idx,2],
 c=BLUE_FILL, marker='o', s=1.5, alpha=0.20, edgecolors='none')

 ax.set_xlabel(ic[0], fontsize=13, fontweight='bold', color=BLACK, labelpad=10)
 ax.set_ylabel(ic[1], fontsize=13, fontweight='bold', color=BLACK, labelpad=10)
 ax.set_zlabel(ic[2], fontsize=13, fontweight='bold', color=BLACK, labelpad=10)
 ax.set_title('3D Feasible Space ,  UF System',
 fontsize=17, fontweight='bold', color=BLACK, pad=14)
 ax.tick_params(colors=BLACK, labelsize=9)
 for pane in [ax.xaxis.pane, ax.yaxis.pane, ax.zaxis.pane]:
 pane.fill = False; pane.set_edgecolor(BLUE_LIGHT)

 plt.tight_layout()
 plt.savefig(f"{save_dir}/Feasible_Space_3D.png",
 dpi=300, bbox_inches='tight', facecolor=FIGBG)
 plt.close(); print(" Feasible_Space_3D.png")

# ══════════════════════════════════════════════════════════════
# 8 & 9. SHAP / ALE IMPORTANCE
# ══════════════════════════════════════════════════════════════

def _importance_bars(imp_df, val_col, xlabel, title, out_name, save_dir):
 imp_df = imp_df.sort_values(val_col, ascending=True)
 n = len(imp_df)

 fig, ax = plt.subplots(figsize=(11, max(6, n*0.9+1)), facecolor=FIGBG)
 _style_ax(ax)

    # Gradient from light to rich cornflower by rank
 cmap = plt.cm.get_cmap('Blues')
 cols = [cmap(0.38 + 0.52*(i/max(n-1,1))) for i in range(n)]
 bars = ax.barh(range(n), imp_df[val_col].values,
 color=cols, edgecolor=BLUE_EDGE, lw=1.4)
    # Top bar in terracotta
 bars[-1].set_facecolor(TERRA); bars[-1].set_edgecolor(BLUE_EDGE)

 ax.set_yticks(range(n))
 ax.set_yticklabels(imp_df['Feature'], fontsize=12, fontweight='bold', color=BLACK)
 ax.set_xlabel(xlabel, fontsize=12, fontweight='bold', color=BLACK)
 ax.set_title(title, fontsize=14, fontweight='bold', pad=12, color=BLACK)
 ax.tick_params(labelsize=10, colors=BLACK)
 ax.set_axisbelow(True)
 ax.xaxis.grid(True, color=BLUE_EDGE, alpha=0.13, lw=0.8)

 xlim = ax.get_xlim()[1]
 for bar, val in zip(bars, imp_df[val_col].values):
 ax.text(val + xlim*0.015, bar.get_y()+bar.get_height()/2,
 f'{val:.4f}', va='center', fontsize=9.5,
 fontweight='bold', color=BLACK)

 plt.tight_layout()
 plt.savefig(f"{save_dir}/{out_name}", dpi=300,
 bbox_inches='tight', facecolor=FIGBG)
 plt.close()

def plot_shap_importance_bars(data, save_dir):
 print("\n8. SHAP importance …")
 shap = data.get('shap_importance')
 if not shap: return
 for out, df in shap.items():
 _importance_bars(df, 'Mean_Abs_SHAP_Feasible',
 'Mean |SHAP Value| (Feasible Region)',
 f'SHAP Feature Importance ,  {out}',
 f'SHAP_Importance_{out}.png', save_dir)
 print(f" SHAP importance for {len(shap)} outputs")

def plot_ale_importance_bars(data, save_dir):
 print("\n9. ALE importance …")
 ale = data.get('ale_importance')
 if not ale: return
 for out, df in ale.items():
 _importance_bars(df, 'ALE_Importance',
 'ALE Importance (Std of ALE values)',
 f'ALE Feature Importance ,  {out}',
 f'ALE_Importance_{out}.png', save_dir)
 print(f" ALE importance for {len(ale)} outputs")

# ══════════════════════════════════════════════════════════════
# 10. METHOD COMPARISON
# ══════════════════════════════════════════════════════════════

def plot_method_comparison(data, save_dir):
 print("\n10. Method comparison …")
 md = data.get('method_comparison')
 if not md: return
 oc = data['output_cols']

    # ── Grouped bar chart
 fig, axes = plt.subplots(1, len(oc), figsize=(11*len(oc), 9), facecolor=FIGBG)
 if len(oc)==1: axes=[axes]

 for idx, out in enumerate(oc):
 if out not in md: continue
 ax = axes[idx]; _style_ax(ax)
 comp = md[out].sort_values('SHAP_Rank')
 feat = comp['Feature'].values
 x = np.arange(len(feat)); w = 0.26

 def norm(v): return v/v.max() if v.max()>0 else v
 pn = norm(comp['Pearson_Abs'].values)
 sn = norm(comp['SHAP_Feasible'].values)
 an = norm(comp['ALE_Importance'].values)

 ax.bar(x-w, pn, w, label='Pearson', color=M_P_FILL, edgecolor=M_P_EDGE, lw=1.6, alpha=0.88)
 ax.bar(x, sn, w, label='SHAP', color=M_S_FILL, edgecolor=M_S_EDGE, lw=1.6, alpha=0.82)
 ax.bar(x+w, an, w, label='ALE', color=M_A_FILL, edgecolor=M_A_EDGE, lw=1.6, alpha=0.85)

 ax.set_xlabel('Feature', fontsize=13, fontweight='bold', color=BLACK)
 ax.set_ylabel('Normalized Importance', fontsize=13, fontweight='bold', color=BLACK)
 ax.set_title(f'{out}: Method Comparison', fontsize=15, fontweight='bold', color=BLACK)
 ax.set_xticks(x)
 ax.set_xticklabels(feat, rotation=38, ha='right', fontsize=11, color=BLACK)
 ax.legend(fontsize=12, framealpha=0.96, edgecolor=BLUE_EDGE, facecolor=ANNO_BG)
 ax.tick_params(labelsize=11, colors=BLACK)
 ax.set_axisbelow(True)
 ax.yaxis.grid(True, color=BLUE_EDGE, alpha=0.12, lw=0.8)

 plt.tight_layout()
 plt.savefig(f"{save_dir}/Method_Comparison_Bar.png",
 dpi=300, bbox_inches='tight', facecolor=FIGBG)
 plt.close()

    # ── Rank heatmaps
 for out in oc:
 if out not in md: continue
 comp = md[out].sort_values('SHAP_Rank')
 feat = comp['Feature'].values
 rm = comp[['Pearson_Rank','SHAP_Rank','ALE_Rank']].values.T

 fig, ax = plt.subplots(figsize=(11, 6), facecolor=FIGBG)
 _style_ax(ax)
 im = ax.imshow(rm, cmap='RdBu_r', aspect='auto', vmin=1, vmax=len(feat))
 ax.set_xticks(range(len(feat))); ax.set_yticks(range(3))
 ax.set_xticklabels(feat, rotation=38, ha='right', fontsize=11, color=BLACK)
 ax.set_yticklabels(['Pearson','SHAP','ALE'], fontsize=13,
 fontweight='bold', color=BLACK)
 for i in range(3):
 for j in range(len(feat)):
 tc = WHITE if rm[i,j] > len(feat)/2 else BLACK
 ax.text(j, i, str(int(rm[i,j])), ha='center', va='center',
 fontsize=13, fontweight='bold', color=tc)
 cb = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
 cb.set_label('Rank', fontsize=11, color=BLACK); cb.ax.tick_params(colors=BLACK)
 ax.set_title(f'{out}: Feature Rank Comparison (1 = Most Important)',
 fontsize=15, fontweight='bold', color=BLACK)
 plt.tight_layout()
 plt.savefig(f"{save_dir}/{out}_Rank_Heatmap.png",
 dpi=300, bbox_inches='tight', facecolor=FIGBG)
 plt.close()

 print(f" Method comparison for {len(md)} outputs")

# ══════════════════════════════════════════════════════════════
# MAIN
# ══════════════════════════════════════════════════════════════

def recreate_all_plots(base_dir=None):
 data = load_all_data(base_dir)
 if data is None: print(" No data"); return

 if base_dir is None:
 try:

 base_dir = './Optimal_Conditions_v2'
 except:
 base_dir = './UF_data_results/Optimal_Conditions_v2'

 out = f'{base_dir}/Elegant_Plots'
 os.makedirs(out, exist_ok=True)
 print(f"\n→ Saving to {out}/")

 plot_input_range_bars(data, out)
 plot_sensitivity_bars(data, out)
 plot_2d_feasible_regions_density(data, out)
 plot_input_distributions(data, out)
 plot_output_distributions(data, out)
 plot_correlation_heatmap(data, out)
 plot_3d_feasible_space(data, out)
 plot_shap_importance_bars(data, out)
 plot_ale_importance_bars(data, out)
 plot_method_comparison(data, out)

 print(f"\n All plots done → {out}/")
 print(f"\n Elegant palette:")
 print(f" BLUE_FILL {BLUE_FILL} (fills, alpha 0.62–0.68)")
 print(f" BLUE_DEEP {BLUE_DEEP} (box fills)")
 print(f" BLUE_EDGE {BLUE_EDGE} (all edges)")
 print(f" TERRA {TERRA} (mean / top feature)")
 print(f" TEAL {TEAL} (P05/P95 reference)")
 print(f" M_A_FILL {M_A_FILL} (ALE bars ,  mint green)")

if __name__ == "__main__":
 recreate_all_plots()


# Visualization of input vs each output

In [ ]:
# -*- coding: utf-8 -*-
"""
Input Feature vs Output Visualization ,  UF Feasible Region
===========================================================
Reads the Feasible_Points_<timestamp>.csv saved by the ML analysis code
and creates, for EACH output separately (tmp, flow, lv):

 Plot 1 ,  Scatter grid : each of 7 inputs vs that output + linear trend + r value
 Plot 2 ,  Hexbin density : same, density-coloured
 Plot 3 ,  Box-by-quintile : input binned into 5 groups → output distribution
 Plot 4 ,  Correlation bar : Pearson r for all 7 inputs vs that output
 Plot 5 ,  Combined panel : all 4 views in one publication-ready figure

File structure created:
 <base>/10_Input_vs_Output/
 tmp/
 tmp_Scatter_vs_Inputs.png
 tmp_Hexbin_vs_Inputs.png
 tmp_BoxByQuintile_vs_Inputs.png
 tmp_Correlation_Summary.png
 Combined_tmp_All_Views.png
 flow/ (same structure)
 lv/ (same structure)

Color palette: Elegant theme
 Cornflower  #5B8DD9 fills
 Terracotta  #C94F2C accent / mean / negative bars
 Teal        #0E9AA7 reference lines
"""

import os
import glob
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from scipy.stats import pearsonr

plt.rcParams['axes.grid'] = False
plt.rcParams['font.family'] = 'DejaVu Sans'

# ══════════════════════════════════════════════════════════════════════════
# COLOUR PALETTE ,  Elegant theme (matches existing scripts)
# ══════════════════════════════════════════════════════════════════════════
BLUE_FILL = '#5B8DD9'   # cornflower ,  fills
BLUE_DEEP = '#2E5FAA'   # deeper blue ,  box fills, positive bars
BLUE_EDGE = '#1C3F6E'   # dark slate ,  all edges
BLUE_LIGHT = '#C5D9F5'   # pale blue ,  band tints

TERRA = '#C94F2C'   # terracotta ,  accent / mean / negative bars
TERRA_LT = '#FAE2DB'   # pale salmon ,  wide band fills

TEAL = '#0E9AA7'   # vivid teal ,  reference lines
TEAL_LT = '#D6F2F5'   # pale teal ,  band fills

ANNO_BG = '#FDF6F0'   # warm off-white ,  annotation boxes
ANNO_BDR = '#C94F2C'   # annotation box border
PANEL_BG = '#F8FAFD'   # barely-blue subplot background
BLACK = '#1A1A2E'
WHITE = '#FFFFFF'
FIGBG = '#FFFFFF'

# ══════════════════════════════════════════════════════════════════════════
# CONFIG ,  must match the column names in Feasible_Points CSV
# ══════════════════════════════════════════════════════════════════════════
INPUT_COLS = ['glu', 'mlss', 'air', 'fm', 'cn', 'hrt', 'srt']
OUTPUT_COLS = ['Predicted_tmp', 'Predicted_flow', 'Predicted_lv']
OUT_LABELS = {
 'Predicted_tmp': 'tmp (Transmembrane Pressure)',
 'Predicted_flow': 'flow (Permeate Flow Rate)',
 'Predicted_lv': 'lv (Level / Volume)',
}
OUT_SHORT = {
 'Predicted_tmp': 'tmp',
 'Predicted_flow': 'flow',
 'Predicted_lv': 'lv',
}

N_QUINTILE_BINS = 5   # number of bins for the box-by-quintile plots

# ══════════════════════════════════════════════════════════════════════════
# SHARED HELPERS
# ══════════════════════════════════════════════════════════════════════════

def _style(ax):
 """Apply the standard elegant subplot styling."""
 ax.set_facecolor(PANEL_BG)
 for sp in ax.spines.values():
 sp.set_color(BLUE_EDGE)
 sp.set_linewidth(1.2)

def _r_box(ax, r, p, loc='upper left'):
 """Draw a Pearson-r annotation box with significance stars."""
 if p < 0.001: sig = '***'
 elif p < 0.01: sig = '**'
 elif p < 0.05: sig = '*'
 else: sig = 'ns'
 txt = f"r = {r:+.3f} {sig}"
 edge = TERRA if abs(r) > 0.30 else BLUE_EDGE
 bbox = dict(boxstyle='round,pad=0.35', facecolor=ANNO_BG,
 edgecolor=edge, lw=1.4, alpha=0.96)
 xp = 0.04 if 'left' in loc else 0.96
 yp = 0.96 if 'upper' in loc else 0.04
 ha = 'left' if 'left' in loc else 'right'
 va = 'top' if 'upper' in loc else 'bottom'
 ax.text(xp, yp, txt, transform=ax.transAxes,
 fontsize=9.5, fontweight='bold',
 va=va, ha=ha, bbox=bbox, color=BLACK)

def _trend(ax, x, y):
 """Fit and draw a linear trend line in terracotta."""
 m, b = np.polyfit(x, y, 1)
 xline = np.linspace(x.min(), x.max(), 300)
 ax.plot(xline, m * xline + b,
 color=TERRA, lw=2.5, ls='-', zorder=6, alpha=0.90)

def _quintile_groups(x, y, n_bins=5):
 """
 Bin x into n_bins quantile groups.
 Returns (groups, used_labels, midpoints).
 Always wraps result in pd.Series so .cat accessor is available
 regardless of whether pd.qcut or pd.cut was used.
 """
 labels = [f'Q{i+1}' for i in range(n_bins)]
 x_series = pd.Series(x)
 y_series = pd.Series(y)

 try:
 cats = pd.qcut(x_series, q=n_bins, labels=labels, duplicates='drop')
 except ValueError:
 cats = pd.cut(x_series, bins=n_bins, labels=labels[:n_bins])

    # Ensure cats is always a Series (pd.cut on a plain ndarray can return Categorical)
 if not isinstance(cats, pd.Series):
 cats = pd.Series(cats)

 used = list(cats.cat.categories)
 groups = [y_series[cats == lbl].values for lbl in used]
 edges = np.percentile(x, np.linspace(0, 100, len(used) + 1))
 mids = [(edges[i] + edges[i+1]) / 2 for i in range(len(used))]
 return groups, used, mids

def _load_csv(csv_dir):
 """Find and load the most recent Feasible_Points CSV."""
 pattern = os.path.join(csv_dir, 'Feasible_Points_*.csv')
 files = sorted(glob.glob(pattern), reverse=True)
 if not files:
 raise FileNotFoundError(
 f"\n No Feasible_Points_*.csv found in:\n {csv_dir}\n"
 " Run the ML analysis code first to generate this file."
 )
 print(f" Loading → {os.path.basename(files[0])}")
 df = pd.read_csv(files[0])
 print(f" Shape → {df.shape[0]:,} rows × {df.shape[1]} columns")
 return df

# ══════════════════════════════════════════════════════════════════════════
# PLOT 1 ,  SCATTER GRID (7 inputs × 1 output)
# ══════════════════════════════════════════════════════════════════════════

def plot_scatter_grid(df, out_col, out_label, out_short, save_path):
 """One scatter subplot per input feature, with trend line and r."""
 y = df[out_col].values
 fig, axes = plt.subplots(2, 4, figsize=(30, 15), facecolor=FIGBG)
 axes = axes.flatten()

 for idx, inp in enumerate(INPUT_COLS):
 ax = axes[idx]
 _style(ax)
 x = df[inp].values
 r, p = pearsonr(x, y)

 ax.scatter(x, y, c=BLUE_FILL, s=5, alpha=0.22,
 edgecolors='none', zorder=3, rasterized=True)
 _trend(ax, x, y)
 ax.axhline(y.mean(), color=TEAL, lw=1.8, ls='--',
 alpha=0.75, zorder=4)
 _r_box(ax, r, p)

 ax.set_xlabel(inp, fontsize=12, fontweight='bold', color=BLACK)
 ax.set_ylabel(out_short, fontsize=12, fontweight='bold', color=BLACK)
 ax.set_title(f'{inp} → {out_short}',
 fontsize=14, fontweight='bold', pad=6, color=BLACK)
 ax.tick_params(labelsize=10, colors=BLACK)

 axes[-1].axis('off')
 fig.legend(handles=[
 Line2D([0],[0], color=TERRA, lw=2.5, label='Linear trend'),
 Line2D([0],[0], color=TEAL, lw=1.8, ls='--',
 label=f'Mean {out_short}'),
 mpatches.Patch(color=BLUE_FILL, alpha=0.55, label='Feasible points'),
 ], loc='lower right', bbox_to_anchor=(0.99, 0.01),
 fontsize=12, framealpha=0.96, edgecolor=BLUE_EDGE, facecolor=ANNO_BG)

 plt.suptitle(
 f'Scatter: All Input Features → {out_label}\n'
 f'Feasible Region Only | n = {len(df):,}',
 fontsize=23, fontweight='bold', y=1.001, color=BLACK)
 plt.tight_layout()
 plt.savefig(save_path, dpi=280, bbox_inches='tight', facecolor=FIGBG)
 plt.close()
 print(f" {os.path.basename(save_path)}")

# ══════════════════════════════════════════════════════════════════════════
# PLOT 2 ,  HEXBIN DENSITY GRID
# ══════════════════════════════════════════════════════════════════════════

def plot_hexbin_grid(df, out_col, out_label, out_short, save_path):
 """One hexbin density subplot per input feature."""
 y = df[out_col].values
 fig, axes = plt.subplots(2, 4, figsize=(30, 15), facecolor=FIGBG)
 axes = axes.flatten()

 for idx, inp in enumerate(INPUT_COLS):
 ax = axes[idx]
 _style(ax)
 x = df[inp].values
 r, p = pearsonr(x, y)

 hb = ax.hexbin(x, y, gridsize=35, cmap='Blues',
 mincnt=1, edgecolors='white',
 linewidths=0.08, alpha=0.93)
 cb = plt.colorbar(hb, ax=ax, pad=0.02, fraction=0.046)
 cb.set_label('Count', fontsize=9, color=BLACK)
 cb.ax.tick_params(labelsize=8, colors=BLACK)
 cb.outline.set_edgecolor(BLUE_EDGE)

        #_trend(ax, x, y)
 _r_box(ax, r, p)

 ax.set_xlabel(inp, fontsize=12, fontweight='bold', color=BLACK)
 ax.set_ylabel(out_short, fontsize=12, fontweight='bold', color=BLACK)
 ax.set_title(f'{inp} → {out_short}',
 fontsize=14, fontweight='bold', pad=6, color=BLACK)
 ax.tick_params(labelsize=10, colors=BLACK)

 axes[-1].axis('off')
 plt.suptitle(
 f'Hexbin Density: All Input Features → {out_label}\n'
 f'Feasible Region Only | n = {len(df):,}',
 fontsize=23, fontweight='bold', y=1.001, color=BLACK)
 plt.tight_layout()
 plt.savefig(save_path, dpi=280, bbox_inches='tight', facecolor=FIGBG)
 plt.close()
 print(f" {os.path.basename(save_path)}")

# ══════════════════════════════════════════════════════════════════════════
# PLOT 3 ,  BOX-BY-QUINTILE GRID
# ══════════════════════════════════════════════════════════════════════════

def plot_boxquintile_grid(df, out_col, out_label, out_short, save_path):
 """
 Bin each input into quintile groups; show output distribution per bin.
 Clearly reveals both linear and non-linear relationships.
 """
 y = df[out_col].values
 fig, axes = plt.subplots(2, 4, figsize=(30, 15), facecolor=FIGBG)
 axes = axes.flatten()

 for idx, inp in enumerate(INPUT_COLS):
 ax = axes[idx]
 _style(ax)
 x = df[inp].values
 r, p = pearsonr(x, y)

 groups, used, mids = _quintile_groups(x, y, N_QUINTILE_BINS)
 n_used = len(used)

 ax.boxplot(
 groups,
 positions=range(n_used),
 widths=0.55,
 patch_artist=True,
 boxprops =dict(facecolor=BLUE_FILL, alpha=0.68,
 linewidth=1.8, edgecolor=BLUE_EDGE),
 medianprops =dict(color=TERRA, linewidth=3.2,
 solid_capstyle='round'),
 whiskerprops=dict(linewidth=1.8, color=BLUE_EDGE),
 capprops =dict(linewidth=2.2, color=BLUE_EDGE),
 flierprops =dict(marker='o', markersize=3,
 markerfacecolor=TERRA, alpha=0.30,
 markeredgecolor='none'),
 showfliers=True,
 )

        # Connect medians with a terracotta trend line
 medians = [np.median(g) for g in groups if len(g) > 0]
 ax.plot(range(len(medians)), medians,
 color=TERRA, lw=2.2, ls='-',
 marker='D', ms=7,
 mec=BLUE_EDGE, mew=1.3, zorder=7)

 tick_lbls = [f"{used[i]}\n≈{mids[i]:.3g}" for i in range(n_used)]
 ax.set_xticks(range(n_used))
 ax.set_xticklabels(tick_lbls, fontsize=9, color=BLACK)

 ax.set_xlabel(f'{inp} (quintile bins)',
 fontsize=11, fontweight='bold', color=BLACK)
 ax.set_ylabel(out_short, fontsize=12, fontweight='bold', color=BLACK)
 ax.set_title(f'{inp} → {out_short}',
 fontsize=14, fontweight='bold', pad=6, color=BLACK)
 ax.tick_params(axis='y', labelsize=10, colors=BLACK)
 _r_box(ax, r, p)

 axes[-1].axis('off')
 plt.suptitle(
 f'Box-by-Quintile: All Input Features → {out_label}\n'
 f'Each box = one quintile of the input | n = {len(df):,}',
 fontsize=23, fontweight='bold', y=1.001, color=BLACK)
 plt.tight_layout()
 plt.savefig(save_path, dpi=280, bbox_inches='tight', facecolor=FIGBG)
 plt.close()
 print(f" {os.path.basename(save_path)}")

# ══════════════════════════════════════════════════════════════════════════
# PLOT 4 ,  CORRELATION SUMMARY BAR
# ══════════════════════════════════════════════════════════════════════════

def plot_correlation_summary(df, out_col, out_label, out_short, save_path):
 """Horizontal bar chart: Pearson r for all 7 inputs vs this output."""
 y = df[out_col].values

 rows = [{'Input': inp,
 'r': pearsonr(df[inp].values, y)[0],
 'abs_r': abs(pearsonr(df[inp].values, y)[0]),
 'p': pearsonr(df[inp].values, y)[1]}
 for inp in INPUT_COLS]
 cdf = pd.DataFrame(rows).sort_values('abs_r', ascending=True)

 fig, ax = plt.subplots(figsize=(11, 7), facecolor=FIGBG)
 _style(ax)

 colors = [BLUE_DEEP if r >= 0 else TERRA for r in cdf['r']]
 bars = ax.barh(cdf['Input'], cdf['r'],
 color=colors, edgecolor=BLUE_EDGE, lw=1.6, alpha=0.82)

 xlim = max(abs(cdf['r'].min()), abs(cdf['r'].max())) * 1.38
 ax.set_xlim(-xlim, xlim)

 for bar, row in zip(bars, cdf.itertuples()):
 if row.p < 0.001: sig = '***'
 elif row.p < 0.01: sig = '**'
 elif row.p < 0.05: sig = '*'
 else: sig = ''
 offset = xlim * 0.022
 ha = 'left' if row.r >= 0 else 'right'
 xpos = row.r + (offset if row.r >= 0 else -offset)
 ax.text(xpos, bar.get_y() + bar.get_height() / 2,
 f'{row.r:+.3f}{sig}',
 va='center', ha=ha,
 fontsize=10.5, fontweight='bold', color=BLACK)

 ax.axvline(0, color=BLUE_EDGE, lw=2.2)
 ax.set_xlabel('Pearson r', fontsize=13, fontweight='bold', color=BLACK)
 ax.set_title(
 f'Input Feature Correlations → {out_label}\n'
 f'* p<0.05 ** p<0.01 *** p<0.001 | n = {len(df):,}',
 fontsize=15, fontweight='bold', pad=10, color=BLACK)
 ax.tick_params(labelsize=12, colors=BLACK)
 ax.set_axisbelow(True)
 ax.xaxis.grid(True, color=BLUE_EDGE, alpha=0.12, lw=0.8)

 ax.legend(handles=[
 mpatches.Patch(color=BLUE_DEEP, alpha=0.82, label='Positive r'),
 mpatches.Patch(color=TERRA, alpha=0.82, label='Negative r'),
 ], fontsize=12, framealpha=0.96, edgecolor=BLUE_EDGE,
 facecolor=ANNO_BG, loc='lower right')

 plt.tight_layout()
 plt.savefig(save_path, dpi=280, bbox_inches='tight', facecolor=FIGBG)
 plt.close()
 print(f" {os.path.basename(save_path)}")

# ══════════════════════════════════════════════════════════════════════════
# PLOT 5 ,  COMBINED PUBLICATION PANEL
# ══════════════════════════════════════════════════════════════════════════

def plot_combined_panel(df, out_col, out_label, out_short, save_path):
 """
 4-row × 7-col publication figure per output:
 Row 0 Scatter + trend (7 cols)
 Row 1 Hexbin density (7 cols)
 Row 2 Box-by-quintile (7 cols)
 Row 3 Correlation bar (left 4 cols) + Stats table (right 3 cols)
 """
 y_all = df[out_col].values
 n_inp = len(INPUT_COLS)

    # Pre-compute correlations
 corr_rows = [{'Input': inp,
 'r': pearsonr(df[inp].values, y_all)[0],
 'abs_r': abs(pearsonr(df[inp].values, y_all)[0]),
 'p': pearsonr(df[inp].values, y_all)[1]}
 for inp in INPUT_COLS]
 corr_df = pd.DataFrame(corr_rows)

 fig = plt.figure(figsize=(36, 50), facecolor=FIGBG)
 gs = gridspec.GridSpec(4, n_inp, figure=fig,
 hspace=0.50, wspace=0.30,
 height_ratios=[1.05, 1.05, 1.05, 0.90])

 row_labels = ['SCATTER', 'HEXBIN', 'BOX–QUINTILE']

    # ── Rows 0-2 ─────────────────────────────────────────────────────
 for row_idx in range(3):
 for ci, inp in enumerate(INPUT_COLS):
 ax = fig.add_subplot(gs[row_idx, ci])
 _style(ax)
 x = df[inp].values
 r, p = pearsonr(x, y_all)

 if row_idx == 0:                         # Scatter
 ax.scatter(x, y_all, c=BLUE_FILL, s=4, alpha=0.22,
 edgecolors='none', zorder=3, rasterized=True)
 _trend(ax, x, y_all)
 ax.axhline(y_all.mean(), color=TEAL, lw=1.5,
 ls='--', alpha=0.70, zorder=4)
 ax.set_title(inp, fontsize=13, fontweight='bold',
 pad=5, color=BLACK)

 elif row_idx == 1:                       # Hexbin
 hb = ax.hexbin(x, y_all, gridsize=28, cmap='Blues',
 mincnt=1, edgecolors='white',
 linewidths=0.06, alpha=0.93)
 _trend(ax, x, y_all)

 else:                                    # Box-by-quintile
 groups, used, mids = _quintile_groups(x, y_all, N_QUINTILE_BINS)
 n_used = len(used)

 ax.boxplot(
 groups, positions=range(n_used), widths=0.52,
 patch_artist=True,
 boxprops =dict(facecolor=BLUE_FILL, alpha=0.66,
 linewidth=1.6, edgecolor=BLUE_EDGE),
 medianprops =dict(color=TERRA, linewidth=3,
 solid_capstyle='round'),
 whiskerprops=dict(linewidth=1.6, color=BLUE_EDGE),
 capprops =dict(linewidth=2, color=BLUE_EDGE),
 flierprops =dict(marker='o', markersize=2.5,
 markerfacecolor=TERRA, alpha=0.25,
 markeredgecolor='none'),
 showfliers=True,
 )
 medians = [np.median(g) for g in groups if len(g) > 0]
 ax.plot(range(len(medians)), medians,
 color=TERRA, lw=2, ls='-',
 marker='D', ms=5.5,
 mec=BLUE_EDGE, mew=1.1, zorder=7)

 tlbls = [f"{used[i]}\n≈{mids[i]:.2g}" for i in range(n_used)]
 ax.set_xticks(range(n_used))
 ax.set_xticklabels(tlbls, fontsize=8, color=BLACK)
 ax.set_xlabel(f'{inp} (Q-bins)',
 fontsize=10, fontweight='bold', color=BLACK)

 _r_box(ax, r, p)
 ax.tick_params(labelsize=9, colors=BLACK)

 if ci == 0:
 ax.set_ylabel(out_short, fontsize=12,
 fontweight='bold', color=BLACK)
 ax.text(-0.26, 0.50, row_labels[row_idx],
 transform=ax.transAxes, fontsize=11,
 fontweight='bold', color=BLUE_EDGE,
 rotation=90, va='center', ha='center')

    # ── Row 3 left: Correlation bar ───────────────────────────────────
 ax_bar = fig.add_subplot(gs[3, :4])
 _style(ax_bar)

 cs = corr_df.sort_values('r', ascending=True)
 bcs = [BLUE_DEEP if r >= 0 else TERRA for r in cs['r']]
 bars = ax_bar.barh(cs['Input'], cs['r'],
 color=bcs, edgecolor=BLUE_EDGE, lw=1.5, alpha=0.82)

 xlim = max(abs(cs['r'].min()), abs(cs['r'].max())) * 1.32
 ax_bar.set_xlim(-xlim, xlim)

 for bar, row in zip(bars, cs.itertuples()):
 if row.p < 0.001: sig = '***'
 elif row.p < 0.01: sig = '**'
 elif row.p < 0.05: sig = '*'
 else: sig = ''
 offset = xlim * 0.025
 ha = 'left' if row.r >= 0 else 'right'
 xpos = row.r + (offset if row.r >= 0 else -offset)
 ax_bar.text(xpos, bar.get_y() + bar.get_height() / 2,
 f'{row.r:+.3f}{sig}',
 va='center', ha=ha,
 fontsize=10, fontweight='bold', color=BLACK)

 ax_bar.axvline(0, color=BLUE_EDGE, lw=2.2)
 ax_bar.set_xlabel('Pearson r', fontsize=13, fontweight='bold', color=BLACK)
 ax_bar.set_title(
 f'Correlations with {out_short} (* p<0.05 ** p<0.01 *** p<0.001)',
 fontsize=13, fontweight='bold', color=BLACK)
 ax_bar.tick_params(labelsize=11, colors=BLACK)
 ax_bar.set_axisbelow(True)
 ax_bar.xaxis.grid(True, color=BLUE_EDGE, alpha=0.12, lw=0.8)

    # ── Row 3 right: Stats table ──────────────────────────────────────
 ax_tbl = fig.add_subplot(gs[3, 4:])
 ax_tbl.axis('off')

 tbl_data = []
 for inp in INPUT_COLS:
 x = df[inp].values
 r, p = pearsonr(x, y_all)
 if p < 0.001: sig = '***'
 elif p < 0.01: sig = '**'
 elif p < 0.05: sig = '*'
 else: sig = 'ns'
 tbl_data.append([inp,
 f'{x.min():.3g}', f'{x.mean():.3g}',
 f'{x.max():.3g}',
 f'{r:+.3f}', f'{p:.2e}', sig])

 col_hdrs = ['Input', 'Min', 'Mean', 'Max', 'r', 'p-value', 'Sig.']
 tbl = ax_tbl.table(cellText=tbl_data, colLabels=col_hdrs,
 loc='center', cellLoc='center')
 tbl.auto_set_font_size(False)
 tbl.set_fontsize(10.5)
 tbl.scale(1.05, 1.65)

 for j in range(len(col_hdrs)):
 tbl[(0, j)].set_facecolor(BLUE_DEEP)
 tbl[(0, j)].set_text_props(color=WHITE, fontweight='bold')
 for i in range(1, len(tbl_data) + 1):
 bg = PANEL_BG if i % 2 == 0 else WHITE
 for j in range(len(col_hdrs)):
 tbl[(i, j)].set_facecolor(bg)

 ax_tbl.set_title('Summary Statistics', fontsize=13,
 fontweight='bold', color=BLACK, pad=10)

    # ── Super-title ───────────────────────────────────────────────────
 plt.suptitle(
 f'Comprehensive Input–Output Analysis\n'
 f'Output: {out_label} | Feasible Region | n = {len(df):,}',
 fontsize=26, fontweight='bold', y=1.001, color=BLACK)

 plt.savefig(save_path, dpi=250, bbox_inches='tight', facecolor=FIGBG)
 plt.close()
 print(f" {os.path.basename(save_path)}")

# ══════════════════════════════════════════════════════════════════════════
# MAIN
# ══════════════════════════════════════════════════════════════════════════

def run_all(csv_dir=None, save_dir=None):
 print("\n" + "="*70)
 print(" INPUT vs OUTPUT VISUALIZATION ,  UF Feasible Region")
 print("="*70)

    # Resolve paths
 if csv_dir is None:
 try:
 import importlib
 importlib.import_module('google.colab')
 csv_dir = ('./'
 'Optimal_Conditions_v2/6_CSV_Data')
 except ModuleNotFoundError:
 csv_dir = './UF_data_results/Optimal_Conditions_v2/6_CSV_Data'

    # Load
 df = _load_csv(csv_dir)

 missing = [c for c in INPUT_COLS + OUTPUT_COLS if c not in df.columns]
 if missing:
 raise ValueError(
 f"\n Missing columns: {missing}\n"
 f" Found: {list(df.columns)}"
 )
 print(f"\n Inputs : {INPUT_COLS}")
 print(f" Outputs : {list(OUT_SHORT.values())}")
 print(f" n (feasible points) = {len(df):,}")

    # Output directory
 if save_dir is None:
 save_dir = os.path.join(
 os.path.dirname(csv_dir), '10_Input_vs_Output')
 os.makedirs(save_dir, exist_ok=True)
 print(f"\n Saving to → {save_dir}\n")

    # Process each output
 for out_col in OUTPUT_COLS:
 short = OUT_SHORT[out_col]
 label = OUT_LABELS[out_col]

 print(f"{'─'*65}")
 print(f" Output: {label}")
 print(f"{'─'*65}")

 sub = os.path.join(save_dir, short)
 os.makedirs(sub, exist_ok=True)

 plot_scatter_grid(
 df, out_col, label, short,
 os.path.join(sub, f'{short}_Scatter_vs_Inputs.png'))

 plot_hexbin_grid(
 df, out_col, label, short,
 os.path.join(sub, f'{short}_Hexbin_vs_Inputs.png'))

 plot_boxquintile_grid(
 df, out_col, label, short,
 os.path.join(sub, f'{short}_BoxByQuintile_vs_Inputs.png'))

 plot_correlation_summary(
 df, out_col, label, short,
 os.path.join(sub, f'{short}_Correlation_Summary.png'))

 plot_combined_panel(
 df, out_col, label, short,
 os.path.join(sub, f'Combined_{short}_All_Views.png'))

 print("\n" + "="*70)
 print(" ALL DONE ")
 print(f"\n Output structure:")
 print(f" {save_dir}/")
 for short in OUT_SHORT.values():
 print(f" {short}/")
 for name in [
 f'{short}_Scatter_vs_Inputs.png',
 f'{short}_Hexbin_vs_Inputs.png',
 f'{short}_BoxByQuintile_vs_Inputs.png',
 f'{short}_Correlation_Summary.png',
 f'Combined_{short}_All_Views.png ← publication-ready composite',
 ]:
 print(f" {name}")
 print("="*70)

if __name__ == "__main__":
    # ─── Set your paths here (or leave None for auto-detect) ─────────
    # csv_dir = folder containing Feasible_Points_<timestamp>.csv
    # save_dir = where figures are written (auto-created)
 run_all(
 csv_dir = None,
 save_dir = None,
 )


#Optimal condition supporting Table 4

In [ ]:
# -*- coding: utf-8 -*-
"""
Table 9 Supporting Evidence Figures ,  
- Each figure = ONE single panel (merged twin-Y where 2 outputs share X)
- 7 individual figures + 1 combined layout
- White bg, black text, full border box, flow in m³/min
- Diverse creative chart types kept
"""

import os, glob, numpy as np, pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.collections import LineCollection
from scipy.stats import pearsonr
from scipy.interpolate import make_interp_spline

BLK = '#000000'
plt.rcParams.update({
 'font.family': 'DejaVu Sans', 'axes.grid': False,
 'figure.facecolor': '#FFFFFF', 'axes.facecolor': '#FFFFFF',
 'text.color': BLK, 'axes.labelcolor': BLK,
 'xtick.color': BLK, 'ytick.color': BLK,
})

# ── Palette ───────────────────────────────────────────────────────────
BLU_DEEP = '#1B3A5C'; BLU_MID = '#2E6BAC'; BLU_STD = '#4A90D9'
BLU_LIGHT = '#7CB9F7'; BLU_PALE = '#C5DFFA'; BLU_WASH = '#E8F1FB'
TEAL = '#0EAAB0'; TEAL_LIGHT = '#5DD9D9'; TEAL_PALE = '#D0F0F0'
CORAL = '#E85D5D'; CORAL_LIGHT = '#F5A0A0'; CORAL_PALE = '#FDE5E5'
AMBER = '#F5A623'; GREEN = '#2AAD7A'; PURPLE = '#7C5CBF'
GRAY_LIGHT = '#B0B0C0'; GRAY_PALE = '#E8E8F0'; WHITE = '#FFFFFF'; FBG = '#FFFFFF'

CMAP_HEAT = LinearSegmentedColormap.from_list('heat',
 ['#FFE74C','#FFAD33','#E85D5D','#7C5CBF','#2E6BAC'])
CMAP_TEAL_BLUE = LinearSegmentedColormap.from_list('tb',
 [TEAL_PALE, TEAL_LIGHT, TEAL, BLU_STD, BLU_DEEP])

INPUT_COLS = ['glu','mlss','air','fm','cn','hrt','srt']
OUT_TMP = 'Predicted_tmp'; OUT_FLOW = 'Predicted_flow'; OUT_LV = 'Predicted_lv'
OUTPUT_COLS = [OUT_TMP, OUT_FLOW, OUT_LV]
FLOW_FLOOR_MIN = 1.5  # m³/min (original unit)
LV_CEIL = 67.0; LV_FLOOR = 65.0

# ── Helpers ───────────────────────────────────────────────────────────

def load_feasible(csv_dir=None):
 if csv_dir is None:
 try:
 import importlib; importlib.import_module('google.colab')
 csv_dir = './Optimal_Conditions_v2/6_CSV_Data'
 except ModuleNotFoundError:
 csv_dir = './UF_data_results/Optimal_Conditions_v2/6_CSV_Data'
 files = sorted(glob.glob(os.path.join(csv_dir, 'Feasible_Points_*.csv')), reverse=True)
 if not files: raise FileNotFoundError(f"No Feasible_Points_*.csv in {csv_dir}")
 df = pd.read_csv(files[0])
 print(f" Loaded : {os.path.basename(files[0])}")
 print(f" Shape : {df.shape[0]:,} rows × {df.shape[1]} cols")
 missing = [c for c in INPUT_COLS + OUTPUT_COLS if c not in df.columns]
 if missing: raise ValueError(f"Missing columns: {missing}")
 return df

def _fmt_num(v):
 a = abs(v)
 if a >= 1000: return f'{v:,.0f}'
 if a >= 100: return f'{v:.0f}'
 if a >= 10: return f'{v:.1f}'
 if a >= 1: return f'{v:.2f}'
 if a >= 0.1: return f'{v:.3f}'
 return f'{v:.4f}'

def _fmt_label(lo, hi): return f'{_fmt_num(lo)}–{_fmt_num(hi)}'

def _box(ax):
 ax.set_facecolor(WHITE)
 for sp in ax.spines.values():
 sp.set_visible(True); sp.set_color(BLK); sp.set_linewidth(0.8)
 ax.tick_params(colors=BLK, labelsize=18, length=4)
 ax.grid(True, axis='y', alpha=0.12, color=GRAY_LIGHT, lw=0.4)

def _box_twin(ax):
 """Style a twin axis so its right spine matches."""
 for sp in ax.spines.values():
 sp.set_visible(True); sp.set_color(BLK); sp.set_linewidth(0.8)
 ax.tick_params(colors=BLK, labelsize=18, length=4)

def qcut_segments(df, col, n_bins, out_cols):
 for nb in range(n_bins, 1, -1):
 try:
 cats, bins = pd.qcut(df[col], q=nb, retbins=True, duplicates='drop'); break
 except ValueError: continue
 df2 = df.copy(); df2['_bin'] = cats
 rows = []
 for interval, sub in df2.groupby('_bin', observed=True):
 row = {'label': _fmt_label(interval.left, interval.right),
 'bin_low': interval.left, 'bin_high': interval.right,
 'mid': (interval.left+interval.right)/2,
 'n': len(sub), 'pct': 100*len(sub)/len(df)}
 for oc in out_cols: row[oc] = sub[oc].mean()
 rows.append(row)
 return pd.DataFrame(rows).reset_index(drop=True)

def savefig(fig, path):
 fig.savefig(path, dpi=300, bbox_inches='tight', facecolor=FBG)
 plt.close(fig); print(f' {os.path.basename(path)}')

def save_csv(df_seg, path, extra_cols=None):
 out = df_seg.copy()
 if extra_cols:
 for k,v in extra_cols.items(): out[k] = v
 out.to_csv(path, index=False, float_format='%.6f')
 print(f' {os.path.basename(path)}')

# ══════════════════════════════════════════════════════════════════════
# S1 ,  MLSS: Single panel ,  bars for |TMP| (left Y) + line for Level (right Y)
# ══════════════════════════════════════════════════════════════════════

def fig_s1_mlss(df, save_path, csv_path):
 seg = qcut_segments(df, 'mlss', 5, [OUT_TMP, OUT_LV])
 n = len(seg); x = np.arange(n)

 fig, ax1 = plt.subplots(figsize=(7, 7), facecolor=FBG)
 ax2 = ax1.twinx()
 _box(ax1); _box_twin(ax2)

    # ── Gradient bars: |TMP| on left Y (F-style teal gradient) ──
 tmp_abs = seg[OUT_TMP].abs().values
 lv = seg[OUT_LV].values
 tmp_norm = (tmp_abs - tmp_abs.min()) / (tmp_abs.max() - tmp_abs.min() + 1e-10)

    # Zoom TMP axis first so imshow extents are correct
 tmp_pad = max((tmp_abs.max()-tmp_abs.min())*1.5, tmp_abs.mean()*0.08)
 tmp_ybot = max(0, tmp_abs.min()-tmp_pad)
 ax1.set_ylim(tmp_ybot, tmp_abs.max()+tmp_pad*3)

 for xi, tv, nv in zip(x, tmp_abs, tmp_norm):
 grad = np.linspace(0, nv, 256).reshape(-1,1)
 ax1.imshow(grad, extent=[xi-0.28, xi+0.28, tmp_ybot, tv], aspect='auto',
 cmap=CMAP_TEAL_BLUE, alpha=0.80, origin='lower', zorder=2, vmin=0, vmax=1)
    # Clean thin-edge bars on top of gradient
 ax1.bar(x, tmp_abs - tmp_ybot, 0.56, bottom=tmp_ybot,
 color='none', edgecolor=BLK, lw=0.8, zorder=3)

    # Bar value labels ,  above bars
 for xi, tv in zip(x, tmp_abs):
 ax1.annotate(f'{tv:.4f}', (xi, tv), textcoords='offset points',
 xytext=(0, 8), ha='center', fontsize=17, fontweight='bold', color=BLK)

    # ── Line: Level on right Y (teal colored like F) ──
 lv_pad = max((lv.max()-lv.min())*2.0, 0.8)
 ax2.set_ylim(min(LV_FLOOR, lv.min()) - lv_pad, max(LV_CEIL, lv.max()) + lv_pad)

    # Smooth line
 if n >= 3:
 xs_lv = np.linspace(0, n-1, 200)
 spl_lv = make_interp_spline(x, lv, k=min(3, n-1))
 ys_lv = spl_lv(xs_lv)
 else:
 xs_lv, ys_lv = x.astype(float), lv.copy()

 ax2.fill_between(xs_lv, lv.min()-0.3, ys_lv, alpha=0.08, color=TEAL, zorder=1)
 pts = np.array([xs_lv, ys_lv]).T.reshape(-1,1,2)
 segs_lv = np.concatenate([pts[:-1], pts[1:]], axis=1)
 norm_lv = Normalize(ys_lv.min(), ys_lv.max())
 lc_lv = LineCollection(segs_lv, cmap=CMAP_TEAL_BLUE, norm=norm_lv, lw=3, zorder=5)
 lc_lv.set_array(ys_lv[:-1]); ax2.add_collection(lc_lv)
 ax2.scatter(x, lv, s=320, c=lv, cmap=CMAP_TEAL_BLUE, norm=norm_lv,
 edgecolors=BLK, lw=1.3, zorder=6)

 for xi, v in zip(x, lv):
 ax2.annotate(f'{v:.2f}', (xi, v), textcoords='offset points',
 xytext=(0, 12), ha='center', fontsize=17, fontweight='bold', color=TEAL)

 ax2.axhline(LV_CEIL, color=CORAL_LIGHT, lw=1.5, ls='--', alpha=0.7)
 ax2.axhline(LV_FLOOR, color=TEAL_LIGHT, lw=1.5, ls='--', alpha=0.7)
 ax2.axhspan(LV_FLOOR, LV_CEIL, alpha=0.04, color=TEAL, zorder=0)

 ax1.set_ylabel('|Mean Pred. TMP| (bar)', fontsize=18, fontweight='bold', color=BLK)
 ax2.set_ylabel('Mean Pred. Level (%)', fontsize=18, fontweight='bold', color=TEAL)
 ax1.set_xlabel('MLSS Segment (mg/L)', fontsize=18, fontweight='bold', color=BLK)
 ax1.set_xticks(x); ax1.set_xticklabels(seg['label'], fontsize=14, color=BLK, rotation=45, ha='right')
 ax1.set_xlim(-0.5, n-0.5)

    # Legend
 leg_els = [mpatches.Patch(fc=TEAL, alpha=0.5, ec=BLK, label='|TMP| (left)'),
 Line2D([0],[0], color=TEAL, marker='o', mfc=TEAL_LIGHT, mec=BLK, lw=2, label='Level (right)'),
 Line2D([0],[0], color=CORAL_LIGHT, ls='--', lw=1.5, label=f'Bounds ({LV_FLOOR}–{LV_CEIL}%)')]
 ax1.legend(handles=leg_els, fontsize=12, loc='upper left',
 edgecolor=BLK, facecolor=WHITE, labelcolor=BLK)

 fig.suptitle(f'(A) MLSS: Non-Monotonic TMP & Level\n'
 f'(Feasible Region, n = {len(df):,})',
 fontsize=20, fontweight='bold', color=BLK, y=1.01)
 savefig(fig, save_path)

 seg_out = seg[['label','bin_low','bin_high','mid','n','pct',OUT_TMP,OUT_LV]].copy()
 seg_out.columns = ['segment_label','bin_low_mg_L','bin_high_mg_L','bin_mid_mg_L',
 'n_points','pct_of_feasible','mean_TMP_bar','mean_Level_pct']
 save_csv(seg_out, csv_path)

# ══════════════════════════════════════════════════════════════════════
# S2 ,  Air: Single panel ,  bars for Flow (left Y) + line for |TMP| (right Y)
# ══════════════════════════════════════════════════════════════════════

def fig_s2_air(df, save_path, csv_path):
 seg = qcut_segments(df, 'air', 5, [OUT_FLOW, OUT_TMP])
 n = len(seg); x = np.arange(n)

 fig, ax1 = plt.subplots(figsize=(7, 7), facecolor=FBG)
 ax2 = ax1.twinx()
 _box(ax1); _box_twin(ax2)

    # ── Gradient bars: Flow (m³/min) on left Y (F-style) ──
 flow_min = seg[OUT_FLOW].values  # already m³/min
 flow_norm = (flow_min - flow_min.min()) / (flow_min.max() - flow_min.min() + 1e-10)

    # Zoom left Y first for imshow extents
 flow_pad = max((flow_min.max()-flow_min.min())*0.8, 0.002)
 flo_ybot = min(flow_min.min(), FLOW_FLOOR_MIN) - flow_pad
 ax1.set_ylim(flo_ybot, flow_min.max() + flow_pad*2)

 for xi, fv, nv in zip(x, flow_min, flow_norm):
 grad = np.linspace(0, nv, 256).reshape(-1,1)
 ax1.imshow(grad, extent=[xi-0.28, xi+0.28, flo_ybot, fv], aspect='auto',
 cmap=CMAP_TEAL_BLUE, alpha=0.80, origin='lower', zorder=2, vmin=0, vmax=1)
    # Clean thin-edge bars on top of gradient
 ax1.bar(x, flow_min - flo_ybot, 0.56, bottom=flo_ybot,
 color='none', edgecolor=BLK, lw=0.8, zorder=3)

    # Flow value labels above bars
 for xi, fv in zip(x, flow_min):
 ax1.annotate(f'{fv:.4f}', (xi, fv), textcoords='offset points',
 xytext=(0, 8), ha='center', fontsize=17, fontweight='bold', color=BLK)

 ax1.axhline(FLOW_FLOOR_MIN, color=CORAL_LIGHT, lw=1.8, ls='--', alpha=0.7)

    # ── Smooth line: |TMP| on right Y (dark teal, F-style) ──
 tmp_abs = seg[OUT_TMP].abs().values
 tmp_pad = max((tmp_abs.max()-tmp_abs.min())*2.0, tmp_abs.mean()*0.08)
 ax2.set_ylim(max(0, tmp_abs.min()-tmp_pad), tmp_abs.max()+tmp_pad)

 if n >= 3:
 xs_t = np.linspace(0, n-1, 200)
 spl_t = make_interp_spline(x, tmp_abs, k=min(3, n-1))
 ys_t = np.clip(spl_t(xs_t), 0, None)
 else:
 xs_t, ys_t = x.astype(float), tmp_abs.copy()

 ax2.fill_between(xs_t, tmp_abs.min()-tmp_pad*0.5, ys_t, alpha=0.08, color=TEAL, zorder=1)
 pts = np.array([xs_t, ys_t]).T.reshape(-1,1,2)
 segs_t = np.concatenate([pts[:-1], pts[1:]], axis=1)
 norm_t = Normalize(ys_t.min(), ys_t.max())
 lc_t = LineCollection(segs_t, cmap=CMAP_TEAL_BLUE, norm=norm_t, lw=3, zorder=5)
 lc_t.set_array(ys_t[:-1]); ax2.add_collection(lc_t)
 ax2.scatter(x, tmp_abs, s=320, c=tmp_abs, cmap=CMAP_TEAL_BLUE, norm=norm_t,
 edgecolors=BLK, lw=1.3, zorder=6)

    # TMP labels ,  offset upward, well clear of bars
 for xi, tv in zip(x, tmp_abs):
 ax2.annotate(f'{tv:.4f}', (xi, tv), textcoords='offset points',
 xytext=(0, 12), ha='center', fontsize=16, fontweight='bold', color=TEAL)

 ax1.set_ylabel('Mean Pred. Flow (m³/min)', fontsize=18, fontweight='bold', color=BLK)
 ax2.set_ylabel('|Mean Pred. TMP| (bar)', fontsize=18, fontweight='bold', color=TEAL)
 ax1.set_xlabel('Air Rate Segment (m³/h)', fontsize=18, fontweight='bold', color=BLK)
 ax1.set_xticks(x); ax1.set_xticklabels(seg['label'], fontsize=14, color=BLK, rotation=45, ha='right')
 ax1.set_xlim(-0.5, n-0.5)

 leg = [mpatches.Patch(fc=TEAL, alpha=0.5, ec=BLK, label='Flow (left)'),
 Line2D([0],[0], color=CORAL_LIGHT, ls='--', lw=1.8, label=f'Floor ({FLOW_FLOOR_MIN:.1f})'),
 Line2D([0],[0], color=TEAL, marker='o', mfc=TEAL_LIGHT, mec=BLK, lw=2, label='|TMP| (right)')]
 ax1.legend(handles=leg, fontsize=12, loc='upper right', edgecolor=BLK, facecolor=WHITE, labelcolor=BLK)

 fig.suptitle(f'(B) Air Rate: Flow vs TMP\n'
 f'(Feasible Region, n = {len(df):,})',
 fontsize=20, fontweight='bold', color=BLK, y=1.01)
 savefig(fig, save_path)

 seg_out = seg[['label','bin_low','bin_high','mid','n','pct',OUT_FLOW,OUT_TMP]].copy()
 seg_out.columns = ['segment_label','bin_low_m3h','bin_high_m3h','bin_mid_m3h',
 'n_points','pct_of_feasible','mean_Flow_m3min','mean_TMP_bar']
 save_csv(seg_out, csv_path)

# ══════════════════════════════════════════════════════════════════════
# S3 ,  SRT: Single panel ,  area+line for |TMP| (left Y) + scatter for Level (right Y)
# ══════════════════════════════════════════════════════════════════════

def fig_s3_srt(df, save_path, csv_path):
 seg = qcut_segments(df, 'srt', 5, [OUT_TMP, OUT_LV])
 n = len(seg); x = np.arange(n)

 fig, ax1 = plt.subplots(figsize=(7, 7), facecolor=FBG)
 ax2 = ax1.twinx()
 _box(ax1); _box_twin(ax2)

    # ── Gradient bars: Level on left Y (F-style) ──
 lv = seg[OUT_LV].values
 lv_norm = (lv - lv.min()) / (lv.max() - lv.min() + 1e-10)
 lv_pad = max((lv.max()-lv.min())*1.5, 0.8)
 lv_ybot = min(LV_FLOOR, lv.min()) - lv_pad
 lv_ytop = max(LV_CEIL, lv.max()) + lv_pad
 ax1.set_ylim(lv_ybot, lv_ytop)

 for xi, v, nv in zip(x, lv, lv_norm):
 grad = np.linspace(0, nv, 256).reshape(-1,1)
 ax1.imshow(grad, extent=[xi-0.28, xi+0.28, lv_ybot, v], aspect='auto',
 cmap=CMAP_TEAL_BLUE, alpha=0.80, origin='lower', zorder=2, vmin=0, vmax=1)
    # Clean thin-edge bars on top of gradient
 ax1.bar(x, lv - lv_ybot, 0.56, bottom=lv_ybot,
 color='none', edgecolor=BLK, lw=0.8, zorder=3)

    # Level labels above bars
 for xi, v in zip(x, lv):
 ax1.annotate(f'{v:.2f}', (xi, v), textcoords='offset points',
 xytext=(0, 8), ha='center', fontsize=18, fontweight='bold', color=BLK)

 ax1.axhspan(LV_FLOOR, LV_CEIL, alpha=0.04, color=TEAL, zorder=0)
 ax1.axhline(LV_CEIL, color=CORAL_LIGHT, lw=1.5, ls='--', alpha=0.7)
 ax1.axhline(LV_FLOOR, color=TEAL_LIGHT, lw=1.5, ls='--', alpha=0.7)

    # ── Smooth line: |TMP| on right Y (F-style teal gradient line) ──
 tmp_abs = np.abs(seg[OUT_TMP].values)
 tmp_pad = max((tmp_abs.max()-tmp_abs.min())*2.5, tmp_abs.mean()*0.12)
 ax2.set_ylim(max(0, tmp_abs.min()-tmp_pad), tmp_abs.max()+tmp_pad)

 if n >= 3:
 xs = np.linspace(0, n-1, 200)
 spl = make_interp_spline(x, tmp_abs, k=min(3, n-1))
 ys = np.clip(spl(xs), 0, None)
 else:
 xs, ys = x.astype(float), tmp_abs.copy()

 ax2.fill_between(xs, tmp_abs.min()-tmp_pad*0.5, ys, alpha=0.08, color=TEAL, zorder=1)
 pts = np.array([xs, ys]).T.reshape(-1,1,2)
 segs_t = np.concatenate([pts[:-1], pts[1:]], axis=1)
 norm_t = Normalize(ys.min(), ys.max())
 lc = LineCollection(segs_t, cmap=CMAP_TEAL_BLUE, norm=norm_t, lw=3, zorder=5)
 lc.set_array(ys[:-1]); ax2.add_collection(lc)
 ax2.scatter(x, tmp_abs, s=320, c=tmp_abs, cmap=CMAP_TEAL_BLUE, norm=norm_t,
 edgecolors=BLK, lw=1.3, zorder=6)

    # TMP labels above the line points
 for xi, tv in zip(x, tmp_abs):
 ax2.annotate(f'{seg[OUT_TMP].iloc[xi]:.4f}', (xi, tv), textcoords='offset points',
 xytext=(0, 12), ha='center', fontsize=16, fontweight='bold', color=TEAL)

 ax1.set_ylabel('Mean Pred. Level (%)', fontsize=18, fontweight='bold', color=BLK)
 ax2.set_ylabel('|Mean Pred. TMP| (bar)', fontsize=18, fontweight='bold', color=TEAL)
 ax1.set_xlabel('SRT Segment (days)', fontsize=18, fontweight='bold', color=BLK)
 ax1.set_xticks(x); ax1.set_xticklabels(seg['label'], fontsize=14, color=BLK, rotation=45, ha='right')
 ax1.set_xlim(-0.4, n-0.6)

 leg = [mpatches.Patch(fc=TEAL, alpha=0.5, ec=BLK, label='Level (left)'),
 Line2D([0],[0], color=CORAL_LIGHT, ls='--', lw=1.5, label=f'Bounds ({LV_FLOOR}–{LV_CEIL}%)'),
 Line2D([0],[0], color=TEAL, marker='o', mfc=TEAL_LIGHT, mec=BLK, lw=2, label='|TMP| (right)')]
 ax1.legend(handles=leg, fontsize=12, loc='upper left', edgecolor=BLK, facecolor=WHITE, labelcolor=BLK)

 fig.suptitle(f'(C) SRT: TMP vs Level Constraint\n'
 f'(Feasible Region, n = {len(df):,})',
 fontsize=20, fontweight='bold', color=BLK, y=1.01)
 savefig(fig, save_path)

 seg_out = seg[['label','bin_low','bin_high','mid','n','pct',OUT_TMP,OUT_LV]].copy()
 seg_out.columns = ['segment_label','bin_low_days','bin_high_days','bin_mid_days',
 'n_points','pct_of_feasible','mean_TMP_bar','mean_Level_pct']
 save_csv(seg_out, csv_path)

# ══════════════════════════════════════════════════════════════════════
# S4 ,  HRT: Single panel ,  lollipops for Flow (left Y) + squares for |TMP| (right Y)
# ══════════════════════════════════════════════════════════════════════

def fig_s4_hrt(df, save_path, csv_path):
 seg = qcut_segments(df, 'hrt', 6, [OUT_FLOW, OUT_TMP])
 n = len(seg); x = np.arange(n)

 fig, ax1 = plt.subplots(figsize=(15, 7), facecolor=FBG)
 ax2 = ax1.twinx()
 _box(ax1); _box_twin(ax2)

 flow_min = seg[OUT_FLOW].values  # already m³/min
 tmp_abs = seg[OUT_TMP].abs().values
 off = 0.18

    # ── Lollipops: Flow on left Y ──
 for xi, fv in zip(x, flow_min):
 c = GREEN if fv >= FLOW_FLOOR_MIN else CORAL
 ax1.plot([xi-off, xi-off], [0, fv], color=c, lw=3, alpha=0.5, zorder=2)
 ax1.scatter(x-off, flow_min, s=180,
 c=[GREEN if f >= FLOW_FLOOR_MIN else CORAL for f in flow_min],
 edgecolors=BLK, lw=1.5, zorder=5, marker='o')
 for xi, fv in zip(x, flow_min):
 ax1.annotate(f'{fv:.4f}', (xi-off, fv), textcoords='offset points',
 xytext=(-22, 8), ha='center', fontsize=16, fontweight='bold', color=BLK)

 ax1.axhline(FLOW_FLOOR_MIN, color=CORAL, lw=2, ls='--', alpha=0.6)
 flo_pad = max((flow_min.max()-flow_min.min())*0.8, 0.002)
 ax1.set_ylim(min(flow_min.min(), FLOW_FLOOR_MIN)-flo_pad, flow_min.max()+flo_pad*2)

    # ── Squares: |TMP| on right Y ──
 for xi, tv in zip(x, tmp_abs):
 ax2.plot([xi+off, xi+off], [0, tv], color=BLU_STD, lw=3, alpha=0.4, zorder=2)
 ax2.scatter(x+off, tmp_abs, s=140, color=BLU_LIGHT, edgecolors=BLK,
 lw=1.5, zorder=5, marker='s')
 for xi, tv in zip(x, tmp_abs):
 ax2.annotate(f'{tv:.4f}', (xi+off, tv), textcoords='offset points',
 xytext=(22, 8), ha='center', fontsize=16, fontweight='bold', color=BLU_MID)

 tmp_pad = max((tmp_abs.max()-tmp_abs.min())*2.0, tmp_abs.mean()*0.08)
 ax2.set_ylim(max(0, tmp_abs.min()-tmp_pad), tmp_abs.max()+tmp_pad)

 ax1.set_ylabel('Mean Predicted Flow (m³/min)', fontsize=22, fontweight='bold', color=BLK)
 ax2.set_ylabel('|Mean Predicted TMP| (bar)', fontsize=22, fontweight='bold', color=BLU_MID)
 ax1.set_xlabel('HRT Segment (h)', fontsize=22, fontweight='bold', color=BLK)
 ax1.set_xticks(x); ax1.set_xticklabels(seg['label'], fontsize=16, color=BLK, rotation=30, ha='right')
 ax1.set_xlim(-0.5, n-0.5)

 leg = [Line2D([0],[0], marker='o', color='w', mfc=GREEN, mec=BLK, ms=10, label='Flow ≥ thr.'),
 Line2D([0],[0], marker='o', color='w', mfc=CORAL, mec=BLK, ms=10, label='Flow < thr.'),
 Line2D([0],[0], color=CORAL, ls='--', lw=2, label=f'Floor ({FLOW_FLOOR_MIN:.1f})'),
 Line2D([0],[0], marker='s', color='w', mfc=BLU_LIGHT, mec=BLK, ms=10, label='|TMP| (right)')]
 ax1.legend(handles=leg, fontsize=13, loc='upper left', edgecolor=BLK, facecolor=WHITE, labelcolor=BLK)

 fig.suptitle(f'(D) HRT: Flow Threshold Effect\n(Feasible Region, n = {len(df):,})',
 fontsize=28, fontweight='bold', color=BLK, y=1.01)
 savefig(fig, save_path)

 seg_out = seg[['label','bin_low','bin_high','mid','n','pct',OUT_FLOW,OUT_TMP]].copy()
 seg_out.columns = ['segment_label','bin_low_h','bin_high_h','bin_mid_h',
 'n_points','pct_of_feasible','mean_Flow_m3min','mean_TMP_bar']
 save_csv(seg_out, csv_path)

# ══════════════════════════════════════════════════════════════════════
# S5 ,  F/M × Glu: Single panel ,  grouped lollipop (already one panel)
# ══════════════════════════════════════════════════════════════════════

def fig_s5_fm(df, save_path, csv_path):
 seg_all = qcut_segments(df, 'fm', 5, [OUT_LV])
 glu_lo = df['glu'].quantile(1/3); glu_hi = df['glu'].quantile(2/3)
 df_lo = df[df['glu'] <= glu_lo]; df_hi = df[df['glu'] >= glu_hi]
 seg_lo = qcut_segments(df_lo, 'fm', 5, [OUT_LV]) if len(df_lo)>50 else seg_all.copy()
 seg_hi = qcut_segments(df_hi, 'fm', 5, [OUT_LV]) if len(df_hi)>50 else seg_all.copy()
 nn = min(len(seg_all), len(seg_lo), len(seg_hi))
 seg_all=seg_all.iloc[:nn].reset_index(drop=True)
 seg_lo=seg_lo.iloc[:nn].reset_index(drop=True)
 seg_hi=seg_hi.iloc[:nn].reset_index(drop=True)
 x = np.arange(nn)
 all_vals = np.concatenate([seg_lo[OUT_LV].values, seg_all[OUT_LV].values, seg_hi[OUT_LV].values])

 fig, ax = plt.subplots(figsize=(15, 7), facecolor=FBG)
 _box(ax)
 off = 0.25
 groups = [
 (x-off, seg_lo[OUT_LV].values, TEAL, 'o', f'Low Glu (≤{_fmt_num(glu_lo)})'),
 (x, seg_all[OUT_LV].values, BLU_STD, 'D', 'All Glu (mean)'),
 (x+off, seg_hi[OUT_LV].values, CORAL, 's', f'High Glu (≥{_fmt_num(glu_hi)})'),
 ]
 all_pad = max((all_vals.max()-all_vals.min())*1.2, 0.6)
 y_bot = min(LV_FLOOR, all_vals.min()) - all_pad
 y_top = max(LV_CEIL, all_vals.max()) + all_pad

 for xg, vals, color, marker, label in groups:
 for xi, v in zip(xg, vals):
 ax.plot([xi, xi], [y_bot+0.1, v], color=color, lw=2.2, alpha=0.3, zorder=2)
 ax.scatter(xg, vals, s=160, color=color, edgecolors=BLK,
 lw=1.5, zorder=5, marker=marker, label=label, alpha=0.85)
 for xi, v in zip(xg, vals):
 ax.annotate(f'{v:.2f}', (xi, v), textcoords='offset points',
 xytext=(0, 11), ha='center', fontsize=15, fontweight='bold', color=color)

 ax.axhspan(LV_FLOOR, LV_CEIL, alpha=0.06, color=TEAL, zorder=0)
 ax.axhline(LV_CEIL, color=CORAL, lw=2, ls='--', alpha=0.7, label=f'Upper ({LV_CEIL}%)')
 ax.axhline(LV_FLOOR, color=TEAL, lw=1.5, ls='--', alpha=0.5, label=f'Lower ({LV_FLOOR}%)')
 ax.set_xticks(x); ax.set_xticklabels(seg_all['label'], fontsize=16, color=BLK, rotation=30, ha='right')
 ax.set_xlabel('F/M Ratio Segment (day⁻¹)', fontsize=22, fontweight='bold', color=BLK)
 ax.set_ylabel('Mean Predicted Level (%)', fontsize=22, fontweight='bold', color=BLK)
 ax.set_xlim(-0.6, nn-0.4); ax.set_ylim(y_bot, y_top)
 ax.legend(fontsize=13, loc='upper right', ncol=2, edgecolor=BLK, facecolor=WHITE, labelcolor=BLK)

 fig.suptitle(f'(E) F/M Ratio: Level Constraint Proximity × Glucose Interaction\n(n = {len(df):,})',
 fontsize=28, fontweight='bold', color=BLK, y=1.01)
 savefig(fig, save_path)

 combined = seg_all[['label','bin_low','bin_high','mid','n','pct']].copy()
 combined.columns = ['segment_label','bin_low','bin_high','bin_mid','n_all','pct_all']
 combined['mean_Level_pct_all_glu']=seg_all[OUT_LV].values
 combined['mean_Level_pct_low_glu']=seg_lo[OUT_LV].values
 combined['mean_Level_pct_high_glu']=seg_hi[OUT_LV].values
 combined['n_low_glu']=seg_lo['n'].values; combined['n_high_glu']=seg_hi['n'].values
 combined.insert(0,'glu_low_threshold_L_min',glu_lo)
 combined.insert(1,'glu_high_threshold_L_min',glu_hi)
 save_csv(combined, csv_path)

# ══════════════════════════════════════════════════════════════════════
# S6 ,  Glu: Single panel ,  gradient bars for Level (left Y) + line for Flow (right Y)
# Uses ALL feasible data (not C/N subset) on one shared X axis
# ══════════════════════════════════════════════════════════════════════

def fig_s6_glu(df, save_path, csv_path):
 seg = qcut_segments(df, 'glu', 5, [OUT_LV, OUT_FLOW])
 CN_THRESH = 7.0
 df_cn = df[df['cn'] < CN_THRESH]
 seg_cn = qcut_segments(df_cn, 'glu', 5, [OUT_FLOW]) if len(df_cn)>50 else None
 n = len(seg); x = np.arange(n)

 fig, ax1 = plt.subplots(figsize=(7, 7), facecolor=FBG)
 ax2 = ax1.twinx()
 _box(ax1); _box_twin(ax2)

    # ── Gradient bars: Level on left Y ──
 lv = seg[OUT_LV].values
 lv_norm = (lv - lv.min()) / (lv.max() - lv.min() + 1e-10)
 lv_pad = max((lv.max()-lv.min())*1.5, 0.8)
 y_bot_lv = min(LV_FLOOR, lv.min()) - lv_pad
 y_top_lv = max(LV_CEIL, lv.max()) + lv_pad

 for xi, v, nv in zip(x, lv, lv_norm):
 grad = np.linspace(0, nv, 256).reshape(-1, 1)
 ax1.imshow(grad, extent=[xi-0.28, xi+0.28, y_bot_lv, v], aspect='auto',
 cmap=CMAP_TEAL_BLUE, alpha=0.80, origin='lower', zorder=2, vmin=0, vmax=1)
    # Clean thin-edge bars on top of gradient
 ax1.bar(x, lv - y_bot_lv, 0.56, bottom=y_bot_lv,
 color='none', edgecolor=BLK, lw=0.8, zorder=3)
 for xi, v in zip(x, lv):
 ax1.annotate(f'{v:.2f}', (xi, v), textcoords='offset points',
 xytext=(0, 8), ha='center', fontsize=18, fontweight='bold', color=BLK)

 ax1.axhspan(LV_FLOOR, LV_CEIL, alpha=0.05, color=TEAL, zorder=0)
 ax1.axhline(LV_CEIL, color=CORAL_LIGHT, lw=1.5, ls='--', alpha=0.7)
 ax1.axhline(LV_FLOOR, color=TEAL_LIGHT, lw=1.5, ls='--', alpha=0.7)
 ax1.set_ylim(y_bot_lv, y_top_lv)

    # ── Line: Flow (m³/min) on right Y ,  use C/N<7 subset if available, else all ──
 if seg_cn is not None and len(seg_cn) == n:
 flow_min = seg_cn[OUT_FLOW].values  # already m³/min
 flow_label = f'Flow at C/N<{CN_THRESH} (m³/min, right)'
 else:
 flow_min = seg[OUT_FLOW].values  # already m³/min
 flow_label = 'Flow (m³/min, right)'

 if n >= 3:
 xs = np.linspace(0, n-1, 200)
 spl = make_interp_spline(x, flow_min, k=min(3, n-1))
 ys = spl(xs)
 else:
 xs, ys = x.astype(float), flow_min.copy()

 ax2.fill_between(xs, flow_min.min()-0.001, ys, alpha=0.10, color=TEAL, zorder=1)
    # Gradient line
 pts = np.array([xs, ys]).T.reshape(-1,1,2)
 segs = np.concatenate([pts[:-1], pts[1:]], axis=1)
 norm = Normalize(ys.min(), ys.max())
 lc = LineCollection(segs, cmap=CMAP_TEAL_BLUE, norm=norm, lw=3, zorder=4)
 lc.set_array(ys[:-1]); ax2.add_collection(lc)
 ax2.scatter(x, flow_min, s=320, c=flow_min, cmap=CMAP_TEAL_BLUE,
 edgecolors=BLK, lw=1.3, zorder=6, norm=norm)
 for xi, fv in zip(x, flow_min):
 ax2.annotate(f'{fv:.4f}', (xi, fv), textcoords='offset points',
 xytext=(0, 12), ha='center', fontsize=17, fontweight='bold', color=TEAL)

 ax2.axhline(FLOW_FLOOR_MIN, color=CORAL, lw=2, ls='--', alpha=0.6)
 flo_pad = max((flow_min.max()-flow_min.min())*0.8, 0.002)
 ax2.set_ylim(min(flow_min.min(), FLOW_FLOOR_MIN)-flo_pad, flow_min.max()+flo_pad*2)

 ax1.set_ylabel('Mean Pred. Level (%)', fontsize=18, fontweight='bold', color=BLK)
 ax2.set_ylabel('Mean Pred. Flow (m³/min)', fontsize=18, fontweight='bold', color=TEAL)
 ax1.set_xlabel('Glucose Segment (L/min)', fontsize=18, fontweight='bold', color=BLK)
 ax1.set_xticks(x); ax1.set_xticklabels(seg['label'], fontsize=14, color=BLK, rotation=45, ha='right')
 ax1.set_xlim(-0.5, n-0.5)

 leg = [mpatches.Patch(fc=TEAL, alpha=0.5, ec=BLK, label='Level (left axis)'),
 Line2D([0],[0], color=CORAL_LIGHT, ls='--', lw=1.5, label=f'Level bounds ({LV_FLOOR}–{LV_CEIL}%)'),
 Line2D([0],[0], color=TEAL, marker='o', mfc=TEAL_LIGHT, mec=BLK, lw=2, label=flow_label),
 Line2D([0],[0], color=CORAL, ls='--', lw=2, label=f'Floor ({FLOW_FLOOR_MIN:.1f})')]
 ax1.legend(handles=leg, fontsize=12, loc='upper left', edgecolor=BLK, facecolor=WHITE, labelcolor=BLK)

 fig.suptitle(f'(F) Glucose: Level & Flow\n'
 f'(Feasible Region, n = {len(df):,})',
 fontsize=20, fontweight='bold', color=BLK, y=1.01)
 savefig(fig, save_path)

 seg_out = seg[['label','bin_low','bin_high','mid','n','pct',OUT_LV,OUT_FLOW]].copy()
 seg_out.columns = ['segment_label','bin_low_L_min','bin_high_L_min','bin_mid_L_min',
 'n_points','pct_of_feasible','mean_Level_pct','mean_Flow_m3min']
 if seg_cn is not None:
 right_out = seg_cn[['n','pct',OUT_FLOW]].copy()
 right_out.columns = ['n_cn_lt7','pct_cn_lt7','mean_Flow_m3min_cn_lt7']
 for col in right_out.columns:
 seg_out[col] = right_out[col].values if len(right_out)==len(seg_out) else np.nan
 save_csv(seg_out, csv_path, extra_cols={'CN_threshold': CN_THRESH})

# ══════════════════════════════════════════════════════════════════════
# S7 ,  C/N: Radar charts + Heatmap (already one panel conceptually)
# ══════════════════════════════════════════════════════════════════════

def fig_s7_cn(df, save_path, csv_path):
 results = {}
 for inp in INPUT_COLS:
 results[inp] = {}
 for out in OUTPUT_COLS:
 r, p = pearsonr(df[inp], df[out])
 results[inp][out] = {'r': r, 'abs_r': abs(r), 'p': p}

 out_labels = {OUT_TMP:'TMP', OUT_FLOW:'Flow', OUT_LV:'Level'}
 out_colors = {OUT_TMP: BLU_STD, OUT_FLOW: TEAL, OUT_LV: CORAL}
 out_fills = {OUT_TMP: BLU_PALE, OUT_FLOW: TEAL_PALE, OUT_LV: CORAL_PALE}

 fig = plt.figure(figsize=(24, 8.5), facecolor=FBG)
 gs = gridspec.GridSpec(1, 3, figure=fig, width_ratios=[1,1,1.15], wspace=0.28)

 nf = len(INPUT_COLS)
 angles = np.linspace(0, 2*np.pi, nf, endpoint=False).tolist()
 angles_closed = angles + [angles[0]]

    # ── Two radar charts with enhanced style ──
 for idx, (out, gi) in enumerate(zip([OUT_TMP, OUT_FLOW], [0, 1])):
 ax_r = fig.add_subplot(gs[gi], polar=True); ax_r.set_facecolor(WHITE)

 vals = [results[inp][out]['abs_r'] for inp in INPUT_COLS]
 vals_c = vals + [vals[0]]
 mx = max(vals)

        # Concentric reference rings
 for ring_val in np.arange(0.1, mx+0.15, 0.1):
 ax_r.plot(np.linspace(0, 2*np.pi, 100), [ring_val]*100,
 color=GRAY_LIGHT, lw=0.4, alpha=0.4, zorder=0)

        # Gradient fill: multiple layers from center outward
 for alpha_i, scale in [(0.06, 0.3), (0.08, 0.6), (0.12, 0.85), (0.10, 1.0)]:
 scaled = [v * scale for v in vals_c]
 ax_r.fill(angles_closed, scaled, alpha=alpha_i, color=out_colors[out], zorder=1)

        # Main polygon with thick colored line
 ax_r.plot(angles_closed, vals_c, '-', color=out_colors[out], lw=3, zorder=4)
        # Outer glow
 ax_r.plot(angles_closed, vals_c, '-', color=out_colors[out], lw=7, alpha=0.12, zorder=3)

        # Data points ,  filled circles
 ax_r.scatter(angles, vals, s=100, color=out_colors[out],
 edgecolors=BLK, lw=1.2, zorder=6)

        # Value labels
 for a, v, inp in zip(angles, vals, INPUT_COLS):
 ax_r.text(a, v + mx*0.16, f'{v:.3f}', ha='center', va='center',
 fontsize=17, fontweight='bold', color=BLK)

 ax_r.set_xticks(angles)
 ax_r.set_xticklabels(INPUT_COLS, fontsize=20, fontweight='bold', color=BLK)
 ax_r.set_yticks([]); ax_r.set_rlim(0, mx*1.38)
 ax_r.spines['polar'].set_color(GRAY_LIGHT); ax_r.spines['polar'].set_linewidth(0.5)
 ax_r.grid(False)  # We draw our own rings
 ax_r.set_title(f'|Pearson r| vs {out_labels[out]}',
 fontsize=26, fontweight='bold', color=BLK, pad=20)

    # ── Heatmap ,  clean, no red box, no CN annotation ──
 ax_h = fig.add_subplot(gs[2])
 for sp in ax_h.spines.values(): sp.set_visible(True); sp.set_color(BLK); sp.set_linewidth(0.8)
 matrix = np.array([[results[inp][out]['abs_r'] for out in OUTPUT_COLS] for inp in INPUT_COLS])
 im = ax_h.imshow(matrix, cmap=CMAP_TEAL_BLUE, aspect='auto', vmin=0, vmax=matrix.max()*1.05)
 for i in range(len(INPUT_COLS)):
 for j in range(len(OUTPUT_COLS)):
 v = matrix[i,j]; c = WHITE if v > matrix.max()*0.6 else BLK
 ax_h.text(j, i, f'{v:.3f}', ha='center', va='center', fontsize=20, fontweight='bold', color=c)
 ax_h.set_xticks(range(len(OUTPUT_COLS)))
 ax_h.set_xticklabels([out_labels[o] for o in OUTPUT_COLS], fontsize=22, fontweight='bold', color=BLK)
 ax_h.set_yticks(range(len(INPUT_COLS)))
 ax_h.set_yticklabels(INPUT_COLS, fontsize=22, fontweight='bold', color=BLK)
 ax_h.set_title('Feature × Output |r| Heatmap', fontsize=26, fontweight='bold', color=BLK, pad=12)
 ax_h.tick_params(colors=BLK)
 cbar = fig.colorbar(im, ax=ax_h, shrink=0.7, pad=0.08)
 cbar.set_label('|Pearson r|', fontsize=20, color=BLK)
 cbar.ax.tick_params(labelsize=16, colors=BLK)

 fig.suptitle(f'(G) C/N Feature Importance (Feasible Region, n = {len(df):,})\n'
 f'|Pearson r| correlation',
 fontsize=30, fontweight='bold', color=BLK, y=1.02)
 savefig(fig, save_path)

 rows = []
 for inp in INPUT_COLS:
 row = {'feature': inp}
 for out in OUTPUT_COLS:
 lbl = out_labels[out]
 row[f'pearson_r_vs_{lbl}'] = results[inp][out]['r']
 row[f'abs_pearson_r_vs_{lbl}'] = results[inp][out]['abs_r']
 row[f'rank_vs_{lbl}'] = sorted(INPUT_COLS, key=lambda k: results[k][out]['abs_r'], reverse=True).index(inp)+1
 rows.append(row)
 pd.DataFrame(rows).to_csv(csv_path, index=False, float_format='%.6f')
 print(f' {os.path.basename(csv_path)}')

# ══════════════════════════════════════════════════════════════════════
# S8 ,  Combined Panel
# ══════════════════════════════════════════════════════════════════════

def fig_s8_combined(panel_paths, save_path):
 from matplotlib.image import imread
 keys = ['s1','s2','s3','s4','s5','s6','s7']
 titles = ['(A) MLSS: TMP + Level','(B) Air: Flow vs TMP',
 '(C) SRT: TMP + Level','(D) HRT: Flow + TMP',
 '(E) F/M × Glu: Level','(F) Glucose: Level + Flow',
 '(G) C/N Importance']
 nc = 4; nr = 2
 fig = plt.figure(figsize=(52, 24), facecolor=FBG)
 gs = gridspec.GridSpec(nr, nc, figure=fig, hspace=0.14, wspace=0.06)

 for i, (k, ttl) in enumerate(zip(keys, titles)):
 r, c = divmod(i, nc)
 ax = fig.add_subplot(gs[r, c]); ax.set_facecolor(FBG)
 path = panel_paths.get(k, '')
 if os.path.exists(path):
 ax.imshow(imread(path))
 for sp in ax.spines.values(): sp.set_color(BLU_STD); sp.set_linewidth(1.5)
 else:
 ax.text(0.5, 0.5, 'MISSING', ha='center', va='center', fontsize=36, color=CORAL, transform=ax.transAxes)
 ax.set_xticks([]); ax.set_yticks([])
 ax.set_title(ttl, fontsize=32, fontweight='bold', color=BLK, pad=8)

 ax_e = fig.add_subplot(gs[1, 3]); ax_e.set_facecolor(BLU_WASH); ax_e.axis('off')
 summ = [("MLSS","Non-monotonic TMP",BLU_STD),("Air","Flow↑ vs TMP↑ trade-off",TEAL),
 ("SRT","TMP improves, level risks",GREEN),("HRT","Sharp flow threshold",CORAL),
 ("F/M","Glu interaction on level",PURPLE),("Glucose","Aids flow at low C/N",AMBER),
 ("C/N","High importance overall",BLU_DEEP)]
 ax_e.text(0.5, 0.96, 'Table 9 Evidence Summary', ha='center', va='top',
 fontsize=32, fontweight='bold', color=BLK, transform=ax_e.transAxes)
 for j, (p, d, c) in enumerate(summ):
 yp = 0.86 - j*0.11
 ax_e.text(0.06, yp, '●', fontsize=36, color=c, ha='center', va='center', transform=ax_e.transAxes)
 ax_e.text(0.12, yp, f'{p}:', fontsize=24, fontweight='bold', color=BLK, ha='left', va='center', transform=ax_e.transAxes)
 ax_e.text(0.32, yp, d, fontsize=22, color=BLK, ha='left', va='center', transform=ax_e.transAxes)
 for sp in ax_e.spines.values(): sp.set_color(BLU_STD); sp.set_linewidth(1.5); sp.set_visible(True)

 fig.suptitle('Supporting Evidence for Table 9 Recommended Operating Conditions',
 fontsize=52, fontweight='bold', color=BLK, y=1.003)
 savefig(fig, save_path)

# ══════════════════════════════════════════════════════════════════════
# MAIN
# ══════════════════════════════════════════════════════════════════════

def run_all(csv_dir=None, save_dir=None):
 print('\n'+'='*65)
 print(' Table 9 Evidence ,   (single panels, twin-Y merged)')
 print('='*65)
 df = load_feasible(csv_dir)
 if save_dir is None:
 try:
 import importlib; importlib.import_module('google.colab')
 base = './Optimal_Conditions_v2'
 except ModuleNotFoundError:
 base = './UF_data_results/Optimal_Conditions_v2'
 save_dir = os.path.join(base, '11_Table9_Evidence')
 os.makedirs(save_dir, exist_ok=True)
 print(f'\n Saving to: {save_dir}\n')

 p = {k: os.path.join(save_dir, f) for k,f in {
 's1':'Fig_S1_MLSS_NonMonotonic.png','s2':'Fig_S2_Air_Tradeoff.png',
 's3':'Fig_S3_SRT_TMP_Level.png','s4':'Fig_S4_HRT_Flow_Threshold.png',
 's5':'Fig_S5_FM_Level_Glu_Interaction.png','s6':'Fig_S6_Glu_Level_Flow.png',
 's7':'Fig_S7_CN_Importance.png','s8':'Fig_S8_Combined_Panel.png'}.items()}
 c = {k: os.path.join(save_dir, f) for k,f in {
 's1':'Data_S1_MLSS.csv','s2':'Data_S2_Air.csv','s3':'Data_S3_SRT.csv',
 's4':'Data_S4_HRT.csv','s5':'Data_S5_FM_Glu.csv','s6':'Data_S6_Glu.csv',
 's7':'Data_S7_CN_Importance.csv'}.items()}

 fig_s1_mlss(df, p['s1'], c['s1'])
 fig_s2_air (df, p['s2'], c['s2'])
 fig_s3_srt (df, p['s3'], c['s3'])
 fig_s4_hrt (df, p['s4'], c['s4'])
 fig_s5_fm (df, p['s5'], c['s5'])
 fig_s6_glu (df, p['s6'], c['s6'])
 fig_s7_cn (df, p['s7'], c['s7'])
 fig_s8_combined(p, p['s8'])

 print('\n'+'='*65+'\n ALL DONE ')
 for k in sorted(p): print(f' {os.path.basename(p[k])}')
 for k in sorted(c): print(f' {os.path.basename(c[k])}')
 print('='*65)

if __name__ == '__main__':
 run_all(csv_dir=None, save_dir=None)
